In [1]:
from pathlib import Path
import shutil

project_root = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution"
)

current_file = project_root / "df40_manifest_pipeline.py"
backup_file = project_root / "df40_manifest_pipeline_before_fix.py"

if current_file.exists() and not backup_file.exists():
    shutil.copy2(current_file, backup_file)
    print("Backup created:", backup_file)
else:
    print("Original file missing or backup already exists.")

Backup created: /home/jovyan/Deepfake_Low Resource_Low Resolution/df40_manifest_pipeline_before_fix.py


In [14]:
from pathlib import Path
import ast

pipeline_path = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution/"
    "df40_manifest_pipeline.py"
)

if not pipeline_path.exists():
    raise FileNotFoundError(pipeline_path)

text = pipeline_path.read_text()
tree = ast.parse(text)

function_node = next(
    (
        node
        for node in tree.body
        if isinstance(node, ast.FunctionDef)
        and node.name == "_norm_token"
    ),
    None,
)

if function_node is None:
    raise RuntimeError(
        "Could not find the _norm_token function."
    )

new_function = '''def _norm_token(value: str) -> str:
    """Normalize folder names for rule matching.

    Examples:
        Real_FF++   -> realff
        Real_Celeb  -> realceleb
        Fake_Test   -> faketest
        Celeb-DF.v2 -> celebdfv2
    """
    value = str(value).strip().lower()
    return re.sub(r"[^a-z0-9]+", "", value)
'''

lines = text.splitlines(keepends=True)

start_index = function_node.lineno - 1
end_index = function_node.end_lineno

updated_lines = (
    lines[:start_index]
    + new_function.splitlines(keepends=True)
    + ["\n"]
    + lines[end_index:]
)

pipeline_path.write_text("".join(updated_lines))

print("Patched:", pipeline_path)

Patched: /home/jovyan/Deepfake_Low Resource_Low Resolution/df40_manifest_pipeline.py


In [16]:
import importlib
from pathlib import Path
import pandas as pd

import df40_manifest_pipeline as manifest_pipeline
importlib.reload(manifest_pipeline)

df40_root = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution/df40"
)

expected = {
    "Train": 1,
    "Real_FF++": 0,
    "Real_Celeb": 0,
    "Fake_Test": 1,
}

rows = []

for folder_name, expected_label in expected.items():
    folder = df40_root / folder_name

    images = [
        path
        for path in folder.rglob("*")
        if path.is_file()
        and path.suffix.lower() in manifest_pipeline.IMAGE_EXTENSIONS
    ]

    if not images:
        rows.append({
            "folder": folder_name,
            "status": "NO IMAGE FOUND",
            "expected_label": expected_label,
        })
        continue

    result = manifest_pipeline._infer_main_root_image(
        df40_root,
        images[0],
    )

    rows.append({
        "folder": folder_name,
        "sample_path": result["relative_path"],
        "expected_label": expected_label,
        "inferred_label": result["label"],
        "label_name": result["label_name"],
        "domain": result["domain"],
        "dataset_partition": result["dataset_partition"],
        "source_layout": result["source_layout"],
        "status": (
            "PASS"
            if result["label"] == expected_label
            else "FAIL"
        ),
    })

audit = pd.DataFrame(rows)
display(audit)

,folder,sample_path,expected_label,inferred_label,label_name,domain,dataset_partition,source_layout,status
0,Train,Train/ddim/001/001_325.png,1,1,fake,unknown,indomain,top_level_train_fake_nested,PASS
1,Real_FF++,Real_FF++/original_sequences/youtube/c23/frame...,0,0,real,ff,indomain,top_level_realff_real,PASS
2,Real_Celeb,Real_Celeb/Celeb-real/frames/id0_0000/000.png,0,0,real,cdf,indomain,top_level_realceleb_real,PASS
3,Fake_Test,Fake_Test/cdf/frames/id9_id0_0006/349.png,1,1,fake,cdf,ood_fake_test,top_level_fake_test_domain_method,PASS


In [21]:
import importlib
from pathlib import Path
import pandas as pd

import df40_manifest_pipeline as manifest_pipeline
importlib.reload(manifest_pipeline)

project_root = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution"
)

df40_root = project_root / "df40"

output_dir = (
    project_root
    / "outputs_df40"
    / "dataset_summary_fixed"
)

manifest, paths = manifest_pipeline.build_df40_dataset(
    df40_roots=[df40_root],
    extra_real_roots=[],
    output_dir=output_dir,
    seed=42,
)

print("\nManifest rows:", len(manifest))

print("\nCounts by split and label:")
display(
    pd.crosstab(
        manifest["split"],
        manifest["label_name"],
        margins=True,
    )
)

print("\nCounts by top-level folder, split, and label:")
display(
    manifest.groupby(
        ["top_level_folder", "split", "label_name"]
    )
    .size()
    .reset_index(name="count")
)

print("\nGenerated files:")
for name, path in paths.items():
    print(f"{name}: {path}")

In-domain group reliable images: 292,907
In-domain file-level fallback images: 0
Reserved OOD Fake_Test images: 175,667

Saved DF40 dataset outputs to: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40/dataset_summary_fixed
  manifest_all: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40/dataset_summary_fixed/df40_manifest_all.csv
  manifest_train: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40/dataset_summary_fixed/df40_manifest_train.csv
  manifest_val: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40/dataset_summary_fixed/df40_manifest_val.csv
  manifest_test: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40/dataset_summary_fixed/df40_manifest_test.csv
  manifest_ood_fake_test: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40/dataset_summary_fixed/df40_manifest_ood_fake_test.csv
  distribution_by_label: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40/dataset_summary_fixed/distribution

label_name,fake,real,All
split,,,
ood_fake_test,175667,0,175667
test,22717,5999,28716
train,187508,48132,235640
val,22554,5997,28551
All,408446,60128,468574



Counts by top-level folder, split, and label:


,top_level_folder,split,label_name,count
0,Fake_Test,ood_fake_test,fake,175667
1,Real_Celeb,test,real,2799
2,Real_Celeb,train,real,22583
3,Real_Celeb,val,real,2797
4,Real_FF++,test,real,3200
5,Real_FF++,train,real,25549
6,Real_FF++,val,real,3200
7,Train,test,fake,22717
8,Train,train,fake,187508
9,Train,val,fake,22554



Generated files:
manifest_all: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40/dataset_summary_fixed/df40_manifest_all.csv
manifest_train: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40/dataset_summary_fixed/df40_manifest_train.csv
manifest_val: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40/dataset_summary_fixed/df40_manifest_val.csv
manifest_test: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40/dataset_summary_fixed/df40_manifest_test.csv
manifest_ood_fake_test: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40/dataset_summary_fixed/df40_manifest_ood_fake_test.csv
distribution_by_label: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40/dataset_summary_fixed/distribution_by_label.csv
distribution_by_method: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40/dataset_summary_fixed/distribution_by_method.csv
distribution_by_domain: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs

In [23]:
from pathlib import Path
from datetime import datetime
import pandas as pd
import re
import shutil

# ============================================================
# Paths
# ============================================================

project_root = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution"
)

combined_manifest_path = (
    project_root
    / "outputs_df40"
    / "dataset_summary_fixed"
    / "df40_manifest_all.csv"
)

canonical_indomain_path = (
    project_root
    / "outputs_df40"
    / "df40_manifest_all_clean.csv"
)

canonical_ood_path = (
    project_root
    / "outputs_df40"
    / "df40_manifest_ood_fake_test.csv"
)

if not combined_manifest_path.exists():
    raise FileNotFoundError(
        f"Corrected combined manifest not found:\n"
        f"{combined_manifest_path}"
    )

# ============================================================
# Load and validate combined manifest
# ============================================================

combined_df = pd.read_csv(combined_manifest_path)

required_columns = {
    "frame_path",
    "label",
    "label_name",
    "split",
    "dataset_partition",
    "top_level_folder",
}

missing_columns = required_columns - set(combined_df.columns)

if missing_columns:
    raise RuntimeError(
        "Missing required columns: "
        f"{sorted(missing_columns)}"
    )

def normalize_folder_name(value):
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(value).strip().lower(),
    )

combined_df["label"] = (
    pd.to_numeric(
        combined_df["label"],
        errors="raise",
    )
    .astype(int)
)

combined_df["dataset_partition"] = (
    combined_df["dataset_partition"]
    .astype(str)
    .str.strip()
    .str.lower()
)

combined_df["split"] = (
    combined_df["split"]
    .astype(str)
    .str.strip()
    .str.lower()
)

combined_df["_normalized_top_folder"] = (
    combined_df["top_level_folder"]
    .apply(normalize_folder_name)
)

observed_partitions = set(
    combined_df["dataset_partition"].unique()
)

allowed_partitions = {
    "indomain",
    "ood_fake_test",
}

unexpected_partitions = (
    observed_partitions - allowed_partitions
)

if unexpected_partitions:
    raise RuntimeError(
        "Unexpected dataset partitions found: "
        f"{sorted(unexpected_partitions)}"
    )

# ============================================================
# Separate in-domain and OOD data
# ============================================================

indomain_df = combined_df[
    combined_df["dataset_partition"].eq("indomain")
].copy()

ood_df = combined_df[
    combined_df["dataset_partition"].eq("ood_fake_test")
].copy()

if len(indomain_df) == 0:
    raise RuntimeError(
        "The in-domain manifest is empty."
    )

if len(ood_df) == 0:
    raise RuntimeError(
        "The OOD Fake_Test manifest is empty."
    )

# ============================================================
# Validate in-domain rows
# ============================================================

expected_indomain_splits = {
    "train",
    "val",
    "test",
}

observed_indomain_splits = set(
    indomain_df["split"].unique()
)

if observed_indomain_splits != expected_indomain_splits:
    raise RuntimeError(
        "Expected in-domain splits train, val, and test, "
        f"but found: {sorted(observed_indomain_splits)}"
    )

if set(indomain_df["label"].unique()) != {0, 1}:
    raise RuntimeError(
        "The in-domain manifest must contain both "
        "label 0 real and label 1 fake."
    )

# Folder rules:
# Train is always fake.
# Real_FF++ is always real.
# Real_Celeb or Real_Celebdf is always real.
expected_folder_labels = {
    "train": 1,
    "realff": 0,
    "realceleb": 0,
    "realcelebdf": 0,
}

observed_indomain_folders = set(
    indomain_df["_normalized_top_folder"].unique()
)

unexpected_indomain_folders = (
    observed_indomain_folders
    - set(expected_folder_labels)
)

if unexpected_indomain_folders:
    raise RuntimeError(
        "Unexpected top-level folders in the in-domain data: "
        f"{sorted(unexpected_indomain_folders)}"
    )

for folder_name in observed_indomain_folders:
    folder_rows = indomain_df[
        indomain_df["_normalized_top_folder"].eq(
            folder_name
        )
    ]

    expected_label = expected_folder_labels[folder_name]

    observed_labels = set(
        folder_rows["label"].unique()
    )

    if observed_labels != {expected_label}:
        raise RuntimeError(
            f"Folder '{folder_name}' should contain only "
            f"label {expected_label}, but found "
            f"{sorted(observed_labels)}."
        )

if indomain_df[
    "_normalized_top_folder"
].eq("faketest").any():
    raise RuntimeError(
        "Fake_Test rows leaked into the in-domain manifest."
    )

# ============================================================
# Validate OOD Fake_Test rows
# ============================================================

if not ood_df["split"].eq("ood_fake_test").all():
    bad_rows = ood_df[
        ~ood_df["split"].eq("ood_fake_test")
    ]

    raise RuntimeError(
        "Some OOD rows have an incorrect split:\n"
        + bad_rows[
            [
                "frame_path",
                "split",
                "dataset_partition",
            ]
        ]
        .head(10)
        .to_string(index=False)
    )

if not ood_df["label"].eq(1).all():
    bad_rows = ood_df[
        ~ood_df["label"].eq(1)
    ]

    raise RuntimeError(
        "Some Fake_Test rows are not labeled fake:\n"
        + bad_rows[
            [
                "frame_path",
                "label",
                "label_name",
            ]
        ]
        .head(10)
        .to_string(index=False)
    )

if not ood_df[
    "_normalized_top_folder"
].eq("faketest").all():
    bad_rows = ood_df[
        ~ood_df[
            "_normalized_top_folder"
        ].eq("faketest")
    ]

    raise RuntimeError(
        "Some OOD rows are not from Fake_Test:\n"
        + bad_rows[
            [
                "frame_path",
                "top_level_folder",
            ]
        ]
        .head(10)
        .to_string(index=False)
    )

# ============================================================
# Remove temporary validation column
# ============================================================

indomain_df = indomain_df.drop(
    columns=["_normalized_top_folder"]
)

ood_df = ood_df.drop(
    columns=["_normalized_top_folder"]
)

# ============================================================
# Create timestamped backups
# ============================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

if canonical_indomain_path.exists():
    backup_path = (
        canonical_indomain_path.parent
        / (
            canonical_indomain_path.stem
            + f"_backup_{timestamp}"
            + canonical_indomain_path.suffix
        )
    )

    shutil.copy2(
        canonical_indomain_path,
        backup_path,
    )

    print(
        "Backed up previous in-domain manifest:"
    )
    print(backup_path)

if canonical_ood_path.exists():
    backup_path = (
        canonical_ood_path.parent
        / (
            canonical_ood_path.stem
            + f"_backup_{timestamp}"
            + canonical_ood_path.suffix
        )
    )

    shutil.copy2(
        canonical_ood_path,
        backup_path,
    )

    print(
        "\nBacked up previous OOD manifest:"
    )
    print(backup_path)

# ============================================================
# Write temporary files first
# ============================================================

canonical_indomain_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

indomain_temp_path = (
    canonical_indomain_path.parent
    / "df40_manifest_all_clean.tmp.csv"
)

ood_temp_path = (
    canonical_ood_path.parent
    / "df40_manifest_ood_fake_test.tmp.csv"
)

indomain_df.to_csv(
    indomain_temp_path,
    index=False,
)

ood_df.to_csv(
    ood_temp_path,
    index=False,
)

# ============================================================
# Read temporary files back and verify them
# ============================================================

indomain_check = pd.read_csv(
    indomain_temp_path
)

ood_check = pd.read_csv(
    ood_temp_path
)

if len(indomain_check) != len(indomain_df):
    raise RuntimeError(
        "In-domain temporary manifest row count mismatch."
    )

if len(ood_check) != len(ood_df):
    raise RuntimeError(
        "OOD temporary manifest row count mismatch."
    )

if set(indomain_check["split"].unique()) != {
    "train",
    "val",
    "test",
}:
    raise RuntimeError(
        "Temporary in-domain manifest has bad splits."
    )

if not ood_check["label"].astype(int).eq(1).all():
    raise RuntimeError(
        "Temporary OOD manifest contains non-fake labels."
    )

# ============================================================
# Promote temporary files to canonical files
# ============================================================

indomain_temp_path.replace(
    canonical_indomain_path
)

ood_temp_path.replace(
    canonical_ood_path
)

# ============================================================
# Final output and audits
# ============================================================

print("\n" + "=" * 80)
print("MANIFEST PROMOTION COMPLETED")
print("=" * 80)

print("\nCanonical in-domain manifest:")
print(canonical_indomain_path)

print("\nCanonical OOD Fake_Test manifest:")
print(canonical_ood_path)

print("\nCombined manifest rows:")
print(f"{len(combined_df):,}")

print("\nIn-domain rows:")
print(f"{len(indomain_df):,}")

print("\nOOD Fake_Test rows:")
print(f"{len(ood_df):,}")

print("\nIn-domain split and label counts:")

display(
    pd.crosstab(
        indomain_df["split"],
        indomain_df["label_name"],
        margins=True,
    )
)

print("\nIn-domain top-level folder audit:")

display(
    indomain_df.groupby(
        [
            "top_level_folder",
            "label",
            "label_name",
        ]
    )
    .size()
    .reset_index(name="count")
)

print("\nOOD Fake_Test audit:")

display(
    ood_df.groupby(
        [
            "top_level_folder",
            "split",
            "label",
            "label_name",
        ]
    )
    .size()
    .reset_index(name="count")
)

/tmp/ipykernel_211/37363911.py:44: DtypeWarning: Columns (0: unclear_reason) have mixed types. Specify dtype option on import or set low_memory=False.
  combined_df = pd.read_csv(combined_manifest_path)


Backed up previous in-domain manifest:
/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40/df40_manifest_all_clean_backup_20260730_212844.csv


/tmp/ipykernel_211/37363911.py:371: DtypeWarning: Columns (0: unclear_reason) have mixed types. Specify dtype option on import or set low_memory=False.
  indomain_check = pd.read_csv(
/tmp/ipykernel_211/37363911.py:375: DtypeWarning: Columns (0: unclear_reason) have mixed types. Specify dtype option on import or set low_memory=False.
  ood_check = pd.read_csv(



MANIFEST PROMOTION COMPLETED

Canonical in-domain manifest:
/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40/df40_manifest_all_clean.csv

Canonical OOD Fake_Test manifest:
/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40/df40_manifest_ood_fake_test.csv

Combined manifest rows:
468,574

In-domain rows:
292,907

OOD Fake_Test rows:
175,667

In-domain split and label counts:


label_name,fake,real,All
split,,,
test,22717,5999,28716
train,187508,48132,235640
val,22554,5997,28551
All,232779,60128,292907



In-domain top-level folder audit:


,top_level_folder,label,label_name,count
0,Real_Celeb,0,real,28179
1,Real_FF++,0,real,31949
2,Train,1,fake,232779



OOD Fake_Test audit:


,top_level_folder,split,label,label_name,count
0,Fake_Test,ood_fake_test,1,fake,175667


In [24]:
from pathlib import Path
from datetime import datetime
import shutil

project_root = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution"
)

current_grfn = (
    project_root
    / "grfn_csf_grfn_only_v6.py"
)

frozen_baseline = (
    project_root
    / "grfn_csf_latefusion_baseline.py"
)

agrfn_working = (
    project_root
    / "agrfn_adaptive_routing_df40.py"
)

if not current_grfn.exists():
    raise FileNotFoundError(
        f"Current GRFN script not found:\n{current_grfn}"
    )

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Back up any existing frozen baseline before replacing it.
if frozen_baseline.exists():
    backup = frozen_baseline.with_name(
        f"{frozen_baseline.stem}_backup_{timestamp}.py"
    )
    shutil.copy2(frozen_baseline, backup)
    print("Backed up existing frozen baseline:")
    print(backup)

# Back up any existing A-GRFN working file before replacing it.
if agrfn_working.exists():
    backup = agrfn_working.with_name(
        f"{agrfn_working.stem}_backup_{timestamp}.py"
    )
    shutil.copy2(agrfn_working, backup)
    print("\nBacked up existing A-GRFN working file:")
    print(backup)

# Create clean copies from the current working GRFN script.
shutil.copy2(current_grfn, frozen_baseline)
shutil.copy2(current_grfn, agrfn_working)

print("\n" + "=" * 80)
print("FILES CREATED")
print("=" * 80)

print("\nFrozen late-fusion baseline:")
print(frozen_baseline)

print("\nA-GRFN working script:")
print(agrfn_working)

print("\nFile sizes:")
print(
    "Baseline:",
    f"{frozen_baseline.stat().st_size:,}",
    "bytes",
)
print(
    "A-GRFN:",
    f"{agrfn_working.stat().st_size:,}",
    "bytes",
)

if frozen_baseline.read_bytes() != agrfn_working.read_bytes():
    raise RuntimeError(
        "The two copied scripts are not identical."
    )

print("\nPASS: both scripts are identical clean copies.")


FILES CREATED

Frozen late-fusion baseline:
/home/jovyan/Deepfake_Low Resource_Low Resolution/grfn_csf_latefusion_baseline.py

A-GRFN working script:
/home/jovyan/Deepfake_Low Resource_Low Resolution/agrfn_adaptive_routing_df40.py

File sizes:
Baseline: 73,018 bytes
A-GRFN: 73,018 bytes

PASS: both scripts are identical clean copies.


In [29]:
from pathlib import Path
from datetime import datetime
import shutil
import py_compile
import subprocess
import sys

# ============================================================
# File locations
# ============================================================

project_root = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution"
)

script_path = (
    project_root
    / "agrfn_adaptive_routing_df40.py"
)

if not script_path.exists():
    raise FileNotFoundError(
        f"A-GRFN working script not found:\n{script_path}"
    )

# ============================================================
# Create a backup
# ============================================================

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

backup_path = script_path.with_name(
    f"{script_path.stem}_before_cli_{timestamp}.py"
)

shutil.copy2(script_path, backup_path)

print("Backup created:")
print(backup_path)

# ============================================================
# Read the working script
# ============================================================

text = script_path.read_text()

# ============================================================
# Add A-GRFN CLI arguments
# ============================================================

cli_block = '''
    # A-GRFN architecture and ablation controls
    parser.add_argument(
        "--fusion_mode",
        type=str,
        default="late",
        choices=[
            "late",
            "average",
            "adaptive_scalar",
            "adaptive_film",
            "adaptive_transformer",
        ],
        help=(
            "Fusion architecture. 'late' preserves the current GRFN-CSF "
            "baseline. Adaptive modes perform fingerprint-conditioned "
            "feature routing before classification."
        ),
    )

    parser.add_argument(
        "--controller_dim",
        type=int,
        default=None,
        help=(
            "Fingerprint controller dimension. Defaults to --feat_dim."
        ),
    )

    parser.add_argument(
        "--routing_temperature",
        type=float,
        default=1.0,
        help=(
            "Softmax temperature used by adaptive branch routing."
        ),
    )

    parser.add_argument(
        "--film_scale",
        type=float,
        default=0.10,
        help=(
            "Maximum initial FiLM modulation scale."
        ),
    )

    parser.add_argument(
        "--router_heads",
        type=int,
        default=4,
        help=(
            "Number of attention heads used by adaptive_transformer."
        ),
    )

    parser.add_argument(
        "--router_layers",
        type=int,
        default=1,
        help=(
            "Number of evidence-router blocks."
        ),
    )

    parser.add_argument(
        "--router_dropout",
        type=float,
        default=0.10,
        help=(
            "Dropout used inside the adaptive evidence router."
        ),
    )

    parser.add_argument(
        "--use_prototypes",
        action="store_true",
        help=(
            "Enable real/fake prototype proximity learning."
        ),
    )

    parser.add_argument(
        "--prototype_temperature",
        type=float,
        default=0.10,
        help=(
            "Temperature used for prototype-distance logits."
        ),
    )

    parser.add_argument(
        "--use_uncertainty",
        action="store_true",
        help=(
            "Enable the A-GRFN uncertainty head."
        ),
    )

    parser.add_argument(
        "--lambda_aux",
        type=float,
        default=0.05,
        help=(
            "Weight for auxiliary branch classification loss."
        ),
    )

    parser.add_argument(
        "--lambda_proto",
        type=float,
        default=0.10,
        help=(
            "Weight for prototype proximity loss."
        ),
    )

    parser.add_argument(
        "--lambda_route",
        type=float,
        default=0.01,
        help=(
            "Weight for routing balance and anti-collapse loss."
        ),
    )

    parser.add_argument(
        "--lambda_unc",
        type=float,
        default=0.00,
        help=(
            "Weight for learned uncertainty supervision."
        ),
    )

    parser.add_argument(
        "--lambda_cons",
        type=float,
        default=0.02,
        help=(
            "Weight for fingerprint and routing consistency "
            "between augmented views."
        ),
    )

'''

cli_anchor = "    # Loss weights\n"

if 'parser.add_argument(\n        "--fusion_mode"' not in text:
    if cli_anchor not in text:
        raise RuntimeError(
            "Could not find the '# Loss weights' insertion point."
        )

    text = text.replace(
        cli_anchor,
        cli_block + cli_anchor,
        1,
    )

    print("\nAdded A-GRFN CLI arguments.")
else:
    print("\nA-GRFN CLI arguments already exist. Skipping insertion.")

# ============================================================
# Add safe default resolution and temporary protection
# ============================================================

safety_block = '''
    # A-GRFN defaults and safety guard
    if args.controller_dim is None:
        args.controller_dim = args.feat_dim

    if args.controller_dim <= 0:
        raise ValueError("--controller_dim must be positive.")

    if args.routing_temperature <= 0:
        raise ValueError("--routing_temperature must be positive.")

    if args.film_scale < 0:
        raise ValueError("--film_scale cannot be negative.")

    if args.router_heads <= 0:
        raise ValueError("--router_heads must be positive.")

    if args.router_layers <= 0:
        raise ValueError("--router_layers must be positive.")

    if not 0.0 <= args.router_dropout < 1.0:
        raise ValueError(
            "--router_dropout must be in the interval [0, 1)."
        )

    if args.prototype_temperature <= 0:
        raise ValueError(
            "--prototype_temperature must be positive."
        )

    active_streams = {
        name.strip()
        for name in args.streams.split(",")
        if name.strip()
    }

    adaptive_modes = {
        "adaptive_scalar",
        "adaptive_film",
        "adaptive_transformer",
    }

    if args.fusion_mode in adaptive_modes:
        required_streams = {"rgb", "fingerprint"}
        missing_streams = required_streams - active_streams

        if missing_streams:
            raise ValueError(
                "Adaptive A-GRFN modes require these streams: "
                + ", ".join(sorted(missing_streams))
            )

    # Temporary guard. This prevents adaptive flags from silently running
    # the unchanged late-fusion model before Stage A is implemented.
    if args.fusion_mode != "late":
        raise NotImplementedError(
            "Only --fusion_mode late is enabled at this checkpoint. "
            "Adaptive modes will be enabled after the Stage A model "
            "and model factory are added."
        )

'''

safety_anchor = (
    "    # Alias old video limits to unit limits "
    "if the new names were not used.\n"
)

if "# A-GRFN defaults and safety guard" not in text:
    if safety_anchor not in text:
        raise RuntimeError(
            "Could not find the argument post-processing insertion point."
        )

    text = text.replace(
        safety_anchor,
        safety_block + safety_anchor,
        1,
    )

    print("Added A-GRFN argument validation and safety guard.")
else:
    print(
        "A-GRFN argument validation already exists. "
        "Skipping insertion."
    )

# ============================================================
# Save the modified script
# ============================================================

script_path.write_text(text)

# ============================================================
# Compile the script
# ============================================================

py_compile.compile(
    str(script_path),
    doraise=True,
)

print("\nPASS: Python compilation succeeded.")

# ============================================================
# Verify that the expected CLI flags exist
# ============================================================

expected_flags = [
    "--fusion_mode",
    "--controller_dim",
    "--routing_temperature",
    "--film_scale",
    "--router_heads",
    "--router_layers",
    "--router_dropout",
    "--use_prototypes",
    "--prototype_temperature",
    "--use_uncertainty",
    "--lambda_aux",
    "--lambda_proto",
    "--lambda_route",
    "--lambda_unc",
    "--lambda_cons",
]

updated_text = script_path.read_text()

missing_flags = [
    flag
    for flag in expected_flags
    if flag not in updated_text
]

if missing_flags:
    raise RuntimeError(
        "These CLI flags were not added correctly: "
        + ", ".join(missing_flags)
    )

# ============================================================
# Run the help command
# ============================================================

help_result = subprocess.run(
    [
        sys.executable,
        str(script_path),
        "--help",
    ],
    cwd=str(project_root),
    capture_output=True,
    text=True,
    timeout=120,
)

if help_result.returncode != 0:
    print(help_result.stdout)
    print(help_result.stderr)

    raise RuntimeError(
        "The script's --help command failed."
    )

# ============================================================
# Final result
# ============================================================

print("\n" + "=" * 80)
print("A-GRFN CLI SCAFFOLDING COMPLETED")
print("=" * 80)

print("\nUpdated script:")
print(script_path)

print("\nVerified CLI flags:")

for flag in expected_flags:
    print(" ", flag)

print(
    "\nCurrent safe mode: --fusion_mode late"
)

print(
    "Adaptive modes are intentionally blocked until "
    "the Stage A routing model is added."
)

Backup created:
/home/jovyan/Deepfake_Low Resource_Low Resolution/agrfn_adaptive_routing_df40_before_cli_20260730_214533.py

A-GRFN CLI arguments already exist. Skipping insertion.
A-GRFN argument validation already exists. Skipping insertion.

PASS: Python compilation succeeded.

A-GRFN CLI SCAFFOLDING COMPLETED

Updated script:
/home/jovyan/Deepfake_Low Resource_Low Resolution/agrfn_adaptive_routing_df40.py

Verified CLI flags:
  --fusion_mode
  --controller_dim
  --routing_temperature
  --film_scale
  --router_heads
  --router_layers
  --router_dropout
  --use_prototypes
  --prototype_temperature
  --use_uncertainty
  --lambda_aux
  --lambda_proto
  --lambda_route
  --lambda_unc
  --lambda_cons

Current safe mode: --fusion_mode late
Adaptive modes are intentionally blocked until the Stage A routing model is added.


In [30]:
from pathlib import Path
from datetime import datetime
import ast
import shutil
import py_compile
import importlib.util
import torch

# ============================================================
# File locations
# ============================================================

project_root = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution"
)

script_path = (
    project_root
    / "agrfn_adaptive_routing_df40.py"
)

if not script_path.exists():
    raise FileNotFoundError(
        f"A-GRFN script not found:\n{script_path}"
    )

# ============================================================
# Create backup
# ============================================================

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

backup_path = script_path.with_name(
    f"{script_path.stem}_before_stage_a_modules_{timestamp}.py"
)

shutil.copy2(script_path, backup_path)

print("Backup created:")
print(backup_path)

# ============================================================
# Parse current script
# ============================================================

text = script_path.read_text()
tree = ast.parse(text)

top_level_names = {
    node.name
    for node in tree.body
    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
            ast.ClassDef,
        ),
    )
}

expected_new_names = {
    "normalized_entropy",
    "compute_branch_disagreement",
    "routing_balance_loss",
    "FingerprintController",
    "ScalarGateRouter",
}

already_present = expected_new_names & top_level_names

if already_present and already_present != expected_new_names:
    raise RuntimeError(
        "Only some Stage A modules already exist:\n"
        + "\n".join(sorted(already_present))
        + "\n\nRestore the latest backup or inspect the script "
          "before continuing."
    )

# ============================================================
# Insert Stage A modules after SmallStreamEncoder
# ============================================================

if expected_new_names.issubset(top_level_names):
    print("\nStage A modules already exist. Skipping insertion.")

else:
    small_encoder_node = next(
        (
            node
            for node in tree.body
            if isinstance(node, ast.ClassDef)
            and node.name == "SmallStreamEncoder"
        ),
        None,
    )

    if small_encoder_node is None:
        raise RuntimeError(
            "Could not find class SmallStreamEncoder."
        )

    stage_a_code = r'''

def normalized_entropy(probs, eps=1e-8):
    """Return entropy normalized to the range [0, 1]."""
    if probs.shape[-1] < 2:
        raise ValueError(
            "normalized_entropy requires at least two classes."
        )

    entropy = -(
        probs * torch.log(probs.clamp_min(eps))
    ).sum(dim=-1)

    return entropy / math.log(probs.shape[-1])


def compute_branch_disagreement(stream_probs):
    """Compute pairwise disagreement between branch fake probabilities.

    Args:
        stream_probs:
            Tensor with shape [batch, branches, classes].

    Returns:
        Tensor with shape [batch, branches, branches].
    """
    if stream_probs.ndim != 3:
        raise ValueError(
            "stream_probs must have shape "
            "[batch, branches, classes]."
        )

    if stream_probs.shape[-1] != 2:
        raise ValueError(
            "Deepfake branch probabilities must contain "
            "two classes: real and fake."
        )

    fake_probs = stream_probs[:, :, 1]

    return torch.abs(
        fake_probs.unsqueeze(2)
        - fake_probs.unsqueeze(1)
    )


def routing_balance_loss(routing_weights):
    """Prevent global routing collapse without forcing each image uniform.

    The loss compares the mean route across the batch with a uniform
    branch distribution. Individual samples may still specialize.
    """
    if routing_weights.ndim != 2:
        raise ValueError(
            "routing_weights must have shape [batch, branches]."
        )

    num_branches = routing_weights.shape[1]

    if num_branches < 1:
        raise ValueError(
            "routing_weights must contain at least one branch."
        )

    mean_route = routing_weights.mean(dim=0)

    uniform_target = torch.full_like(
        mean_route,
        1.0 / num_branches,
    )

    return F.mse_loss(
        mean_route,
        uniform_target,
    )


class FingerprintController(nn.Module):
    """Convert the fingerprint embedding into a routing controller."""

    def __init__(
        self,
        feat_dim=128,
        controller_dim=128,
        dropout=0.10,
    ):
        super().__init__()

        if feat_dim <= 0:
            raise ValueError("feat_dim must be positive.")

        if controller_dim <= 0:
            raise ValueError(
                "controller_dim must be positive."
            )

        if not 0.0 <= dropout < 1.0:
            raise ValueError(
                "dropout must be in the interval [0, 1)."
            )

        self.feat_dim = int(feat_dim)
        self.controller_dim = int(controller_dim)

        self.net = nn.Sequential(
            nn.Linear(
                self.feat_dim,
                self.controller_dim,
            ),
            nn.LayerNorm(self.controller_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(
                self.controller_dim,
                self.feat_dim,
            ),
            nn.LayerNorm(self.feat_dim),
        )

    def forward(self, fingerprint):
        if fingerprint.ndim != 2:
            raise ValueError(
                "fingerprint must have shape "
                "[batch, feat_dim]."
            )

        if fingerprint.shape[1] != self.feat_dim:
            raise ValueError(
                "Fingerprint feature dimension mismatch. "
                f"Expected {self.feat_dim}, "
                f"received {fingerprint.shape[1]}."
            )

        return self.net(fingerprint)


class ScalarGateRouter(nn.Module):
    """Generate fingerprint-conditioned scalar evidence weights."""

    def __init__(
        self,
        feat_dim=128,
        num_branches=3,
        temperature=1.0,
    ):
        super().__init__()

        if feat_dim <= 0:
            raise ValueError("feat_dim must be positive.")

        if num_branches <= 0:
            raise ValueError(
                "num_branches must be positive."
            )

        if temperature <= 0:
            raise ValueError(
                "temperature must be positive."
            )

        self.feat_dim = int(feat_dim)
        self.num_branches = int(num_branches)
        self.temperature = float(temperature)

        self.router = nn.Sequential(
            nn.Linear(
                self.feat_dim,
                self.feat_dim,
            ),
            nn.LayerNorm(self.feat_dim),
            nn.GELU(),
            nn.Linear(
                self.feat_dim,
                self.num_branches,
            ),
        )

        # Start close to uniform routing.
        final_layer = self.router[-1]
        nn.init.zeros_(final_layer.weight)
        nn.init.zeros_(final_layer.bias)

    def forward(self, controller):
        if controller.ndim != 2:
            raise ValueError(
                "controller must have shape "
                "[batch, feat_dim]."
            )

        if controller.shape[1] != self.feat_dim:
            raise ValueError(
                "Controller feature dimension mismatch. "
                f"Expected {self.feat_dim}, "
                f"received {controller.shape[1]}."
            )

        gate_logits = self.router(controller)

        routing_weights = F.softmax(
            gate_logits / self.temperature,
            dim=1,
        )

        return {
            "gate_logits": gate_logits,
            "routing_weights": routing_weights,
        }

'''

    lines = text.splitlines(keepends=True)

    insertion_index = small_encoder_node.end_lineno

    updated_lines = (
        lines[:insertion_index]
        + stage_a_code.splitlines(keepends=True)
        + lines[insertion_index:]
    )

    script_path.write_text(
        "".join(updated_lines)
    )

    print("\nInserted Stage A modules:")
    for name in sorted(expected_new_names):
        print(" ", name)

# ============================================================
# Compile updated script
# ============================================================

py_compile.compile(
    str(script_path),
    doraise=True,
)

print("\nPASS: Python compilation succeeded.")

# ============================================================
# Import updated module
# ============================================================

module_name = (
    "agrfn_stage_a_test_"
    + timestamp
)

spec = importlib.util.spec_from_file_location(
    module_name,
    script_path,
)

agrfn_module = importlib.util.module_from_spec(spec)

if spec.loader is None:
    raise RuntimeError(
        "Could not create a module loader."
    )

spec.loader.exec_module(agrfn_module)

# ============================================================
# Instantiate Stage A modules
# ============================================================

feat_dim = 128
controller_dim = 128
num_evidence_branches = 3
batch_size = 4

controller_module = agrfn_module.FingerprintController(
    feat_dim=feat_dim,
    controller_dim=controller_dim,
    dropout=0.10,
)

router_module = agrfn_module.ScalarGateRouter(
    feat_dim=feat_dim,
    num_branches=num_evidence_branches,
    temperature=1.0,
)

controller_module.train()
router_module.train()

dummy_fingerprint = torch.randn(
    batch_size,
    feat_dim,
    requires_grad=True,
)

controller = controller_module(
    dummy_fingerprint
)

routing_output = router_module(
    controller
)

routing_weights = routing_output[
    "routing_weights"
]

gate_logits = routing_output[
    "gate_logits"
]

# ============================================================
# Validate output shapes and values
# ============================================================

assert controller.shape == (
    batch_size,
    feat_dim,
), controller.shape

assert gate_logits.shape == (
    batch_size,
    num_evidence_branches,
), gate_logits.shape

assert routing_weights.shape == (
    batch_size,
    num_evidence_branches,
), routing_weights.shape

route_sums = routing_weights.sum(dim=1)

if not torch.allclose(
    route_sums,
    torch.ones_like(route_sums),
    atol=1e-6,
):
    raise RuntimeError(
        "Routing weights do not sum to one."
    )

if not torch.isfinite(
    routing_weights
).all():
    raise RuntimeError(
        "Routing weights contain NaN or infinity."
    )

if (
    routing_weights < 0
).any():
    raise RuntimeError(
        "Routing weights contain negative values."
    )

balance_loss = (
    agrfn_module.routing_balance_loss(
        routing_weights
    )
)

if balance_loss.ndim != 0:
    raise RuntimeError(
        "routing_balance_loss must return a scalar."
    )

if not torch.isfinite(balance_loss):
    raise RuntimeError(
        "routing_balance_loss is not finite."
    )

# ============================================================
# Gradient test
# ============================================================

test_loss = (
    controller.pow(2).mean()
    + routing_weights.pow(2).mean()
    + balance_loss
)

test_loss.backward()

controller_has_grad = any(
    parameter.grad is not None
    and torch.isfinite(parameter.grad).all()
    for parameter in controller_module.parameters()
)

router_has_grad = any(
    parameter.grad is not None
    and torch.isfinite(parameter.grad).all()
    for parameter in router_module.parameters()
)

if not controller_has_grad:
    raise RuntimeError(
        "FingerprintController did not receive gradients."
    )

if not router_has_grad:
    raise RuntimeError(
        "ScalarGateRouter did not receive gradients."
    )

# ============================================================
# Test helper functions
# ============================================================

dummy_stream_logits = torch.randn(
    batch_size,
    4,
    2,
)

dummy_stream_probs = torch.softmax(
    dummy_stream_logits,
    dim=-1,
)

stream_entropy = (
    agrfn_module.normalized_entropy(
        dummy_stream_probs
    )
)

disagreement = (
    agrfn_module.compute_branch_disagreement(
        dummy_stream_probs
    )
)

assert stream_entropy.shape == (
    batch_size,
    4,
)

assert disagreement.shape == (
    batch_size,
    4,
    4,
)

if not torch.allclose(
    disagreement,
    disagreement.transpose(1, 2),
    atol=1e-6,
):
    raise RuntimeError(
        "Disagreement matrix is not symmetric."
    )

# ============================================================
# Final report
# ============================================================

print("\n" + "=" * 80)
print("STAGE A MODULE TEST PASSED")
print("=" * 80)

print("\nController shape:")
print(tuple(controller.shape))

print("\nGate-logit shape:")
print(tuple(gate_logits.shape))

print("\nRouting-weight shape:")
print(tuple(routing_weights.shape))

print("\nRouting weights:")
print(routing_weights.detach())

print("\nRouting sums:")
print(route_sums.detach())

print("\nRouting balance loss:")
print(float(balance_loss.detach()))

print("\nStream entropy shape:")
print(tuple(stream_entropy.shape))

print("\nDisagreement shape:")
print(tuple(disagreement.shape))

print("\nGradient checks:")
print("FingerprintController:", controller_has_grad)
print("ScalarGateRouter:", router_has_grad)

print("\nThe adaptive modules were added successfully.")
print(
    "The current GRFN model and --fusion_mode late "
    "behavior have not been changed."
)

Backup created:
/home/jovyan/Deepfake_Low Resource_Low Resolution/agrfn_adaptive_routing_df40_before_stage_a_modules_20260730_214637.py

Inserted Stage A modules:
  FingerprintController
  ScalarGateRouter
  compute_branch_disagreement
  normalized_entropy
  routing_balance_loss

PASS: Python compilation succeeded.

STAGE A MODULE TEST PASSED

Controller shape:
(4, 128)

Gate-logit shape:
(4, 3)

Routing-weight shape:
(4, 3)

Routing weights:
tensor([[0.3333, 0.3333, 0.3333],
        [0.3333, 0.3333, 0.3333],
        [0.3333, 0.3333, 0.3333],
        [0.3333, 0.3333, 0.3333]])

Routing sums:
tensor([1., 1., 1., 1.])

Routing balance loss:
0.0

Stream entropy shape:
(4, 4)

Disagreement shape:
(4, 4, 4)

Gradient checks:
FingerprintController: True
ScalarGateRouter: True

The adaptive modules were added successfully.
The current GRFN model and --fusion_mode late behavior have not been changed.


/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [31]:
from pathlib import Path
from datetime import datetime
from types import SimpleNamespace
import ast
import shutil
import py_compile
import importlib.util
import torch

# ============================================================
# Paths and backup
# ============================================================

project_root = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution"
)

script_path = (
    project_root
    / "agrfn_adaptive_routing_df40.py"
)

if not script_path.exists():
    raise FileNotFoundError(
        f"A-GRFN script not found:\n{script_path}"
    )

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

backup_path = script_path.with_name(
    f"{script_path.stem}_before_scalar_model_{timestamp}.py"
)

shutil.copy2(script_path, backup_path)

print("Backup created:")
print(backup_path)

text = script_path.read_text()
tree = ast.parse(text)

top_level_names = {
    node.name
    for node in tree.body
    if isinstance(node, (ast.ClassDef, ast.FunctionDef))
}

required_stage_a_modules = {
    "FingerprintController",
    "ScalarGateRouter",
    "normalized_entropy",
    "compute_branch_disagreement",
}

missing_stage_a_modules = (
    required_stage_a_modules - top_level_names
)

if missing_stage_a_modules:
    raise RuntimeError(
        "Missing Stage A modules:\n"
        + "\n".join(sorted(missing_stage_a_modules))
    )

# ============================================================
# Add AGRFN class
# ============================================================

if "AGRFN" not in top_level_names:
    simple_model_node = next(
        (
            node
            for node in tree.body
            if isinstance(node, ast.ClassDef)
            and node.name == "SimpleStreamModel"
        ),
        None,
    )

    if simple_model_node is None:
        raise RuntimeError(
            "Could not find SimpleStreamModel."
        )

    agrfn_code = r'''

class AGRFN(nn.Module):
    """Adaptive Generalizable Residual Fingerprint Network.

    The learned fingerprint embedding acts as a controller. It generates
    scalar routing weights for the forensic evidence streams before the
    final classifier. Branch heads remain auxiliary and do not contribute
    directly to the final logits.
    """

    VALID_STREAMS = [
        "rgb",
        "fingerprint",
        "rec",
        "den",
        "jpeg",
        "freq",
        "noise",
    ]

    def __init__(
        self,
        feat_dim=128,
        controller_dim=128,
        stream_names=None,
        num_classes=2,
        routing_temperature=1.0,
        dropout=0.20,
    ):
        super().__init__()

        if stream_names is None:
            stream_names = [
                "rgb",
                "fingerprint",
                "jpeg",
                "freq",
            ]

        stream_names = [
            name.strip()
            for name in stream_names
            if name.strip()
        ]

        unknown = [
            name
            for name in stream_names
            if name not in self.VALID_STREAMS
        ]

        if unknown:
            raise ValueError(
                f"Unknown streams: {unknown}. "
                f"Valid streams: {self.VALID_STREAMS}"
            )

        if "rgb" not in stream_names:
            raise ValueError(
                "A-GRFN requires the RGB stream."
            )

        if "fingerprint" not in stream_names:
            raise ValueError(
                "A-GRFN requires the fingerprint controller."
            )

        self.stream_names = stream_names
        self.evidence_streams = [
            name
            for name in self.stream_names
            if name != "fingerprint"
        ]

        if len(self.evidence_streams) == 0:
            raise ValueError(
                "A-GRFN requires at least one evidence stream."
            )

        self.num_streams = len(self.stream_names)
        self.num_evidence_streams = len(
            self.evidence_streams
        )

        self.feat_dim = int(feat_dim)
        self.controller_dim = int(controller_dim)
        self.num_classes = int(num_classes)

        # Preserve the same stream encoders used by GRFN-CSF.
        self.rgb_encoder = SmallStreamEncoder(
            3,
            self.feat_dim,
        )

        self.rec_encoder = SmallStreamEncoder(
            3,
            self.feat_dim,
        )

        self.den_encoder = SmallStreamEncoder(
            3,
            self.feat_dim,
        )

        self.jpeg_encoder = SmallStreamEncoder(
            3,
            self.feat_dim,
        )

        self.freq_encoder = SmallStreamEncoder(
            1,
            self.feat_dim,
        )

        self.noise_encoder = SmallStreamEncoder(
            1,
            self.feat_dim,
        )

        self.fingerprint_head = nn.Sequential(
            nn.Linear(
                self.feat_dim,
                self.feat_dim,
            ),
            nn.LayerNorm(self.feat_dim),
            nn.ReLU(inplace=True),
            nn.Linear(
                self.feat_dim,
                self.feat_dim,
            ),
        )

        self.fingerprint_controller = (
            FingerprintController(
                feat_dim=self.feat_dim,
                controller_dim=self.controller_dim,
                dropout=0.10,
            )
        )

        self.scalar_router = ScalarGateRouter(
            feat_dim=self.feat_dim,
            num_branches=self.num_evidence_streams,
            temperature=routing_temperature,
        )

        # These heads are used only for auxiliary branch supervision,
        # disagreement analysis, and interpretability.
        self.stream_heads = nn.ModuleDict({
            name: nn.Linear(
                self.feat_dim,
                self.num_classes,
            )
            for name in self.stream_names
        })

        # The routed feature and fingerprint controller are combined
        # before the final classification decision.
        self.joint_projector = nn.Sequential(
            nn.Linear(
                self.feat_dim * 2,
                self.feat_dim,
            ),
            nn.LayerNorm(self.feat_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(
                self.feat_dim,
                self.feat_dim,
            ),
            nn.LayerNorm(self.feat_dim),
            nn.GELU(),
        )

        self.final_head = nn.Linear(
            self.feat_dim,
            self.num_classes,
        )

    def encode_streams(self, view):
        features = {}

        rgb_feature = self.rgb_encoder(
            view["rgb"]
        )

        if "rgb" in self.stream_names:
            features["rgb"] = rgb_feature

        fingerprint_raw = self.fingerprint_head(
            rgb_feature
        )

        features["fingerprint"] = F.normalize(
            fingerprint_raw,
            dim=1,
        )

        if "rec" in self.stream_names:
            features["rec"] = self.rec_encoder(
                view["rec"]
            )

        if "den" in self.stream_names:
            features["den"] = self.den_encoder(
                view["den"]
            )

        if "jpeg" in self.stream_names:
            features["jpeg"] = self.jpeg_encoder(
                view["jpeg"]
            )

        if "freq" in self.stream_names:
            features["freq"] = self.freq_encoder(
                view["freq"]
            )

        if "noise" in self.stream_names:
            features["noise"] = self.noise_encoder(
                view["noise"]
            )

        return features

    def forward(self, view):
        features = self.encode_streams(view)

        fingerprint = features["fingerprint"]

        controller = self.fingerprint_controller(
            fingerprint
        )

        # Auxiliary branch predictions.
        stream_logits = torch.stack(
            [
                self.stream_heads[name](
                    features[name]
                )
                for name in self.stream_names
            ],
            dim=1,
        )

        stream_probs = F.softmax(
            stream_logits,
            dim=-1,
        )

        stream_entropy = normalized_entropy(
            stream_probs
        )

        disagreement = (
            compute_branch_disagreement(
                stream_probs
            )
        )

        # Only non-fingerprint branches are evidence tokens.
        evidence_features = torch.stack(
            [
                features[name]
                for name in self.evidence_streams
            ],
            dim=1,
        )

        routing_output = self.scalar_router(
            controller
        )

        routing_weights = routing_output[
            "routing_weights"
        ]

        gate_logits = routing_output[
            "gate_logits"
        ]

        modulated_features = (
            evidence_features
            * routing_weights.unsqueeze(-1)
        )

        routed_feature = (
            modulated_features.sum(dim=1)
        )

        joint_embedding = self.joint_projector(
            torch.cat(
                [
                    routed_feature,
                    controller,
                ],
                dim=1,
            )
        )

        # Final prediction comes only from the routed joint embedding.
        logits = self.final_head(
            joint_embedding
        )

        probs = F.softmax(
            logits,
            dim=1,
        )

        classification_entropy = (
            normalized_entropy(probs)
        )

        mean_disagreement = disagreement.mean(
            dim=(1, 2)
        )

        if self.num_evidence_streams > 1:
            routing_entropy = (
                normalized_entropy(
                    routing_weights
                )
            )
        else:
            routing_entropy = torch.zeros_like(
                classification_entropy
            )

        uncertainty = torch.clamp(
            0.50 * classification_entropy
            + 0.30 * mean_disagreement
            + 0.20 * routing_entropy,
            0.0,
            1.0,
        )

        # Backward-compatible reliability tensor aligned with
        # model.stream_names. Fingerprint is a controller, so its
        # routing weight is zero.
        reliability = torch.zeros(
            logits.shape[0],
            self.num_streams,
            dtype=logits.dtype,
            device=logits.device,
        )

        for evidence_index, stream_name in enumerate(
            self.evidence_streams
        ):
            stream_index = self.stream_names.index(
                stream_name
            )

            reliability[:, stream_index] = (
                routing_weights[
                    :,
                    evidence_index,
                ]
            )

        return {
            "logits": logits,
            "probs": probs,
            "fake_prob": probs[:, 1],
            "uncertainty": uncertainty,

            "stream_logits": stream_logits,
            "stream_probs": stream_probs,
            "stream_entropy": stream_entropy,
            "disagreement": disagreement,

            "fingerprint": fingerprint,
            "controller": controller,
            "joint_embedding": joint_embedding,

            "routing_weights": routing_weights,
            "gate_weights": routing_weights,
            "gate_logits": gate_logits,
            "routing_entropy": routing_entropy,

            # Compatibility with the current prediction logger.
            "reliability": reliability,

            "evidence_streams": self.evidence_streams,
        }

'''

    lines = text.splitlines(keepends=True)

    insertion_index = simple_model_node.lineno - 1

    updated_lines = (
        lines[:insertion_index]
        + agrfn_code.splitlines(keepends=True)
        + lines[insertion_index:]
    )

    text = "".join(updated_lines)

    print("\nInserted class AGRFN.")

else:
    print("\nClass AGRFN already exists. Skipping insertion.")

# ============================================================
# Reparse after adding AGRFN
# ============================================================

tree = ast.parse(text)

# ============================================================
# Add model factory before the Losses section
# ============================================================

top_level_names = {
    node.name
    for node in tree.body
    if isinstance(node, (ast.ClassDef, ast.FunctionDef))
}

if "build_model" not in top_level_names:
    losses_marker = (
        "# ============================================================\n"
        "# Losses\n"
        "# ============================================================\n"
    )

    if losses_marker not in text:
        raise RuntimeError(
            "Could not find the Losses section."
        )

    factory_code = r'''

def build_model(args, stream_names):
    """Build a model while preserving late-fusion compatibility."""

    if args.fusion_mode == "late":
        return GRFNCSF(
            feat_dim=args.feat_dim,
            stream_names=stream_names,
            stream_logit_residual_weight=(
                args.stream_logit_residual_weight
            ),
        )

    if args.fusion_mode == "average":
        return SimpleStreamModel(
            feat_dim=args.feat_dim,
            stream_names=stream_names,
        )

    if args.fusion_mode == "adaptive_scalar":
        return AGRFN(
            feat_dim=args.feat_dim,
            controller_dim=args.controller_dim,
            stream_names=stream_names,
            routing_temperature=(
                args.routing_temperature
            ),
        )

    raise NotImplementedError(
        f"Fusion mode '{args.fusion_mode}' has not "
        "been implemented yet."
    )


'''

    text = text.replace(
        losses_marker,
        factory_code + losses_marker,
        1,
    )

    print("Inserted build_model factory.")

else:
    print(
        "build_model already exists. "
        "Skipping insertion."
    )

# ============================================================
# Replace the temporary CLI guard
# ============================================================

tree = ast.parse(text)

parse_args_node = next(
    (
        node
        for node in tree.body
        if isinstance(node, ast.FunctionDef)
        and node.name == "parse_args"
    ),
    None,
)

if parse_args_node is None:
    raise RuntimeError(
        "Could not find parse_args()."
    )

temporary_guard = None

for node in ast.walk(parse_args_node):
    if not isinstance(node, ast.If):
        continue

    source = ast.get_source_segment(
        text,
        node.test,
    )

    if source and (
        'args.fusion_mode != "late"' in source
        or "args.fusion_mode != 'late'" in source
    ):
        temporary_guard = node
        break

if temporary_guard is not None:
    lines = text.splitlines(keepends=True)

    replacement = '''    if args.fusion_mode in {
        "adaptive_film",
        "adaptive_transformer",
    }:
        raise NotImplementedError(
            f"Fusion mode '{args.fusion_mode}' will be enabled "
            "in a later implementation stage."
        )

'''

    start_index = temporary_guard.lineno - 1
    end_index = temporary_guard.end_lineno

    lines = (
        lines[:start_index]
        + replacement.splitlines(keepends=True)
        + lines[end_index:]
    )

    text = "".join(lines)

    print(
        "Updated safety guard. "
        "average and adaptive_scalar are now enabled."
    )

else:
    print(
        "Temporary late-only guard was not found. "
        "It may already be updated."
    )

# ============================================================
# Replace hardcoded GRFNCSF construction in main()
# ============================================================

tree = ast.parse(text)

main_node = next(
    (
        node
        for node in tree.body
        if isinstance(node, ast.FunctionDef)
        and node.name == "main"
    ),
    None,
)

if main_node is None:
    raise RuntimeError(
        "Could not find main()."
    )

model_assignment = None

for node in ast.walk(main_node):
    if not isinstance(node, ast.Assign):
        continue

    if len(node.targets) != 1:
        continue

    target = node.targets[0]

    if not (
        isinstance(target, ast.Name)
        and target.id == "model"
    ):
        continue

    value_source = ast.get_source_segment(
        text,
        node.value,
    )

    if value_source and "GRFNCSF" in value_source:
        model_assignment = node
        break

if model_assignment is not None:
    lines = text.splitlines(keepends=True)

    replacement = (
        "    model = build_model(\n"
        "        args,\n"
        "        stream_names,\n"
        "    ).to(device)\n"
    )

    start_index = model_assignment.lineno - 1
    end_index = model_assignment.end_lineno

    lines = (
        lines[:start_index]
        + replacement.splitlines(keepends=True)
        + lines[end_index:]
    )

    text = "".join(lines)

    print(
        "Replaced hardcoded GRFNCSF construction "
        "with build_model()."
    )

elif "model = build_model(" in text:
    print(
        "main() already uses build_model()."
    )

else:
    raise RuntimeError(
        "Could not locate the model construction "
        "inside main()."
    )

# ============================================================
# Save and compile
# ============================================================

script_path.write_text(text)

py_compile.compile(
    str(script_path),
    doraise=True,
)

print("\nPASS: Python compilation succeeded.")

# ============================================================
# Import the updated module
# ============================================================

module_name = (
    "agrfn_scalar_test_"
    + timestamp
)

spec = importlib.util.spec_from_file_location(
    module_name,
    script_path,
)

module = importlib.util.module_from_spec(spec)

if spec.loader is None:
    raise RuntimeError(
        "Could not create module loader."
    )

spec.loader.exec_module(module)

# ============================================================
# Build adaptive scalar model through the factory
# ============================================================

args = SimpleNamespace(
    fusion_mode="adaptive_scalar",
    feat_dim=128,
    controller_dim=128,
    routing_temperature=1.0,
    stream_logit_residual_weight=0.25,
)

stream_names = [
    "rgb",
    "fingerprint",
    "jpeg",
    "freq",
]

model = module.build_model(
    args,
    stream_names,
)

model.train()

# ============================================================
# Dummy forward test
# ============================================================

batch_size = 2
image_size = 224

dummy_view = {
    "rgb": torch.randn(
        batch_size,
        3,
        image_size,
        image_size,
    ),
    "rec": torch.randn(
        batch_size,
        3,
        image_size,
        image_size,
    ),
    "den": torch.randn(
        batch_size,
        3,
        image_size,
        image_size,
    ),
    "jpeg": torch.randn(
        batch_size,
        3,
        image_size,
        image_size,
    ),
    "freq": torch.randn(
        batch_size,
        1,
        image_size,
        image_size,
    ),
    "noise": torch.randn(
        batch_size,
        1,
        image_size,
        image_size,
    ),
}

output = model(dummy_view)

expected_shapes = {
    "logits": (batch_size, 2),
    "stream_logits": (
        batch_size,
        len(stream_names),
        2,
    ),
    "routing_weights": (
        batch_size,
        3,
    ),
    "reliability": (
        batch_size,
        len(stream_names),
    ),
    "joint_embedding": (
        batch_size,
        128,
    ),
    "controller": (
        batch_size,
        128,
    ),
}

for key, expected_shape in expected_shapes.items():
    actual_shape = tuple(
        output[key].shape
    )

    if actual_shape != expected_shape:
        raise RuntimeError(
            f"{key} shape mismatch. "
            f"Expected {expected_shape}, "
            f"received {actual_shape}."
        )

route_sums = output[
    "routing_weights"
].sum(dim=1)

if not torch.allclose(
    route_sums,
    torch.ones_like(route_sums),
    atol=1e-6,
):
    raise RuntimeError(
        "Routing weights do not sum to one."
    )

if not torch.isfinite(
    output["logits"]
).all():
    raise RuntimeError(
        "Final logits contain NaN or infinity."
    )

# ============================================================
# Verify final logits do not depend on branch heads
# ============================================================

model.zero_grad(set_to_none=True)

output = model(dummy_view)

final_only_loss = (
    output["logits"].sum()
)

final_only_loss.backward()

branch_head_gradients = {}

for name, head in model.stream_heads.items():
    has_gradient = any(
        parameter.grad is not None
        for parameter in head.parameters()
    )

    branch_head_gradients[name] = has_gradient

if any(branch_head_gradients.values()):
    raise RuntimeError(
        "Final logits still depend on auxiliary "
        f"branch heads: {branch_head_gradients}"
    )

controller_has_gradient = any(
    parameter.grad is not None
    and torch.isfinite(parameter.grad).all()
    for parameter in (
        model.fingerprint_controller.parameters()
    )
)

router_has_gradient = any(
    parameter.grad is not None
    and torch.isfinite(parameter.grad).all()
    for parameter in (
        model.scalar_router.parameters()
    )
)

final_head_has_gradient = any(
    parameter.grad is not None
    and torch.isfinite(parameter.grad).all()
    for parameter in (
        model.final_head.parameters()
    )
)

if not controller_has_gradient:
    raise RuntimeError(
        "Fingerprint controller received no gradient."
    )

if not router_has_gradient:
    raise RuntimeError(
        "Scalar router received no gradient."
    )

if not final_head_has_gradient:
    raise RuntimeError(
        "Final classifier received no gradient."
    )

# ============================================================
# Verify late mode still builds
# ============================================================

late_args = SimpleNamespace(
    fusion_mode="late",
    feat_dim=128,
    controller_dim=128,
    routing_temperature=1.0,
    stream_logit_residual_weight=0.25,
)

late_model = module.build_model(
    late_args,
    stream_names,
)

if not isinstance(
    late_model,
    module.GRFNCSF,
):
    raise RuntimeError(
        "Late mode did not build GRFNCSF."
    )

# ============================================================
# Final report
# ============================================================

print("\n" + "=" * 80)
print("A-GRFN ADAPTIVE SCALAR MODEL TEST PASSED")
print("=" * 80)

print("\nModel class:")
print(type(model).__name__)

print("\nActive streams:")
print(model.stream_names)

print("\nEvidence streams:")
print(model.evidence_streams)

print("\nFinal logits shape:")
print(tuple(output["logits"].shape))

print("\nJoint embedding shape:")
print(tuple(output["joint_embedding"].shape))

print("\nRouting weights:")
print(
    output["routing_weights"]
    .detach()
    .cpu()
)

print("\nRouting sums:")
print(
    output["routing_weights"]
    .sum(dim=1)
    .detach()
    .cpu()
)

print("\nAuxiliary branch-head gradients from final logits:")
print(branch_head_gradients)

print("\nController gradient:")
print(controller_has_gradient)

print("Scalar-router gradient:")
print(router_has_gradient)

print("Final-head gradient:")
print(final_head_has_gradient)

print("\nLate-fusion model still available:")
print(type(late_model).__name__)

print(
    "\nPASS: final A-GRFN logits come from the "
    "fingerprint-conditioned routed joint embedding, "
    "not from auxiliary branch predictions."
)

Backup created:
/home/jovyan/Deepfake_Low Resource_Low Resolution/agrfn_adaptive_routing_df40_before_scalar_model_20260730_214941.py

Inserted class AGRFN.
Inserted build_model factory.
Updated safety guard. average and adaptive_scalar are now enabled.
Replaced hardcoded GRFNCSF construction with build_model().

PASS: Python compilation succeeded.

A-GRFN ADAPTIVE SCALAR MODEL TEST PASSED

Model class:
AGRFN

Active streams:
['rgb', 'fingerprint', 'jpeg', 'freq']

Evidence streams:
['rgb', 'jpeg', 'freq']

Final logits shape:
(2, 2)

Joint embedding shape:
(2, 128)

Routing weights:
tensor([[0.3333, 0.3333, 0.3333],
        [0.3333, 0.3333, 0.3333]])

Routing sums:
tensor([1., 1.])

Auxiliary branch-head gradients from final logits:
{'rgb': False, 'fingerprint': False, 'jpeg': False, 'freq': False}

Controller gradient:
True
Scalar-router gradient:
True
Final-head gradient:
True

Late-fusion model still available:
GRFNCSF

PASS: final A-GRFN logits come from the fingerprint-conditioned

In [32]:
from pathlib import Path
from datetime import datetime
from types import SimpleNamespace
import ast
import shutil
import py_compile
import importlib.util
import torch

# ============================================================
# Paths and backup
# ============================================================

project_root = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution"
)

script_path = (
    project_root
    / "agrfn_adaptive_routing_df40.py"
)

if not script_path.exists():
    raise FileNotFoundError(
        f"A-GRFN script not found:\n{script_path}"
    )

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

backup_path = script_path.with_name(
    f"{script_path.stem}_before_adaptive_loss_{timestamp}.py"
)

shutil.copy2(script_path, backup_path)

print("Backup created:")
print(backup_path)

# ============================================================
# Read and parse the script
# ============================================================

text = script_path.read_text()
tree = ast.parse(text)

grfn_loss_node = next(
    (
        node
        for node in tree.body
        if isinstance(node, ast.FunctionDef)
        and node.name == "grfn_loss"
    ),
    None,
)

if grfn_loss_node is None:
    raise RuntimeError(
        "Could not find grfn_loss()."
    )

# ============================================================
# Replace grfn_loss with a mode-aware implementation
# ============================================================

new_loss_function = r'''def grfn_loss(
    model,
    batch,
    args,
    device,
    class_weights=None,
):
    """Compute late-fusion or adaptive A-GRFN training losses."""

    view_a = move_view_to_device(
        batch["a"],
        device,
    )

    view_b = move_view_to_device(
        batch["b"],
        device,
    )

    y = batch["label"].to(
        device,
        non_blocking=True,
    )

    out_a = model(view_a)
    out_b = model(view_b)

    # --------------------------------------------------------
    # Main final-classification loss
    # --------------------------------------------------------

    cls_loss = 0.5 * (
        F.cross_entropy(
            out_a["logits"],
            y,
            weight=class_weights,
        )
        +
        F.cross_entropy(
            out_b["logits"],
            y,
            weight=class_weights,
        )
    )

    loss_mode = getattr(
        args,
        "loss_mode",
        "grfn",
    )

    if loss_mode == "auto":
        loss_mode = (
            "ce"
            if getattr(args, "model_mode", "grfn") == "simple"
            else "grfn"
        )

    zero = torch.zeros(
        (),
        dtype=cls_loss.dtype,
        device=device,
    )

    if loss_mode == "ce":
        logs = {
            "total": float(cls_loss.detach().cpu()),
            "cls": float(cls_loss.detach().cpu()),
            "stream": 0.0,
            "pred_cons": 0.0,
            "feat_cons": 0.0,
            "supcon": 0.0,
            "route_balance": 0.0,
            "route_cons": 0.0,
            "aug_cons": 0.0,
            "proto": 0.0,
            "unc": 0.0,
        }

        return cls_loss, logs

    # --------------------------------------------------------
    # Auxiliary branch classification
    # --------------------------------------------------------

    stream_logits_a = out_a["stream_logits"]
    stream_logits_b = out_b["stream_logits"]

    batch_size, num_streams, num_classes = (
        stream_logits_a.shape
    )

    if num_classes != 2:
        raise ValueError(
            "Expected two output classes: real and fake."
        )

    y_repeated = (
        y.unsqueeze(1)
        .repeat(1, num_streams)
        .reshape(-1)
    )

    stream_loss_a = F.cross_entropy(
        stream_logits_a.reshape(
            batch_size * num_streams,
            num_classes,
        ),
        y_repeated,
        weight=class_weights,
    )

    stream_loss_b = F.cross_entropy(
        stream_logits_b.reshape(
            batch_size * num_streams,
            num_classes,
        ),
        y_repeated,
        weight=class_weights,
    )

    stream_loss = 0.5 * (
        stream_loss_a + stream_loss_b
    )

    # --------------------------------------------------------
    # Prediction consistency between augmented views
    # --------------------------------------------------------

    final_prediction_consistency = 0.5 * (
        soft_kl(
            out_a["logits"],
            out_b["logits"],
        )
        +
        soft_kl(
            out_b["logits"],
            out_a["logits"],
        )
    )

    stream_prediction_consistency = 0.5 * (
        F.mse_loss(
            out_a["stream_probs"],
            out_b["stream_probs"].detach(),
        )
        +
        F.mse_loss(
            out_b["stream_probs"],
            out_a["stream_probs"].detach(),
        )
    )

    pred_cons = (
        final_prediction_consistency
        + stream_prediction_consistency
    )

    # --------------------------------------------------------
    # Fingerprint consistency
    # --------------------------------------------------------

    feat_cons = (
        1.0
        - F.cosine_similarity(
            out_a["fingerprint"],
            out_b["fingerprint"],
            dim=1,
        ).mean()
    )

    # --------------------------------------------------------
    # Optional supervised contrastive fingerprint loss
    # --------------------------------------------------------

    fingerprint_all = torch.cat(
        [
            out_a["fingerprint"],
            out_b["fingerprint"],
        ],
        dim=0,
    )

    labels_all = torch.cat(
        [y, y],
        dim=0,
    )

    supcon = supervised_contrastive_loss(
        fingerprint_all,
        labels_all,
    )

    # --------------------------------------------------------
    # Adaptive routing losses
    # --------------------------------------------------------

    fusion_mode = getattr(
        args,
        "fusion_mode",
        "late",
    )

    is_adaptive = fusion_mode in {
        "adaptive_scalar",
        "adaptive_film",
        "adaptive_transformer",
    }

    route_balance = zero
    route_cons = zero
    aug_cons = feat_cons

    if is_adaptive:
        if (
            "routing_weights" not in out_a
            or "routing_weights" not in out_b
        ):
            raise KeyError(
                "Adaptive fusion mode requires "
                "routing_weights in the model output."
            )

        routing_a = out_a["routing_weights"]
        routing_b = out_b["routing_weights"]

        if routing_a.shape != routing_b.shape:
            raise ValueError(
                "Routing shapes differ between augmented views: "
                f"{tuple(routing_a.shape)} versus "
                f"{tuple(routing_b.shape)}."
            )

        route_balance = 0.5 * (
            routing_balance_loss(routing_a)
            +
            routing_balance_loss(routing_b)
        )

        route_cons = 0.5 * (
            F.mse_loss(
                routing_a,
                routing_b.detach(),
            )
            +
            F.mse_loss(
                routing_b,
                routing_a.detach(),
            )
        )

        # Combine fingerprint stability and route stability.
        aug_cons = 0.5 * (
            feat_cons + route_cons
        )

    # --------------------------------------------------------
    # Mode-aware total objective
    # --------------------------------------------------------

    if is_adaptive:
        lambda_aux = float(
            getattr(
                args,
                "lambda_aux",
                getattr(args, "lambda_stream", 0.05),
            )
        )

        lambda_route = float(
            getattr(args, "lambda_route", 0.01)
        )

        lambda_cons = float(
            getattr(
                args,
                "lambda_cons",
                getattr(args, "lambda_feat_cons", 0.02),
            )
        )

        total = (
            cls_loss
            + lambda_aux * stream_loss
            + args.lambda_pred_cons * pred_cons
            + lambda_cons * aug_cons
            + lambda_route * route_balance
            + args.lambda_supcon * supcon
        )

    else:
        # Preserve the original GRFN-CSF loss exactly for
        # late-fusion and average-fusion baselines.
        total = (
            cls_loss
            + args.lambda_stream * stream_loss
            + args.lambda_pred_cons * pred_cons
            + args.lambda_feat_cons * feat_cons
            + args.lambda_supcon * supcon
        )

    logs = {
        "total": float(total.detach().cpu()),
        "cls": float(cls_loss.detach().cpu()),
        "stream": float(stream_loss.detach().cpu()),
        "pred_cons": float(pred_cons.detach().cpu()),
        "feat_cons": float(feat_cons.detach().cpu()),
        "supcon": float(supcon.detach().cpu()),
        "route_balance": float(
            route_balance.detach().cpu()
        ),
        "route_cons": float(
            route_cons.detach().cpu()
        ),
        "aug_cons": float(
            aug_cons.detach().cpu()
        ),
        "proto": 0.0,
        "unc": 0.0,
    }

    return total, logs
'''

lines = text.splitlines(keepends=True)

start_index = grfn_loss_node.lineno - 1
end_index = grfn_loss_node.end_lineno

lines = (
    lines[:start_index]
    + new_loss_function.splitlines(keepends=True)
    + ["\n\n"]
    + lines[end_index:]
)

text = "".join(lines)

print("\nReplaced grfn_loss() with adaptive mode support.")

# ============================================================
# Update train_one_epoch logging keys
# ============================================================

tree = ast.parse(text)

train_node = next(
    (
        node
        for node in tree.body
        if isinstance(node, ast.FunctionDef)
        and node.name == "train_one_epoch"
    ),
    None,
)

if train_node is None:
    raise RuntimeError(
        "Could not find train_one_epoch()."
    )

running_assignment = next(
    (
        node
        for node in ast.walk(train_node)
        if isinstance(node, ast.Assign)
        and any(
            isinstance(target, ast.Name)
            and target.id == "running"
            for target in node.targets
        )
    ),
    None,
)

if running_assignment is None:
    raise RuntimeError(
        "Could not find the running-loss dictionary "
        "inside train_one_epoch()."
    )

running_replacement = '''    running = {
        "total": [],
        "cls": [],
        "stream": [],
        "pred_cons": [],
        "feat_cons": [],
        "supcon": [],
        "route_balance": [],
        "route_cons": [],
        "aug_cons": [],
        "proto": [],
        "unc": [],
    }
'''

lines = text.splitlines(keepends=True)

start_index = running_assignment.lineno - 1
end_index = running_assignment.end_lineno

lines = (
    lines[:start_index]
    + running_replacement.splitlines(keepends=True)
    + lines[end_index:]
)

text = "".join(lines)

print(
    "Updated train_one_epoch() loss logging."
)

# ============================================================
# Save and compile
# ============================================================

script_path.write_text(text)

py_compile.compile(
    str(script_path),
    doraise=True,
)

print("\nPASS: Python compilation succeeded.")

# ============================================================
# Import updated module
# ============================================================

module_name = (
    "agrfn_loss_test_"
    + timestamp
)

spec = importlib.util.spec_from_file_location(
    module_name,
    script_path,
)

module = importlib.util.module_from_spec(spec)

if spec.loader is None:
    raise RuntimeError(
        "Could not create module loader."
    )

spec.loader.exec_module(module)

# ============================================================
# Shared dummy inputs
# ============================================================

device = torch.device("cpu")
batch_size = 2
image_size = 64

def make_dummy_view():
    return {
        "rgb": torch.randn(
            batch_size,
            3,
            image_size,
            image_size,
        ),
        "rec": torch.randn(
            batch_size,
            3,
            image_size,
            image_size,
        ),
        "den": torch.randn(
            batch_size,
            3,
            image_size,
            image_size,
        ),
        "jpeg": torch.randn(
            batch_size,
            3,
            image_size,
            image_size,
        ),
        "freq": torch.randn(
            batch_size,
            1,
            image_size,
            image_size,
        ),
        "noise": torch.randn(
            batch_size,
            1,
            image_size,
            image_size,
        ),
    }

dummy_batch = {
    "a": make_dummy_view(),
    "b": make_dummy_view(),
    "label": torch.tensor(
        [0, 1],
        dtype=torch.long,
    ),
}

stream_names = [
    "rgb",
    "fingerprint",
    "jpeg",
    "freq",
]

# ============================================================
# Adaptive scalar loss test
# ============================================================

adaptive_args = SimpleNamespace(
    fusion_mode="adaptive_scalar",
    feat_dim=128,
    controller_dim=128,
    routing_temperature=1.0,
    stream_logit_residual_weight=0.25,

    loss_mode="grfn",
    model_mode="grfn",

    lambda_stream=0.05,
    lambda_aux=0.05,

    lambda_pred_cons=0.02,
    lambda_feat_cons=0.02,
    lambda_cons=0.02,

    lambda_route=0.01,
    lambda_supcon=0.00,
)

adaptive_model = module.build_model(
    adaptive_args,
    stream_names,
).to(device)

adaptive_model.train()
adaptive_model.zero_grad(set_to_none=True)

adaptive_loss, adaptive_logs = module.grfn_loss(
    adaptive_model,
    dummy_batch,
    adaptive_args,
    device,
    class_weights=None,
)

if not torch.isfinite(adaptive_loss):
    raise RuntimeError(
        "Adaptive loss is not finite."
    )

adaptive_loss.backward()

router_gradients = [
    parameter.grad
    for parameter in adaptive_model.scalar_router.parameters()
    if parameter.grad is not None
]

if not router_gradients:
    raise RuntimeError(
        "Scalar router received no gradients "
        "from the adaptive training loss."
    )

if not all(
    torch.isfinite(gradient).all()
    for gradient in router_gradients
):
    raise RuntimeError(
        "Scalar-router gradients contain NaN or infinity."
    )

expected_log_keys = {
    "total",
    "cls",
    "stream",
    "pred_cons",
    "feat_cons",
    "supcon",
    "route_balance",
    "route_cons",
    "aug_cons",
    "proto",
    "unc",
}

missing_log_keys = (
    expected_log_keys - set(adaptive_logs)
)

if missing_log_keys:
    raise RuntimeError(
        "Adaptive loss logs are missing: "
        f"{sorted(missing_log_keys)}"
    )

# ============================================================
# Late-fusion compatibility test
# ============================================================

late_args = SimpleNamespace(
    fusion_mode="late",
    feat_dim=128,
    controller_dim=128,
    routing_temperature=1.0,
    stream_logit_residual_weight=0.25,

    loss_mode="grfn",
    model_mode="grfn",

    lambda_stream=0.05,
    lambda_aux=0.05,

    lambda_pred_cons=0.02,
    lambda_feat_cons=0.02,
    lambda_cons=0.02,

    lambda_route=0.01,
    lambda_supcon=0.00,
)

late_model = module.build_model(
    late_args,
    stream_names,
).to(device)

late_model.train()

late_loss, late_logs = module.grfn_loss(
    late_model,
    dummy_batch,
    late_args,
    device,
    class_weights=None,
)

if not torch.isfinite(late_loss):
    raise RuntimeError(
        "Late-fusion loss is not finite."
    )

if late_logs["route_balance"] != 0.0:
    raise RuntimeError(
        "Late fusion unexpectedly received "
        "routing-balance loss."
    )

if late_logs["route_cons"] != 0.0:
    raise RuntimeError(
        "Late fusion unexpectedly received "
        "routing-consistency loss."
    )

# ============================================================
# Final report
# ============================================================

print("\n" + "=" * 80)
print("ADAPTIVE LOSS TEST PASSED")
print("=" * 80)

print("\nAdaptive scalar loss:")
print(float(adaptive_loss.detach()))

print("\nAdaptive loss components:")

for key, value in adaptive_logs.items():
    print(f"{key:16s}: {value:.6f}")

print("\nScalar-router gradient tensors:")
print(len(router_gradients))

print("\nLate-fusion loss:")
print(float(late_loss.detach()))

print("\nLate routing losses:")
print(
    "route_balance =",
    late_logs["route_balance"],
)
print(
    "route_cons =",
    late_logs["route_cons"],
)

print(
    "\nPASS: adaptive scalar now trains with auxiliary, "
    "routing-balance, and augmentation-consistency losses."
)

print(
    "PASS: late fusion keeps its original loss behavior."
)

Backup created:
/home/jovyan/Deepfake_Low Resource_Low Resolution/agrfn_adaptive_routing_df40_before_adaptive_loss_20260730_215134.py

Replaced grfn_loss() with adaptive mode support.
Updated train_one_epoch() loss logging.

PASS: Python compilation succeeded.

ADAPTIVE LOSS TEST PASSED

Adaptive scalar loss:
0.7709503173828125

Adaptive loss components:
total           : 0.770950
cls             : 0.731682
stream          : 0.776251
pred_cons       : 0.003803
feat_cons       : 0.038001
supcon          : 0.942558
route_balance   : 0.000000
route_cons      : 0.000000
aug_cons        : 0.019000
proto           : 0.000000
unc             : 0.000000

Scalar-router gradient tensors:
6

Late-fusion loss:
0.7232771515846252

Late routing losses:
route_balance = 0.0
route_cons = 0.0

PASS: adaptive scalar now trains with auxiliary, routing-balance, and augmentation-consistency losses.
PASS: late fusion keeps its original loss behavior.


In [33]:
from pathlib import Path
from datetime import datetime
import subprocess
import sys
import pandas as pd
import numpy as np

project_root = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution"
)

script_path = project_root / "agrfn_adaptive_routing_df40.py"
manifest_path = project_root / "outputs_df40/df40_manifest_all_clean.csv"

if not script_path.exists():
    raise FileNotFoundError(script_path)

if not manifest_path.exists():
    raise FileNotFoundError(manifest_path)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

out_dir = (
    project_root
    / f"outputs_agrfn_scalar_df40_smoke_{timestamp}"
)

command = [
    sys.executable,
    str(script_path),

    "--metadata_csv",
    str(manifest_path),

    "--out_dir",
    str(out_dir),

    "--dataset_name",
    "DF40",

    "--fusion_mode",
    "adaptive_scalar",

    "--streams",
    "rgb,fingerprint,jpeg,freq",

    "--frames_per_video",
    "2",

    "--limit_train_units",
    "40",

    "--limit_val_units",
    "20",

    "--limit_test_units",
    "20",

    "--balanced_unit_limits",

    "--image_size",
    "224",

    "--batch_size",
    "4",

    "--num_workers",
    "0",

    "--epochs",
    "1",

    "--lr",
    "1e-4",

    "--feat_dim",
    "128",

    "--lambda_aux",
    "0.05",

    "--lambda_route",
    "0.01",

    "--lambda_cons",
    "0.02",

    "--lambda_pred_cons",
    "0.02",

    "--balance_strategy",
    "auto",

    "--score_direction",
    "normal",

    "--decision_threshold",
    "0.5",

    "--amp",
]

print("=" * 80)
print("RUNNING A-GRFN SCALAR SMOKE TEST")
print("=" * 80)

print("\nOutput directory:")
print(out_dir)

print("\nCommand:")
print(" ".join(f'"{item}"' if " " in item else item for item in command))

result = subprocess.run(
    command,
    cwd=str(project_root),
)

if result.returncode != 0:
    raise RuntimeError(
        f"Smoke test failed with return code {result.returncode}."
    )

# ============================================================
# Validate generated files
# ============================================================

required_files = [
    "best_checkpoint.pt",
    "latest_checkpoint.pt",
    "training_history.csv",
    "test_frame_predictions.csv",
    "test_video_predictions.csv",
    "test_metrics.csv",
    "test_inference_time.csv",
    "experiment_summary.csv",
]

missing_files = [
    name
    for name in required_files
    if not (out_dir / name).exists()
]

if missing_files:
    raise RuntimeError(
        "Smoke test finished, but these files are missing:\n"
        + "\n".join(missing_files)
    )

history = pd.read_csv(
    out_dir / "training_history.csv"
)

metrics = pd.read_csv(
    out_dir / "test_metrics.csv"
)

predictions = pd.read_csv(
    out_dir / "test_frame_predictions.csv"
)

summary = pd.read_csv(
    out_dir / "experiment_summary.csv"
)

# ============================================================
# Basic safety checks
# ============================================================

required_prediction_columns = [
    "label",
    "fake_prob",
    "uncertainty",
    "mean_branch_disagreement",
]

missing_prediction_columns = [
    column
    for column in required_prediction_columns
    if column not in predictions.columns
]

if missing_prediction_columns:
    raise RuntimeError(
        "Missing prediction columns: "
        + ", ".join(missing_prediction_columns)
    )

numeric_columns = [
    "fake_prob",
    "uncertainty",
    "mean_branch_disagreement",
]

if not np.isfinite(
    predictions[numeric_columns].to_numpy()
).all():
    raise RuntimeError(
        "Prediction output contains NaN or infinity."
    )

if not predictions["fake_prob"].between(0, 1).all():
    raise RuntimeError(
        "fake_prob contains values outside [0, 1]."
    )

if not predictions["uncertainty"].between(0, 1).all():
    raise RuntimeError(
        "uncertainty contains values outside [0, 1]."
    )

reliability_columns = [
    column
    for column in predictions.columns
    if column.startswith("rel_")
]

if reliability_columns:
    reliability_sums = predictions[
        reliability_columns
    ].sum(axis=1)

    max_route_sum_error = float(
        np.abs(reliability_sums - 1.0).max()
    )

    if max_route_sum_error > 1e-4:
        raise RuntimeError(
            "Adaptive reliability/routing weights do not sum to one. "
            f"Maximum error: {max_route_sum_error}"
        )
else:
    max_route_sum_error = np.nan

# ============================================================
# Report
# ============================================================

print("\n" + "=" * 80)
print("A-GRFN SCALAR SMOKE TEST PASSED")
print("=" * 80)

print("\nOutput directory:")
print(out_dir)

print("\nTraining history:")
display(history)

print("\nTest metrics:")
display(metrics)

print("\nExperiment summary:")
display(summary)

print("\nPrediction probability summary:")
display(
    predictions[
        [
            "label",
            "fake_prob",
            "uncertainty",
            "mean_branch_disagreement",
        ]
    ].describe()
)

print("\nRouting/reliability columns:")
print(reliability_columns)

if reliability_columns:
    print("\nMean routing weights:")
    display(
        predictions[
            reliability_columns
        ].mean().to_frame("mean_weight")
    )

    print("\nRouting-weight standard deviations:")
    display(
        predictions[
            reliability_columns
        ].std().to_frame("std_weight")
    )

    print("\nMaximum routing-sum error:")
    print(max_route_sum_error)

predicted_fake = int(
    (predictions["fake_prob"] >= 0.5).sum()
)

predicted_real = int(
    (predictions["fake_prob"] < 0.5).sum()
)

print("\nPredicted real:", predicted_real)
print("Predicted fake:", predicted_fake)

print(
    "\nPASS: adaptive scalar completed a full "
    "train, validation, checkpoint, and test cycle."
)

RUNNING A-GRFN SCALAR SMOKE TEST

Output directory:
/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_scalar_df40_smoke_20260730_215319

Command:
/opt/conda/bin/python "/home/jovyan/Deepfake_Low Resource_Low Resolution/agrfn_adaptive_routing_df40.py" --metadata_csv "/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40/df40_manifest_all_clean.csv" --out_dir "/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_scalar_df40_smoke_20260730_215319" --dataset_name DF40 --fusion_mode adaptive_scalar --streams rgb,fingerprint,jpeg,freq --frames_per_video 2 --limit_train_units 40 --limit_val_units 20 --limit_test_units 20 --balanced_unit_limits --image_size 224 --batch_size 4 --num_workers 0 --epochs 1 --lr 1e-4 --feat_dim 128 --lambda_aux 0.05 --lambda_route 0.01 --lambda_cons 0.02 --lambda_pred_cons 0.02 --balance_strategy auto --score_direction normal --decision_threshold 0.5 --amp

GRFN-CSF GRFN-only training and evaluation
Device: cuda
Output directory: /

Sample train: 100%|██████████| 40/40 [00:00<00:00, 65.15unit/s]


Sample train: frames=78, units=40, real=40, fake=38
Sample val: frames=39, units=20, real=20, fake=19


Sample test: 100%|██████████| 20/20 [00:00<00:00, 355.74unit/s]


Sample test: frames=39, units=20, real=20, fake=19

Balancing strategy
Train class counts [real, fake]: [40, 38]
Train imbalance ratio: 1.053
Use class-weighted loss: False
Use weighted sampler: False

Building dataloaders
Train batches: 19
Val batches: 10
Test batches: 10

Building model
Model mode: grfn
Loss mode: grfn
Active streams: rgb, fingerprint, jpeg, freq
Total params: 1,681,741
Trainable params: 1,681,741

Training


Predict test:   0%|          | 0/10 [00:00<?, ?it/s]

... (11 outputs left out)

,mean_weight
rel_rgb,0.339667
rel_fingerprint,0.000000
rel_jpeg,0.332183
rel_freq,0.328149



Routing-weight standard deviations:


,std_weight
rel_rgb,0.000549
rel_fingerprint,0.000000
rel_jpeg,0.000611
rel_freq,0.000276



Maximum routing-sum error:
5.960464488641293e-08

Predicted real: 35
Predicted fake: 4

PASS: adaptive scalar completed a full train, validation, checkpoint, and test cycle.


In [34]:
from pathlib import Path
from datetime import datetime
import ast
import shutil
import py_compile
import importlib.util
import tempfile
import pandas as pd

# ============================================================
# Paths and backup
# ============================================================

project_root = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution"
)

script_path = (
    project_root
    / "agrfn_adaptive_routing_df40.py"
)

if not script_path.exists():
    raise FileNotFoundError(script_path)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

backup_path = script_path.with_name(
    f"{script_path.stem}_before_diagnostic_logging_{timestamp}.py"
)

shutil.copy2(script_path, backup_path)

print("Backup created:")
print(backup_path)

text = script_path.read_text()

# ============================================================
# Add metadata columns preserved from the DF40 manifest
# ============================================================

metadata_block = '''

PASS_THROUGH_COLUMNS = [
    "method",
    "domain",
    "source_layout",
    "top_level_folder",
    "extension",
    "video_id",
    "dataset",
    "dataset_name",
    "root_type",
    "relative_path",
    "group_reliable",
    "dataset_partition",
]

'''

if "PASS_THROUGH_COLUMNS = [" not in text:
    anchor = 'DEFAULT_OUT_DIR = "./outputs_grfn_csf_universal"\n'

    if anchor not in text:
        raise RuntimeError(
            "Could not find DEFAULT_OUT_DIR insertion point."
        )

    text = text.replace(
        anchor,
        anchor + metadata_block,
        1,
    )

    print("\nAdded DF40 pass-through metadata columns.")
else:
    print("\nPass-through metadata columns already exist.")

# ============================================================
# Replace GRFNDataset.__getitem__
# ============================================================

tree = ast.parse(text)

dataset_class = next(
    (
        node
        for node in tree.body
        if isinstance(node, ast.ClassDef)
        and node.name == "GRFNDataset"
    ),
    None,
)

if dataset_class is None:
    raise RuntimeError("Could not find GRFNDataset.")

getitem_node = next(
    (
        node
        for node in dataset_class.body
        if isinstance(node, ast.FunctionDef)
        and node.name == "__getitem__"
    ),
    None,
)

if getitem_node is None:
    raise RuntimeError(
        "Could not find GRFNDataset.__getitem__()."
    )

new_getitem = '''    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img = self.load_image(
            row["image_path"]
        )

        label = int(row["label"])

        view_a = make_views_from_pil(
            img,
            self.image_size,
            train_aug=False,
        )

        view_b = make_views_from_pil(
            img,
            self.image_size,
            train_aug=(
                self.train
                and self.train_aug
            ),
        )

        sample = {
            "a": view_a,
            "b": view_b,
            "label": torch.tensor(
                label,
                dtype=torch.long,
            ),
            "group_id": str(
                row["group_id"]
            ),
            "sample_unit_id": str(
                row["sample_unit_id"]
            ),
            "image_path": str(
                row["image_path"]
            ),
        }

        for column in PASS_THROUGH_COLUMNS:
            if column not in row.index:
                continue

            value = row[column]

            if pd.isna(value):
                sample[column] = ""
            else:
                sample[column] = str(value)

        return sample
'''

lines = text.splitlines(keepends=True)

lines = (
    lines[: getitem_node.lineno - 1]
    + new_getitem.splitlines(keepends=True)
    + ["\n"]
    + lines[getitem_node.end_lineno :]
)

text = "".join(lines)

print("Updated GRFNDataset metadata pass-through.")

# ============================================================
# Replace predict_loader with diagnostic logging
# ============================================================

tree = ast.parse(text)

predict_node = next(
    (
        node
        for node in tree.body
        if isinstance(node, ast.FunctionDef)
        and node.name == "predict_loader"
    ),
    None,
)

if predict_node is None:
    raise RuntimeError(
        "Could not find predict_loader()."
    )

new_predict_loader = '''def predict_loader(
    model,
    loader,
    args,
    device,
    split_name="test",
):
    model.eval()
    rows = []

    def batch_value(batch, key, index, default=""):
        if key not in batch:
            return default

        value = batch[key][index]

        if torch.is_tensor(value):
            if value.numel() == 1:
                return value.item()

            return value.detach().cpu().tolist()

        return value

    pbar = tqdm(
        loader,
        desc=f"Predict {split_name}",
        dynamic_ncols=True,
    )

    for batch in pbar:
        view = move_view_to_device(
            batch["a"],
            device,
        )

        labels = (
            batch["label"]
            .cpu()
            .numpy()
            .astype(int)
        )

        out = model(view)

        fake_prob = (
            out["fake_prob"]
            .detach()
            .cpu()
            .numpy()
        )

        uncertainty = (
            out["uncertainty"]
            .detach()
            .cpu()
            .numpy()
        )

        stream_probs_tensor = out["stream_probs"]

        stream_probs = (
            stream_probs_tensor
            .detach()
            .cpu()
            .numpy()
        )

        if "stream_entropy" in out:
            stream_entropy_tensor = out[
                "stream_entropy"
            ]
        else:
            stream_entropy_tensor = (
                normalized_entropy(
                    stream_probs_tensor
                )
            )

        stream_entropy = (
            stream_entropy_tensor
            .detach()
            .cpu()
            .numpy()
        )

        disagreement_tensor = out[
            "disagreement"
        ]

        disagreement = (
            disagreement_tensor
            .detach()
            .cpu()
            .numpy()
        )

        reliability_tensor = out.get(
            "reliability"
        )

        reliability = (
            reliability_tensor
            .detach()
            .cpu()
            .numpy()
            if reliability_tensor is not None
            else None
        )

        routing_tensor = out.get(
            "routing_weights"
        )

        routing_weights = (
            routing_tensor
            .detach()
            .cpu()
            .numpy()
            if routing_tensor is not None
            else None
        )

        gate_tensor = out.get(
            "gate_weights"
        )

        gate_weights = (
            gate_tensor
            .detach()
            .cpu()
            .numpy()
            if gate_tensor is not None
            else None
        )

        attention_tensor = out.get(
            "attention_weights"
        )

        attention_weights = (
            attention_tensor
            .detach()
            .cpu()
            .numpy()
            if attention_tensor is not None
            else None
        )

        routing_entropy_tensor = out.get(
            "routing_entropy"
        )

        routing_entropy = (
            routing_entropy_tensor
            .detach()
            .cpu()
            .numpy()
            if routing_entropy_tensor is not None
            else None
        )

        evidence_streams = out.get(
            "evidence_streams",
            getattr(
                model,
                "evidence_streams",
                [],
            ),
        )

        evidence_streams = list(
            evidence_streams
        )

        if (
            routing_weights is not None
            and len(evidence_streams)
            != routing_weights.shape[1]
        ):
            evidence_streams = [
                f"branch_{index}"
                for index in range(
                    routing_weights.shape[1]
                )
            ]

        prototype_distances_tensor = (
            out.get("prototype_distances")
        )

        prototype_distances = (
            prototype_distances_tensor
            .detach()
            .cpu()
            .numpy()
            if prototype_distances_tensor is not None
            else None
        )

        prototype_margin_tensor = out.get(
            "prototype_margin"
        )

        prototype_margin = (
            prototype_margin_tensor
            .detach()
            .cpu()
            .numpy()
            if prototype_margin_tensor is not None
            else None
        )

        prototype_ambiguity_tensor = out.get(
            "prototype_ambiguity"
        )

        prototype_ambiguity = (
            prototype_ambiguity_tensor
            .detach()
            .cpu()
            .numpy()
            if prototype_ambiguity_tensor is not None
            else None
        )

        unknown_tensor = out.get(
            "unknown_suspiciousness"
        )

        unknown_suspiciousness = (
            unknown_tensor
            .detach()
            .cpu()
            .numpy()
            if unknown_tensor is not None
            else None
        )

        for index in range(len(fake_prob)):
            mean_disagreement = float(
                disagreement[index].mean()
            )

            max_disagreement = float(
                disagreement[index].max()
            )

            row = {
                "split": split_name,
                "image_path": batch_value(
                    batch,
                    "image_path",
                    index,
                ),
                "group_id": batch_value(
                    batch,
                    "group_id",
                    index,
                ),
                "sample_unit_id": batch_value(
                    batch,
                    "sample_unit_id",
                    index,
                ),
                "label": int(labels[index]),
                "fake_prob": float(
                    fake_prob[index]
                ),
                "uncertainty": float(
                    uncertainty[index]
                ),
                "classification_entropy": float(
                    normalized_entropy(
                        out["probs"]
                    )[index]
                    .detach()
                    .cpu()
                ),
                "mean_branch_entropy": float(
                    stream_entropy[index].mean()
                ),
                "mean_branch_disagreement": (
                    mean_disagreement
                ),
                "max_branch_disagreement": (
                    max_disagreement
                ),
            }

            for column in PASS_THROUGH_COLUMNS:
                row[column] = batch_value(
                    batch,
                    column,
                    index,
                    "",
                )

            for stream_index, stream_name in enumerate(
                model.stream_names
            ):
                row[
                    f"prob_{stream_name}"
                ] = float(
                    stream_probs[
                        index,
                        stream_index,
                        1,
                    ]
                )

                row[
                    f"entropy_{stream_name}"
                ] = float(
                    stream_entropy[
                        index,
                        stream_index,
                    ]
                )

                if reliability is not None:
                    row[
                        f"rel_{stream_name}"
                    ] = float(
                        reliability[
                            index,
                            stream_index,
                        ]
                    )

            if routing_weights is not None:
                for route_index, stream_name in enumerate(
                    evidence_streams
                ):
                    row[
                        f"route_{stream_name}"
                    ] = float(
                        routing_weights[
                            index,
                            route_index,
                        ]
                    )

            if gate_weights is not None:
                for gate_index, stream_name in enumerate(
                    evidence_streams
                ):
                    row[
                        f"gate_{stream_name}"
                    ] = float(
                        gate_weights[
                            index,
                            gate_index,
                        ]
                    )

            if attention_weights is not None:
                for attention_index, stream_name in enumerate(
                    evidence_streams
                ):
                    row[
                        f"attention_{stream_name}"
                    ] = float(
                        attention_weights[
                            index,
                            attention_index,
                        ]
                    )

            if routing_entropy is not None:
                row["routing_entropy"] = float(
                    routing_entropy[index]
                )

            for first_index, first_name in enumerate(
                model.stream_names
            ):
                for second_index in range(
                    first_index + 1,
                    len(model.stream_names),
                ):
                    second_name = (
                        model.stream_names[
                            second_index
                        ]
                    )

                    row[
                        f"disagree_{first_name}__{second_name}"
                    ] = float(
                        disagreement[
                            index,
                            first_index,
                            second_index,
                        ]
                    )

            if prototype_distances is not None:
                row["distance_real"] = float(
                    prototype_distances[
                        index,
                        0,
                    ]
                )

                row["distance_fake"] = float(
                    prototype_distances[
                        index,
                        1,
                    ]
                )

                row[
                    "minimum_prototype_distance"
                ] = float(
                    prototype_distances[
                        index
                    ].min()
                )

            if prototype_margin is not None:
                row["prototype_margin"] = float(
                    prototype_margin[index]
                )

            if prototype_ambiguity is not None:
                row[
                    "prototype_ambiguity"
                ] = float(
                    prototype_ambiguity[index]
                )

            if unknown_suspiciousness is not None:
                row[
                    "unknown_suspiciousness"
                ] = float(
                    unknown_suspiciousness[
                        index
                    ]
                )

            rows.append(row)

    return pd.DataFrame(rows)
'''

lines = text.splitlines(keepends=True)

# Keep the existing @torch.no_grad() decorator.
lines = (
    lines[: predict_node.lineno - 1]
    + new_predict_loader.splitlines(
        keepends=True
    )
    + ["\n\n"]
    + lines[predict_node.end_lineno :]
)

text = "".join(lines)

print("Updated predict_loader diagnostic logging.")

# ============================================================
# Add helper to save separate diagnostic CSV files
# ============================================================

tree = ast.parse(text)

existing_names = {
    node.name
    for node in tree.body
    if isinstance(node, ast.FunctionDef)
}

if "save_diagnostic_csvs" not in existing_names:
    prediction_debug_marker = (
        "def prediction_debug_summary("
    )

    marker_index = text.find(
        prediction_debug_marker
    )

    if marker_index < 0:
        raise RuntimeError(
            "Could not find prediction_debug_summary()."
        )

    helper_code = '''def save_diagnostic_csvs(
    prediction_df,
    out_dir,
):
    """Save A-GRFN diagnostic tables from the master prediction CSV."""

    out_dir = Path(out_dir)
    out_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    identifier_columns = [
        "split",
        "image_path",
        "group_id",
        "sample_unit_id",
        "label",
        "method",
        "domain",
        "top_level_folder",
        "source_layout",
        "relative_path",
    ]

    identifier_columns = [
        column
        for column in identifier_columns
        if column in prediction_df.columns
    ]

    diagnostic_groups = {
        "branch_predictions.csv": [
            "prob_",
        ],
        "branch_uncertainties.csv": [
            "entropy_",
        ],
        "routing_weights.csv": [
            "route_",
            "gate_",
            "attention_",
            "rel_",
        ],
        "disagreement_matrices.csv": [
            "disagree_",
        ],
    }

    uncertainty_columns = [
        "fake_prob",
        "uncertainty",
        "classification_entropy",
        "mean_branch_entropy",
        "mean_branch_disagreement",
        "max_branch_disagreement",
        "routing_entropy",
        "prototype_margin",
        "prototype_ambiguity",
        "minimum_prototype_distance",
        "unknown_suspiciousness",
    ]

    prototype_columns = [
        "distance_real",
        "distance_fake",
        "minimum_prototype_distance",
        "prototype_margin",
        "prototype_ambiguity",
        "unknown_suspiciousness",
    ]

    saved_paths = {}

    for filename, prefixes in diagnostic_groups.items():
        diagnostic_columns = [
            column
            for column in prediction_df.columns
            if any(
                column.startswith(prefix)
                for prefix in prefixes
            )
        ]

        if not diagnostic_columns:
            continue

        output_path = out_dir / filename

        prediction_df[
            identifier_columns
            + diagnostic_columns
        ].to_csv(
            output_path,
            index=False,
        )

        saved_paths[filename] = output_path

    existing_uncertainty_columns = [
        column
        for column in uncertainty_columns
        if column in prediction_df.columns
    ]

    if existing_uncertainty_columns:
        output_path = (
            out_dir
            / "uncertainty_breakdown.csv"
        )

        prediction_df[
            identifier_columns
            + existing_uncertainty_columns
        ].to_csv(
            output_path,
            index=False,
        )

        saved_paths[
            "uncertainty_breakdown.csv"
        ] = output_path

    existing_prototype_columns = [
        column
        for column in prototype_columns
        if column in prediction_df.columns
    ]

    if existing_prototype_columns:
        output_path = (
            out_dir
            / "prototype_distances.csv"
        )

        prediction_df[
            identifier_columns
            + existing_prototype_columns
        ].to_csv(
            output_path,
            index=False,
        )

        saved_paths[
            "prototype_distances.csv"
        ] = output_path

    return saved_paths


'''

    text = (
        text[:marker_index]
        + helper_code
        + text[marker_index:]
    )

    print("Added save_diagnostic_csvs().")
else:
    print("save_diagnostic_csvs() already exists.")

# ============================================================
# Call the diagnostic saver during final evaluation
# ============================================================

save_anchor = (
    '    summary_df.to_csv('
    'Path(args.out_dir) / '
    '"experiment_summary.csv", '
    'index=False)\n'
)

save_call = '''    diagnostic_paths = save_diagnostic_csvs(
        test_frame_df,
        args.out_dir,
    )

'''

if "diagnostic_paths = save_diagnostic_csvs(" not in text:
    if save_anchor not in text:
        raise RuntimeError(
            "Could not find experiment_summary.csv save point."
        )

    text = text.replace(
        save_anchor,
        save_anchor + save_call,
        1,
    )

    print("Connected diagnostic CSV saving to final evaluation.")
else:
    print("Diagnostic saver already connected.")

# ============================================================
# Save and compile
# ============================================================

script_path.write_text(text)

py_compile.compile(
    str(script_path),
    doraise=True,
)

print("\nPASS: Python compilation succeeded.")

# ============================================================
# Import and test CSV helper
# ============================================================

module_name = (
    "agrfn_logging_test_"
    + timestamp
)

spec = importlib.util.spec_from_file_location(
    module_name,
    script_path,
)

module = importlib.util.module_from_spec(spec)

if spec.loader is None:
    raise RuntimeError(
        "Could not create module loader."
    )

spec.loader.exec_module(module)

mock_predictions = pd.DataFrame({
    "split": ["test", "test"],
    "image_path": ["a.png", "b.png"],
    "group_id": ["a", "b"],
    "sample_unit_id": ["a", "b"],
    "label": [0, 1],
    "method": ["real", "ddim"],
    "domain": ["ff", "unknown"],
    "fake_prob": [0.2, 0.8],
    "uncertainty": [0.3, 0.2],
    "classification_entropy": [0.4, 0.3],
    "mean_branch_entropy": [0.5, 0.4],
    "mean_branch_disagreement": [0.1, 0.2],
    "max_branch_disagreement": [0.2, 0.3],
    "routing_entropy": [0.9, 0.8],
    "prob_rgb": [0.3, 0.7],
    "prob_jpeg": [0.2, 0.8],
    "entropy_rgb": [0.5, 0.4],
    "entropy_jpeg": [0.4, 0.3],
    "route_rgb": [0.5, 0.4],
    "route_jpeg": [0.5, 0.6],
    "disagree_rgb__jpeg": [0.1, 0.1],
})

with tempfile.TemporaryDirectory() as temp_dir:
    saved = module.save_diagnostic_csvs(
        mock_predictions,
        temp_dir,
    )

    expected_files = {
        "branch_predictions.csv",
        "branch_uncertainties.csv",
        "routing_weights.csv",
        "disagreement_matrices.csv",
        "uncertainty_breakdown.csv",
    }

    missing = (
        expected_files - set(saved)
    )

    if missing:
        raise RuntimeError(
            "Diagnostic helper did not create: "
            + ", ".join(sorted(missing))
        )

print("\n" + "=" * 80)
print("DIAGNOSTIC LOGGING PATCH PASSED")
print("=" * 80)

print("\nThe next A-GRFN run will save:")

for filename in [
    "branch_predictions.csv",
    "branch_uncertainties.csv",
    "routing_weights.csv",
    "disagreement_matrices.csv",
    "uncertainty_breakdown.csv",
]:
    print(" ", filename)

print(
    "\nDF40 method, domain, top-level folder, "
    "and source-layout metadata will also be preserved."
)

print(
    "\nNo model architecture or training behavior was changed."
)

Backup created:
/home/jovyan/Deepfake_Low Resource_Low Resolution/agrfn_adaptive_routing_df40_before_diagnostic_logging_20260730_215627.py

Added DF40 pass-through metadata columns.
Updated GRFNDataset metadata pass-through.
Updated predict_loader diagnostic logging.
Added save_diagnostic_csvs().
Connected diagnostic CSV saving to final evaluation.

PASS: Python compilation succeeded.

DIAGNOSTIC LOGGING PATCH PASSED

The next A-GRFN run will save:
  branch_predictions.csv
  branch_uncertainties.csv
  routing_weights.csv
  disagreement_matrices.csv
  uncertainty_breakdown.csv

DF40 method, domain, top-level folder, and source-layout metadata will also be preserved.

No model architecture or training behavior was changed.


In [35]:
from pathlib import Path
from datetime import datetime
import subprocess
import sys
import pandas as pd
import numpy as np

# ============================================================
# Locate project files
# ============================================================

project_root = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution"
)

script_path = (
    project_root
    / "agrfn_adaptive_routing_df40.py"
)

manifest_path = (
    project_root
    / "outputs_df40"
    / "df40_manifest_all_clean.csv"
)

if not script_path.exists():
    raise FileNotFoundError(script_path)

if not manifest_path.exists():
    raise FileNotFoundError(manifest_path)

# ============================================================
# Find the latest successful scalar smoke checkpoint
# ============================================================

candidate_dirs = sorted(
    [
        path
        for path in project_root.glob(
            "outputs_agrfn_scalar_df40_smoke_*"
        )
        if path.is_dir()
        and (path / "best_checkpoint.pt").exists()
    ],
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)

if not candidate_dirs:
    raise RuntimeError(
        "No successful scalar smoke-test checkpoint was found."
    )

source_dir = candidate_dirs[0]
checkpoint_path = source_dir / "best_checkpoint.pt"

print("Using checkpoint:")
print(checkpoint_path)

# ============================================================
# Create a fresh evaluation output directory
# ============================================================

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

out_dir = (
    project_root
    / f"outputs_agrfn_scalar_logging_check_{timestamp}"
)

# ============================================================
# Run evaluation only
# ============================================================

command = [
    sys.executable,
    str(script_path),

    "--metadata_csv",
    str(manifest_path),

    "--out_dir",
    str(out_dir),

    "--dataset_name",
    "DF40",

    "--fusion_mode",
    "adaptive_scalar",

    "--streams",
    "rgb,fingerprint,jpeg,freq",

    "--feat_dim",
    "128",

    "--controller_dim",
    "128",

    "--routing_temperature",
    "1.0",

    "--frames_per_video",
    "2",

    "--limit_train_units",
    "40",

    "--limit_val_units",
    "20",

    "--limit_test_units",
    "20",

    "--balanced_unit_limits",

    "--image_size",
    "224",

    "--batch_size",
    "4",

    "--num_workers",
    "0",

    "--decision_threshold",
    "0.5",

    "--score_direction",
    "normal",

    "--resume",
    str(checkpoint_path),

    "--eval_only",
]

print("\n" + "=" * 80)
print("RUNNING DIAGNOSTIC LOGGING CHECK")
print("=" * 80)

print("\nOutput directory:")
print(out_dir)

result = subprocess.run(
    command,
    cwd=str(project_root),
)

if result.returncode != 0:
    raise RuntimeError(
        "Diagnostic evaluation failed with return code "
        f"{result.returncode}."
    )

# ============================================================
# Validate expected outputs
# ============================================================

required_files = [
    "test_frame_predictions.csv",
    "test_video_predictions.csv",
    "test_metrics.csv",
    "test_inference_time.csv",
    "experiment_summary.csv",
    "branch_predictions.csv",
    "branch_uncertainties.csv",
    "routing_weights.csv",
    "disagreement_matrices.csv",
    "uncertainty_breakdown.csv",
]

missing_files = [
    name
    for name in required_files
    if not (out_dir / name).exists()
]

if missing_files:
    raise RuntimeError(
        "The following files were not created:\n"
        + "\n".join(missing_files)
    )

predictions = pd.read_csv(
    out_dir / "test_frame_predictions.csv"
)

routing_df = pd.read_csv(
    out_dir / "routing_weights.csv"
)

branch_df = pd.read_csv(
    out_dir / "branch_predictions.csv"
)

uncertainty_df = pd.read_csv(
    out_dir / "uncertainty_breakdown.csv"
)

disagreement_df = pd.read_csv(
    out_dir / "disagreement_matrices.csv"
)

# ============================================================
# Validate diagnostic columns
# ============================================================

expected_route_columns = [
    "route_rgb",
    "route_jpeg",
    "route_freq",
]

missing_route_columns = [
    column
    for column in expected_route_columns
    if column not in predictions.columns
]

if missing_route_columns:
    raise RuntimeError(
        "Missing route columns: "
        + ", ".join(missing_route_columns)
    )

expected_branch_columns = [
    "prob_rgb",
    "prob_fingerprint",
    "prob_jpeg",
    "prob_freq",
]

missing_branch_columns = [
    column
    for column in expected_branch_columns
    if column not in predictions.columns
]

if missing_branch_columns:
    raise RuntimeError(
        "Missing branch columns: "
        + ", ".join(missing_branch_columns)
    )

expected_uncertainty_columns = [
    "classification_entropy",
    "mean_branch_entropy",
    "mean_branch_disagreement",
    "max_branch_disagreement",
    "routing_entropy",
    "uncertainty",
]

missing_uncertainty_columns = [
    column
    for column in expected_uncertainty_columns
    if column not in predictions.columns
]

if missing_uncertainty_columns:
    raise RuntimeError(
        "Missing uncertainty columns: "
        + ", ".join(missing_uncertainty_columns)
    )

# ============================================================
# Validate routing values
# ============================================================

route_values = predictions[
    expected_route_columns
].to_numpy(dtype=float)

if not np.isfinite(route_values).all():
    raise RuntimeError(
        "Routing output contains NaN or infinity."
    )

if (route_values < 0).any():
    raise RuntimeError(
        "Routing output contains negative values."
    )

route_sums = route_values.sum(axis=1)

maximum_route_sum_error = float(
    np.abs(route_sums - 1.0).max()
)

if maximum_route_sum_error > 1e-5:
    raise RuntimeError(
        "Routing weights do not sum to one. "
        f"Maximum error: {maximum_route_sum_error}"
    )

# Fingerprint is the controller, not an evidence route.
if "route_fingerprint" in predictions.columns:
    raise RuntimeError(
        "Fingerprint incorrectly appears as an evidence route."
    )

if "rel_fingerprint" in predictions.columns:
    fingerprint_reliability_max = float(
        predictions["rel_fingerprint"]
        .abs()
        .max()
    )

    if fingerprint_reliability_max > 1e-7:
        raise RuntimeError(
            "rel_fingerprint should remain zero because "
            "fingerprint is the controller."
        )

# ============================================================
# Validate DF40 metadata pass-through
# ============================================================

metadata_columns = [
    "method",
    "domain",
    "top_level_folder",
    "source_layout",
    "relative_path",
]

available_metadata = [
    column
    for column in metadata_columns
    if column in predictions.columns
]

if not available_metadata:
    raise RuntimeError(
        "No DF40 metadata columns were preserved."
    )

# ============================================================
# Final report
# ============================================================

print("\n" + "=" * 80)
print("DIAGNOSTIC LOGGING CHECK PASSED")
print("=" * 80)

print("\nOutput directory:")
print(out_dir)

print("\nSaved diagnostic files:")

for name in required_files:
    print(" ", name)

print("\nPrediction rows:")
print(len(predictions))

print("\nRoute columns:")
print(expected_route_columns)

print("\nMaximum routing-sum error:")
print(maximum_route_sum_error)

print("\nMean routing weights:")
display(
    predictions[
        expected_route_columns
    ]
    .mean()
    .to_frame("mean_weight")
)

print("\nRouting standard deviations:")
display(
    predictions[
        expected_route_columns
    ]
    .std()
    .to_frame("standard_deviation")
)

print("\nAvailable DF40 metadata:")
print(available_metadata)

if "method" in predictions.columns:
    print("\nMean routing weights by method:")

    display(
        predictions.groupby("method")[
            expected_route_columns
        ]
        .mean()
    )

print("\nDiagnostic table sizes:")
print("Branch predictions:", len(branch_df))
print("Routing weights:", len(routing_df))
print("Uncertainty breakdown:", len(uncertainty_df))
print("Disagreement matrices:", len(disagreement_df))

print(
    "\nPASS: real DF40 predictions now include branch, "
    "routing, disagreement, uncertainty, and dataset metadata."
)

Using checkpoint:
/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_scalar_df40_smoke_20260730_215319/best_checkpoint.pt

RUNNING DIAGNOSTIC LOGGING CHECK

Output directory:
/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_scalar_logging_check_20260730_215740

GRFN-CSF GRFN-only training and evaluation
Device: cuda
Output directory: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_scalar_logging_check_20260730_215740
Metadata CSV: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40/df40_manifest_all_clean.csv
Streams: rgb,fingerprint,jpeg,freq

Metadata loaded
Metadata CSV: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40/df40_manifest_all_clean.csv
Path column: frame_path
Label column: label
Split column: split
Group column: group_id
Usable frames: 292907
train: frames=235640, units=7843, real=48132, fake=187508
val: frames=28551, units=980, real=5997, fake=22554
test: frames=28716, units=981, real=5999, fake=22717
S

Sample train: 100%|██████████| 40/40 [00:00<00:00, 65.01unit/s]


Sample train: frames=78, units=40, real=40, fake=38
Sample val: frames=39, units=20, real=20, fake=19


Sample test: 100%|██████████| 20/20 [00:00<00:00, 353.34unit/s]


Sample test: frames=39, units=20, real=20, fake=19

Balancing strategy
Train class counts [real, fake]: [40, 38]
Train imbalance ratio: 1.053
Use class-weighted loss: False
Use weighted sampler: False

Building dataloaders
Train batches: 19
Val batches: 10
Test batches: 10

Building model
Model mode: grfn
Loss mode: grfn
Active streams: rgb, fingerprint, jpeg, freq
Total params: 1,681,741
Trainable params: 1,681,741
Loading checkpoint: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_scalar_df40_smoke_20260730_215319/best_checkpoint.pt
Resumed from epoch: 1 best metric: 0.481578947368421

Final test evaluation


Predict test: 100%|██████████| 10/10 [00:01<00:00,  8.66it/s]


... (3 outputs left out)

,standard_deviation
route_rgb,0.000549
route_jpeg,0.000611
route_freq,0.000276



Available DF40 metadata:
['method', 'domain', 'top_level_folder', 'source_layout', 'relative_path']

Mean routing weights by method:


,route_rgb,route_jpeg,route_freq
method,,,
CollabDiff,0.338984,0.332922,0.328094
ddim,0.338887,0.332749,0.328364
fomm,0.339437,0.332368,0.328195
inswap,0.339760,0.332304,0.327936
real,0.340044,0.331872,0.328084
sadtalker,0.339482,0.332786,0.327733
sd2.1,0.339045,0.332814,0.328142
simswap,0.339177,0.332319,0.328504
wav2lip,0.339466,0.331939,0.328595



Diagnostic table sizes:
Branch predictions: 39
Routing weights: 39
Uncertainty breakdown: 39
Disagreement matrices: 39

PASS: real DF40 predictions now include branch, routing, disagreement, uncertainty, and dataset metadata.


In [36]:
from pathlib import Path
from datetime import datetime
from types import SimpleNamespace
import ast
import shutil
import py_compile
import importlib.util

import torch
import torch.nn.functional as F


# ============================================================
# Paths and backup
# ============================================================

project_root = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution"
)

script_path = (
    project_root
    / "agrfn_adaptive_routing_df40.py"
)

if not script_path.exists():
    raise FileNotFoundError(script_path)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

backup_path = script_path.with_name(
    f"{script_path.stem}_before_film_{timestamp}.py"
)

shutil.copy2(script_path, backup_path)

print("Backup created:")
print(backup_path)

text = script_path.read_text()


# ============================================================
# Add FingerprintFiLMRouter
# ============================================================

tree = ast.parse(text)

top_level_names = {
    node.name
    for node in tree.body
    if isinstance(
        node,
        (ast.ClassDef, ast.FunctionDef),
    )
}

if "FingerprintFiLMRouter" not in top_level_names:
    scalar_router_node = next(
        (
            node
            for node in tree.body
            if isinstance(node, ast.ClassDef)
            and node.name == "ScalarGateRouter"
        ),
        None,
    )

    if scalar_router_node is None:
        raise RuntimeError(
            "Could not find ScalarGateRouter."
        )

    film_router_code = r'''

class FingerprintFiLMRouter(nn.Module):
    """Generate fingerprint-conditioned gates and FiLM parameters."""

    def __init__(
        self,
        feat_dim=128,
        num_branches=3,
        film_scale=0.10,
        temperature=1.0,
    ):
        super().__init__()

        if feat_dim <= 0:
            raise ValueError(
                "feat_dim must be positive."
            )

        if num_branches <= 0:
            raise ValueError(
                "num_branches must be positive."
            )

        if film_scale < 0:
            raise ValueError(
                "film_scale cannot be negative."
            )

        if temperature <= 0:
            raise ValueError(
                "temperature must be positive."
            )

        self.feat_dim = int(feat_dim)
        self.num_branches = int(num_branches)
        self.film_scale = float(film_scale)
        self.temperature = float(temperature)

        self.shared = nn.Sequential(
            nn.Linear(
                self.feat_dim,
                self.feat_dim,
            ),
            nn.LayerNorm(self.feat_dim),
            nn.GELU(),
        )

        self.gate_head = nn.Linear(
            self.feat_dim,
            self.num_branches,
        )

        self.gamma_head = nn.Linear(
            self.feat_dim,
            self.num_branches
            * self.feat_dim,
        )

        self.beta_head = nn.Linear(
            self.feat_dim,
            self.num_branches
            * self.feat_dim,
        )

        # Begin with uniform gates and identity FiLM.
        nn.init.zeros_(self.gate_head.weight)
        nn.init.zeros_(self.gate_head.bias)

        nn.init.zeros_(self.gamma_head.weight)
        nn.init.zeros_(self.gamma_head.bias)

        nn.init.zeros_(self.beta_head.weight)
        nn.init.zeros_(self.beta_head.bias)

    def forward(self, controller):
        if controller.ndim != 2:
            raise ValueError(
                "controller must have shape "
                "[batch, feat_dim]."
            )

        if controller.shape[1] != self.feat_dim:
            raise ValueError(
                "Controller feature dimension mismatch. "
                f"Expected {self.feat_dim}, "
                f"received {controller.shape[1]}."
            )

        hidden = self.shared(controller)

        gate_logits = self.gate_head(hidden)

        routing_weights = F.softmax(
            gate_logits / self.temperature,
            dim=1,
        )

        batch_size = controller.shape[0]

        gamma_raw = self.gamma_head(hidden).view(
            batch_size,
            self.num_branches,
            self.feat_dim,
        )

        beta_raw = self.beta_head(hidden).view(
            batch_size,
            self.num_branches,
            self.feat_dim,
        )

        # Identity-centered and bounded modulation.
        gamma = (
            1.0
            + self.film_scale
            * torch.tanh(gamma_raw)
        )

        beta = (
            self.film_scale
            * torch.tanh(beta_raw)
        )

        return {
            "gate_logits": gate_logits,
            "routing_weights": routing_weights,
            "gamma": gamma,
            "beta": beta,
        }

'''

    lines = text.splitlines(keepends=True)

    insertion_index = scalar_router_node.end_lineno

    lines = (
        lines[:insertion_index]
        + film_router_code.splitlines(
            keepends=True
        )
        + lines[insertion_index:]
    )

    text = "".join(lines)

    print("\nInserted FingerprintFiLMRouter.")
else:
    print(
        "\nFingerprintFiLMRouter already exists."
    )


# ============================================================
# Replace AGRFN with scalar and FiLM support
# ============================================================

tree = ast.parse(text)

agrfn_node = next(
    (
        node
        for node in tree.body
        if isinstance(node, ast.ClassDef)
        and node.name == "AGRFN"
    ),
    None,
)

if agrfn_node is None:
    raise RuntimeError(
        "Could not find class AGRFN."
    )

new_agrfn_class = r'''class AGRFN(nn.Module):
    """Adaptive Generalizable Residual Fingerprint Network.

    The learned fingerprint embedding controls forensic evidence
    routing before the final classification decision.

    Supported modes:
        adaptive_scalar
        adaptive_film
    """

    VALID_STREAMS = [
        "rgb",
        "fingerprint",
        "rec",
        "den",
        "jpeg",
        "freq",
        "noise",
    ]

    VALID_FUSION_MODES = {
        "adaptive_scalar",
        "adaptive_film",
    }

    def __init__(
        self,
        feat_dim=128,
        controller_dim=128,
        stream_names=None,
        num_classes=2,
        fusion_mode="adaptive_scalar",
        routing_temperature=1.0,
        film_scale=0.10,
        dropout=0.20,
    ):
        super().__init__()

        if stream_names is None:
            stream_names = [
                "rgb",
                "fingerprint",
                "jpeg",
                "freq",
            ]

        stream_names = [
            name.strip()
            for name in stream_names
            if name.strip()
        ]

        unknown = [
            name
            for name in stream_names
            if name not in self.VALID_STREAMS
        ]

        if unknown:
            raise ValueError(
                f"Unknown streams: {unknown}. "
                f"Valid streams: {self.VALID_STREAMS}"
            )

        if fusion_mode not in self.VALID_FUSION_MODES:
            raise ValueError(
                f"Unsupported AGRFN fusion mode: "
                f"{fusion_mode}"
            )

        if "rgb" not in stream_names:
            raise ValueError(
                "A-GRFN requires the RGB stream."
            )

        if "fingerprint" not in stream_names:
            raise ValueError(
                "A-GRFN requires the fingerprint controller."
            )

        self.stream_names = stream_names

        self.evidence_streams = [
            name
            for name in self.stream_names
            if name != "fingerprint"
        ]

        if not self.evidence_streams:
            raise ValueError(
                "A-GRFN requires at least one evidence stream."
            )

        self.num_streams = len(
            self.stream_names
        )

        self.num_evidence_streams = len(
            self.evidence_streams
        )

        self.feat_dim = int(feat_dim)
        self.controller_dim = int(controller_dim)
        self.num_classes = int(num_classes)
        self.fusion_mode = str(fusion_mode)

        # Same stream encoders as the late-fusion baseline.
        self.rgb_encoder = SmallStreamEncoder(
            3,
            self.feat_dim,
        )

        self.rec_encoder = SmallStreamEncoder(
            3,
            self.feat_dim,
        )

        self.den_encoder = SmallStreamEncoder(
            3,
            self.feat_dim,
        )

        self.jpeg_encoder = SmallStreamEncoder(
            3,
            self.feat_dim,
        )

        self.freq_encoder = SmallStreamEncoder(
            1,
            self.feat_dim,
        )

        self.noise_encoder = SmallStreamEncoder(
            1,
            self.feat_dim,
        )

        self.fingerprint_head = nn.Sequential(
            nn.Linear(
                self.feat_dim,
                self.feat_dim,
            ),
            nn.LayerNorm(self.feat_dim),
            nn.ReLU(inplace=True),
            nn.Linear(
                self.feat_dim,
                self.feat_dim,
            ),
        )

        self.fingerprint_controller = (
            FingerprintController(
                feat_dim=self.feat_dim,
                controller_dim=self.controller_dim,
                dropout=0.10,
            )
        )

        # Instantiate only the router used by this mode.
        # This preserves scalar-checkpoint compatibility.
        if self.fusion_mode == "adaptive_scalar":
            self.scalar_router = ScalarGateRouter(
                feat_dim=self.feat_dim,
                num_branches=(
                    self.num_evidence_streams
                ),
                temperature=routing_temperature,
            )

        elif self.fusion_mode == "adaptive_film":
            self.film_router = FingerprintFiLMRouter(
                feat_dim=self.feat_dim,
                num_branches=(
                    self.num_evidence_streams
                ),
                film_scale=film_scale,
                temperature=routing_temperature,
            )

        # Auxiliary heads only.
        self.stream_heads = nn.ModuleDict({
            name: nn.Linear(
                self.feat_dim,
                self.num_classes,
            )
            for name in self.stream_names
        })

        self.joint_projector = nn.Sequential(
            nn.Linear(
                self.feat_dim * 2,
                self.feat_dim,
            ),
            nn.LayerNorm(self.feat_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(
                self.feat_dim,
                self.feat_dim,
            ),
            nn.LayerNorm(self.feat_dim),
            nn.GELU(),
        )

        self.final_head = nn.Linear(
            self.feat_dim,
            self.num_classes,
        )

    def encode_streams(self, view):
        features = {}

        rgb_feature = self.rgb_encoder(
            view["rgb"]
        )

        if "rgb" in self.stream_names:
            features["rgb"] = rgb_feature

        fingerprint_raw = (
            self.fingerprint_head(
                rgb_feature
            )
        )

        features["fingerprint"] = (
            F.normalize(
                fingerprint_raw,
                dim=1,
            )
        )

        if "rec" in self.stream_names:
            features["rec"] = (
                self.rec_encoder(
                    view["rec"]
                )
            )

        if "den" in self.stream_names:
            features["den"] = (
                self.den_encoder(
                    view["den"]
                )
            )

        if "jpeg" in self.stream_names:
            features["jpeg"] = (
                self.jpeg_encoder(
                    view["jpeg"]
                )
            )

        if "freq" in self.stream_names:
            features["freq"] = (
                self.freq_encoder(
                    view["freq"]
                )
            )

        if "noise" in self.stream_names:
            features["noise"] = (
                self.noise_encoder(
                    view["noise"]
                )
            )

        return features

    def forward(self, view):
        features = self.encode_streams(view)

        fingerprint = features[
            "fingerprint"
        ]

        controller = (
            self.fingerprint_controller(
                fingerprint
            )
        )

        # Auxiliary branch predictions.
        stream_logits = torch.stack(
            [
                self.stream_heads[name](
                    features[name]
                )
                for name in self.stream_names
            ],
            dim=1,
        )

        stream_probs = F.softmax(
            stream_logits,
            dim=-1,
        )

        stream_entropy = (
            normalized_entropy(
                stream_probs
            )
        )

        disagreement = (
            compute_branch_disagreement(
                stream_probs
            )
        )

        evidence_features = torch.stack(
            [
                features[name]
                for name in self.evidence_streams
            ],
            dim=1,
        )

        if self.fusion_mode == "adaptive_scalar":
            routing_output = (
                self.scalar_router(
                    controller
                )
            )

            routing_weights = (
                routing_output[
                    "routing_weights"
                ]
            )

            gamma = torch.ones_like(
                evidence_features
            )

            beta = torch.zeros_like(
                evidence_features
            )

        elif self.fusion_mode == "adaptive_film":
            routing_output = (
                self.film_router(
                    controller
                )
            )

            routing_weights = (
                routing_output[
                    "routing_weights"
                ]
            )

            gamma = routing_output["gamma"]
            beta = routing_output["beta"]

        else:
            raise RuntimeError(
                f"Unhandled fusion mode: "
                f"{self.fusion_mode}"
            )

        gate_logits = routing_output[
            "gate_logits"
        ]

        modulated_features = (
            routing_weights.unsqueeze(-1)
            * (
                gamma * evidence_features
                + beta
            )
        )

        routed_feature = (
            modulated_features.sum(dim=1)
        )

        joint_embedding = (
            self.joint_projector(
                torch.cat(
                    [
                        routed_feature,
                        controller,
                    ],
                    dim=1,
                )
            )
        )

        # No branch logits enter this classifier.
        logits = self.final_head(
            joint_embedding
        )

        probs = F.softmax(
            logits,
            dim=1,
        )

        classification_entropy = (
            normalized_entropy(probs)
        )

        mean_disagreement = (
            disagreement.mean(
                dim=(1, 2)
            )
        )

        if self.num_evidence_streams > 1:
            routing_entropy = (
                normalized_entropy(
                    routing_weights
                )
            )
        else:
            routing_entropy = (
                torch.zeros_like(
                    classification_entropy
                )
            )

        uncertainty = torch.clamp(
            0.50 * classification_entropy
            + 0.30 * mean_disagreement
            + 0.20 * routing_entropy,
            0.0,
            1.0,
        )

        reliability = torch.zeros(
            logits.shape[0],
            self.num_streams,
            dtype=logits.dtype,
            device=logits.device,
        )

        for evidence_index, stream_name in enumerate(
            self.evidence_streams
        ):
            stream_index = (
                self.stream_names.index(
                    stream_name
                )
            )

            reliability[:, stream_index] = (
                routing_weights[
                    :,
                    evidence_index,
                ]
            )

        gamma_abs_delta = (
            torch.abs(gamma - 1.0)
            .mean(dim=-1)
        )

        beta_abs_mean = (
            torch.abs(beta)
            .mean(dim=-1)
        )

        return {
            "logits": logits,
            "probs": probs,
            "fake_prob": probs[:, 1],
            "uncertainty": uncertainty,

            "stream_logits": stream_logits,
            "stream_probs": stream_probs,
            "stream_entropy": stream_entropy,
            "disagreement": disagreement,

            "fingerprint": fingerprint,
            "controller": controller,
            "joint_embedding": joint_embedding,

            "routing_weights": routing_weights,
            "gate_weights": routing_weights,
            "gate_logits": gate_logits,
            "routing_entropy": routing_entropy,

            "film_gamma": gamma,
            "film_beta": beta,
            "film_gamma_abs_delta": (
                gamma_abs_delta
            ),
            "film_beta_abs_mean": (
                beta_abs_mean
            ),

            "reliability": reliability,
            "evidence_streams": (
                self.evidence_streams
            ),
            "fusion_mode": self.fusion_mode,
        }
'''

lines = text.splitlines(keepends=True)

lines = (
    lines[:agrfn_node.lineno - 1]
    + new_agrfn_class.splitlines(
        keepends=True
    )
    + ["\n\n"]
    + lines[agrfn_node.end_lineno:]
)

text = "".join(lines)

print("Updated AGRFN with FiLM support.")


# ============================================================
# Replace build_model()
# ============================================================

tree = ast.parse(text)

build_model_node = next(
    (
        node
        for node in tree.body
        if isinstance(node, ast.FunctionDef)
        and node.name == "build_model"
    ),
    None,
)

if build_model_node is None:
    raise RuntimeError(
        "Could not find build_model()."
    )

new_build_model = r'''def build_model(args, stream_names):
    """Build late, average, scalar, or FiLM models."""

    if args.fusion_mode == "late":
        return GRFNCSF(
            feat_dim=args.feat_dim,
            stream_names=stream_names,
            stream_logit_residual_weight=(
                args.stream_logit_residual_weight
            ),
        )

    if args.fusion_mode == "average":
        return SimpleStreamModel(
            feat_dim=args.feat_dim,
            stream_names=stream_names,
        )

    if args.fusion_mode in {
        "adaptive_scalar",
        "adaptive_film",
    }:
        return AGRFN(
            feat_dim=args.feat_dim,
            controller_dim=(
                args.controller_dim
            ),
            stream_names=stream_names,
            fusion_mode=args.fusion_mode,
            routing_temperature=(
                args.routing_temperature
            ),
            film_scale=getattr(
                args,
                "film_scale",
                0.10,
            ),
        )

    raise NotImplementedError(
        f"Fusion mode '{args.fusion_mode}' "
        "has not been implemented yet."
    )
'''

lines = text.splitlines(keepends=True)

lines = (
    lines[:build_model_node.lineno - 1]
    + new_build_model.splitlines(
        keepends=True
    )
    + ["\n\n"]
    + lines[build_model_node.end_lineno:]
)

text = "".join(lines)

print("Updated build_model().")


# ============================================================
# Update CLI guard to allow adaptive_film
# ============================================================

tree = ast.parse(text)

parse_args_node = next(
    (
        node
        for node in tree.body
        if isinstance(node, ast.FunctionDef)
        and node.name == "parse_args"
    ),
    None,
)

if parse_args_node is None:
    raise RuntimeError(
        "Could not find parse_args()."
    )

guard_node = None

for node in ast.walk(parse_args_node):
    if not isinstance(node, ast.If):
        continue

    source = ast.get_source_segment(
        text,
        node,
    )

    if (
        source
        and "adaptive_film" in source
        and "adaptive_transformer" in source
        and "NotImplementedError" in source
    ):
        guard_node = node
        break

if guard_node is not None:
    replacement = '''    if args.fusion_mode == "adaptive_transformer":
        raise NotImplementedError(
            "adaptive_transformer will be enabled "
            "in the next architecture stage."
        )

'''

    lines = text.splitlines(keepends=True)

    lines = (
        lines[:guard_node.lineno - 1]
        + replacement.splitlines(
            keepends=True
        )
        + lines[guard_node.end_lineno:]
    )

    text = "".join(lines)

    print(
        "Enabled adaptive_film in parse_args()."
    )
else:
    print(
        "FiLM CLI guard was already updated "
        "or not present."
    )


# ============================================================
# Save and compile
# ============================================================

script_path.write_text(text)

py_compile.compile(
    str(script_path),
    doraise=True,
)

print("\nPASS: Python compilation succeeded.")


# ============================================================
# Import updated module
# ============================================================

module_name = (
    "agrfn_film_test_"
    + timestamp
)

spec = importlib.util.spec_from_file_location(
    module_name,
    script_path,
)

module = importlib.util.module_from_spec(
    spec
)

if spec.loader is None:
    raise RuntimeError(
        "Could not create module loader."
    )

spec.loader.exec_module(module)


# ============================================================
# Dummy inputs
# ============================================================

batch_size = 2
image_size = 64

dummy_view = {
    "rgb": torch.randn(
        batch_size,
        3,
        image_size,
        image_size,
    ),
    "rec": torch.randn(
        batch_size,
        3,
        image_size,
        image_size,
    ),
    "den": torch.randn(
        batch_size,
        3,
        image_size,
        image_size,
    ),
    "jpeg": torch.randn(
        batch_size,
        3,
        image_size,
        image_size,
    ),
    "freq": torch.randn(
        batch_size,
        1,
        image_size,
        image_size,
    ),
    "noise": torch.randn(
        batch_size,
        1,
        image_size,
        image_size,
    ),
}

stream_names = [
    "rgb",
    "fingerprint",
    "jpeg",
    "freq",
]

common_args = {
    "feat_dim": 128,
    "controller_dim": 128,
    "routing_temperature": 1.0,
    "film_scale": 0.10,
    "stream_logit_residual_weight": 0.25,
}


# ============================================================
# Build and test FiLM model
# ============================================================

film_args = SimpleNamespace(
    fusion_mode="adaptive_film",
    **common_args,
)

film_model = module.build_model(
    film_args,
    stream_names,
)

film_model.train()

film_output = film_model(
    dummy_view
)

expected_shapes = {
    "logits": (batch_size, 2),
    "routing_weights": (
        batch_size,
        3,
    ),
    "film_gamma": (
        batch_size,
        3,
        128,
    ),
    "film_beta": (
        batch_size,
        3,
        128,
    ),
    "joint_embedding": (
        batch_size,
        128,
    ),
}

for key, expected_shape in expected_shapes.items():
    actual_shape = tuple(
        film_output[key].shape
    )

    if actual_shape != expected_shape:
        raise RuntimeError(
            f"{key} shape mismatch. "
            f"Expected {expected_shape}, "
            f"received {actual_shape}."
        )

route_sums = film_output[
    "routing_weights"
].sum(dim=1)

if not torch.allclose(
    route_sums,
    torch.ones_like(route_sums),
    atol=1e-6,
):
    raise RuntimeError(
        "FiLM routing weights do not sum to one."
    )

# FiLM must begin as an identity transformation.
if not torch.allclose(
    film_output["film_gamma"],
    torch.ones_like(
        film_output["film_gamma"]
    ),
    atol=1e-7,
):
    raise RuntimeError(
        "FiLM gamma did not initialize to one."
    )

if not torch.allclose(
    film_output["film_beta"],
    torch.zeros_like(
        film_output["film_beta"]
    ),
    atol=1e-7,
):
    raise RuntimeError(
        "FiLM beta did not initialize to zero."
    )


# ============================================================
# Gradient and branch-independence test
# ============================================================

film_model.zero_grad(
    set_to_none=True
)

film_output = film_model(
    dummy_view
)

labels = torch.tensor(
    [0, 1],
    dtype=torch.long,
)

loss = F.cross_entropy(
    film_output["logits"],
    labels,
)

loss.backward()

branch_head_gradients = {
    name: any(
        parameter.grad is not None
        for parameter in head.parameters()
    )
    for name, head
    in film_model.stream_heads.items()
}

if any(
    branch_head_gradients.values()
):
    raise RuntimeError(
        "Final FiLM logits depend on auxiliary "
        f"branch heads: {branch_head_gradients}"
    )

film_router_gradients = {
    "gate_head": (
        film_model
        .film_router
        .gate_head
        .weight
        .grad
    ),
    "gamma_head": (
        film_model
        .film_router
        .gamma_head
        .weight
        .grad
    ),
    "beta_head": (
        film_model
        .film_router
        .beta_head
        .weight
        .grad
    ),
}

for name, gradient in film_router_gradients.items():
    if gradient is None:
        raise RuntimeError(
            f"{name} received no gradient."
        )

    if not torch.isfinite(
        gradient
    ).all():
        raise RuntimeError(
            f"{name} gradient contains "
            "NaN or infinity."
        )

    if float(
        gradient.abs().sum()
    ) == 0.0:
        raise RuntimeError(
            f"{name} gradient is entirely zero."
        )


# ============================================================
# Confirm scalar mode still works
# ============================================================

scalar_args = SimpleNamespace(
    fusion_mode="adaptive_scalar",
    **common_args,
)

scalar_model = module.build_model(
    scalar_args,
    stream_names,
)

scalar_model.eval()

with torch.no_grad():
    scalar_output = scalar_model(
        dummy_view
    )

if tuple(
    scalar_output[
        "routing_weights"
    ].shape
) != (
    batch_size,
    3,
):
    raise RuntimeError(
        "Scalar mode was broken by the FiLM patch."
    )

if hasattr(
    scalar_model,
    "film_router",
):
    raise RuntimeError(
        "Scalar model should not instantiate "
        "the FiLM router."
    )


# ============================================================
# Final report
# ============================================================

print("\n" + "=" * 80)
print("A-GRFN FILM MODULE TEST PASSED")
print("=" * 80)

print("\nFiLM model class:")
print(type(film_model).__name__)

print("\nFusion mode:")
print(film_model.fusion_mode)

print("\nEvidence streams:")
print(film_model.evidence_streams)

print("\nInitial routing weights:")
print(
    film_output[
        "routing_weights"
    ]
    .detach()
    .cpu()
)

print("\nRouting sums:")
print(
    route_sums
    .detach()
    .cpu()
)

print("\nInitial gamma deviation from one:")
print(
    float(
        torch.abs(
            film_output["film_gamma"]
            - 1.0
        ).max()
    )
)

print("\nInitial beta magnitude:")
print(
    float(
        torch.abs(
            film_output["film_beta"]
        ).max()
    )
)

print("\nAuxiliary branch-head gradients:")
print(branch_head_gradients)

print("\nFiLM-router gradient magnitudes:")

for name, gradient in film_router_gradients.items():
    print(
        f"{name:12s}: "
        f"{float(gradient.abs().sum()):.8f}"
    )

print("\nScalar mode remains available:")
print(type(scalar_model).__name__)

print(
    "\nPASS: fingerprint-conditioned FiLM "
    "modulation is active before classification."
)

print(
    "PASS: final logits remain independent "
    "of auxiliary branch decisions."
)

Backup created:
/home/jovyan/Deepfake_Low Resource_Low Resolution/agrfn_adaptive_routing_df40_before_film_20260730_215953.py

Inserted FingerprintFiLMRouter.
Updated AGRFN with FiLM support.
Updated build_model().
Enabled adaptive_film in parse_args().

PASS: Python compilation succeeded.

A-GRFN FILM MODULE TEST PASSED

FiLM model class:
AGRFN

Fusion mode:
adaptive_film

Evidence streams:
['rgb', 'jpeg', 'freq']

Initial routing weights:
tensor([[0.3333, 0.3333, 0.3333],
        [0.3333, 0.3333, 0.3333]])

Routing sums:
tensor([1., 1.])

Initial gamma deviation from one:
0.0

Initial beta magnitude:
0.0

Auxiliary branch-head gradients:
{'rgb': False, 'fingerprint': False, 'jpeg': False, 'freq': False}

FiLM-router gradient magnitudes:
gate_head   : 4.34919310
gamma_head  : 2.83662844
beta_head   : 6.58966303

Scalar mode remains available:
AGRFN

PASS: fingerprint-conditioned FiLM modulation is active before classification.
PASS: final logits remain independent of auxiliary branch d

In [37]:
from pathlib import Path
from datetime import datetime
import subprocess
import sys

import numpy as np
import pandas as pd

# ============================================================
# Paths
# ============================================================

project_root = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution"
)

script_path = project_root / "agrfn_adaptive_routing_df40.py"

manifest_path = (
    project_root
    / "outputs_df40"
    / "df40_manifest_all_clean.csv"
)

if not script_path.exists():
    raise FileNotFoundError(script_path)

if not manifest_path.exists():
    raise FileNotFoundError(manifest_path)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

out_dir = (
    project_root
    / f"outputs_agrfn_film_df40_smoke_{timestamp}"
)

# ============================================================
# Run one-epoch FiLM smoke training
# ============================================================

command = [
    sys.executable,
    str(script_path),

    "--metadata_csv",
    str(manifest_path),

    "--out_dir",
    str(out_dir),

    "--dataset_name",
    "DF40",

    "--fusion_mode",
    "adaptive_film",

    "--streams",
    "rgb,fingerprint,jpeg,freq",

    "--feat_dim",
    "128",

    "--controller_dim",
    "128",

    "--routing_temperature",
    "1.0",

    "--film_scale",
    "0.10",

    "--frames_per_video",
    "2",

    "--limit_train_units",
    "40",

    "--limit_val_units",
    "20",

    "--limit_test_units",
    "20",

    "--balanced_unit_limits",

    "--image_size",
    "224",

    "--batch_size",
    "4",

    "--num_workers",
    "0",

    "--epochs",
    "1",

    "--lr",
    "1e-4",

    "--lambda_aux",
    "0.05",

    "--lambda_route",
    "0.01",

    "--lambda_cons",
    "0.02",

    "--lambda_pred_cons",
    "0.02",

    "--balance_strategy",
    "auto",

    "--decision_threshold",
    "0.5",

    "--score_direction",
    "normal",

    "--amp",
]

print("=" * 80)
print("RUNNING A-GRFN FILM SMOKE TEST")
print("=" * 80)

print("\nOutput directory:")
print(out_dir)

result = subprocess.run(
    command,
    cwd=str(project_root),
)

if result.returncode != 0:
    raise RuntimeError(
        "FiLM smoke test failed with return code "
        f"{result.returncode}."
    )

# ============================================================
# Validate output files
# ============================================================

required_files = [
    "best_checkpoint.pt",
    "latest_checkpoint.pt",
    "training_history.csv",
    "test_frame_predictions.csv",
    "test_video_predictions.csv",
    "test_metrics.csv",
    "test_inference_time.csv",
    "experiment_summary.csv",
    "branch_predictions.csv",
    "branch_uncertainties.csv",
    "routing_weights.csv",
    "disagreement_matrices.csv",
    "uncertainty_breakdown.csv",
]

missing_files = [
    name
    for name in required_files
    if not (out_dir / name).exists()
]

if missing_files:
    raise RuntimeError(
        "Missing output files:\n"
        + "\n".join(missing_files)
    )

history = pd.read_csv(
    out_dir / "training_history.csv"
)

predictions = pd.read_csv(
    out_dir / "test_frame_predictions.csv"
)

metrics = pd.read_csv(
    out_dir / "test_metrics.csv"
)

routing = pd.read_csv(
    out_dir / "routing_weights.csv"
)

# ============================================================
# Validate training losses
# ============================================================

required_history_columns = [
    "train_total",
    "train_cls",
    "train_stream",
    "train_route_balance",
    "train_route_cons",
    "train_aug_cons",
]

missing_history_columns = [
    column
    for column in required_history_columns
    if column not in history.columns
]

if missing_history_columns:
    raise RuntimeError(
        "Missing training-history columns: "
        + ", ".join(missing_history_columns)
    )

if not np.isfinite(
    history[required_history_columns].to_numpy(dtype=float)
).all():
    raise RuntimeError(
        "Training history contains NaN or infinity."
    )

# ============================================================
# Validate prediction and routing outputs
# ============================================================

route_columns = [
    "route_rgb",
    "route_jpeg",
    "route_freq",
]

missing_route_columns = [
    column
    for column in route_columns
    if column not in predictions.columns
]

if missing_route_columns:
    raise RuntimeError(
        "Missing routing columns: "
        + ", ".join(missing_route_columns)
    )

prediction_columns = [
    "fake_prob",
    "uncertainty",
    "mean_branch_disagreement",
    "routing_entropy",
]

missing_prediction_columns = [
    column
    for column in prediction_columns
    if column not in predictions.columns
]

if missing_prediction_columns:
    raise RuntimeError(
        "Missing prediction columns: "
        + ", ".join(missing_prediction_columns)
    )

numeric_values = predictions[
    prediction_columns + route_columns
].to_numpy(dtype=float)

if not np.isfinite(numeric_values).all():
    raise RuntimeError(
        "Predictions contain NaN or infinity."
    )

if not predictions["fake_prob"].between(0, 1).all():
    raise RuntimeError(
        "fake_prob contains values outside [0, 1]."
    )

if not predictions["uncertainty"].between(0, 1).all():
    raise RuntimeError(
        "uncertainty contains values outside [0, 1]."
    )

route_values = predictions[
    route_columns
].to_numpy(dtype=float)

route_sums = route_values.sum(axis=1)

maximum_route_sum_error = float(
    np.abs(route_sums - 1.0).max()
)

if maximum_route_sum_error > 1e-5:
    raise RuntimeError(
        "FiLM routing weights do not sum to one. "
        f"Maximum error: {maximum_route_sum_error}"
    )

if "route_fingerprint" in predictions.columns:
    raise RuntimeError(
        "Fingerprint incorrectly appears as an evidence route."
    )

if "rel_fingerprint" in predictions.columns:
    fingerprint_route_max = float(
        predictions["rel_fingerprint"].abs().max()
    )

    if fingerprint_route_max > 1e-7:
        raise RuntimeError(
            "rel_fingerprint should be zero because "
            "fingerprint is the controller."
        )

# ============================================================
# Report
# ============================================================

print("\n" + "=" * 80)
print("A-GRFN FILM SMOKE TEST PASSED")
print("=" * 80)

print("\nOutput directory:")
print(out_dir)

print("\nTraining losses:")
display(
    history[required_history_columns]
)

print("\nTest metrics:")
display(metrics)

print("\nMean routing weights:")
display(
    predictions[
        route_columns
    ]
    .mean()
    .to_frame("mean_weight")
)

print("\nRouting standard deviations:")
display(
    predictions[
        route_columns
    ]
    .std()
    .to_frame("standard_deviation")
)

print("\nMaximum routing-sum error:")
print(maximum_route_sum_error)

print("\nPrediction summary:")
display(
    predictions[
        [
            "label",
            "fake_prob",
            "uncertainty",
            "mean_branch_disagreement",
            "routing_entropy",
        ]
    ].describe()
)

print("\nRouting CSV rows:")
print(len(routing))

print(
    "\nPASS: adaptive FiLM completed a full DF40 "
    "training, validation, checkpoint, and test cycle."
)

RUNNING A-GRFN FILM SMOKE TEST

Output directory:
/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_film_df40_smoke_20260730_220055

GRFN-CSF GRFN-only training and evaluation
Device: cuda
Output directory: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_film_df40_smoke_20260730_220055
Metadata CSV: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40/df40_manifest_all_clean.csv
Streams: rgb,fingerprint,jpeg,freq

Metadata loaded
Metadata CSV: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40/df40_manifest_all_clean.csv
Path column: frame_path
Label column: label
Split column: split
Group column: group_id
Usable frames: 292907
train: frames=235640, units=7843, real=48132, fake=187508
val: frames=28551, units=980, real=5997, fake=22554
test: frames=28716, units=981, real=5999, fake=22717
Saved label audit samples to: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_film_df40_smoke_20260730_220055/label_audit_samples.csv

Sample train: 100%|██████████| 40/40 [00:00<00:00, 65.55unit/s]


Sample train: frames=78, units=40, real=40, fake=38
Sample val: frames=39, units=20, real=20, fake=19


Sample test: 100%|██████████| 20/20 [00:00<00:00, 354.50unit/s]


Sample test: frames=39, units=20, real=20, fake=19

Balancing strategy
Train class counts [real, fake]: [40, 38]
Train imbalance ratio: 1.053
Use class-weighted loss: False
Use weighted sampler: False

Building dataloaders
Train batches: 19
Val batches: 10
Test batches: 10

Building model
Model mode: grfn
Loss mode: grfn
Active streams: rgb, fingerprint, jpeg, freq
Total params: 1,780,813
Trainable params: 1,780,813

Training


Predict val: 100%|██████████| 10/10 [00:00<00:00, 10.03it/s]


... (9 outputs left out)

,standard_deviation
route_rgb,0.000630
route_jpeg,0.000297
route_freq,0.000390



Maximum routing-sum error:
5.960464499743523e-08

Prediction summary:


,label,fake_prob,uncertainty,mean_branch_disagreement,routing_entropy
count,39.000000,39.000000,39.000000,39.000000,39.000000
mean,0.487179,0.319958,0.657479,0.035380,0.999998
std,0.506370,0.057077,0.034681,0.016696,0.000003
min,0.000000,0.230001,0.593579,0.013328,0.999988
25%,0.000000,0.279936,0.638171,0.024239,0.999997
50%,0.000000,0.319496,0.662917,0.031917,0.999999
75%,1.000000,0.358188,0.682232,0.042556,1.000000
max,1.000000,0.429592,0.710719,0.089225,1.000000



Routing CSV rows:
39

PASS: adaptive FiLM completed a full DF40 training, validation, checkpoint, and test cycle.


In [38]:
from pathlib import Path
from datetime import datetime
from types import SimpleNamespace
import ast
import shutil
import py_compile
import importlib.util

import torch
import torch.nn.functional as F


# ============================================================
# Paths and backup
# ============================================================

project_root = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution"
)

script_path = (
    project_root
    / "agrfn_adaptive_routing_df40.py"
)

if not script_path.exists():
    raise FileNotFoundError(script_path)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

backup_path = script_path.with_name(
    f"{script_path.stem}_before_transformer_{timestamp}.py"
)

shutil.copy2(script_path, backup_path)

print("Backup created:")
print(backup_path)

text = script_path.read_text()


# ============================================================
# Add FingerprintTransformerRouter
# ============================================================

tree = ast.parse(text)

top_level_names = {
    node.name
    for node in tree.body
    if isinstance(
        node,
        (ast.ClassDef, ast.FunctionDef),
    )
}

if "FingerprintTransformerRouter" not in top_level_names:
    film_router_node = next(
        (
            node
            for node in tree.body
            if isinstance(node, ast.ClassDef)
            and node.name == "FingerprintFiLMRouter"
        ),
        None,
    )

    if film_router_node is None:
        raise RuntimeError(
            "Could not find FingerprintFiLMRouter."
        )

    transformer_router_code = r'''

class FingerprintTransformerRouter(nn.Module):
    """Contextualize evidence streams using the fingerprint controller."""

    def __init__(
        self,
        feat_dim=128,
        num_branches=3,
        num_heads=4,
        num_layers=1,
        dropout=0.10,
        film_scale=0.10,
        temperature=1.0,
    ):
        super().__init__()

        if feat_dim <= 0:
            raise ValueError(
                "feat_dim must be positive."
            )

        if num_branches <= 0:
            raise ValueError(
                "num_branches must be positive."
            )

        if num_heads <= 0:
            raise ValueError(
                "num_heads must be positive."
            )

        if feat_dim % num_heads != 0:
            raise ValueError(
                "feat_dim must be divisible by num_heads. "
                f"Received feat_dim={feat_dim} and "
                f"num_heads={num_heads}."
            )

        if num_layers <= 0:
            raise ValueError(
                "num_layers must be positive."
            )

        if not 0.0 <= dropout < 1.0:
            raise ValueError(
                "dropout must be in [0, 1)."
            )

        if film_scale < 0:
            raise ValueError(
                "film_scale cannot be negative."
            )

        if temperature <= 0:
            raise ValueError(
                "temperature must be positive."
            )

        self.feat_dim = int(feat_dim)
        self.num_branches = int(num_branches)
        self.num_heads = int(num_heads)
        self.num_layers = int(num_layers)
        self.film_scale = float(film_scale)
        self.temperature = float(temperature)

        self.controller_projection = nn.Sequential(
            nn.Linear(
                self.feat_dim,
                self.feat_dim,
            ),
            nn.LayerNorm(self.feat_dim),
            nn.GELU(),
        )

        self.gamma_head = nn.Linear(
            self.feat_dim,
            self.num_branches * self.feat_dim,
        )

        self.beta_head = nn.Linear(
            self.feat_dim,
            self.num_branches * self.feat_dim,
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.feat_dim,
            nhead=self.num_heads,
            dim_feedforward=self.feat_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=self.num_layers,
            norm=nn.LayerNorm(self.feat_dim),
        )

        # Controller/evidence pair scoring.
        self.route_head = nn.Sequential(
            nn.Linear(
                self.feat_dim * 4,
                self.feat_dim,
            ),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(
                self.feat_dim,
                1,
            ),
        )

        # Identity FiLM initialization.
        nn.init.zeros_(self.gamma_head.weight)
        nn.init.zeros_(self.gamma_head.bias)

        nn.init.zeros_(self.beta_head.weight)
        nn.init.zeros_(self.beta_head.bias)

        # Uniform initial routing.
        final_route_layer = self.route_head[-1]
        nn.init.zeros_(final_route_layer.weight)
        nn.init.zeros_(final_route_layer.bias)

    def forward(
        self,
        evidence_features,
        controller,
    ):
        if evidence_features.ndim != 3:
            raise ValueError(
                "evidence_features must have shape "
                "[batch, branches, feat_dim]."
            )

        if controller.ndim != 2:
            raise ValueError(
                "controller must have shape "
                "[batch, feat_dim]."
            )

        batch_size, num_branches, feat_dim = (
            evidence_features.shape
        )

        if num_branches != self.num_branches:
            raise ValueError(
                "Evidence branch count mismatch. "
                f"Expected {self.num_branches}, "
                f"received {num_branches}."
            )

        if feat_dim != self.feat_dim:
            raise ValueError(
                "Evidence feature dimension mismatch. "
                f"Expected {self.feat_dim}, "
                f"received {feat_dim}."
            )

        if controller.shape != (
            batch_size,
            self.feat_dim,
        ):
            raise ValueError(
                "Controller shape mismatch. "
                f"Expected {(batch_size, self.feat_dim)}, "
                f"received {tuple(controller.shape)}."
            )

        projected_controller = (
            self.controller_projection(
                controller
            )
        )

        gamma_raw = self.gamma_head(
            projected_controller
        ).view(
            batch_size,
            self.num_branches,
            self.feat_dim,
        )

        beta_raw = self.beta_head(
            projected_controller
        ).view(
            batch_size,
            self.num_branches,
            self.feat_dim,
        )

        gamma = (
            1.0
            + self.film_scale
            * torch.tanh(gamma_raw)
        )

        beta = (
            self.film_scale
            * torch.tanh(beta_raw)
        )

        modulated_evidence = (
            gamma * evidence_features
            + beta
        )

        # The first token is the fingerprint controller.
        tokens = torch.cat(
            [
                projected_controller.unsqueeze(1),
                modulated_evidence,
            ],
            dim=1,
        )

        contextual_tokens = self.transformer(
            tokens
        )

        controller_token = contextual_tokens[
            :,
            0,
        ]

        contextual_evidence = contextual_tokens[
            :,
            1:,
        ]

        expanded_controller = (
            controller_token
            .unsqueeze(1)
            .expand(
                -1,
                self.num_branches,
                -1,
            )
        )

        route_features = torch.cat(
            [
                expanded_controller,
                contextual_evidence,
                expanded_controller
                * contextual_evidence,
                torch.abs(
                    expanded_controller
                    - contextual_evidence
                ),
            ],
            dim=-1,
        )

        gate_logits = self.route_head(
            route_features
        ).squeeze(-1)

        routing_weights = F.softmax(
            gate_logits / self.temperature,
            dim=1,
        )

        routed_feature = (
            routing_weights.unsqueeze(-1)
            * contextual_evidence
        ).sum(dim=1)

        return {
            "gate_logits": gate_logits,
            "routing_weights": routing_weights,
            "attention_weights": routing_weights,
            "gamma": gamma,
            "beta": beta,
            "controller_token": controller_token,
            "contextual_evidence": contextual_evidence,
            "routed_feature": routed_feature,
        }

'''

    lines = text.splitlines(keepends=True)

    insertion_index = film_router_node.end_lineno

    lines = (
        lines[:insertion_index]
        + transformer_router_code.splitlines(
            keepends=True
        )
        + lines[insertion_index:]
    )

    text = "".join(lines)

    print("\nInserted FingerprintTransformerRouter.")
else:
    print(
        "\nFingerprintTransformerRouter already exists."
    )


# ============================================================
# Update AGRFN
# ============================================================

tree = ast.parse(text)

agrfn_node = next(
    (
        node
        for node in tree.body
        if isinstance(node, ast.ClassDef)
        and node.name == "AGRFN"
    ),
    None,
)

if agrfn_node is None:
    raise RuntimeError(
        "Could not find class AGRFN."
    )

lines = text.splitlines(keepends=True)

agrfn_source = "".join(
    lines[
        agrfn_node.lineno - 1:
        agrfn_node.end_lineno
    ]
)


def replace_required(
    source,
    old,
    new,
    description,
):
    if old in source:
        print(f"Updating {description}.")
        return source.replace(
            old,
            new,
            1,
        )

    if new in source:
        print(
            f"{description} already updated."
        )
        return source

    raise RuntimeError(
        f"Could not find the expected block for "
        f"{description}."
    )


# ------------------------------------------------------------
# Supported modes
# ------------------------------------------------------------

agrfn_source = replace_required(
    agrfn_source,
    '''    VALID_FUSION_MODES = {
        "adaptive_scalar",
        "adaptive_film",
    }
''',
    '''    VALID_FUSION_MODES = {
        "adaptive_scalar",
        "adaptive_film",
        "adaptive_transformer",
    }
''',
    "AGRFN fusion modes",
)


# ------------------------------------------------------------
# Constructor arguments
# ------------------------------------------------------------

agrfn_source = replace_required(
    agrfn_source,
    '''        routing_temperature=1.0,
        film_scale=0.10,
        dropout=0.20,
''',
    '''        routing_temperature=1.0,
        film_scale=0.10,
        router_heads=4,
        router_layers=1,
        router_dropout=0.10,
        dropout=0.20,
''',
    "AGRFN transformer arguments",
)


# ------------------------------------------------------------
# Router construction
# ------------------------------------------------------------

router_start_marker = (
    "        # Instantiate only the router used by this mode.\n"
)

router_end_marker = (
    "        # Auxiliary heads only.\n"
)

router_start = agrfn_source.find(
    router_start_marker
)

router_end = agrfn_source.find(
    router_end_marker,
    router_start,
)

if router_start < 0 or router_end < 0:
    raise RuntimeError(
        "Could not find the AGRFN router-construction block."
    )

new_router_block = '''        # Instantiate only the router used by this mode.
        if self.fusion_mode == "adaptive_scalar":
            self.scalar_router = ScalarGateRouter(
                feat_dim=self.feat_dim,
                num_branches=(
                    self.num_evidence_streams
                ),
                temperature=routing_temperature,
            )

        elif self.fusion_mode == "adaptive_film":
            self.film_router = FingerprintFiLMRouter(
                feat_dim=self.feat_dim,
                num_branches=(
                    self.num_evidence_streams
                ),
                film_scale=film_scale,
                temperature=routing_temperature,
            )

        elif self.fusion_mode == "adaptive_transformer":
            self.transformer_router = (
                FingerprintTransformerRouter(
                    feat_dim=self.feat_dim,
                    num_branches=(
                        self.num_evidence_streams
                    ),
                    num_heads=router_heads,
                    num_layers=router_layers,
                    dropout=router_dropout,
                    film_scale=film_scale,
                    temperature=routing_temperature,
                )
            )

'''

agrfn_source = (
    agrfn_source[:router_start]
    + new_router_block
    + agrfn_source[router_end:]
)

print("Updated AGRFN router construction.")


# ------------------------------------------------------------
# Forward routing logic
# ------------------------------------------------------------

evidence_marker = (
    "        evidence_features = torch.stack(\n"
)

evidence_position = agrfn_source.find(
    evidence_marker
)

if evidence_position < 0:
    raise RuntimeError(
        "Could not find evidence feature construction."
    )

forward_route_start = agrfn_source.find(
    '        if self.fusion_mode == "adaptive_scalar":\n',
    evidence_position,
)

forward_route_end = agrfn_source.find(
    "        joint_embedding = (\n",
    forward_route_start,
)

if forward_route_start < 0 or forward_route_end < 0:
    raise RuntimeError(
        "Could not find AGRFN forward routing block."
    )

new_forward_routing = '''        attention_weights = None
        routing_controller = controller

        if self.fusion_mode == "adaptive_scalar":
            routing_output = (
                self.scalar_router(
                    controller
                )
            )

            routing_weights = (
                routing_output[
                    "routing_weights"
                ]
            )

            gamma = torch.ones_like(
                evidence_features
            )

            beta = torch.zeros_like(
                evidence_features
            )

            modulated_features = (
                routing_weights.unsqueeze(-1)
                * evidence_features
            )

            routed_feature = (
                modulated_features.sum(dim=1)
            )

        elif self.fusion_mode == "adaptive_film":
            routing_output = (
                self.film_router(
                    controller
                )
            )

            routing_weights = (
                routing_output[
                    "routing_weights"
                ]
            )

            gamma = routing_output["gamma"]
            beta = routing_output["beta"]

            modulated_features = (
                routing_weights.unsqueeze(-1)
                * (
                    gamma * evidence_features
                    + beta
                )
            )

            routed_feature = (
                modulated_features.sum(dim=1)
            )

        elif self.fusion_mode == "adaptive_transformer":
            routing_output = (
                self.transformer_router(
                    evidence_features,
                    controller,
                )
            )

            routing_weights = (
                routing_output[
                    "routing_weights"
                ]
            )

            attention_weights = (
                routing_output[
                    "attention_weights"
                ]
            )

            gamma = routing_output["gamma"]
            beta = routing_output["beta"]

            routed_feature = (
                routing_output[
                    "routed_feature"
                ]
            )

            routing_controller = (
                routing_output[
                    "controller_token"
                ]
            )

        else:
            raise RuntimeError(
                f"Unhandled fusion mode: "
                f"{self.fusion_mode}"
            )

        gate_logits = routing_output[
            "gate_logits"
        ]

'''

agrfn_source = (
    agrfn_source[:forward_route_start]
    + new_forward_routing
    + agrfn_source[forward_route_end:]
)

print("Updated AGRFN forward routing.")


# ------------------------------------------------------------
# Use contextual controller in final fusion
# ------------------------------------------------------------

agrfn_source = replace_required(
    agrfn_source,
    '''                        routed_feature,
                        controller,
''',
    '''                        routed_feature,
                        routing_controller,
''',
    "contextual controller fusion",
)


# ------------------------------------------------------------
# Add transformer diagnostics to output
# ------------------------------------------------------------

agrfn_source = replace_required(
    agrfn_source,
    '''            "controller": controller,
            "joint_embedding": joint_embedding,
''',
    '''            "controller": controller,
            "routing_controller": routing_controller,
            "joint_embedding": joint_embedding,
''',
    "routing-controller output",
)

agrfn_source = replace_required(
    agrfn_source,
    '''            "routing_entropy": routing_entropy,

            "film_gamma": gamma,
''',
    '''            "routing_entropy": routing_entropy,
            "attention_weights": attention_weights,

            "film_gamma": gamma,
''',
    "attention output",
)


# ------------------------------------------------------------
# Reinsert updated AGRFN class
# ------------------------------------------------------------

lines = text.splitlines(keepends=True)

lines = (
    lines[:agrfn_node.lineno - 1]
    + agrfn_source.splitlines(
        keepends=True
    )
    + lines[agrfn_node.end_lineno:]
)

text = "".join(lines)


# ============================================================
# Replace build_model()
# ============================================================

tree = ast.parse(text)

build_model_node = next(
    (
        node
        for node in tree.body
        if isinstance(node, ast.FunctionDef)
        and node.name == "build_model"
    ),
    None,
)

if build_model_node is None:
    raise RuntimeError(
        "Could not find build_model()."
    )

new_build_model = r'''def build_model(args, stream_names):
    """Build all supported GRFN and A-GRFN variants."""

    if args.fusion_mode == "late":
        return GRFNCSF(
            feat_dim=args.feat_dim,
            stream_names=stream_names,
            stream_logit_residual_weight=(
                args.stream_logit_residual_weight
            ),
        )

    if args.fusion_mode == "average":
        return SimpleStreamModel(
            feat_dim=args.feat_dim,
            stream_names=stream_names,
        )

    if args.fusion_mode in {
        "adaptive_scalar",
        "adaptive_film",
        "adaptive_transformer",
    }:
        return AGRFN(
            feat_dim=args.feat_dim,
            controller_dim=(
                args.controller_dim
            ),
            stream_names=stream_names,
            fusion_mode=args.fusion_mode,
            routing_temperature=(
                args.routing_temperature
            ),
            film_scale=getattr(
                args,
                "film_scale",
                0.10,
            ),
            router_heads=getattr(
                args,
                "router_heads",
                4,
            ),
            router_layers=getattr(
                args,
                "router_layers",
                1,
            ),
            router_dropout=getattr(
                args,
                "router_dropout",
                0.10,
            ),
        )

    raise NotImplementedError(
        f"Fusion mode '{args.fusion_mode}' "
        "has not been implemented."
    )
'''

lines = text.splitlines(keepends=True)

lines = (
    lines[:build_model_node.lineno - 1]
    + new_build_model.splitlines(
        keepends=True
    )
    + ["\n\n"]
    + lines[build_model_node.end_lineno:]
)

text = "".join(lines)

print("Updated build_model().")


# ============================================================
# Remove the temporary transformer CLI guard
# ============================================================

tree = ast.parse(text)

parse_args_node = next(
    (
        node
        for node in tree.body
        if isinstance(node, ast.FunctionDef)
        and node.name == "parse_args"
    ),
    None,
)

if parse_args_node is None:
    raise RuntimeError(
        "Could not find parse_args()."
    )

guard_node = None

for node in ast.walk(parse_args_node):
    if not isinstance(node, ast.If):
        continue

    source = ast.get_source_segment(
        text,
        node,
    )

    if (
        source
        and "adaptive_transformer" in source
        and "NotImplementedError" in source
    ):
        guard_node = node
        break

if guard_node is not None:
    lines = text.splitlines(keepends=True)

    lines = (
        lines[:guard_node.lineno - 1]
        + lines[guard_node.end_lineno:]
    )

    text = "".join(lines)

    print(
        "Enabled adaptive_transformer in parse_args()."
    )
else:
    print(
        "Transformer CLI guard was already removed."
    )


# ============================================================
# Save and compile
# ============================================================

script_path.write_text(text)

py_compile.compile(
    str(script_path),
    doraise=True,
)

print("\nPASS: Python compilation succeeded.")


# ============================================================
# Import updated module
# ============================================================

module_name = (
    "agrfn_transformer_test_"
    + timestamp
)

spec = importlib.util.spec_from_file_location(
    module_name,
    script_path,
)

module = importlib.util.module_from_spec(
    spec
)

if spec.loader is None:
    raise RuntimeError(
        "Could not create module loader."
    )

spec.loader.exec_module(module)


# ============================================================
# Dummy inputs
# ============================================================

batch_size = 2
image_size = 64

dummy_view = {
    "rgb": torch.randn(
        batch_size,
        3,
        image_size,
        image_size,
    ),
    "rec": torch.randn(
        batch_size,
        3,
        image_size,
        image_size,
    ),
    "den": torch.randn(
        batch_size,
        3,
        image_size,
        image_size,
    ),
    "jpeg": torch.randn(
        batch_size,
        3,
        image_size,
        image_size,
    ),
    "freq": torch.randn(
        batch_size,
        1,
        image_size,
        image_size,
    ),
    "noise": torch.randn(
        batch_size,
        1,
        image_size,
        image_size,
    ),
}

stream_names = [
    "rgb",
    "fingerprint",
    "jpeg",
    "freq",
]

common_args = {
    "feat_dim": 128,
    "controller_dim": 128,
    "routing_temperature": 1.0,
    "film_scale": 0.10,
    "router_heads": 4,
    "router_layers": 1,
    "router_dropout": 0.10,
    "stream_logit_residual_weight": 0.25,
}


# ============================================================
# Transformer forward test
# ============================================================

transformer_args = SimpleNamespace(
    fusion_mode="adaptive_transformer",
    **common_args,
)

transformer_model = module.build_model(
    transformer_args,
    stream_names,
)

transformer_model.train()

output = transformer_model(
    dummy_view
)

expected_shapes = {
    "logits": (
        batch_size,
        2,
    ),
    "routing_weights": (
        batch_size,
        3,
    ),
    "attention_weights": (
        batch_size,
        3,
    ),
    "film_gamma": (
        batch_size,
        3,
        128,
    ),
    "film_beta": (
        batch_size,
        3,
        128,
    ),
    "routing_controller": (
        batch_size,
        128,
    ),
    "joint_embedding": (
        batch_size,
        128,
    ),
}

for key, expected_shape in expected_shapes.items():
    actual_shape = tuple(
        output[key].shape
    )

    if actual_shape != expected_shape:
        raise RuntimeError(
            f"{key} shape mismatch. "
            f"Expected {expected_shape}, "
            f"received {actual_shape}."
        )

route_sums = output[
    "routing_weights"
].sum(dim=1)

if not torch.allclose(
    route_sums,
    torch.ones_like(route_sums),
    atol=1e-6,
):
    raise RuntimeError(
        "Transformer routes do not sum to one."
    )

if not torch.allclose(
    output["attention_weights"],
    output["routing_weights"],
    atol=1e-7,
):
    raise RuntimeError(
        "Logged attention and routing weights differ."
    )

if not torch.allclose(
    output["film_gamma"],
    torch.ones_like(
        output["film_gamma"]
    ),
    atol=1e-7,
):
    raise RuntimeError(
        "Transformer FiLM gamma did not "
        "initialize to one."
    )

if not torch.allclose(
    output["film_beta"],
    torch.zeros_like(
        output["film_beta"]
    ),
    atol=1e-7,
):
    raise RuntimeError(
        "Transformer FiLM beta did not "
        "initialize to zero."
    )


# ============================================================
# Gradient and branch-independence test
# ============================================================

transformer_model.zero_grad(
    set_to_none=True
)

output = transformer_model(
    dummy_view
)

labels = torch.tensor(
    [0, 1],
    dtype=torch.long,
)

loss = F.cross_entropy(
    output["logits"],
    labels,
)

loss.backward()

branch_head_gradients = {
    name: any(
        parameter.grad is not None
        for parameter in head.parameters()
    )
    for name, head
    in transformer_model.stream_heads.items()
}

if any(branch_head_gradients.values()):
    raise RuntimeError(
        "Transformer final logits depend on "
        "auxiliary branch heads: "
        f"{branch_head_gradients}"
    )

transformer_parameters_with_grad = [
    parameter
    for parameter in (
        transformer_model
        .transformer_router
        .transformer
        .parameters()
    )
    if (
        parameter.grad is not None
        and torch.isfinite(
            parameter.grad
        ).all()
    )
]

if not transformer_parameters_with_grad:
    raise RuntimeError(
        "Transformer layers received no gradients."
    )

route_head_gradient = (
    transformer_model
    .transformer_router
    .route_head[-1]
    .weight
    .grad
)

if route_head_gradient is None:
    raise RuntimeError(
        "Transformer route head received no gradient."
    )

if not torch.isfinite(
    route_head_gradient
).all():
    raise RuntimeError(
        "Route-head gradient contains NaN or infinity."
    )

if float(
    route_head_gradient.abs().sum()
) == 0.0:
    raise RuntimeError(
        "Route-head gradient is entirely zero."
    )

gamma_gradient = (
    transformer_model
    .transformer_router
    .gamma_head
    .weight
    .grad
)

beta_gradient = (
    transformer_model
    .transformer_router
    .beta_head
    .weight
    .grad
)

for name, gradient in {
    "gamma_head": gamma_gradient,
    "beta_head": beta_gradient,
}.items():
    if gradient is None:
        raise RuntimeError(
            f"{name} received no gradient."
        )

    if not torch.isfinite(
        gradient
    ).all():
        raise RuntimeError(
            f"{name} gradient contains NaN or infinity."
        )


# ============================================================
# Confirm scalar and FiLM still build
# ============================================================

scalar_args = SimpleNamespace(
    fusion_mode="adaptive_scalar",
    **common_args,
)

film_args = SimpleNamespace(
    fusion_mode="adaptive_film",
    **common_args,
)

scalar_model = module.build_model(
    scalar_args,
    stream_names,
)

film_model = module.build_model(
    film_args,
    stream_names,
)

with torch.no_grad():
    scalar_output = scalar_model(
        dummy_view
    )

    film_output = film_model(
        dummy_view
    )

if tuple(
    scalar_output["logits"].shape
) != (
    batch_size,
    2,
):
    raise RuntimeError(
        "Scalar mode was broken."
    )

if tuple(
    film_output["logits"].shape
) != (
    batch_size,
    2,
):
    raise RuntimeError(
        "FiLM mode was broken."
    )


# ============================================================
# Final report
# ============================================================

print("\n" + "=" * 80)
print("A-GRFN TRANSFORMER MODULE TEST PASSED")
print("=" * 80)

print("\nFusion mode:")
print(transformer_model.fusion_mode)

print("\nEvidence streams:")
print(transformer_model.evidence_streams)

print("\nInitial transformer routing:")
print(
    output[
        "routing_weights"
    ]
    .detach()
    .cpu()
)

print("\nRouting sums:")
print(
    route_sums
    .detach()
    .cpu()
)

print("\nInitial gamma deviation:")
print(
    float(
        torch.abs(
            output["film_gamma"]
            - 1.0
        ).max()
    )
)

print("\nInitial beta magnitude:")
print(
    float(
        torch.abs(
            output["film_beta"]
        ).max()
    )
)

print("\nAuxiliary branch-head gradients:")
print(branch_head_gradients)

print("\nTransformer parameters with gradients:")
print(len(transformer_parameters_with_grad))

print("\nRoute-head gradient magnitude:")
print(
    float(
        route_head_gradient
        .abs()
        .sum()
    )
)

print("\nScalar mode remains available:")
print(type(scalar_model).__name__)

print("\nFiLM mode remains available:")
print(type(film_model).__name__)

print(
    "\nPASS: fingerprint and evidence tokens now "
    "interact through transformer routing."
)

print(
    "PASS: final transformer logits remain independent "
    "of auxiliary branch decisions."
)

Backup created:
/home/jovyan/Deepfake_Low Resource_Low Resolution/agrfn_adaptive_routing_df40_before_transformer_20260730_220351.py

Inserted FingerprintTransformerRouter.
Updating AGRFN fusion modes.
Updating AGRFN transformer arguments.
Updated AGRFN router construction.
Updated AGRFN forward routing.
Updating contextual controller fusion.
Updating routing-controller output.
Updating attention output.
Updated build_model().
Enabled adaptive_transformer in parse_args().

PASS: Python compilation succeeded.

A-GRFN TRANSFORMER MODULE TEST PASSED

Fusion mode:
adaptive_transformer

Evidence streams:
['rgb', 'jpeg', 'freq']

Initial transformer routing:
tensor([[0.3333, 0.3333, 0.3333],
        [0.3333, 0.3333, 0.3333]])

Routing sums:
tensor([1., 1.])

Initial gamma deviation:
0.0

Initial beta magnitude:
0.0

Auxiliary branch-head gradients:
{'rgb': False, 'fingerprint': False, 'jpeg': False, 'freq': False}

Transformer parameters with gradients:
14

Route-head gradient magnitude:
0.57

In [39]:
from pathlib import Path
from datetime import datetime
import subprocess
import sys

import numpy as np
import pandas as pd


# ============================================================
# Paths
# ============================================================

project_root = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution"
)

script_path = (
    project_root
    / "agrfn_adaptive_routing_df40.py"
)

manifest_path = (
    project_root
    / "outputs_df40"
    / "df40_manifest_all_clean.csv"
)

if not script_path.exists():
    raise FileNotFoundError(script_path)

if not manifest_path.exists():
    raise FileNotFoundError(manifest_path)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

out_dir = (
    project_root
    / f"outputs_agrfn_transformer_df40_smoke_{timestamp}"
)


# ============================================================
# Run one-epoch transformer smoke training
# ============================================================

command = [
    sys.executable,
    str(script_path),

    "--metadata_csv",
    str(manifest_path),

    "--out_dir",
    str(out_dir),

    "--dataset_name",
    "DF40",

    "--fusion_mode",
    "adaptive_transformer",

    "--streams",
    "rgb,fingerprint,jpeg,freq",

    "--feat_dim",
    "128",

    "--controller_dim",
    "128",

    "--routing_temperature",
    "1.0",

    "--film_scale",
    "0.10",

    "--router_heads",
    "4",

    "--router_layers",
    "1",

    "--router_dropout",
    "0.10",

    "--frames_per_video",
    "2",

    "--limit_train_units",
    "40",

    "--limit_val_units",
    "20",

    "--limit_test_units",
    "20",

    "--balanced_unit_limits",

    "--image_size",
    "224",

    "--batch_size",
    "4",

    "--num_workers",
    "0",

    "--epochs",
    "1",

    "--lr",
    "1e-4",

    "--lambda_aux",
    "0.05",

    "--lambda_route",
    "0.01",

    "--lambda_cons",
    "0.02",

    "--lambda_pred_cons",
    "0.02",

    "--balance_strategy",
    "auto",

    "--decision_threshold",
    "0.5",

    "--score_direction",
    "normal",

    "--amp",
]

print("=" * 80)
print("RUNNING A-GRFN TRANSFORMER SMOKE TEST")
print("=" * 80)

print("\nOutput directory:")
print(out_dir)

result = subprocess.run(
    command,
    cwd=str(project_root),
)

if result.returncode != 0:
    raise RuntimeError(
        "Transformer smoke test failed with return code "
        f"{result.returncode}."
    )


# ============================================================
# Validate output files
# ============================================================

required_files = [
    "best_checkpoint.pt",
    "latest_checkpoint.pt",
    "training_history.csv",
    "test_frame_predictions.csv",
    "test_video_predictions.csv",
    "test_metrics.csv",
    "test_inference_time.csv",
    "experiment_summary.csv",
    "branch_predictions.csv",
    "branch_uncertainties.csv",
    "routing_weights.csv",
    "disagreement_matrices.csv",
    "uncertainty_breakdown.csv",
]

missing_files = [
    filename
    for filename in required_files
    if not (out_dir / filename).exists()
]

if missing_files:
    raise RuntimeError(
        "Missing transformer output files:\n"
        + "\n".join(missing_files)
    )


# ============================================================
# Load outputs
# ============================================================

history = pd.read_csv(
    out_dir / "training_history.csv"
)

predictions = pd.read_csv(
    out_dir / "test_frame_predictions.csv"
)

metrics = pd.read_csv(
    out_dir / "test_metrics.csv"
)

routing_table = pd.read_csv(
    out_dir / "routing_weights.csv"
)

inference_table = pd.read_csv(
    out_dir / "test_inference_time.csv"
)


# ============================================================
# Validate training losses
# ============================================================

required_history_columns = [
    "train_total",
    "train_cls",
    "train_stream",
    "train_pred_cons",
    "train_feat_cons",
    "train_route_balance",
    "train_route_cons",
    "train_aug_cons",
]

missing_history_columns = [
    column
    for column in required_history_columns
    if column not in history.columns
]

if missing_history_columns:
    raise RuntimeError(
        "Missing training-history columns: "
        + ", ".join(missing_history_columns)
    )

history_values = history[
    required_history_columns
].to_numpy(dtype=float)

if not np.isfinite(history_values).all():
    raise RuntimeError(
        "Transformer training history contains NaN or infinity."
    )


# ============================================================
# Validate routing and attention diagnostics
# ============================================================

route_columns = [
    "route_rgb",
    "route_jpeg",
    "route_freq",
]

attention_columns = [
    "attention_rgb",
    "attention_jpeg",
    "attention_freq",
]

missing_route_columns = [
    column
    for column in route_columns
    if column not in predictions.columns
]

missing_attention_columns = [
    column
    for column in attention_columns
    if column not in predictions.columns
]

if missing_route_columns:
    raise RuntimeError(
        "Missing routing columns: "
        + ", ".join(missing_route_columns)
    )

if missing_attention_columns:
    raise RuntimeError(
        "Missing transformer attention columns: "
        + ", ".join(missing_attention_columns)
    )

diagnostic_columns = [
    "fake_prob",
    "uncertainty",
    "classification_entropy",
    "mean_branch_entropy",
    "mean_branch_disagreement",
    "max_branch_disagreement",
    "routing_entropy",
] + route_columns + attention_columns

missing_diagnostic_columns = [
    column
    for column in diagnostic_columns
    if column not in predictions.columns
]

if missing_diagnostic_columns:
    raise RuntimeError(
        "Missing diagnostic columns: "
        + ", ".join(missing_diagnostic_columns)
    )

diagnostic_values = predictions[
    diagnostic_columns
].to_numpy(dtype=float)

if not np.isfinite(diagnostic_values).all():
    raise RuntimeError(
        "Transformer predictions contain NaN or infinity."
    )


# ============================================================
# Probability and routing checks
# ============================================================

if not predictions["fake_prob"].between(0, 1).all():
    raise RuntimeError(
        "fake_prob contains values outside [0, 1]."
    )

if not predictions["uncertainty"].between(0, 1).all():
    raise RuntimeError(
        "uncertainty contains values outside [0, 1]."
    )

route_values = predictions[
    route_columns
].to_numpy(dtype=float)

attention_values = predictions[
    attention_columns
].to_numpy(dtype=float)

if (route_values < 0).any():
    raise RuntimeError(
        "Routing weights contain negative values."
    )

if (attention_values < 0).any():
    raise RuntimeError(
        "Attention weights contain negative values."
    )

route_sums = route_values.sum(axis=1)
attention_sums = attention_values.sum(axis=1)

maximum_route_sum_error = float(
    np.abs(route_sums - 1.0).max()
)

maximum_attention_sum_error = float(
    np.abs(attention_sums - 1.0).max()
)

if maximum_route_sum_error > 1e-5:
    raise RuntimeError(
        "Transformer routes do not sum to one. "
        f"Maximum error: {maximum_route_sum_error}"
    )

if maximum_attention_sum_error > 1e-5:
    raise RuntimeError(
        "Transformer attention values do not sum to one. "
        f"Maximum error: {maximum_attention_sum_error}"
    )

maximum_route_attention_difference = float(
    np.abs(
        route_values - attention_values
    ).max()
)

if maximum_route_attention_difference > 1e-6:
    raise RuntimeError(
        "Logged routing and attention values differ. "
        f"Maximum difference: "
        f"{maximum_route_attention_difference}"
    )


# ============================================================
# Controller isolation checks
# ============================================================

if "route_fingerprint" in predictions.columns:
    raise RuntimeError(
        "Fingerprint incorrectly appears as an evidence route."
    )

if "attention_fingerprint" in predictions.columns:
    raise RuntimeError(
        "Fingerprint incorrectly appears as an evidence attention branch."
    )

if "rel_fingerprint" in predictions.columns:
    maximum_fingerprint_reliability = float(
        predictions[
            "rel_fingerprint"
        ].abs().max()
    )

    if maximum_fingerprint_reliability > 1e-7:
        raise RuntimeError(
            "rel_fingerprint must remain zero because "
            "the fingerprint is the controller."
        )


# ============================================================
# Validate metadata and output row alignment
# ============================================================

metadata_candidates = [
    "method",
    "domain",
    "top_level_folder",
    "source_layout",
    "relative_path",
]

available_metadata = [
    column
    for column in metadata_candidates
    if column in predictions.columns
]

if not available_metadata:
    raise RuntimeError(
        "No DF40 metadata columns were preserved."
    )

if len(routing_table) != len(predictions):
    raise RuntimeError(
        "routing_weights.csv and test_frame_predictions.csv "
        "have different row counts."
    )


# ============================================================
# Final report
# ============================================================

print("\n" + "=" * 80)
print("A-GRFN TRANSFORMER SMOKE TEST PASSED")
print("=" * 80)

print("\nOutput directory:")
print(out_dir)

print("\nTraining losses:")
display(
    history[
        required_history_columns
    ]
)

print("\nTest metrics:")
display(metrics)

print("\nInference timing:")
display(inference_table)

print("\nMean transformer routing weights:")
display(
    predictions[
        route_columns
    ]
    .mean()
    .to_frame("mean_weight")
)

print("\nRouting standard deviations:")
display(
    predictions[
        route_columns
    ]
    .std()
    .to_frame("standard_deviation")
)

print("\nMean transformer attention weights:")
display(
    predictions[
        attention_columns
    ]
    .mean()
    .to_frame("mean_attention")
)

print("\nMaximum routing-sum error:")
print(maximum_route_sum_error)

print("\nMaximum attention-sum error:")
print(maximum_attention_sum_error)

print("\nMaximum route/attention difference:")
print(maximum_route_attention_difference)

print("\nPrediction summary:")
display(
    predictions[
        [
            "label",
            "fake_prob",
            "uncertainty",
            "mean_branch_disagreement",
            "routing_entropy",
        ]
    ].describe()
)

print("\nAvailable DF40 metadata:")
print(available_metadata)

if "method" in predictions.columns:
    print("\nMean routing weights by manipulation method:")

    display(
        predictions.groupby(
            "method"
        )[route_columns].mean()
    )

print("\nRouting diagnostic rows:")
print(len(routing_table))

print(
    "\nPASS: adaptive transformer completed a full DF40 "
    "training, validation, checkpoint, and testing cycle."
)

print(
    "PASS: routing and attention diagnostics were saved "
    "without including the fingerprint controller as evidence."
)

RUNNING A-GRFN TRANSFORMER SMOKE TEST

Output directory:
/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_transformer_df40_smoke_20260730_220612

GRFN-CSF GRFN-only training and evaluation
Device: cuda
Output directory: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_transformer_df40_smoke_20260730_220612
Metadata CSV: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40/df40_manifest_all_clean.csv
Streams: rgb,fingerprint,jpeg,freq

Metadata loaded
Metadata CSV: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40/df40_manifest_all_clean.csv
Path column: frame_path
Label column: label
Split column: split
Group column: group_id
Usable frames: 292907
train: frames=235640, units=7843, real=48132, fake=187508
val: frames=28551, units=980, real=5997, fake=22554
test: frames=28716, units=981, real=5999, fake=22717
Saved label audit samples to: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_transformer_df40_smoke_20260730_22

Sample train: 100%|██████████| 40/40 [00:00<00:00, 65.26unit/s]


Sample train: frames=78, units=40, real=40, fake=38
Sample val: frames=39, units=20, real=20, fake=19


Sample test: 100%|██████████| 20/20 [00:00<00:00, 353.02unit/s]


Sample test: frames=39, units=20, real=20, fake=19

Balancing strategy
Train class counts [real, fake]: [40, 38]
Train imbalance ratio: 1.053
Use class-weighted loss: False
Use weighted sampler: False

Building dataloaders
Train batches: 19
Val batches: 10
Test batches: 10

Building model
Model mode: grfn
Loss mode: grfn
Active streams: rgb, fingerprint, jpeg, freq
Total params: 2,044,747
Trainable params: 2,044,747

Training


Predict test:   0%|          | 0/10 [00:00<?, ?it/s]

... (15 outputs left out)

,label,fake_prob,uncertainty,mean_branch_disagreement,routing_entropy
count,39.000000,39.000000,39.000000,39.000000,3.900000e+01
mean,0.487179,0.600428,0.664314,0.058464,9.999993e-01
std,0.506370,0.161662,0.038864,0.017829,4.743572e-07
min,0.000000,0.309304,0.580676,0.035208,9.999982e-01
25%,0.000000,0.447078,0.647996,0.042373,9.999990e-01
50%,0.000000,0.643512,0.670478,0.056014,9.999993e-01
75%,1.000000,0.710084,0.693740,0.070317,9.999997e-01
max,1.000000,0.791275,0.718583,0.101175,1.000000e+00



Available DF40 metadata:
['method', 'domain', 'top_level_folder', 'source_layout', 'relative_path']

Mean routing weights by manipulation method:


,route_rgb,route_jpeg,route_freq
method,,,
CollabDiff,0.333781,0.332758,0.333461
ddim,0.333663,0.332840,0.333497
fomm,0.333449,0.333028,0.333524
inswap,0.333509,0.332977,0.333515
real,0.333236,0.333106,0.333659
sadtalker,0.333517,0.332956,0.333527
sd2.1,0.333961,0.332838,0.333201
simswap,0.333590,0.332961,0.333449
wav2lip,0.333324,0.333189,0.333487



Routing diagnostic rows:
39

PASS: adaptive transformer completed a full DF40 training, validation, checkpoint, and testing cycle.
PASS: routing and attention diagnostics were saved without including the fingerprint controller as evidence.


In [41]:
from pathlib import Path
from datetime import datetime
from types import SimpleNamespace
import importlib.util

import torch
import torch.nn.functional as F


# ============================================================
# Import the already-patched script
# ============================================================

project_root = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution"
)

script_path = (
    project_root
    / "agrfn_adaptive_routing_df40.py"
)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

module_name = f"agrfn_prototype_retest_{timestamp}"

spec = importlib.util.spec_from_file_location(
    module_name,
    script_path,
)

module = importlib.util.module_from_spec(spec)

if spec.loader is None:
    raise RuntimeError("Could not create module loader.")

spec.loader.exec_module(module)


# ============================================================
# Confirm PrototypeBank was added
# ============================================================

if not hasattr(module, "PrototypeBank"):
    raise RuntimeError(
        "PrototypeBank was not found in the script. "
        "The previous patch may not have been saved."
    )


# ============================================================
# Dummy input
# ============================================================

batch_size = 4
image_size = 64

dummy_view = {
    "rgb": torch.randn(
        batch_size, 3, image_size, image_size
    ),
    "rec": torch.randn(
        batch_size, 3, image_size, image_size
    ),
    "den": torch.randn(
        batch_size, 3, image_size, image_size
    ),
    "jpeg": torch.randn(
        batch_size, 3, image_size, image_size
    ),
    "freq": torch.randn(
        batch_size, 1, image_size, image_size
    ),
    "noise": torch.randn(
        batch_size, 1, image_size, image_size
    ),
}

stream_names = [
    "rgb",
    "fingerprint",
    "jpeg",
    "freq",
]


# ============================================================
# Build prototype-enabled transformer model
# ============================================================

prototype_args = SimpleNamespace(
    fusion_mode="adaptive_transformer",
    feat_dim=128,
    controller_dim=128,
    routing_temperature=1.0,
    film_scale=0.10,
    router_heads=4,
    router_layers=1,
    router_dropout=0.10,
    stream_logit_residual_weight=0.25,
    use_prototypes=True,
    prototype_temperature=0.10,
)

model = module.build_model(
    prototype_args,
    stream_names,
)

model.train()

output = model(dummy_view)


# ============================================================
# Shape checks
# ============================================================

expected_shapes = {
    "logits": (batch_size, 2),
    "prototype_logits": (batch_size, 2),
    "prototype_probs": (batch_size, 2),
    "prototype_distances": (batch_size, 2),
    "prototype_margin": (batch_size,),
    "prototype_ambiguity": (batch_size,),
    "minimum_prototype_distance": (batch_size,),
    "unknown_suspiciousness": (batch_size,),
}

for key, expected_shape in expected_shapes.items():
    value = output[key]

    if value is None:
        raise RuntimeError(
            f"{key} was unexpectedly None."
        )

    actual_shape = tuple(value.shape)

    if actual_shape != expected_shape:
        raise RuntimeError(
            f"{key} shape mismatch. "
            f"Expected {expected_shape}, "
            f"received {actual_shape}."
        )


# ============================================================
# Numerical checks
# ============================================================

prototype_prob_sums = (
    output["prototype_probs"].sum(dim=1)
)

if not torch.allclose(
    prototype_prob_sums,
    torch.ones_like(prototype_prob_sums),
    atol=1e-6,
):
    raise RuntimeError(
        "Prototype probabilities do not sum to one."
    )

for key in expected_shapes:
    if not torch.isfinite(output[key]).all():
        raise RuntimeError(
            f"{key} contains NaN or infinity."
        )

if (output["prototype_distances"] < 0).any():
    raise RuntimeError(
        "Prototype distances contain negative values."
    )

prototype_ambiguity = output[
    "prototype_ambiguity"
]

ambiguity_in_range = (
    (prototype_ambiguity >= 0.0)
    & (prototype_ambiguity <= 1.0)
).all()

if not bool(ambiguity_in_range):
    raise RuntimeError(
        "Prototype ambiguity is outside [0, 1]."
    )

unknown_suspiciousness = output[
    "unknown_suspiciousness"
]

suspiciousness_in_range = (
    (unknown_suspiciousness >= 0.0)
    & (unknown_suspiciousness <= 1.0)
).all()

if not bool(suspiciousness_in_range):
    raise RuntimeError(
        "Unknown suspiciousness is outside [0, 1]."
    )


# ============================================================
# Prototype gradient test
# ============================================================

model.zero_grad(set_to_none=True)

output = model(dummy_view)

labels = torch.tensor(
    [0, 1, 0, 1],
    dtype=torch.long,
)

sample_indices = torch.arange(
    batch_size,
    device=labels.device,
)

target_distances = output[
    "prototype_distances"
][sample_indices, labels]

prototype_loss = (
    F.cross_entropy(
        output["prototype_logits"],
        labels,
    )
    + target_distances.mean()
)

classifier_loss = F.cross_entropy(
    output["logits"],
    labels,
)

total_loss = (
    classifier_loss
    + 0.10 * prototype_loss
)

total_loss.backward()

prototype_gradient = (
    model.prototype_bank.prototypes.grad
)

if prototype_gradient is None:
    raise RuntimeError(
        "Prototype parameters received no gradient."
    )

if not torch.isfinite(
    prototype_gradient
).all():
    raise RuntimeError(
        "Prototype gradients contain NaN or infinity."
    )

if float(
    prototype_gradient.abs().sum()
) == 0.0:
    raise RuntimeError(
        "Prototype gradient is entirely zero."
    )

branch_head_gradients = {
    name: any(
        parameter.grad is not None
        for parameter in head.parameters()
    )
    for name, head in model.stream_heads.items()
}

if any(branch_head_gradients.values()):
    raise RuntimeError(
        "Prototype/classifier losses unexpectedly depend "
        f"on branch heads: {branch_head_gradients}"
    )


# ============================================================
# Confirm disabled mode remains unchanged
# ============================================================

disabled_args = SimpleNamespace(
    fusion_mode="adaptive_scalar",
    feat_dim=128,
    controller_dim=128,
    routing_temperature=1.0,
    film_scale=0.10,
    router_heads=4,
    router_layers=1,
    router_dropout=0.10,
    stream_logit_residual_weight=0.25,
    use_prototypes=False,
    prototype_temperature=0.10,
)

disabled_model = module.build_model(
    disabled_args,
    stream_names,
)

disabled_model.eval()

with torch.no_grad():
    disabled_output = disabled_model(
        dummy_view
    )

if hasattr(disabled_model, "prototype_bank"):
    raise RuntimeError(
        "Prototype-disabled model unexpectedly "
        "created a prototype bank."
    )

prototype_output_keys = [
    "prototype_logits",
    "prototype_probs",
    "prototype_distances",
    "prototype_margin",
    "prototype_ambiguity",
    "minimum_prototype_distance",
    "unknown_suspiciousness",
]

for key in prototype_output_keys:
    if disabled_output[key] is not None:
        raise RuntimeError(
            f"{key} should be None when prototypes "
            "are disabled."
        )


# ============================================================
# Final report
# ============================================================

print("\n" + "=" * 80)
print("A-GRFN PROTOTYPE MODULE TEST PASSED")
print("=" * 80)

print("\nPrototype shape:")
print(
    tuple(
        model.prototype_bank.prototypes.shape
    )
)

print("\nPrototype probabilities:")
print(
    output["prototype_probs"]
    .detach()
    .cpu()
)

print("\nPrototype distances:")
print(
    output["prototype_distances"]
    .detach()
    .cpu()
)

print("\nPrototype margins:")
print(
    output["prototype_margin"]
    .detach()
    .cpu()
)

print("\nPrototype ambiguity:")
print(
    output["prototype_ambiguity"]
    .detach()
    .cpu()
)

print("\nUnknown suspiciousness:")
print(
    output["unknown_suspiciousness"]
    .detach()
    .cpu()
)

print("\nPrototype gradient magnitude:")
print(
    float(
        prototype_gradient.abs().sum()
    )
)

print("\nAuxiliary branch-head gradients:")
print(branch_head_gradients)

print(
    "\nPASS: real/fake prototypes produce "
    "distance and ambiguity diagnostics."
)

print(
    "PASS: prototype-disabled models retain "
    "their original architecture."
)

print(
    "PASS: final classifier logits remain separate "
    "from prototype logits at this stage."
)


A-GRFN PROTOTYPE MODULE TEST PASSED

Prototype shape:
(2, 128)

Prototype probabilities:
tensor([[0.9128, 0.0872],
        [0.9232, 0.0768],
        [0.9168, 0.0832],
        [0.9636, 0.0364]])

Prototype distances:
tensor([[0.8098, 1.0447],
        [0.7487, 0.9973],
        [0.7756, 1.0155],
        [0.7847, 1.1124]])

Prototype margins:
tensor([0.2349, 0.2487, 0.2399, 0.3277])

Prototype ambiguity:
tensor([0.4269, 0.3907, 0.4135, 0.2254])

Unknown suspiciousness:
tensor([0.4049, 0.3743, 0.3878, 0.3923])

Prototype gradient magnitude:
5.745031356811523

Auxiliary branch-head gradients:
{'rgb': False, 'fingerprint': False, 'jpeg': False, 'freq': False}

PASS: real/fake prototypes produce distance and ambiguity diagnostics.
PASS: prototype-disabled models retain their original architecture.
PASS: final classifier logits remain separate from prototype logits at this stage.


In [42]:
from pathlib import Path
from datetime import datetime
from types import SimpleNamespace
import ast
import shutil
import py_compile
import importlib.util

import torch


# ============================================================
# Paths and backup
# ============================================================

project_root = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution"
)

script_path = (
    project_root
    / "agrfn_adaptive_routing_df40.py"
)

if not script_path.exists():
    raise FileNotFoundError(script_path)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

backup_path = script_path.with_name(
    f"{script_path.stem}_before_prototype_loss_{timestamp}.py"
)

shutil.copy2(script_path, backup_path)

print("Backup created:")
print(backup_path)

text = script_path.read_text()
tree = ast.parse(text)


# ============================================================
# Locate grfn_loss()
# ============================================================

grfn_loss_node = next(
    (
        node
        for node in tree.body
        if isinstance(node, ast.FunctionDef)
        and node.name == "grfn_loss"
    ),
    None,
)

if grfn_loss_node is None:
    raise RuntimeError(
        "Could not find grfn_loss()."
    )


# ============================================================
# Replace grfn_loss() with prototype-aware version
# ============================================================

new_grfn_loss = r'''def grfn_loss(
    model,
    batch,
    args,
    device,
    class_weights=None,
):
    """Compute GRFN or adaptive A-GRFN training losses."""

    view_a = move_view_to_device(
        batch["a"],
        device,
    )

    view_b = move_view_to_device(
        batch["b"],
        device,
    )

    y = batch["label"].to(
        device,
        non_blocking=True,
    )

    out_a = model(view_a)
    out_b = model(view_b)

    # --------------------------------------------------------
    # Main final-classification loss
    # --------------------------------------------------------

    cls_loss = 0.5 * (
        F.cross_entropy(
            out_a["logits"],
            y,
            weight=class_weights,
        )
        +
        F.cross_entropy(
            out_b["logits"],
            y,
            weight=class_weights,
        )
    )

    loss_mode = getattr(
        args,
        "loss_mode",
        "grfn",
    )

    if loss_mode == "auto":
        loss_mode = (
            "ce"
            if getattr(
                args,
                "model_mode",
                "grfn",
            ) == "simple"
            else "grfn"
        )

    zero = torch.zeros(
        (),
        dtype=cls_loss.dtype,
        device=device,
    )

    if loss_mode == "ce":
        logs = {
            "total": float(
                cls_loss.detach().cpu()
            ),
            "cls": float(
                cls_loss.detach().cpu()
            ),
            "stream": 0.0,
            "pred_cons": 0.0,
            "feat_cons": 0.0,
            "supcon": 0.0,
            "route_balance": 0.0,
            "route_cons": 0.0,
            "aug_cons": 0.0,
            "proto": 0.0,
            "proto_ce": 0.0,
            "proto_pull": 0.0,
            "unc": 0.0,
        }

        return cls_loss, logs

    # --------------------------------------------------------
    # Auxiliary branch classification
    # --------------------------------------------------------

    stream_logits_a = out_a[
        "stream_logits"
    ]

    stream_logits_b = out_b[
        "stream_logits"
    ]

    batch_size, num_streams, num_classes = (
        stream_logits_a.shape
    )

    if num_classes != 2:
        raise ValueError(
            "Expected two classes: real and fake."
        )

    y_repeated = (
        y.unsqueeze(1)
        .repeat(1, num_streams)
        .reshape(-1)
    )

    stream_loss_a = F.cross_entropy(
        stream_logits_a.reshape(
            batch_size * num_streams,
            num_classes,
        ),
        y_repeated,
        weight=class_weights,
    )

    stream_loss_b = F.cross_entropy(
        stream_logits_b.reshape(
            batch_size * num_streams,
            num_classes,
        ),
        y_repeated,
        weight=class_weights,
    )

    stream_loss = 0.5 * (
        stream_loss_a
        + stream_loss_b
    )

    # --------------------------------------------------------
    # Prediction consistency
    # --------------------------------------------------------

    final_prediction_consistency = 0.5 * (
        soft_kl(
            out_a["logits"],
            out_b["logits"],
        )
        +
        soft_kl(
            out_b["logits"],
            out_a["logits"],
        )
    )

    stream_prediction_consistency = 0.5 * (
        F.mse_loss(
            out_a["stream_probs"],
            out_b["stream_probs"].detach(),
        )
        +
        F.mse_loss(
            out_b["stream_probs"],
            out_a["stream_probs"].detach(),
        )
    )

    pred_cons = (
        final_prediction_consistency
        + stream_prediction_consistency
    )

    # --------------------------------------------------------
    # Fingerprint consistency
    # --------------------------------------------------------

    feat_cons = (
        1.0
        - F.cosine_similarity(
            out_a["fingerprint"],
            out_b["fingerprint"],
            dim=1,
        ).mean()
    )

    # --------------------------------------------------------
    # Supervised contrastive fingerprint loss
    # --------------------------------------------------------

    fingerprint_all = torch.cat(
        [
            out_a["fingerprint"],
            out_b["fingerprint"],
        ],
        dim=0,
    )

    labels_all = torch.cat(
        [y, y],
        dim=0,
    )

    supcon = supervised_contrastive_loss(
        fingerprint_all,
        labels_all,
    )

    # --------------------------------------------------------
    # Adaptive routing losses
    # --------------------------------------------------------

    fusion_mode = getattr(
        args,
        "fusion_mode",
        "late",
    )

    is_adaptive = fusion_mode in {
        "adaptive_scalar",
        "adaptive_film",
        "adaptive_transformer",
    }

    route_balance = zero
    route_cons = zero
    aug_cons = feat_cons

    if is_adaptive:
        if (
            "routing_weights" not in out_a
            or "routing_weights" not in out_b
        ):
            raise KeyError(
                "Adaptive fusion requires "
                "routing_weights."
            )

        routing_a = out_a[
            "routing_weights"
        ]

        routing_b = out_b[
            "routing_weights"
        ]

        if routing_a.shape != routing_b.shape:
            raise ValueError(
                "Routing shapes differ between views: "
                f"{tuple(routing_a.shape)} and "
                f"{tuple(routing_b.shape)}."
            )

        route_balance = 0.5 * (
            routing_balance_loss(
                routing_a
            )
            +
            routing_balance_loss(
                routing_b
            )
        )

        route_cons = 0.5 * (
            F.mse_loss(
                routing_a,
                routing_b.detach(),
            )
            +
            F.mse_loss(
                routing_b,
                routing_a.detach(),
            )
        )

        aug_cons = 0.5 * (
            feat_cons
            + route_cons
        )

    # --------------------------------------------------------
    # Prototype proximity loss
    # --------------------------------------------------------

    proto_loss = zero
    proto_ce = zero
    proto_pull = zero

    prototypes_enabled = (
        is_adaptive
        and bool(
            getattr(
                args,
                "use_prototypes",
                False,
            )
        )
    )

    if prototypes_enabled:
        required_prototype_outputs = [
            "prototype_logits",
            "prototype_distances",
        ]

        for output_name, output in [
            ("view_a", out_a),
            ("view_b", out_b),
        ]:
            for key in required_prototype_outputs:
                if (
                    key not in output
                    or output[key] is None
                ):
                    raise KeyError(
                        f"{output_name} is missing "
                        f"prototype output '{key}'."
                    )

        proto_ce = 0.5 * (
            F.cross_entropy(
                out_a["prototype_logits"],
                y,
                weight=class_weights,
            )
            +
            F.cross_entropy(
                out_b["prototype_logits"],
                y,
                weight=class_weights,
            )
        )

        sample_indices = torch.arange(
            y.shape[0],
            device=y.device,
        )

        target_distance_a = (
            out_a["prototype_distances"][
                sample_indices,
                y,
            ]
        )

        target_distance_b = (
            out_b["prototype_distances"][
                sample_indices,
                y,
            ]
        )

        proto_pull = 0.5 * (
            target_distance_a.mean()
            + target_distance_b.mean()
        )

        proto_loss = (
            proto_ce
            + proto_pull
        )

    # --------------------------------------------------------
    # Total objective
    # --------------------------------------------------------

    if is_adaptive:
        lambda_aux = float(
            getattr(
                args,
                "lambda_aux",
                getattr(
                    args,
                    "lambda_stream",
                    0.05,
                ),
            )
        )

        lambda_route = float(
            getattr(
                args,
                "lambda_route",
                0.01,
            )
        )

        lambda_cons = float(
            getattr(
                args,
                "lambda_cons",
                getattr(
                    args,
                    "lambda_feat_cons",
                    0.02,
                ),
            )
        )

        lambda_proto = float(
            getattr(
                args,
                "lambda_proto",
                0.10,
            )
        )

        total = (
            cls_loss
            + lambda_aux * stream_loss
            + args.lambda_pred_cons
            * pred_cons
            + lambda_cons
            * aug_cons
            + lambda_route
            * route_balance
            + args.lambda_supcon
            * supcon
            + lambda_proto
            * proto_loss
        )

    else:
        # Preserve the original late-fusion objective.
        total = (
            cls_loss
            + args.lambda_stream
            * stream_loss
            + args.lambda_pred_cons
            * pred_cons
            + args.lambda_feat_cons
            * feat_cons
            + args.lambda_supcon
            * supcon
        )

    logs = {
        "total": float(
            total.detach().cpu()
        ),
        "cls": float(
            cls_loss.detach().cpu()
        ),
        "stream": float(
            stream_loss.detach().cpu()
        ),
        "pred_cons": float(
            pred_cons.detach().cpu()
        ),
        "feat_cons": float(
            feat_cons.detach().cpu()
        ),
        "supcon": float(
            supcon.detach().cpu()
        ),
        "route_balance": float(
            route_balance.detach().cpu()
        ),
        "route_cons": float(
            route_cons.detach().cpu()
        ),
        "aug_cons": float(
            aug_cons.detach().cpu()
        ),
        "proto": float(
            proto_loss.detach().cpu()
        ),
        "proto_ce": float(
            proto_ce.detach().cpu()
        ),
        "proto_pull": float(
            proto_pull.detach().cpu()
        ),
        "unc": 0.0,
    }

    return total, logs
'''

lines = text.splitlines(keepends=True)

lines = (
    lines[:grfn_loss_node.lineno - 1]
    + new_grfn_loss.splitlines(
        keepends=True
    )
    + ["\n\n"]
    + lines[grfn_loss_node.end_lineno:]
)

text = "".join(lines)

print("\nUpdated grfn_loss() with prototype learning.")


# ============================================================
# Update train_one_epoch logging
# ============================================================

tree = ast.parse(text)

train_node = next(
    (
        node
        for node in tree.body
        if isinstance(node, ast.FunctionDef)
        and node.name == "train_one_epoch"
    ),
    None,
)

if train_node is None:
    raise RuntimeError(
        "Could not find train_one_epoch()."
    )

running_assignment = next(
    (
        node
        for node in ast.walk(train_node)
        if isinstance(node, ast.Assign)
        and any(
            isinstance(target, ast.Name)
            and target.id == "running"
            for target in node.targets
        )
    ),
    None,
)

if running_assignment is None:
    raise RuntimeError(
        "Could not find the running-loss dictionary."
    )

new_running_dictionary = '''    running = {
        "total": [],
        "cls": [],
        "stream": [],
        "pred_cons": [],
        "feat_cons": [],
        "supcon": [],
        "route_balance": [],
        "route_cons": [],
        "aug_cons": [],
        "proto": [],
        "proto_ce": [],
        "proto_pull": [],
        "unc": [],
    }
'''

lines = text.splitlines(keepends=True)

lines = (
    lines[:running_assignment.lineno - 1]
    + new_running_dictionary.splitlines(
        keepends=True
    )
    + lines[running_assignment.end_lineno:]
)

text = "".join(lines)

print(
    "Updated training-history prototype logging."
)


# ============================================================
# Save and compile
# ============================================================

script_path.write_text(text)

py_compile.compile(
    str(script_path),
    doraise=True,
)

print("\nPASS: Python compilation succeeded.")


# ============================================================
# Import patched module
# ============================================================

module_name = (
    "agrfn_prototype_loss_test_"
    + timestamp
)

spec = importlib.util.spec_from_file_location(
    module_name,
    script_path,
)

module = importlib.util.module_from_spec(
    spec
)

if spec.loader is None:
    raise RuntimeError(
        "Could not create module loader."
    )

spec.loader.exec_module(module)


# ============================================================
# Dummy batch
# ============================================================

device = torch.device("cpu")
batch_size = 2
image_size = 64

def make_dummy_view():
    return {
        "rgb": torch.randn(
            batch_size,
            3,
            image_size,
            image_size,
        ),
        "rec": torch.randn(
            batch_size,
            3,
            image_size,
            image_size,
        ),
        "den": torch.randn(
            batch_size,
            3,
            image_size,
            image_size,
        ),
        "jpeg": torch.randn(
            batch_size,
            3,
            image_size,
            image_size,
        ),
        "freq": torch.randn(
            batch_size,
            1,
            image_size,
            image_size,
        ),
        "noise": torch.randn(
            batch_size,
            1,
            image_size,
            image_size,
        ),
    }

dummy_batch = {
    "a": make_dummy_view(),
    "b": make_dummy_view(),
    "label": torch.tensor(
        [0, 1],
        dtype=torch.long,
    ),
}

stream_names = [
    "rgb",
    "fingerprint",
    "jpeg",
    "freq",
]


# ============================================================
# Prototype-enabled loss test
# ============================================================

prototype_args = SimpleNamespace(
    fusion_mode="adaptive_transformer",

    feat_dim=128,
    controller_dim=128,

    routing_temperature=1.0,
    film_scale=0.10,

    router_heads=4,
    router_layers=1,
    router_dropout=0.10,

    stream_logit_residual_weight=0.25,

    use_prototypes=True,
    prototype_temperature=0.10,

    loss_mode="grfn",
    model_mode="grfn",

    lambda_stream=0.05,
    lambda_aux=0.05,

    lambda_pred_cons=0.02,
    lambda_feat_cons=0.02,
    lambda_cons=0.02,

    lambda_route=0.01,
    lambda_proto=0.10,
    lambda_supcon=0.00,
)

prototype_model = module.build_model(
    prototype_args,
    stream_names,
).to(device)

prototype_model.train()
prototype_model.zero_grad(
    set_to_none=True
)

prototype_loss, prototype_logs = (
    module.grfn_loss(
        prototype_model,
        dummy_batch,
        prototype_args,
        device,
        class_weights=None,
    )
)

if not torch.isfinite(
    prototype_loss
):
    raise RuntimeError(
        "Prototype-enabled loss is not finite."
    )

if prototype_logs["proto"] <= 0:
    raise RuntimeError(
        "Prototype loss was not activated."
    )

if prototype_logs["proto_ce"] <= 0:
    raise RuntimeError(
        "Prototype classification loss "
        "was not activated."
    )

if prototype_logs["proto_pull"] < 0:
    raise RuntimeError(
        "Prototype pull loss is negative."
    )

prototype_loss.backward()

prototype_gradient = (
    prototype_model
    .prototype_bank
    .prototypes
    .grad
)

if prototype_gradient is None:
    raise RuntimeError(
        "Prototype parameters received no gradient."
    )

if not torch.isfinite(
    prototype_gradient
).all():
    raise RuntimeError(
        "Prototype gradient contains NaN or infinity."
    )

if float(
    prototype_gradient.abs().sum()
) == 0.0:
    raise RuntimeError(
        "Prototype gradient is entirely zero."
    )

final_head_gradient = (
    prototype_model
    .final_head
    .weight
    .grad
)

if final_head_gradient is None:
    raise RuntimeError(
        "Final classifier received no gradient."
    )


# ============================================================
# Prototype-disabled compatibility test
# ============================================================

disabled_args = SimpleNamespace(
    fusion_mode="adaptive_scalar",

    feat_dim=128,
    controller_dim=128,

    routing_temperature=1.0,
    film_scale=0.10,

    router_heads=4,
    router_layers=1,
    router_dropout=0.10,

    stream_logit_residual_weight=0.25,

    use_prototypes=False,
    prototype_temperature=0.10,

    loss_mode="grfn",
    model_mode="grfn",

    lambda_stream=0.05,
    lambda_aux=0.05,

    lambda_pred_cons=0.02,
    lambda_feat_cons=0.02,
    lambda_cons=0.02,

    lambda_route=0.01,
    lambda_proto=0.10,
    lambda_supcon=0.00,
)

disabled_model = module.build_model(
    disabled_args,
    stream_names,
).to(device)

disabled_model.train()

disabled_loss, disabled_logs = (
    module.grfn_loss(
        disabled_model,
        dummy_batch,
        disabled_args,
        device,
        class_weights=None,
    )
)

if not torch.isfinite(
    disabled_loss
):
    raise RuntimeError(
        "Prototype-disabled loss is not finite."
    )

if disabled_logs["proto"] != 0.0:
    raise RuntimeError(
        "Prototype-disabled run received "
        "a prototype loss."
    )

if disabled_logs["proto_ce"] != 0.0:
    raise RuntimeError(
        "Prototype-disabled run received "
        "prototype CE loss."
    )

if disabled_logs["proto_pull"] != 0.0:
    raise RuntimeError(
        "Prototype-disabled run received "
        "prototype pull loss."
    )


# ============================================================
# Final report
# ============================================================

print("\n" + "=" * 80)
print("A-GRFN PROTOTYPE LOSS TEST PASSED")
print("=" * 80)

print("\nPrototype-enabled total loss:")
print(
    float(
        prototype_loss.detach()
    )
)

print("\nPrototype loss components:")

for key in [
    "proto",
    "proto_ce",
    "proto_pull",
]:
    print(
        f"{key:12s}: "
        f"{prototype_logs[key]:.6f}"
    )

print("\nPrototype gradient magnitude:")
print(
    float(
        prototype_gradient
        .abs()
        .sum()
    )
)

print("\nPrototype-disabled components:")
print(
    "proto =",
    disabled_logs["proto"],
)
print(
    "proto_ce =",
    disabled_logs["proto_ce"],
)
print(
    "proto_pull =",
    disabled_logs["proto_pull"],
)

print(
    "\nPASS: prototype learning is now connected "
    "through --lambda_proto."
)

print(
    "PASS: prototype-disabled training retains "
    "zero prototype loss."
)

Backup created:
/home/jovyan/Deepfake_Low Resource_Low Resolution/agrfn_adaptive_routing_df40_before_prototype_loss_20260730_221021.py

Updated grfn_loss() with prototype learning.
Updated training-history prototype logging.

PASS: Python compilation succeeded.

A-GRFN PROTOTYPE LOSS TEST PASSED

Prototype-enabled total loss:
0.987717866897583

Prototype loss components:
proto       : 2.180903
proto_ce    : 1.194865
proto_pull  : 0.986038

Prototype gradient magnitude:
5.291346073150635

Prototype-disabled components:
proto = 0.0
proto_ce = 0.0
proto_pull = 0.0

PASS: prototype learning is now connected through --lambda_proto.
PASS: prototype-disabled training retains zero prototype loss.


In [43]:
from pathlib import Path
from datetime import datetime
import subprocess
import sys

import numpy as np
import pandas as pd


# ============================================================
# Paths
# ============================================================

project_root = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution"
)

script_path = (
    project_root
    / "agrfn_adaptive_routing_df40.py"
)

manifest_path = (
    project_root
    / "outputs_df40"
    / "df40_manifest_all_clean.csv"
)

if not script_path.exists():
    raise FileNotFoundError(script_path)

if not manifest_path.exists():
    raise FileNotFoundError(manifest_path)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

out_dir = (
    project_root
    / f"outputs_agrfn_transformer_proto_smoke_{timestamp}"
)


# ============================================================
# Run one-epoch prototype smoke training
# ============================================================

command = [
    sys.executable,
    str(script_path),

    "--metadata_csv",
    str(manifest_path),

    "--out_dir",
    str(out_dir),

    "--dataset_name",
    "DF40",

    "--fusion_mode",
    "adaptive_transformer",

    "--streams",
    "rgb,fingerprint,jpeg,freq",

    "--feat_dim",
    "128",

    "--controller_dim",
    "128",

    "--routing_temperature",
    "1.0",

    "--film_scale",
    "0.10",

    "--router_heads",
    "4",

    "--router_layers",
    "1",

    "--router_dropout",
    "0.10",

    "--use_prototypes",

    "--prototype_temperature",
    "0.10",

    "--frames_per_video",
    "2",

    "--limit_train_units",
    "40",

    "--limit_val_units",
    "20",

    "--limit_test_units",
    "20",

    "--balanced_unit_limits",

    "--image_size",
    "224",

    "--batch_size",
    "4",

    "--num_workers",
    "0",

    "--epochs",
    "1",

    "--lr",
    "1e-4",

    "--lambda_aux",
    "0.05",

    "--lambda_route",
    "0.01",

    "--lambda_cons",
    "0.02",

    "--lambda_pred_cons",
    "0.02",

    "--lambda_proto",
    "0.10",

    "--balance_strategy",
    "auto",

    "--decision_threshold",
    "0.5",

    "--score_direction",
    "normal",

    "--amp",
]

print("=" * 80)
print("RUNNING A-GRFN PROTOTYPE SMOKE TEST")
print("=" * 80)

print("\nOutput directory:")
print(out_dir)

result = subprocess.run(
    command,
    cwd=str(project_root),
)

if result.returncode != 0:
    raise RuntimeError(
        "Prototype smoke test failed with return code "
        f"{result.returncode}."
    )


# ============================================================
# Validate output files
# ============================================================

required_files = [
    "best_checkpoint.pt",
    "latest_checkpoint.pt",
    "training_history.csv",
    "test_frame_predictions.csv",
    "test_video_predictions.csv",
    "test_metrics.csv",
    "test_inference_time.csv",
    "experiment_summary.csv",
    "branch_predictions.csv",
    "branch_uncertainties.csv",
    "routing_weights.csv",
    "disagreement_matrices.csv",
    "uncertainty_breakdown.csv",
    "prototype_distances.csv",
]

missing_files = [
    filename
    for filename in required_files
    if not (out_dir / filename).exists()
]

if missing_files:
    raise RuntimeError(
        "Missing prototype output files:\n"
        + "\n".join(missing_files)
    )


# ============================================================
# Load results
# ============================================================

history = pd.read_csv(
    out_dir / "training_history.csv"
)

predictions = pd.read_csv(
    out_dir / "test_frame_predictions.csv"
)

prototype_table = pd.read_csv(
    out_dir / "prototype_distances.csv"
)

metrics = pd.read_csv(
    out_dir / "test_metrics.csv"
)

inference_table = pd.read_csv(
    out_dir / "test_inference_time.csv"
)


# ============================================================
# Validate prototype training losses
# ============================================================

required_history_columns = [
    "train_total",
    "train_cls",
    "train_proto",
    "train_proto_ce",
    "train_proto_pull",
]

missing_history_columns = [
    column
    for column in required_history_columns
    if column not in history.columns
]

if missing_history_columns:
    raise RuntimeError(
        "Missing prototype training-history columns: "
        + ", ".join(missing_history_columns)
    )

history_values = history[
    required_history_columns
].to_numpy(dtype=float)

if not np.isfinite(history_values).all():
    raise RuntimeError(
        "Prototype training history contains NaN or infinity."
    )

if float(history["train_proto"].iloc[-1]) <= 0:
    raise RuntimeError(
        "Prototype loss was not active during training."
    )

if float(history["train_proto_ce"].iloc[-1]) <= 0:
    raise RuntimeError(
        "Prototype classification loss was not active."
    )

if float(history["train_proto_pull"].iloc[-1]) < 0:
    raise RuntimeError(
        "Prototype pull loss is negative."
    )


# ============================================================
# Validate prediction diagnostics
# ============================================================

prototype_columns = [
    "distance_real",
    "distance_fake",
    "minimum_prototype_distance",
    "prototype_margin",
    "prototype_ambiguity",
    "unknown_suspiciousness",
]

missing_prototype_columns = [
    column
    for column in prototype_columns
    if column not in predictions.columns
]

if missing_prototype_columns:
    raise RuntimeError(
        "Missing prototype prediction columns: "
        + ", ".join(missing_prototype_columns)
    )

prototype_values = predictions[
    prototype_columns
].to_numpy(dtype=float)

if not np.isfinite(prototype_values).all():
    raise RuntimeError(
        "Prototype diagnostics contain NaN or infinity."
    )

if (
    predictions[
        [
            "distance_real",
            "distance_fake",
            "minimum_prototype_distance",
        ]
    ].to_numpy(dtype=float) < 0
).any():
    raise RuntimeError(
        "Prototype distances contain negative values."
    )

if not predictions[
    "prototype_ambiguity"
].between(0.0, 1.0).all():
    raise RuntimeError(
        "Prototype ambiguity contains values outside [0, 1]."
    )

if not predictions[
    "unknown_suspiciousness"
].between(0.0, 1.0).all():
    raise RuntimeError(
        "Unknown suspiciousness contains values outside [0, 1]."
    )


# ============================================================
# Validate distance relationships
# ============================================================

expected_minimum_distance = predictions[
    [
        "distance_real",
        "distance_fake",
    ]
].min(axis=1)

maximum_minimum_distance_error = float(
    np.abs(
        predictions["minimum_prototype_distance"]
        - expected_minimum_distance
    ).max()
)

if maximum_minimum_distance_error > 1e-6:
    raise RuntimeError(
        "minimum_prototype_distance does not match "
        "the nearest class prototype. Maximum error: "
        f"{maximum_minimum_distance_error}"
    )

expected_margin = np.abs(
    predictions["distance_real"]
    - predictions["distance_fake"]
)

maximum_margin_error = float(
    np.abs(
        predictions["prototype_margin"]
        - expected_margin
    ).max()
)

if maximum_margin_error > 1e-6:
    raise RuntimeError(
        "prototype_margin does not match the distance gap. "
        f"Maximum error: {maximum_margin_error}"
    )


# ============================================================
# Validate dedicated prototype CSV
# ============================================================

if len(prototype_table) != len(predictions):
    raise RuntimeError(
        "prototype_distances.csv and frame predictions "
        "have different row counts."
    )

missing_table_columns = [
    column
    for column in prototype_columns
    if column not in prototype_table.columns
]

if missing_table_columns:
    raise RuntimeError(
        "prototype_distances.csv is missing columns: "
        + ", ".join(missing_table_columns)
    )


# ============================================================
# Final report
# ============================================================

print("\n" + "=" * 80)
print("A-GRFN PROTOTYPE SMOKE TEST PASSED")
print("=" * 80)

print("\nOutput directory:")
print(out_dir)

print("\nPrototype training losses:")
display(
    history[
        required_history_columns
    ]
)

print("\nTest metrics:")
display(metrics)

print("\nInference timing:")
display(inference_table)

print("\nPrototype diagnostic summary:")
display(
    predictions[
        prototype_columns
    ].describe()
)

print("\nMean prototype diagnostics:")
display(
    predictions[
        prototype_columns
    ]
    .mean()
    .to_frame("mean")
)

print("\nMaximum minimum-distance error:")
print(maximum_minimum_distance_error)

print("\nMaximum prototype-margin error:")
print(maximum_margin_error)

print("\nPrototype diagnostic rows:")
print(len(prototype_table))

if "method" in predictions.columns:
    print("\nPrototype diagnostics by manipulation method:")

    display(
        predictions.groupby("method")[
            [
                "distance_real",
                "distance_fake",
                "prototype_margin",
                "prototype_ambiguity",
                "unknown_suspiciousness",
            ]
        ].mean()
    )

print(
    "\nPASS: prototype learning completed a full "
    "training, validation, checkpoint, and testing cycle."
)

print(
    "PASS: real/fake distances, margins, ambiguity, "
    "and unknown-suspiciousness were logged correctly."
)

RUNNING A-GRFN PROTOTYPE SMOKE TEST

Output directory:
/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_transformer_proto_smoke_20260730_221237

GRFN-CSF GRFN-only training and evaluation
Device: cuda
Output directory: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_transformer_proto_smoke_20260730_221237
Metadata CSV: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40/df40_manifest_all_clean.csv
Streams: rgb,fingerprint,jpeg,freq

Metadata loaded
Metadata CSV: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40/df40_manifest_all_clean.csv
Path column: frame_path
Label column: label
Split column: split
Group column: group_id
Usable frames: 292907
train: frames=235640, units=7843, real=48132, fake=187508
val: frames=28551, units=980, real=5997, fake=22554
test: frames=28716, units=981, real=5999, fake=22717
Saved label audit samples to: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_transformer_proto_smoke_20260730_2

Sample train: 100%|██████████| 40/40 [00:00<00:00, 65.60unit/s]


Sample train: frames=78, units=40, real=40, fake=38
Sample val: frames=39, units=20, real=20, fake=19


Sample test: 100%|██████████| 20/20 [00:00<00:00, 353.91unit/s]


Sample test: frames=39, units=20, real=20, fake=19

Balancing strategy
Train class counts [real, fake]: [40, 38]
Train imbalance ratio: 1.053
Use class-weighted loss: False
Use weighted sampler: False

Building dataloaders
Train batches: 19
Val batches: 10
Test batches: 10

Building model
Model mode: grfn
Loss mode: grfn
Active streams: rgb, fingerprint, jpeg, freq
Total params: 2,045,003
Trainable params: 2,045,003

Training


Predict test:   0%|          | 0/10 [00:00<?, ?it/s]

... (11 outputs left out)

,mean
distance_real,0.873036
distance_fake,0.916278
minimum_prototype_distance,0.872225
prototype_margin,0.044863
prototype_ambiguity,0.954591
unknown_suspiciousness,0.436113



Maximum minimum-distance error:
0.0

Maximum prototype-margin error:
1.97758476261356e-16

Prototype diagnostic rows:
39

Prototype diagnostics by manipulation method:


,distance_real,distance_fake,prototype_margin,prototype_ambiguity,unknown_suspiciousness
method,,,,,
CollabDiff,0.864664,0.928816,0.064152,0.929433,0.432332
ddim,0.852319,0.874517,0.027373,0.982486,0.424866
fomm,0.901140,0.924029,0.033522,0.971310,0.447912
inswap,0.857820,0.926013,0.068193,0.920476,0.428910
real,0.880627,0.926392,0.045765,0.955880,0.440314
sadtalker,0.885581,0.896176,0.010595,0.997838,0.442791
sd2.1,0.842000,0.941527,0.099527,0.839907,0.421000
simswap,0.859579,0.883787,0.024208,0.989476,0.429789
wav2lip,0.833696,0.894499,0.060802,0.935727,0.416848



PASS: prototype learning completed a full training, validation, checkpoint, and testing cycle.
PASS: real/fake distances, margins, ambiguity, and unknown-suspiciousness were logged correctly.


In [44]:
from pathlib import Path
from datetime import datetime
from types import SimpleNamespace
import ast
import shutil
import py_compile
import importlib.util

import torch
import torch.nn.functional as F


# ============================================================
# Paths and backup
# ============================================================

project_root = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution"
)

script_path = (
    project_root
    / "agrfn_adaptive_routing_df40.py"
)

if not script_path.exists():
    raise FileNotFoundError(script_path)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

backup_path = script_path.with_name(
    f"{script_path.stem}_before_uncertainty_head_{timestamp}.py"
)

shutil.copy2(script_path, backup_path)

print("Backup created:")
print(backup_path)

text = script_path.read_text()


# ============================================================
# Add EvidenceUncertaintyHead
# ============================================================

tree = ast.parse(text)

top_level_names = {
    node.name
    for node in tree.body
    if isinstance(
        node,
        (ast.ClassDef, ast.FunctionDef),
    )
}

if "EvidenceUncertaintyHead" not in top_level_names:
    prototype_bank_node = next(
        (
            node
            for node in tree.body
            if isinstance(node, ast.ClassDef)
            and node.name == "PrototypeBank"
        ),
        None,
    )

    if prototype_bank_node is None:
        raise RuntimeError(
            "Could not find PrototypeBank."
        )

    uncertainty_head_code = r'''

class EvidenceUncertaintyHead(nn.Module):
    """Predict sample-level uncertainty from routed evidence."""

    def __init__(
        self,
        feat_dim=128,
        hidden_dim=None,
        dropout=0.10,
    ):
        super().__init__()

        if feat_dim <= 0:
            raise ValueError(
                "feat_dim must be positive."
            )

        if hidden_dim is None:
            hidden_dim = feat_dim

        if hidden_dim <= 0:
            raise ValueError(
                "hidden_dim must be positive."
            )

        if not 0.0 <= dropout < 1.0:
            raise ValueError(
                "dropout must be in [0, 1)."
            )

        self.feat_dim = int(feat_dim)
        self.hidden_dim = int(hidden_dim)

        # Five scalar diagnostics are concatenated with
        # the routed joint embedding.
        self.num_diagnostics = 5

        self.network = nn.Sequential(
            nn.Linear(
                self.feat_dim
                + self.num_diagnostics,
                self.hidden_dim,
            ),
            nn.LayerNorm(
                self.hidden_dim
            ),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(
                self.hidden_dim,
                1,
            ),
        )

        # Begin near 0.5 uncertainty.
        final_layer = self.network[-1]

        nn.init.normal_(
            final_layer.weight,
            mean=0.0,
            std=0.01,
        )

        nn.init.zeros_(
            final_layer.bias
        )

    def forward(
        self,
        joint_embedding,
        classification_entropy,
        mean_branch_disagreement,
        routing_entropy,
        prototype_ambiguity,
        unknown_suspiciousness,
    ):
        if joint_embedding.ndim != 2:
            raise ValueError(
                "joint_embedding must have shape "
                "[batch, feat_dim]."
            )

        if joint_embedding.shape[1] != self.feat_dim:
            raise ValueError(
                "Joint embedding dimension mismatch. "
                f"Expected {self.feat_dim}, "
                f"received {joint_embedding.shape[1]}."
            )

        diagnostics = [
            classification_entropy,
            mean_branch_disagreement,
            routing_entropy,
            prototype_ambiguity,
            unknown_suspiciousness,
        ]

        batch_size = joint_embedding.shape[0]

        for index, diagnostic in enumerate(
            diagnostics
        ):
            if diagnostic.ndim != 1:
                raise ValueError(
                    "Each uncertainty diagnostic must "
                    "have shape [batch]. "
                    f"Diagnostic {index} has shape "
                    f"{tuple(diagnostic.shape)}."
                )

            if diagnostic.shape[0] != batch_size:
                raise ValueError(
                    "Uncertainty diagnostic batch-size "
                    "mismatch."
                )

        diagnostic_tensor = torch.stack(
            diagnostics,
            dim=1,
        )

        head_input = torch.cat(
            [
                joint_embedding,
                diagnostic_tensor,
            ],
            dim=1,
        )

        uncertainty_logit = (
            self.network(
                head_input
            ).squeeze(1)
        )

        learned_uncertainty = torch.sigmoid(
            uncertainty_logit
        )

        return {
            "uncertainty_logit": (
                uncertainty_logit
            ),
            "learned_uncertainty": (
                learned_uncertainty
            ),
            "uncertainty_diagnostics": (
                diagnostic_tensor
            ),
        }

'''

    lines = text.splitlines(
        keepends=True
    )

    insertion_index = (
        prototype_bank_node.end_lineno
    )

    lines = (
        lines[:insertion_index]
        + uncertainty_head_code.splitlines(
            keepends=True
        )
        + lines[insertion_index:]
    )

    text = "".join(lines)

    print(
        "\nInserted EvidenceUncertaintyHead."
    )

else:
    print(
        "\nEvidenceUncertaintyHead already exists."
    )


# ============================================================
# Update AGRFN
# ============================================================

tree = ast.parse(text)

agrfn_node = next(
    (
        node
        for node in tree.body
        if isinstance(node, ast.ClassDef)
        and node.name == "AGRFN"
    ),
    None,
)

if agrfn_node is None:
    raise RuntimeError(
        "Could not find class AGRFN."
    )

lines = text.splitlines(
    keepends=True
)

agrfn_source = "".join(
    lines[
        agrfn_node.lineno - 1:
        agrfn_node.end_lineno
    ]
)


def replace_required(
    source,
    old,
    new,
    description,
):
    if old in source:
        print(
            f"Updating {description}."
        )

        return source.replace(
            old,
            new,
            1,
        )

    if new in source:
        print(
            f"{description} already updated."
        )

        return source

    raise RuntimeError(
        "Could not find expected block for "
        f"{description}."
    )


# ------------------------------------------------------------
# Constructor arguments
# ------------------------------------------------------------

agrfn_source = replace_required(
    agrfn_source,
    '''        use_prototypes=False,
        prototype_temperature=0.10,
        dropout=0.20,
''',
    '''        use_prototypes=False,
        prototype_temperature=0.10,
        use_uncertainty=False,
        dropout=0.20,
''',
    "uncertainty constructor argument",
)


# ------------------------------------------------------------
# Store configuration
# ------------------------------------------------------------

agrfn_source = replace_required(
    agrfn_source,
    '''        self.prototype_temperature = float(
            prototype_temperature
        )
''',
    '''        self.prototype_temperature = float(
            prototype_temperature
        )
        self.use_uncertainty = bool(
            use_uncertainty
        )
''',
    "uncertainty configuration",
)


# ------------------------------------------------------------
# Create optional uncertainty head
# ------------------------------------------------------------

prototype_construction = '''        if self.use_prototypes:
            self.prototype_bank = PrototypeBank(
                feat_dim=self.feat_dim,
                num_classes=self.num_classes,
                temperature=(
                    self.prototype_temperature
                ),
            )
'''

prototype_and_uncertainty_construction = '''        if self.use_prototypes:
            self.prototype_bank = PrototypeBank(
                feat_dim=self.feat_dim,
                num_classes=self.num_classes,
                temperature=(
                    self.prototype_temperature
                ),
            )

        if self.use_uncertainty:
            self.uncertainty_head = (
                EvidenceUncertaintyHead(
                    feat_dim=self.feat_dim,
                    hidden_dim=self.feat_dim,
                    dropout=0.10,
                )
            )
'''

agrfn_source = replace_required(
    agrfn_source,
    prototype_construction,
    prototype_and_uncertainty_construction,
    "uncertainty-head construction",
)


# ------------------------------------------------------------
# Replace heuristic-only uncertainty calculation
# ------------------------------------------------------------

old_uncertainty_block = '''        uncertainty = torch.clamp(
            0.50 * classification_entropy
            + 0.30 * mean_disagreement
            + 0.20 * routing_entropy,
            0.0,
            1.0,
        )
'''

new_uncertainty_block = '''        heuristic_uncertainty = torch.clamp(
            0.50 * classification_entropy
            + 0.30 * mean_disagreement
            + 0.20 * routing_entropy,
            0.0,
            1.0,
        )

        uncertainty_logit = None
        learned_uncertainty = None
        uncertainty_diagnostics = None

        if self.use_uncertainty:
            if prototype_ambiguity is None:
                uncertainty_prototype_ambiguity = (
                    torch.zeros_like(
                        classification_entropy
                    )
                )
            else:
                uncertainty_prototype_ambiguity = (
                    prototype_ambiguity
                )

            if unknown_suspiciousness is None:
                uncertainty_unknown_score = (
                    torch.zeros_like(
                        classification_entropy
                    )
                )
            else:
                uncertainty_unknown_score = (
                    unknown_suspiciousness
                )

            uncertainty_output = (
                self.uncertainty_head(
                    joint_embedding=joint_embedding,

                    # Detaching these diagnostics prevents
                    # uncertainty supervision from directly
                    # distorting branch, routing, or prototype
                    # predictions.
                    classification_entropy=(
                        classification_entropy.detach()
                    ),
                    mean_branch_disagreement=(
                        mean_disagreement.detach()
                    ),
                    routing_entropy=(
                        routing_entropy.detach()
                    ),
                    prototype_ambiguity=(
                        uncertainty_prototype_ambiguity
                        .detach()
                    ),
                    unknown_suspiciousness=(
                        uncertainty_unknown_score
                        .detach()
                    ),
                )
            )

            uncertainty_logit = (
                uncertainty_output[
                    "uncertainty_logit"
                ]
            )

            learned_uncertainty = (
                uncertainty_output[
                    "learned_uncertainty"
                ]
            )

            uncertainty_diagnostics = (
                uncertainty_output[
                    "uncertainty_diagnostics"
                ]
            )

            uncertainty = learned_uncertainty

        else:
            uncertainty = heuristic_uncertainty
'''

agrfn_source = replace_required(
    agrfn_source,
    old_uncertainty_block,
    new_uncertainty_block,
    "learned uncertainty forward pass",
)


# ------------------------------------------------------------
# Add uncertainty outputs
# ------------------------------------------------------------

agrfn_source = replace_required(
    agrfn_source,
    '''            "fake_prob": probs[:, 1],
            "uncertainty": uncertainty,

            "stream_logits": stream_logits,
''',
    '''            "fake_prob": probs[:, 1],
            "uncertainty": uncertainty,
            "heuristic_uncertainty": (
                heuristic_uncertainty
            ),
            "learned_uncertainty": (
                learned_uncertainty
            ),
            "uncertainty_logit": (
                uncertainty_logit
            ),
            "uncertainty_diagnostics": (
                uncertainty_diagnostics
            ),

            "stream_logits": stream_logits,
''',
    "uncertainty model outputs",
)


# ------------------------------------------------------------
# Reinsert AGRFN
# ------------------------------------------------------------

lines = text.splitlines(
    keepends=True
)

lines = (
    lines[:agrfn_node.lineno - 1]
    + agrfn_source.splitlines(
        keepends=True
    )
    + lines[agrfn_node.end_lineno:]
)

text = "".join(lines)


# ============================================================
# Update build_model()
# ============================================================

tree = ast.parse(text)

build_model_node = next(
    (
        node
        for node in tree.body
        if isinstance(
            node,
            ast.FunctionDef,
        )
        and node.name == "build_model"
    ),
    None,
)

if build_model_node is None:
    raise RuntimeError(
        "Could not find build_model()."
    )

lines = text.splitlines(
    keepends=True
)

build_source = "".join(
    lines[
        build_model_node.lineno - 1:
        build_model_node.end_lineno
    ]
)

build_source = replace_required(
    build_source,
    '''            prototype_temperature=getattr(
                args,
                "prototype_temperature",
                0.10,
            ),
''',
    '''            prototype_temperature=getattr(
                args,
                "prototype_temperature",
                0.10,
            ),
            use_uncertainty=getattr(
                args,
                "use_uncertainty",
                False,
            ),
''',
    "build_model uncertainty argument",
)

lines = (
    lines[:build_model_node.lineno - 1]
    + build_source.splitlines(
        keepends=True
    )
    + lines[build_model_node.end_lineno:]
)

text = "".join(lines)


# ============================================================
# Save and compile
# ============================================================

script_path.write_text(text)

py_compile.compile(
    str(script_path),
    doraise=True,
)

print(
    "\nPASS: Python compilation succeeded."
)


# ============================================================
# Import updated module
# ============================================================

module_name = (
    "agrfn_uncertainty_test_"
    + timestamp
)

spec = importlib.util.spec_from_file_location(
    module_name,
    script_path,
)

module = importlib.util.module_from_spec(
    spec
)

if spec.loader is None:
    raise RuntimeError(
        "Could not create module loader."
    )

spec.loader.exec_module(module)


# ============================================================
# Dummy inputs
# ============================================================

batch_size = 4
image_size = 64

dummy_view = {
    "rgb": torch.randn(
        batch_size,
        3,
        image_size,
        image_size,
    ),
    "rec": torch.randn(
        batch_size,
        3,
        image_size,
        image_size,
    ),
    "den": torch.randn(
        batch_size,
        3,
        image_size,
        image_size,
    ),
    "jpeg": torch.randn(
        batch_size,
        3,
        image_size,
        image_size,
    ),
    "freq": torch.randn(
        batch_size,
        1,
        image_size,
        image_size,
    ),
    "noise": torch.randn(
        batch_size,
        1,
        image_size,
        image_size,
    ),
}

stream_names = [
    "rgb",
    "fingerprint",
    "jpeg",
    "freq",
]


# ============================================================
# Build uncertainty-enabled model
# ============================================================

enabled_args = SimpleNamespace(
    fusion_mode="adaptive_transformer",

    feat_dim=128,
    controller_dim=128,

    routing_temperature=1.0,
    film_scale=0.10,

    router_heads=4,
    router_layers=1,
    router_dropout=0.10,

    stream_logit_residual_weight=0.25,

    use_prototypes=True,
    prototype_temperature=0.10,

    use_uncertainty=True,
)

enabled_model = module.build_model(
    enabled_args,
    stream_names,
)

enabled_model.train()

enabled_output = enabled_model(
    dummy_view
)


# ============================================================
# Shape and range checks
# ============================================================

expected_shapes = {
    "uncertainty": (
        batch_size,
    ),
    "heuristic_uncertainty": (
        batch_size,
    ),
    "learned_uncertainty": (
        batch_size,
    ),
    "uncertainty_logit": (
        batch_size,
    ),
    "uncertainty_diagnostics": (
        batch_size,
        5,
    ),
}

for key, expected_shape in expected_shapes.items():
    value = enabled_output[key]

    if value is None:
        raise RuntimeError(
            f"{key} was unexpectedly None."
        )

    actual_shape = tuple(
        value.shape
    )

    if actual_shape != expected_shape:
        raise RuntimeError(
            f"{key} shape mismatch. "
            f"Expected {expected_shape}, "
            f"received {actual_shape}."
        )

    if not torch.isfinite(
        value
    ).all():
        raise RuntimeError(
            f"{key} contains NaN or infinity."
        )

for key in [
    "uncertainty",
    "heuristic_uncertainty",
    "learned_uncertainty",
]:
    value = enabled_output[key]

    in_range = (
        (value >= 0.0)
        & (value <= 1.0)
    ).all()

    if not bool(in_range):
        raise RuntimeError(
            f"{key} contains values outside [0, 1]."
        )

if not torch.allclose(
    enabled_output["uncertainty"],
    enabled_output[
        "learned_uncertainty"
    ],
    atol=1e-7,
):
    raise RuntimeError(
        "Enabled model did not expose learned "
        "uncertainty as its main uncertainty output."
    )


# ============================================================
# Verify final logits do not depend on uncertainty head
# ============================================================

enabled_model.zero_grad(
    set_to_none=True
)

output = enabled_model(
    dummy_view
)

classification_loss = (
    output["logits"].sum()
)

classification_loss.backward()

uncertainty_head_received_cls_grad = any(
    parameter.grad is not None
    for parameter in (
        enabled_model
        .uncertainty_head
        .parameters()
    )
)

if uncertainty_head_received_cls_grad:
    raise RuntimeError(
        "Final logits unexpectedly depend on "
        "the uncertainty head."
    )


# ============================================================
# Verify uncertainty head receives its own gradients
# ============================================================

enabled_model.zero_grad(
    set_to_none=True
)

output = enabled_model(
    dummy_view
)

synthetic_uncertainty_targets = torch.tensor(
    [0.0, 1.0, 0.0, 1.0],
    dtype=torch.float32,
)

uncertainty_loss = (
    F.binary_cross_entropy_with_logits(
        output["uncertainty_logit"],
        synthetic_uncertainty_targets,
    )
)

uncertainty_loss.backward()

uncertainty_gradients = [
    parameter.grad
    for parameter in (
        enabled_model
        .uncertainty_head
        .parameters()
    )
    if parameter.grad is not None
]

if not uncertainty_gradients:
    raise RuntimeError(
        "Uncertainty head received no gradients."
    )

if not all(
    torch.isfinite(
        gradient
    ).all()
    for gradient in uncertainty_gradients
):
    raise RuntimeError(
        "Uncertainty-head gradients contain "
        "NaN or infinity."
    )

if sum(
    float(
        gradient.abs().sum()
    )
    for gradient in uncertainty_gradients
) == 0.0:
    raise RuntimeError(
        "Uncertainty-head gradients are all zero."
    )


# ============================================================
# Confirm disabled architecture remains unchanged
# ============================================================

disabled_args = SimpleNamespace(
    fusion_mode="adaptive_scalar",

    feat_dim=128,
    controller_dim=128,

    routing_temperature=1.0,
    film_scale=0.10,

    router_heads=4,
    router_layers=1,
    router_dropout=0.10,

    stream_logit_residual_weight=0.25,

    use_prototypes=False,
    prototype_temperature=0.10,

    use_uncertainty=False,
)

disabled_model = module.build_model(
    disabled_args,
    stream_names,
)

disabled_model.eval()

with torch.no_grad():
    disabled_output = disabled_model(
        dummy_view
    )

if hasattr(
    disabled_model,
    "uncertainty_head",
):
    raise RuntimeError(
        "Uncertainty-disabled model unexpectedly "
        "created an uncertainty head."
    )

if (
    disabled_output[
        "learned_uncertainty"
    ]
    is not None
):
    raise RuntimeError(
        "learned_uncertainty must be None "
        "when the head is disabled."
    )

if (
    disabled_output[
        "uncertainty_logit"
    ]
    is not None
):
    raise RuntimeError(
        "uncertainty_logit must be None "
        "when the head is disabled."
    )

if not torch.allclose(
    disabled_output["uncertainty"],
    disabled_output[
        "heuristic_uncertainty"
    ],
    atol=1e-7,
):
    raise RuntimeError(
        "Disabled model did not preserve "
        "heuristic uncertainty behavior."
    )


# ============================================================
# Final report
# ============================================================

gradient_magnitude = sum(
    float(
        gradient.abs().sum()
    )
    for gradient in uncertainty_gradients
)

print("\n" + "=" * 80)
print("A-GRFN UNCERTAINTY HEAD TEST PASSED")
print("=" * 80)

print("\nLearned uncertainty:")
print(
    enabled_output[
        "learned_uncertainty"
    ]
    .detach()
    .cpu()
)

print("\nHeuristic uncertainty:")
print(
    enabled_output[
        "heuristic_uncertainty"
    ]
    .detach()
    .cpu()
)

print("\nUncertainty diagnostic shape:")
print(
    tuple(
        enabled_output[
            "uncertainty_diagnostics"
        ].shape
    )
)

print("\nUncertainty-head gradient magnitude:")
print(gradient_magnitude)

print(
    "\nUncertainty head received classification gradient:"
)
print(
    uncertainty_head_received_cls_grad
)

print("\nDisabled mode uses heuristic uncertainty:")
print(
    torch.allclose(
        disabled_output["uncertainty"],
        disabled_output[
            "heuristic_uncertainty"
        ],
        atol=1e-7,
    )
)

print(
    "\nPASS: learned uncertainty is available "
    "through --use_uncertainty."
)

print(
    "PASS: final classification logits remain "
    "independent of the uncertainty head."
)

print(
    "PASS: uncertainty-disabled models preserve "
    "the original heuristic behavior."
)

Backup created:
/home/jovyan/Deepfake_Low Resource_Low Resolution/agrfn_adaptive_routing_df40_before_uncertainty_head_20260730_221500.py

Inserted EvidenceUncertaintyHead.
Updating uncertainty constructor argument.
Updating uncertainty configuration.
Updating uncertainty-head construction.
Updating learned uncertainty forward pass.
Updating uncertainty model outputs.
Updating build_model uncertainty argument.

PASS: Python compilation succeeded.

A-GRFN UNCERTAINTY HEAD TEST PASSED

Learned uncertainty:
tensor([0.4423, 0.4675, 0.4973, 0.5020])

Heuristic uncertainty:
tensor([0.7233, 0.7343, 0.7331, 0.7090])

Uncertainty diagnostic shape:
(4, 5)

Uncertainty-head gradient magnitude:
28.600204899907112

Uncertainty head received classification gradient:
False

Disabled mode uses heuristic uncertainty:
True

PASS: learned uncertainty is available through --use_uncertainty.
PASS: final classification logits remain independent of the uncertainty head.
PASS: uncertainty-disabled models prese

In [47]:
from pathlib import Path
from datetime import datetime
from types import SimpleNamespace
import ast
import copy
import shutil
import py_compile
import importlib.util

import torch


# ============================================================
# Paths and backup
# ============================================================

project_root = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution"
)

script_path = (
    project_root
    / "agrfn_adaptive_routing_df40.py"
)

if not script_path.exists():
    raise FileNotFoundError(script_path)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

backup_path = script_path.with_name(
    f"{script_path.stem}_before_unc_log_fix_{timestamp}.py"
)

shutil.copy2(
    script_path,
    backup_path,
)

print("Backup created:")
print(backup_path)


# ============================================================
# Locate both logs dictionaries inside grfn_loss()
# ============================================================

text = script_path.read_text()
tree = ast.parse(text)

grfn_loss_node = next(
    (
        node
        for node in tree.body
        if isinstance(node, ast.FunctionDef)
        and node.name == "grfn_loss"
    ),
    None,
)

if grfn_loss_node is None:
    raise RuntimeError(
        "Could not find grfn_loss()."
    )

log_assignments = []

for node in ast.walk(grfn_loss_node):
    if not isinstance(node, ast.Assign):
        continue

    assigns_logs = any(
        isinstance(target, ast.Name)
        and target.id == "logs"
        for target in node.targets
    )

    if (
        assigns_logs
        and isinstance(node.value, ast.Dict)
    ):
        log_assignments.append(node)

log_assignments.sort(
    key=lambda node: node.lineno
)

if len(log_assignments) < 2:
    raise RuntimeError(
        "Expected an early CE log dictionary and "
        "a final training log dictionary."
    )

early_logs = log_assignments[0]
final_logs = log_assignments[-1]


# ============================================================
# Find the value associated with "unc"
# ============================================================

def find_dictionary_value(
    dictionary_node,
    requested_key,
):
    for key_node, value_node in zip(
        dictionary_node.value.keys,
        dictionary_node.value.values,
    ):
        if (
            isinstance(key_node, ast.Constant)
            and key_node.value == requested_key
        ):
            return value_node

    raise RuntimeError(
        f"Could not find '{requested_key}' "
        "inside a logs dictionary."
    )


early_unc_value = find_dictionary_value(
    early_logs,
    "unc",
)

final_unc_value = find_dictionary_value(
    final_logs,
    "unc",
)


# ============================================================
# Convert AST locations to absolute character positions
# ============================================================

source_lines = text.splitlines(
    keepends=True
)

line_offsets = []
current_offset = 0

for line in source_lines:
    line_offsets.append(current_offset)
    current_offset += len(line)


def absolute_span(node):
    start = (
        line_offsets[node.lineno - 1]
        + node.col_offset
    )

    end = (
        line_offsets[node.end_lineno - 1]
        + node.end_col_offset
    )

    return start, end


early_start, early_end = absolute_span(
    early_unc_value
)

final_start, final_end = absolute_span(
    final_unc_value
)

print("\nCurrent early CE uncertainty log:")
print(text[early_start:early_end])

print("\nCurrent final uncertainty log:")
print(text[final_start:final_end])


# ============================================================
# Apply replacements from bottom to top
# ============================================================

replacements = [
    (
        final_start,
        final_end,
        "float(unc_loss.detach().cpu())",
    ),
    (
        early_start,
        early_end,
        "0.0",
    ),
]

for start, end, replacement in sorted(
    replacements,
    key=lambda item: item[0],
    reverse=True,
):
    text = (
        text[:start]
        + replacement
        + text[end:]
    )

script_path.write_text(text)

print(
    "\nFixed early and final uncertainty logging."
)


# ============================================================
# Compile and import
# ============================================================

py_compile.compile(
    str(script_path),
    doraise=True,
)

print("PASS: Python compilation succeeded.")

module_name = (
    "agrfn_unc_log_fix_test_"
    + timestamp
)

spec = importlib.util.spec_from_file_location(
    module_name,
    script_path,
)

module = importlib.util.module_from_spec(
    spec
)

if spec.loader is None:
    raise RuntimeError(
        "Could not create module loader."
    )

spec.loader.exec_module(module)


# ============================================================
# Create a small deterministic test batch
# ============================================================

device = torch.device("cpu")
batch_size = 4
image_size = 64


def make_dummy_view():
    return {
        "rgb": torch.randn(
            batch_size,
            3,
            image_size,
            image_size,
        ),
        "rec": torch.randn(
            batch_size,
            3,
            image_size,
            image_size,
        ),
        "den": torch.randn(
            batch_size,
            3,
            image_size,
            image_size,
        ),
        "jpeg": torch.randn(
            batch_size,
            3,
            image_size,
            image_size,
        ),
        "freq": torch.randn(
            batch_size,
            1,
            image_size,
            image_size,
        ),
        "noise": torch.randn(
            batch_size,
            1,
            image_size,
            image_size,
        ),
    }


dummy_batch = {
    "a": make_dummy_view(),
    "b": make_dummy_view(),
    "label": torch.tensor(
        [0, 1, 0, 1],
        dtype=torch.long,
    ),
}

stream_names = [
    "rgb",
    "fingerprint",
    "jpeg",
    "freq",
]


# ============================================================
# Build uncertainty-enabled model
# ============================================================

enabled_args = SimpleNamespace(
    fusion_mode="adaptive_transformer",

    feat_dim=128,
    controller_dim=128,

    routing_temperature=1.0,
    film_scale=0.10,

    router_heads=4,
    router_layers=1,
    router_dropout=0.10,

    stream_logit_residual_weight=0.25,

    use_prototypes=True,
    prototype_temperature=0.10,

    use_uncertainty=True,

    loss_mode="grfn",
    model_mode="grfn",

    lambda_stream=0.05,
    lambda_aux=0.05,

    lambda_pred_cons=0.02,
    lambda_feat_cons=0.02,
    lambda_cons=0.02,

    lambda_route=0.01,
    lambda_proto=0.10,
    lambda_unc=0.20,
    lambda_supcon=0.00,
)

model = module.build_model(
    enabled_args,
    stream_names,
).to(device)

model.eval()


# ============================================================
# Compare lambda_unc = 0 against lambda_unc = 0.20
# ============================================================

zero_weight_args = copy.copy(
    enabled_args
)

zero_weight_args.lambda_unc = 0.0

with torch.no_grad():
    loss_without_unc, logs_without = (
        module.grfn_loss(
            model,
            dummy_batch,
            zero_weight_args,
            device,
            class_weights=None,
        )
    )

    loss_with_unc, logs_with = (
        module.grfn_loss(
            model,
            dummy_batch,
            enabled_args,
            device,
            class_weights=None,
        )
    )

if logs_with["unc"] <= 0.0:
    raise RuntimeError(
        "Uncertainty loss is still reported as zero."
    )

observed_increase = float(
    (
        loss_with_unc
        - loss_without_unc
    ).detach()
)

expected_increase = (
    enabled_args.lambda_unc
    * logs_with["unc"]
)

if abs(
    observed_increase
    - expected_increase
) > 1e-5:
    raise RuntimeError(
        "lambda_unc does not match the total-loss increase. "
        f"Expected {expected_increase}, "
        f"observed {observed_increase}."
    )


# ============================================================
# Verify the early CE branch remains safe
# ============================================================

ce_args = copy.copy(
    enabled_args
)

ce_args.loss_mode = "ce"

with torch.no_grad():
    ce_loss, ce_logs = module.grfn_loss(
        model,
        dummy_batch,
        ce_args,
        device,
        class_weights=None,
    )

if ce_logs["unc"] != 0.0:
    raise RuntimeError(
        "CE-only mode should report zero "
        "uncertainty training loss."
    )


# ============================================================
# Final report
# ============================================================

print("\n" + "=" * 80)
print("UNCERTAINTY LOSS LOGGING FIX PASSED")
print("=" * 80)

print("\nComputed uncertainty loss:")
print(logs_with["unc"])

print("\nConfigured lambda_unc:")
print(enabled_args.lambda_unc)

print("\nExpected total-loss increase:")
print(expected_increase)

print("\nObserved total-loss increase:")
print(observed_increase)

print("\nCE-only uncertainty log:")
print(ce_logs["unc"])

print(
    "\nPASS: uncertainty loss is computed and "
    "reported by the final training logs."
)

print(
    "PASS: lambda_unc contributes the expected "
    "amount to the total objective."
)

print(
    "PASS: the early CE-only path remains safe."
)

Backup created:
/home/jovyan/Deepfake_Low Resource_Low Resolution/agrfn_adaptive_routing_df40_before_unc_log_fix_20260730_221814.py

Current early CE uncertainty log:
float(
            unc_loss.detach().cpu()
        )

Current final uncertainty log:
0.0

Fixed early and final uncertainty logging.
PASS: Python compilation succeeded.

UNCERTAINTY LOSS LOGGING FIX PASSED

Computed uncertainty loss:
0.6931553483009338

Configured lambda_unc:
0.2

Expected total-loss increase:
0.13863106966018676

Observed total-loss increase:
0.13863110542297363

CE-only uncertainty log:
0.0

PASS: uncertainty loss is computed and reported by the final training logs.
PASS: lambda_unc contributes the expected amount to the total objective.
PASS: the early CE-only path remains safe.


In [48]:
from pathlib import Path
from datetime import datetime
import subprocess
import sys

import numpy as np
import pandas as pd


# ============================================================
# Paths
# ============================================================

project_root = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution"
)

script_path = (
    project_root
    / "agrfn_adaptive_routing_df40.py"
)

manifest_path = (
    project_root
    / "outputs_df40"
    / "df40_manifest_all_clean.csv"
)

if not script_path.exists():
    raise FileNotFoundError(script_path)

if not manifest_path.exists():
    raise FileNotFoundError(manifest_path)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

out_dir = (
    project_root
    / f"outputs_agrfn_full_uncertainty_smoke_{timestamp}"
)


# ============================================================
# Run one-epoch full A-GRFN smoke training
# ============================================================

command = [
    sys.executable,
    str(script_path),

    "--metadata_csv",
    str(manifest_path),

    "--out_dir",
    str(out_dir),

    "--dataset_name",
    "DF40",

    "--fusion_mode",
    "adaptive_transformer",

    "--streams",
    "rgb,fingerprint,jpeg,freq",

    "--feat_dim",
    "128",

    "--controller_dim",
    "128",

    "--routing_temperature",
    "1.0",

    "--film_scale",
    "0.10",

    "--router_heads",
    "4",

    "--router_layers",
    "1",

    "--router_dropout",
    "0.10",

    "--use_prototypes",

    "--prototype_temperature",
    "0.10",

    "--use_uncertainty",

    "--frames_per_video",
    "2",

    "--limit_train_units",
    "40",

    "--limit_val_units",
    "20",

    "--limit_test_units",
    "20",

    "--balanced_unit_limits",

    "--image_size",
    "224",

    "--batch_size",
    "4",

    "--num_workers",
    "0",

    "--epochs",
    "1",

    "--lr",
    "1e-4",

    "--lambda_aux",
    "0.05",

    "--lambda_route",
    "0.01",

    "--lambda_cons",
    "0.02",

    "--lambda_pred_cons",
    "0.02",

    "--lambda_proto",
    "0.10",

    "--lambda_unc",
    "0.10",

    "--balance_strategy",
    "auto",

    "--decision_threshold",
    "0.5",

    "--score_direction",
    "normal",

    "--amp",
]

print("=" * 80)
print("RUNNING FULL A-GRFN UNCERTAINTY SMOKE TEST")
print("=" * 80)

print("\nOutput directory:")
print(out_dir)

result = subprocess.run(
    command,
    cwd=str(project_root),
)

if result.returncode != 0:
    raise RuntimeError(
        "Full uncertainty smoke test failed with return code "
        f"{result.returncode}."
    )


# ============================================================
# Validate output files
# ============================================================

required_files = [
    "best_checkpoint.pt",
    "latest_checkpoint.pt",
    "training_history.csv",
    "test_frame_predictions.csv",
    "test_video_predictions.csv",
    "test_metrics.csv",
    "test_inference_time.csv",
    "experiment_summary.csv",
    "branch_predictions.csv",
    "branch_uncertainties.csv",
    "routing_weights.csv",
    "disagreement_matrices.csv",
    "uncertainty_breakdown.csv",
    "prototype_distances.csv",
]

missing_files = [
    filename
    for filename in required_files
    if not (out_dir / filename).exists()
]

if missing_files:
    raise RuntimeError(
        "Missing full-model output files:\n"
        + "\n".join(missing_files)
    )


# ============================================================
# Load outputs
# ============================================================

history = pd.read_csv(
    out_dir / "training_history.csv"
)

predictions = pd.read_csv(
    out_dir / "test_frame_predictions.csv"
)

uncertainty_table = pd.read_csv(
    out_dir / "uncertainty_breakdown.csv"
)

prototype_table = pd.read_csv(
    out_dir / "prototype_distances.csv"
)

routing_table = pd.read_csv(
    out_dir / "routing_weights.csv"
)

metrics = pd.read_csv(
    out_dir / "test_metrics.csv"
)

inference_table = pd.read_csv(
    out_dir / "test_inference_time.csv"
)


# ============================================================
# Validate all training losses
# ============================================================

required_history_columns = [
    "train_total",
    "train_cls",
    "train_stream",
    "train_pred_cons",
    "train_route_balance",
    "train_route_cons",
    "train_aug_cons",
    "train_proto",
    "train_proto_ce",
    "train_proto_pull",
    "train_unc",
]

missing_history_columns = [
    column
    for column in required_history_columns
    if column not in history.columns
]

if missing_history_columns:
    raise RuntimeError(
        "Missing full-model training columns: "
        + ", ".join(missing_history_columns)
    )

history_values = history[
    required_history_columns
].to_numpy(dtype=float)

if not np.isfinite(history_values).all():
    raise RuntimeError(
        "Training history contains NaN or infinity."
    )

if float(history["train_proto"].iloc[-1]) <= 0:
    raise RuntimeError(
        "Prototype loss was not active."
    )

if float(history["train_unc"].iloc[-1]) <= 0:
    raise RuntimeError(
        "Learned uncertainty loss was not active."
    )


# ============================================================
# Validate predictions
# ============================================================

required_prediction_columns = [
    "label",
    "fake_prob",
    "uncertainty",
    "classification_entropy",
    "mean_branch_entropy",
    "mean_branch_disagreement",
    "max_branch_disagreement",
    "routing_entropy",
    "distance_real",
    "distance_fake",
    "prototype_margin",
    "prototype_ambiguity",
    "unknown_suspiciousness",
    "route_rgb",
    "route_jpeg",
    "route_freq",
    "attention_rgb",
    "attention_jpeg",
    "attention_freq",
]

missing_prediction_columns = [
    column
    for column in required_prediction_columns
    if column not in predictions.columns
]

if missing_prediction_columns:
    raise RuntimeError(
        "Missing full-model prediction columns: "
        + ", ".join(missing_prediction_columns)
    )

prediction_values = predictions[
    required_prediction_columns
].to_numpy(dtype=float)

if not np.isfinite(prediction_values).all():
    raise RuntimeError(
        "Full-model predictions contain NaN or infinity."
    )


# ============================================================
# Probability and uncertainty ranges
# ============================================================

bounded_columns = [
    "fake_prob",
    "uncertainty",
    "classification_entropy",
    "mean_branch_entropy",
    "routing_entropy",
    "prototype_ambiguity",
    "unknown_suspiciousness",
]

for column in bounded_columns:
    if not predictions[column].between(
        0.0,
        1.0,
    ).all():
        raise RuntimeError(
            f"{column} contains values outside [0, 1]."
        )


# ============================================================
# Routing and attention checks
# ============================================================

route_columns = [
    "route_rgb",
    "route_jpeg",
    "route_freq",
]

attention_columns = [
    "attention_rgb",
    "attention_jpeg",
    "attention_freq",
]

route_values = predictions[
    route_columns
].to_numpy(dtype=float)

attention_values = predictions[
    attention_columns
].to_numpy(dtype=float)

route_sums = route_values.sum(axis=1)
attention_sums = attention_values.sum(axis=1)

maximum_route_sum_error = float(
    np.abs(route_sums - 1.0).max()
)

maximum_attention_sum_error = float(
    np.abs(attention_sums - 1.0).max()
)

maximum_route_attention_difference = float(
    np.abs(
        route_values
        - attention_values
    ).max()
)

if maximum_route_sum_error > 1e-5:
    raise RuntimeError(
        "Routing weights do not sum to one. "
        f"Maximum error: {maximum_route_sum_error}"
    )

if maximum_attention_sum_error > 1e-5:
    raise RuntimeError(
        "Attention weights do not sum to one. "
        f"Maximum error: {maximum_attention_sum_error}"
    )

if maximum_route_attention_difference > 1e-6:
    raise RuntimeError(
        "Routing and attention logs differ. "
        f"Maximum difference: "
        f"{maximum_route_attention_difference}"
    )


# ============================================================
# Prototype checks
# ============================================================

expected_minimum_distance = predictions[
    [
        "distance_real",
        "distance_fake",
    ]
].min(axis=1)

minimum_distance_error = float(
    np.abs(
        predictions["minimum_prototype_distance"]
        - expected_minimum_distance
    ).max()
)

expected_margin = np.abs(
    predictions["distance_real"]
    - predictions["distance_fake"]
)

margin_error = float(
    np.abs(
        predictions["prototype_margin"]
        - expected_margin
    ).max()
)

if minimum_distance_error > 1e-6:
    raise RuntimeError(
        "Minimum prototype-distance logging is inconsistent."
    )

if margin_error > 1e-6:
    raise RuntimeError(
        "Prototype-margin logging is inconsistent."
    )


# ============================================================
# Row-alignment checks
# ============================================================

expected_rows = len(predictions)

for table_name, table in [
    ("uncertainty_breakdown.csv", uncertainty_table),
    ("prototype_distances.csv", prototype_table),
    ("routing_weights.csv", routing_table),
]:
    if len(table) != expected_rows:
        raise RuntimeError(
            f"{table_name} has {len(table)} rows, "
            f"but predictions contain {expected_rows}."
        )


# ============================================================
# Basic uncertainty behavior analysis
# ============================================================

predicted_labels = (
    predictions["fake_prob"] >= 0.5
).astype(int)

predictions = predictions.copy()

predictions["is_error"] = (
    predicted_labels
    != predictions["label"].astype(int)
).astype(int)

correct_uncertainty = predictions.loc[
    predictions["is_error"] == 0,
    "uncertainty",
]

error_uncertainty = predictions.loc[
    predictions["is_error"] == 1,
    "uncertainty",
]

mean_correct_uncertainty = (
    float(correct_uncertainty.mean())
    if len(correct_uncertainty) > 0
    else np.nan
)

mean_error_uncertainty = (
    float(error_uncertainty.mean())
    if len(error_uncertainty) > 0
    else np.nan
)


# ============================================================
# Final report
# ============================================================

print("\n" + "=" * 80)
print("FULL A-GRFN UNCERTAINTY SMOKE TEST PASSED")
print("=" * 80)

print("\nOutput directory:")
print(out_dir)

print("\nTraining losses:")
display(
    history[
        required_history_columns
    ]
)

print("\nTest metrics:")
display(metrics)

print("\nInference timing:")
display(inference_table)

print("\nUncertainty summary:")
display(
    predictions[
        [
            "uncertainty",
            "classification_entropy",
            "mean_branch_entropy",
            "mean_branch_disagreement",
            "prototype_ambiguity",
            "unknown_suspiciousness",
        ]
    ].describe()
)

print("\nMean routing weights:")
display(
    predictions[
        route_columns
    ]
    .mean()
    .to_frame("mean_weight")
)

print("\nMean uncertainty for correct predictions:")
print(mean_correct_uncertainty)

print("\nMean uncertainty for incorrect predictions:")
print(mean_error_uncertainty)

print("\nNumber of correct predictions:")
print(int((predictions["is_error"] == 0).sum()))

print("\nNumber of incorrect predictions:")
print(int((predictions["is_error"] == 1).sum()))

print("\nMaximum routing-sum error:")
print(maximum_route_sum_error)

print("\nMaximum route/attention difference:")
print(maximum_route_attention_difference)

print("\nPrototype minimum-distance error:")
print(minimum_distance_error)

print("\nPrototype margin error:")
print(margin_error)

print(
    "\nPASS: the complete A-GRFN architecture completed "
    "training, validation, checkpointing, and testing."
)

print(
    "PASS: transformer routing, prototypes, and learned "
    "uncertainty were active together."
)

RUNNING FULL A-GRFN UNCERTAINTY SMOKE TEST

Output directory:
/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_full_uncertainty_smoke_20260730_221903

GRFN-CSF GRFN-only training and evaluation
Device: cuda
Output directory: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_full_uncertainty_smoke_20260730_221903
Metadata CSV: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40/df40_manifest_all_clean.csv
Streams: rgb,fingerprint,jpeg,freq

Metadata loaded
Metadata CSV: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40/df40_manifest_all_clean.csv
Path column: frame_path
Label column: label
Split column: split
Group column: group_id
Usable frames: 292907
train: frames=235640, units=7843, real=48132, fake=187508
val: frames=28551, units=980, real=5997, fake=22554
test: frames=28716, units=981, real=5999, fake=22717
Saved label audit samples to: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_full_uncertainty_smoke_202607

Sample train: 100%|██████████| 40/40 [00:00<00:00, 65.39unit/s]


Sample train: frames=78, units=40, real=40, fake=38
Sample val: frames=39, units=20, real=20, fake=19


Sample test: 100%|██████████| 20/20 [00:00<00:00, 353.90unit/s]


Sample test: frames=39, units=20, real=20, fake=19

Balancing strategy
Train class counts [real, fake]: [40, 38]
Train imbalance ratio: 1.053
Use class-weighted loss: False
Use weighted sampler: False

Building dataloaders
Train batches: 19
Val batches: 10
Test batches: 10

Building model
Model mode: grfn
Loss mode: grfn
Active streams: rgb, fingerprint, jpeg, freq
Total params: 2,062,540
Trainable params: 2,062,540

Training


Predict test:   0%|          | 0/10 [00:00<?, ?it/s]

... (9 outputs left out)

,uncertainty,classification_entropy,mean_branch_entropy,mean_branch_disagreement,prototype_ambiguity,unknown_suspiciousness
count,39.000000,39.000000,39.000000,39.000000,39.000000,39.000000
mean,0.475668,0.879574,0.987450,0.057903,0.887996,0.427263
std,0.007497,0.103337,0.008083,0.020890,0.105875,0.020577
min,0.463438,0.692781,0.970370,0.017082,0.713149,0.397639
25%,0.470739,0.771257,0.981151,0.043268,0.792014,0.407417
50%,0.474199,0.920785,0.987755,0.064976,0.914581,0.432167
75%,0.479634,0.964340,0.995152,0.073477,0.990900,0.444200
max,0.491415,0.998470,0.999147,0.091672,0.999929,0.457231



Mean routing weights:


,mean_weight
route_rgb,0.334577
route_jpeg,0.333276
route_freq,0.332147



Mean uncertainty for correct predictions:
0.4754140092567964

Mean uncertainty for incorrect predictions:
0.47599610510994406

Number of correct predictions:
22

Number of incorrect predictions:
17

Maximum routing-sum error:
5.960464477539063e-08

Maximum route/attention difference:
0.0

Prototype minimum-distance error:
0.0

Prototype margin error:
2.0816681711721685e-16

PASS: the complete A-GRFN architecture completed training, validation, checkpointing, and testing.
PASS: transformer routing, prototypes, and learned uncertainty were active together.


In [49]:
from pathlib import Path
from datetime import datetime
from types import SimpleNamespace
from torch.utils.data import DataLoader
import importlib.util
import time

import numpy as np
import pandas as pd
import torch


# ============================================================
# Project paths
# ============================================================

project_root = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution"
)

script_path = (
    project_root
    / "agrfn_adaptive_routing_df40.py"
)

ood_manifest_path = (
    project_root
    / "outputs_df40"
    / "df40_manifest_ood_fake_test.csv"
)

if not script_path.exists():
    raise FileNotFoundError(script_path)

if not ood_manifest_path.exists():
    raise FileNotFoundError(ood_manifest_path)


# ============================================================
# Locate the latest complete A-GRFN smoke checkpoint
# ============================================================

candidate_dirs = sorted(
    [
        path
        for path in project_root.glob(
            "outputs_agrfn_full_uncertainty_smoke_*"
        )
        if (
            path.is_dir()
            and (path / "best_checkpoint.pt").exists()
            and (path / "test_frame_predictions.csv").exists()
            and (path / "test_metrics.csv").exists()
        )
    ],
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)

if not candidate_dirs:
    raise RuntimeError(
        "No completed full A-GRFN uncertainty smoke run "
        "was found."
    )

indomain_dir = candidate_dirs[0]

checkpoint_path = (
    indomain_dir
    / "best_checkpoint.pt"
)

print("Using in-domain experiment:")
print(indomain_dir)

print("\nUsing checkpoint:")
print(checkpoint_path)


# ============================================================
# Recover the in-domain decision policy
# ============================================================

def normalized_column_name(value):
    return "".join(
        character.lower()
        for character in str(value)
        if character.isalnum()
    )


def extract_threshold(csv_paths):
    for csv_path in csv_paths:
        if not csv_path.exists():
            continue

        table = pd.read_csv(csv_path)

        for column in table.columns:
            normalized = normalized_column_name(
                column
            )

            if (
                "threshold" in normalized
                and "uncertainty" not in normalized
            ):
                values = pd.to_numeric(
                    table[column],
                    errors="coerce",
                ).dropna()

                if len(values) > 0:
                    threshold = float(
                        values.iloc[0]
                    )

                    if 0.0 <= threshold <= 1.0:
                        return threshold, (
                            f"{csv_path.name}:{column}"
                        )

    return 0.50, "fallback_0.50"


decision_threshold, threshold_source = (
    extract_threshold(
        [
            indomain_dir
            / "test_metrics.csv",

            indomain_dir
            / "experiment_summary.csv",

            indomain_dir
            / "test_threshold_report.csv",
        ]
    )
)

indomain_predictions = pd.read_csv(
    indomain_dir
    / "test_frame_predictions.csv"
)

invert_scores = False

if "score_inverted" in indomain_predictions.columns:
    score_inverted_values = (
        indomain_predictions[
            "score_inverted"
        ]
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(
            [
                "true",
                "1",
                "yes",
            ]
        )
    )

    invert_scores = bool(
        score_inverted_values.mean() >= 0.50
    )

print("\nDecision threshold:")
print(decision_threshold)

print("Threshold source:")
print(threshold_source)

print("Invert OOD scores:")
print(invert_scores)


# ============================================================
# Create OOD output directory
# ============================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

out_dir = (
    project_root
    / f"outputs_agrfn_ood_fake_test_smoke_{timestamp}"
)

out_dir.mkdir(
    parents=True,
    exist_ok=True,
)

print("\nOOD output directory:")
print(out_dir)


# ============================================================
# Import the updated A-GRFN module
# ============================================================

module_name = (
    "agrfn_ood_test_"
    + timestamp
)

spec = importlib.util.spec_from_file_location(
    module_name,
    script_path,
)

module = importlib.util.module_from_spec(
    spec
)

if spec.loader is None:
    raise RuntimeError(
        "Could not create module loader."
    )

spec.loader.exec_module(module)


# ============================================================
# Load checkpoint and reconstruct its exact architecture
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

try:
    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=False,
    )
except TypeError:
    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
    )

checkpoint_args = checkpoint.get(
    "args",
    {},
)

if not isinstance(
    checkpoint_args,
    dict,
):
    raise RuntimeError(
        "Checkpoint args are not stored as a dictionary."
    )

args = SimpleNamespace(
    **checkpoint_args
)

# Safe defaults for older checkpoint fields.
default_arguments = {
    "fusion_mode": "adaptive_transformer",
    "feat_dim": 128,
    "controller_dim": 128,
    "routing_temperature": 1.0,
    "film_scale": 0.10,
    "router_heads": 4,
    "router_layers": 1,
    "router_dropout": 0.10,
    "stream_logit_residual_weight": 0.25,
    "use_prototypes": True,
    "prototype_temperature": 0.10,
    "use_uncertainty": True,
    "image_size": 224,
    "batch_size": 4,
    "num_workers": 0,
    "uncertainty_threshold": 0.50,
    "data_root": None,
}

for name, default_value in default_arguments.items():
    if not hasattr(args, name):
        setattr(
            args,
            name,
            default_value,
        )

stream_names = checkpoint.get(
    "stream_names",
    None,
)

if stream_names is None:
    stream_names = [
        name.strip()
        for name in str(
            getattr(
                args,
                "streams",
                "rgb,fingerprint,jpeg,freq",
            )
        ).split(",")
        if name.strip()
    ]

model = module.build_model(
    args,
    stream_names,
).to(device)

model.load_state_dict(
    checkpoint["model"],
    strict=True,
)

model.eval()

print("\nDevice:")
print(device)

print("\nRecovered fusion mode:")
print(args.fusion_mode)

print("\nRecovered streams:")
print(stream_names)

print("\nPrototypes enabled:")
print(args.use_prototypes)

print("\nLearned uncertainty enabled:")
print(args.use_uncertainty)


# ============================================================
# Load and normalize the fake-only OOD manifest
# ============================================================

raw_ood = pd.read_csv(
    ood_manifest_path
)

if len(raw_ood) == 0:
    raise RuntimeError(
        "The OOD manifest is empty."
    )

path_candidates = [
    "image_path",
    "frame_path",
    "path",
    "filepath",
    "file_path",
]

path_column = next(
    (
        column
        for column in path_candidates
        if column in raw_ood.columns
    ),
    None,
)

if path_column is None:
    raise RuntimeError(
        "Could not find an image-path column "
        "in the OOD manifest."
    )

ood_df = raw_ood.copy()

def resolve_image_path(value):
    path = Path(str(value))

    if path.is_absolute():
        return str(path)

    data_root = getattr(
        args,
        "data_root",
        None,
    )

    if data_root:
        return str(
            Path(data_root)
            / path
        )

    return str(
        project_root
        / path
    )


ood_df["image_path"] = (
    ood_df[path_column]
    .map(resolve_image_path)
)

# Fake_Test must remain fake-only.
ood_df["label"] = 1
ood_df["label_name"] = "fake"
ood_df["split"] = "ood_fake_test"

group_candidates = [
    "sample_unit_id",
    "group_id",
    "video_id",
    "clip_id",
]

group_column = next(
    (
        column
        for column in group_candidates
        if column in ood_df.columns
    ),
    None,
)

if group_column is None:
    ood_df["group_id"] = (
        ood_df["image_path"]
        .astype(str)
    )
else:
    ood_df["group_id"] = (
        ood_df[group_column]
        .astype(str)
    )

if "sample_unit_id" not in ood_df.columns:
    ood_df["sample_unit_id"] = (
        ood_df["group_id"]
        .astype(str)
    )
else:
    ood_df["sample_unit_id"] = (
        ood_df["sample_unit_id"]
        .astype(str)
    )

if "method" not in ood_df.columns:
    ood_df["method"] = "unknown"

if "domain" not in ood_df.columns:
    ood_df["domain"] = "ood_fake_test"

if "dataset_partition" not in ood_df.columns:
    ood_df[
        "dataset_partition"
    ] = "ood_fake_test"

if set(
    ood_df["label"].unique()
) != {1}:
    raise RuntimeError(
        "Fake_Test OOD evaluation must contain only label 1."
    )


# ============================================================
# Validate paths and sample a tiny OOD subset
# ============================================================

existing_mask = (
    ood_df["image_path"]
    .map(lambda value: Path(value).exists())
)

missing_count = int(
    (~existing_mask).sum()
)

if missing_count > 0:
    missing_examples = (
        ood_df.loc[
            ~existing_mask,
            "image_path",
        ]
        .head(10)
        .tolist()
    )

    raise FileNotFoundError(
        f"{missing_count} OOD images were not found. "
        f"Examples:\n"
        + "\n".join(missing_examples)
    )

ood_df = module.sample_frames_per_unit(
    ood_df,
    frames_per_unit=2,
    limit_units=20,
    seed=42,
    desc="Sample OOD Fake_Test",
    balanced_units=False,
)

if len(ood_df) == 0:
    raise RuntimeError(
        "No OOD frames remained after sampling."
    )

print("\nOOD smoke-test frames:")
print(len(ood_df))

print("OOD smoke-test units:")
print(
    ood_df[
        "sample_unit_id"
    ].nunique()
)

print("OOD methods:")
print(
    sorted(
        ood_df["method"]
        .astype(str)
        .unique()
        .tolist()
    )
)


# ============================================================
# Build OOD loader
# ============================================================

ood_dataset = module.GRFNDataset(
    ood_df,
    image_size=args.image_size,
    train=False,
    train_aug=False,
)

ood_loader = DataLoader(
    ood_dataset,
    batch_size=args.batch_size,
    shuffle=False,
    num_workers=args.num_workers,
    pin_memory=(
        device.type == "cuda"
    ),
)


# ============================================================
# Run OOD inference
# ============================================================

start_time = time.perf_counter()

ood_raw_predictions = (
    module.predict_loader(
        model,
        ood_loader,
        args,
        device,
        split_name="ood_fake_test",
    )
)

elapsed_seconds = (
    time.perf_counter()
    - start_time
)

ood_predictions = (
    module.apply_score_direction(
        ood_raw_predictions,
        invert_scores=invert_scores,
    )
)

ood_predictions[
    "predicted_label"
] = (
    ood_predictions[
        "fake_prob"
    ]
    >= decision_threshold
).astype(int)

ood_predictions[
    "detected_fake"
] = (
    ood_predictions[
        "predicted_label"
    ] == 1
).astype(int)

ood_predictions[
    "missed_fake"
] = (
    ood_predictions[
        "predicted_label"
    ] == 0
).astype(int)

ood_predictions[
    "is_uncertain"
] = (
    ood_predictions[
        "uncertainty"
    ]
    >= float(
        args.uncertainty_threshold
    )
).astype(int)


# ============================================================
# Build unit/video-level predictions
# ============================================================

unit_aggregation = {
    "group_id": "first",
    "label": "first",
    "fake_prob": "mean",
    "uncertainty": "mean",
    "method": "first",
    "domain": "first",
    "image_path": "count",
}

available_aggregation = {
    key: value
    for key, value in unit_aggregation.items()
    if key in ood_predictions.columns
}

ood_video_predictions = (
    ood_predictions
    .groupby(
        "sample_unit_id",
        as_index=False,
    )
    .agg(
        available_aggregation
    )
)

if "image_path" in ood_video_predictions.columns:
    ood_video_predictions = (
        ood_video_predictions.rename(
            columns={
                "image_path": "num_frames"
            }
        )
    )

ood_video_predictions[
    "predicted_label"
] = (
    ood_video_predictions[
        "fake_prob"
    ]
    >= decision_threshold
).astype(int)

ood_video_predictions[
    "detected_fake"
] = (
    ood_video_predictions[
        "predicted_label"
    ] == 1
).astype(int)

ood_video_predictions[
    "missed_fake"
] = (
    ood_video_predictions[
        "predicted_label"
    ] == 0
).astype(int)

ood_video_predictions[
    "is_uncertain"
] = (
    ood_video_predictions[
        "uncertainty"
    ]
    >= float(
        args.uncertainty_threshold
    )
).astype(int)


# ============================================================
# Fake-only metric helpers
# ============================================================

def fake_only_metrics(
    dataframe,
    level,
    method="ALL",
):
    total = int(
        len(dataframe)
    )

    fake_detection_rate = float(
        dataframe[
            "detected_fake"
        ].mean()
    )

    missed_fake_rate = float(
        dataframe[
            "missed_fake"
        ].mean()
    )

    return {
        "Level": level,
        "Method": method,
        "Samples": total,
        "Decision Threshold": (
            decision_threshold
        ),
        "Score Inverted": (
            invert_scores
        ),
        "Fake Detection Rate": (
            fake_detection_rate
        ),
        "Recall (1)": (
            fake_detection_rate
        ),
        "Missed Fake Rate": (
            missed_fake_rate
        ),
        "Mean Fake Probability": float(
            dataframe[
                "fake_prob"
            ].mean()
        ),
        "Median Fake Probability": float(
            dataframe[
                "fake_prob"
            ].median()
        ),
        "Mean Uncertainty": float(
            dataframe[
                "uncertainty"
            ].mean()
        ),
        "Uncertain Rate": float(
            dataframe[
                "is_uncertain"
            ].mean()
        ),
        "Uncertainty Threshold": float(
            args.uncertainty_threshold
        ),
    }


overall_metrics = pd.DataFrame(
    [
        fake_only_metrics(
            ood_predictions,
            "Frame",
        ),
        fake_only_metrics(
            ood_video_predictions,
            "Unit",
        ),
    ]
)

per_method_rows = []

if "method" in ood_predictions.columns:
    for method, method_df in (
        ood_predictions.groupby(
            "method",
            dropna=False,
        )
    ):
        per_method_rows.append(
            fake_only_metrics(
                method_df,
                "Frame",
                method=str(method),
            )
        )

if (
    "method"
    in ood_video_predictions.columns
):
    for method, method_df in (
        ood_video_predictions.groupby(
            "method",
            dropna=False,
        )
    ):
        per_method_rows.append(
            fake_only_metrics(
                method_df,
                "Unit",
                method=str(method),
            )
        )

per_method_metrics = pd.DataFrame(
    per_method_rows
)


# ============================================================
# Inference-time table
# ============================================================

inference_table = pd.DataFrame(
    [
        {
            "Evaluation": (
                "DF40 Fake_Test OOD"
            ),
            "Device": str(device),
            "Fusion Mode": (
                args.fusion_mode
            ),
            "Image Size": (
                args.image_size
            ),
            "Batch Size": (
                args.batch_size
            ),
            "Frames Scored": (
                len(ood_predictions)
            ),
            "Units Scored": (
                len(
                    ood_video_predictions
                )
            ),
            "Total Seconds": (
                elapsed_seconds
            ),
            "Time Per Frame (ms)": (
                elapsed_seconds
                / max(
                    len(
                        ood_predictions
                    ),
                    1,
                )
                * 1000.0
            ),
            "FPS": (
                len(
                    ood_predictions
                )
                / max(
                    elapsed_seconds,
                    1e-8,
                )
            ),
        }
    ]
)


# ============================================================
# Save all OOD outputs
# ============================================================

ood_predictions.to_csv(
    out_dir
    / "ood_frame_predictions.csv",
    index=False,
)

ood_video_predictions.to_csv(
    out_dir
    / "ood_video_predictions.csv",
    index=False,
)

overall_metrics.to_csv(
    out_dir
    / "ood_fake_only_metrics.csv",
    index=False,
)

per_method_metrics.to_csv(
    out_dir
    / "ood_per_method_metrics.csv",
    index=False,
)

inference_table.to_csv(
    out_dir
    / "ood_inference_time.csv",
    index=False,
)

ood_df.to_csv(
    out_dir
    / "ood_sampled_manifest.csv",
    index=False,
)

diagnostic_paths = (
    module.save_diagnostic_csvs(
        ood_predictions,
        out_dir,
    )
)


# ============================================================
# Validate outputs
# ============================================================

required_output_files = [
    "ood_frame_predictions.csv",
    "ood_video_predictions.csv",
    "ood_fake_only_metrics.csv",
    "ood_per_method_metrics.csv",
    "ood_inference_time.csv",
    "ood_sampled_manifest.csv",
    "routing_weights.csv",
    "branch_predictions.csv",
    "branch_uncertainties.csv",
    "disagreement_matrices.csv",
    "uncertainty_breakdown.csv",
    "prototype_distances.csv",
]

missing_output_files = [
    filename
    for filename in required_output_files
    if not (
        out_dir
        / filename
    ).exists()
]

if missing_output_files:
    raise RuntimeError(
        "Missing OOD output files:\n"
        + "\n".join(
            missing_output_files
        )
    )

if not (
    ood_predictions[
        "fake_prob"
    ].between(
        0.0,
        1.0,
    ).all()
):
    raise RuntimeError(
        "OOD fake probabilities are outside [0, 1]."
    )

if not (
    ood_predictions[
        "uncertainty"
    ].between(
        0.0,
        1.0,
    ).all()
):
    raise RuntimeError(
        "OOD uncertainty values are outside [0, 1]."
    )

if not np.isfinite(
    ood_predictions[
        [
            "fake_prob",
            "uncertainty",
        ]
    ].to_numpy(
        dtype=float
    )
).all():
    raise RuntimeError(
        "OOD predictions contain NaN or infinity."
    )


# ============================================================
# Final report
# ============================================================

print("\n" + "=" * 80)
print("DF40 FAKE_TEST OOD SMOKE TEST PASSED")
print("=" * 80)

print("\nOutput directory:")
print(out_dir)

print("\nFake-only OOD metrics:")
display(overall_metrics)

print("\nPer-method OOD metrics:")
display(per_method_metrics)

print("\nInference timing:")
display(inference_table)

print("\nPrediction summary:")
display(
    ood_predictions[
        [
            "fake_prob",
            "uncertainty",
            "detected_fake",
            "missed_fake",
            "is_uncertain",
        ]
    ].describe()
)

print("\nDiagnostic files:")

for filename in sorted(
    diagnostic_paths
):
    print(" ", filename)

print(
    "\nNo AUCROC, AUPRC, balanced accuracy, or EER "
    "was reported because Fake_Test contains only fake samples."
)

print(
    "\nPASS: the full A-GRFN checkpoint completed "
    "fake-only OOD evaluation with the in-domain "
    "decision policy."
)

Using in-domain experiment:
/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_full_uncertainty_smoke_20260730_221903

Using checkpoint:
/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_full_uncertainty_smoke_20260730_221903/best_checkpoint.pt

Decision threshold:
0.5
Threshold source:
test_metrics.csv:Decision Threshold
Invert OOD scores:
False

OOD output directory:
/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_ood_fake_test_smoke_20260730_222042

Device:
cuda

Recovered fusion mode:
adaptive_transformer

Recovered streams:
['rgb', 'fingerprint', 'jpeg', 'freq']

Prototypes enabled:
True

Learned uncertainty enabled:
True


Sample OOD Fake_Test: 100%|██████████| 20/20 [00:00<00:00, 101.82unit/s]


Sample OOD Fake_Test: frames=38, units=20, real=0, fake=38

OOD smoke-test frames:
38
OOD smoke-test units:
20
OOD methods:
['CollabDiff', 'DiT', 'blendface', 'danet', 'ddim', 'deepfacelab', 'frames']


Predict ood_fake_test: 100%|██████████| 10/10 [00:02<00:00,  3.93it/s]


DF40 FAKE_TEST OOD SMOKE TEST PASSED

Output directory:
/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_ood_fake_test_smoke_20260730_222042

Fake-only OOD metrics:


,Level,Method,Samples,Decision Threshold,Score Inverted,Fake Detection Rate,Recall (1),Missed Fake Rate,Mean Fake Probability,Median Fake Probability,Mean Uncertainty,Uncertain Rate,Uncertainty Threshold
0,Frame,ALL,38,0.5,False,0.315789,0.315789,0.684211,0.392918,0.369202,0.477931,0.0,0.5
1,Unit,ALL,20,0.5,False,0.200000,0.200000,0.800000,0.395957,0.401559,0.478130,0.0,0.5


... (3 outputs left out)

,Evaluation,Device,Fusion Mode,Image Size,Batch Size,Frames Scored,Units Scored,Total Seconds,Time Per Frame (ms),FPS
0,DF40 Fake_Test OOD,cuda,adaptive_transformer,224,4,38,20,2.549926,67.10332,14.902392



Prediction summary:


,fake_prob,uncertainty,detected_fake,missed_fake,is_uncertain
count,38.000000,38.000000,38.000000,38.000000,38.0
mean,0.392918,0.477931,0.315789,0.684211,0.0
std,0.177558,0.008300,0.471069,0.471069,0.0
min,0.178321,0.464508,0.000000,0.000000,0.0
25%,0.220077,0.471227,0.000000,0.000000,0.0
50%,0.369202,0.478650,0.000000,1.000000,0.0
75%,0.537832,0.484670,1.000000,1.000000,0.0
max,0.748386,0.494404,1.000000,1.000000,0.0



Diagnostic files:
  branch_predictions.csv
  branch_uncertainties.csv
  disagreement_matrices.csv
  prototype_distances.csv
  routing_weights.csv
  uncertainty_breakdown.csv

No AUCROC, AUPRC, balanced accuracy, or EER was reported because Fake_Test contains only fake samples.

PASS: the full A-GRFN checkpoint completed fake-only OOD evaluation with the in-domain decision policy.


In [50]:
from pathlib import Path
from datetime import datetime
import py_compile
import subprocess
import sys
import textwrap

project_root = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution"
)

evaluator_path = (
    project_root
    / "evaluate_agrfn_ood.py"
)

# ============================================================
# Preserve an existing evaluator
# ============================================================

if evaluator_path.exists():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    backup_path = evaluator_path.with_name(
        f"{evaluator_path.stem}_backup_{timestamp}.py"
    )

    backup_path.write_bytes(
        evaluator_path.read_bytes()
    )

    print("Existing evaluator backed up:")
    print(backup_path)

# ============================================================
# Reusable fake-only OOD evaluator
# ============================================================

evaluator_code = r'''
#!/usr/bin/env python3

import argparse
import importlib.util
import json
import time
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader


def parse_args():
    parser = argparse.ArgumentParser(
        description=(
            "Evaluate an A-GRFN checkpoint on the fake-only "
            "DF40 Fake_Test OOD partition."
        )
    )

    parser.add_argument(
        "--checkpoint",
        type=str,
        required=True,
        help="Path to best_checkpoint.pt.",
    )

    parser.add_argument(
        "--model_script",
        type=str,
        default="agrfn_adaptive_routing_df40.py",
        help="Path to the A-GRFN model/training script.",
    )

    parser.add_argument(
        "--ood_metadata_csv",
        type=str,
        default=(
            "outputs_df40/"
            "df40_manifest_ood_fake_test.csv"
        ),
        help="Fake-only OOD manifest.",
    )

    parser.add_argument(
        "--indomain_dir",
        type=str,
        default=None,
        help=(
            "In-domain experiment directory used to recover "
            "the validation-tuned threshold and score direction."
        ),
    )

    parser.add_argument(
        "--out_dir",
        type=str,
        default=None,
        help="OOD output directory.",
    )

    parser.add_argument(
        "--decision_threshold",
        type=float,
        default=None,
        help=(
            "Explicit threshold override. When omitted, the "
            "threshold is recovered from --indomain_dir."
        ),
    )

    parser.add_argument(
        "--score_direction",
        type=str,
        default="auto",
        choices=[
            "auto",
            "normal",
            "inverted",
        ],
    )

    parser.add_argument(
        "--uncertainty_threshold",
        type=float,
        default=0.50,
    )

    parser.add_argument(
        "--frames_per_unit",
        type=int,
        default=0,
        help=(
            "Frames sampled per OOD unit. Zero keeps all frames."
        ),
    )

    parser.add_argument(
        "--limit_units",
        type=int,
        default=0,
        help=(
            "Maximum OOD units. Zero evaluates all units."
        ),
    )

    parser.add_argument(
        "--batch_size",
        type=int,
        default=None,
    )

    parser.add_argument(
        "--num_workers",
        type=int,
        default=None,
    )

    parser.add_argument(
        "--seed",
        type=int,
        default=42,
    )

    args = parser.parse_args()

    if not 0.0 <= args.uncertainty_threshold <= 1.0:
        parser.error(
            "--uncertainty_threshold must be in [0, 1]."
        )

    if (
        args.decision_threshold is not None
        and not 0.0 <= args.decision_threshold <= 1.0
    ):
        parser.error(
            "--decision_threshold must be in [0, 1]."
        )

    if args.frames_per_unit < 0:
        parser.error(
            "--frames_per_unit cannot be negative."
        )

    if args.limit_units < 0:
        parser.error(
            "--limit_units cannot be negative."
        )

    return args


def resolve_path(value, project_root):
    path = Path(value).expanduser()

    if not path.is_absolute():
        path = project_root / path

    return path.resolve()


def import_model_module(script_path):
    module_name = (
        "agrfn_ood_runtime_"
        + str(abs(hash(str(script_path))))
    )

    spec = importlib.util.spec_from_file_location(
        module_name,
        script_path,
    )

    if spec is None or spec.loader is None:
        raise RuntimeError(
            f"Could not import model script: {script_path}"
        )

    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)

    return module


def load_checkpoint(checkpoint_path, device):
    try:
        return torch.load(
            checkpoint_path,
            map_location=device,
            weights_only=False,
        )
    except TypeError:
        return torch.load(
            checkpoint_path,
            map_location=device,
        )


def normalized_column_name(value):
    return "".join(
        character.lower()
        for character in str(value)
        if character.isalnum()
    )


def recover_threshold(indomain_dir):
    if indomain_dir is None:
        return 0.50, "fallback_0.50"

    candidate_files = [
        indomain_dir / "test_metrics.csv",
        indomain_dir / "experiment_summary.csv",
        indomain_dir / "test_threshold_report.csv",
    ]

    for csv_path in candidate_files:
        if not csv_path.exists():
            continue

        table = pd.read_csv(csv_path)

        for column in table.columns:
            normalized = normalized_column_name(
                column
            )

            if (
                "threshold" in normalized
                and "uncertainty" not in normalized
            ):
                values = pd.to_numeric(
                    table[column],
                    errors="coerce",
                ).dropna()

                if len(values) == 0:
                    continue

                threshold = float(values.iloc[0])

                if 0.0 <= threshold <= 1.0:
                    return (
                        threshold,
                        f"{csv_path.name}:{column}",
                    )

    return 0.50, "fallback_0.50"


def recover_score_inversion(
    indomain_dir,
    requested_direction,
):
    if requested_direction == "normal":
        return False, "explicit_normal"

    if requested_direction == "inverted":
        return True, "explicit_inverted"

    if indomain_dir is None:
        return False, "auto_fallback_normal"

    prediction_path = (
        indomain_dir
        / "test_frame_predictions.csv"
    )

    if prediction_path.exists():
        predictions = pd.read_csv(
            prediction_path
        )

        if "score_inverted" in predictions.columns:
            values = (
                predictions["score_inverted"]
                .astype(str)
                .str.strip()
                .str.lower()
                .isin(
                    [
                        "true",
                        "1",
                        "yes",
                    ]
                )
            )

            return (
                bool(values.mean() >= 0.50),
                "test_frame_predictions.csv",
            )

    summary_path = (
        indomain_dir
        / "experiment_summary.csv"
    )

    if summary_path.exists():
        summary = pd.read_csv(summary_path)

        for column in summary.columns:
            normalized = normalized_column_name(
                column
            )

            if (
                "score" in normalized
                and "direction" in normalized
            ):
                value = str(
                    summary[column].iloc[0]
                ).strip().lower()

                if "invert" in value:
                    return True, (
                        "experiment_summary.csv"
                    )

                if "normal" in value:
                    return False, (
                        "experiment_summary.csv"
                    )

    return False, "auto_fallback_normal"


def checkpoint_arguments(checkpoint):
    stored_args = checkpoint.get(
        "args",
        {},
    )

    if isinstance(stored_args, argparse.Namespace):
        stored_args = vars(stored_args)

    if not isinstance(stored_args, dict):
        raise RuntimeError(
            "Checkpoint args must be a dictionary or Namespace."
        )

    model_args = SimpleNamespace(
        **stored_args
    )

    defaults = {
        "fusion_mode": "adaptive_transformer",
        "feat_dim": 128,
        "controller_dim": 128,
        "routing_temperature": 1.0,
        "film_scale": 0.10,
        "router_heads": 4,
        "router_layers": 1,
        "router_dropout": 0.10,
        "stream_logit_residual_weight": 0.25,
        "use_prototypes": True,
        "prototype_temperature": 0.10,
        "use_uncertainty": True,
        "image_size": 224,
        "batch_size": 8,
        "num_workers": 0,
        "data_root": None,
        "streams": (
            "rgb,fingerprint,jpeg,freq"
        ),
    }

    for name, value in defaults.items():
        if not hasattr(model_args, name):
            setattr(
                model_args,
                name,
                value,
            )

    return model_args


def recover_stream_names(
    checkpoint,
    model_args,
):
    stream_names = checkpoint.get(
        "stream_names"
    )

    if stream_names is None:
        stream_names = [
            name.strip()
            for name in str(
                model_args.streams
            ).split(",")
            if name.strip()
        ]

    return list(stream_names)


def find_path_column(dataframe):
    candidates = [
        "image_path",
        "frame_path",
        "path",
        "filepath",
        "file_path",
    ]

    for column in candidates:
        if column in dataframe.columns:
            return column

    raise RuntimeError(
        "Could not locate an image-path column "
        "in the OOD manifest."
    )


def prepare_ood_manifest(
    raw_manifest,
    project_root,
    model_args,
):
    dataframe = raw_manifest.copy()

    if len(dataframe) == 0:
        raise RuntimeError(
            "The OOD manifest is empty."
        )

    path_column = find_path_column(
        dataframe
    )

    def resolve_image_path(value):
        path = Path(str(value))

        if path.is_absolute():
            return str(path)

        if model_args.data_root:
            candidate = (
                Path(model_args.data_root)
                / path
            )

            if candidate.exists():
                return str(candidate.resolve())

        return str(
            (
                project_root
                / path
            ).resolve()
        )

    dataframe["image_path"] = (
        dataframe[path_column]
        .map(resolve_image_path)
    )

    dataframe["label"] = 1
    dataframe["label_name"] = "fake"
    dataframe["split"] = "ood_fake_test"

    group_candidates = [
        "sample_unit_id",
        "group_id",
        "video_id",
        "clip_id",
    ]

    group_column = next(
        (
            column
            for column in group_candidates
            if column in dataframe.columns
        ),
        None,
    )

    if group_column is None:
        dataframe["group_id"] = (
            dataframe["image_path"]
            .astype(str)
        )
    else:
        dataframe["group_id"] = (
            dataframe[group_column]
            .astype(str)
        )

    if "sample_unit_id" not in dataframe.columns:
        dataframe["sample_unit_id"] = (
            dataframe["group_id"]
            .astype(str)
        )
    else:
        dataframe["sample_unit_id"] = (
            dataframe["sample_unit_id"]
            .astype(str)
        )

    defaults = {
        "method": "unknown",
        "domain": "ood_fake_test",
        "dataset_partition": "ood_fake_test",
        "top_level_folder": "Fake_Test",
    }

    for column, value in defaults.items():
        if column not in dataframe.columns:
            dataframe[column] = value

    missing_mask = ~dataframe[
        "image_path"
    ].map(
        lambda value: Path(value).exists()
    )

    if missing_mask.any():
        examples = (
            dataframe.loc[
                missing_mask,
                "image_path",
            ]
            .head(10)
            .tolist()
        )

        raise FileNotFoundError(
            f"{int(missing_mask.sum())} OOD images "
            "were not found. Examples:\n"
            + "\n".join(examples)
        )

    return dataframe


def aggregate_units(predictions):
    aggregation = {
        "group_id": "first",
        "label": "first",
        "fake_prob": "mean",
        "uncertainty": "mean",
        "method": "first",
        "domain": "first",
        "image_path": "count",
    }

    aggregation = {
        key: operation
        for key, operation in aggregation.items()
        if key in predictions.columns
    }

    units = (
        predictions
        .groupby(
            "sample_unit_id",
            as_index=False,
        )
        .agg(aggregation)
    )

    if "image_path" in units.columns:
        units = units.rename(
            columns={
                "image_path": "num_frames"
            }
        )

    return units


def add_decisions(
    dataframe,
    decision_threshold,
    uncertainty_threshold,
):
    dataframe = dataframe.copy()

    dataframe["predicted_label"] = (
        dataframe["fake_prob"]
        >= decision_threshold
    ).astype(int)

    dataframe["detected_fake"] = (
        dataframe["predicted_label"] == 1
    ).astype(int)

    dataframe["missed_fake"] = (
        dataframe["predicted_label"] == 0
    ).astype(int)

    dataframe["is_uncertain"] = (
        dataframe["uncertainty"]
        >= uncertainty_threshold
    ).astype(int)

    return dataframe


def fake_only_metric_row(
    dataframe,
    level,
    decision_threshold,
    uncertainty_threshold,
    score_inverted,
    method="ALL",
):
    return {
        "Level": level,
        "Method": method,
        "Samples": int(len(dataframe)),
        "Decision Threshold": float(
            decision_threshold
        ),
        "Score Inverted": bool(
            score_inverted
        ),
        "Fake Detection Rate": float(
            dataframe["detected_fake"].mean()
        ),
        "Recall (1)": float(
            dataframe["detected_fake"].mean()
        ),
        "Missed Fake Rate": float(
            dataframe["missed_fake"].mean()
        ),
        "Mean Fake Probability": float(
            dataframe["fake_prob"].mean()
        ),
        "Median Fake Probability": float(
            dataframe["fake_prob"].median()
        ),
        "Mean Uncertainty": float(
            dataframe["uncertainty"].mean()
        ),
        "Uncertain Rate": float(
            dataframe["is_uncertain"].mean()
        ),
        "Uncertainty Threshold": float(
            uncertainty_threshold
        ),
    }


def build_fake_only_metrics(
    frame_predictions,
    unit_predictions,
    decision_threshold,
    uncertainty_threshold,
    score_inverted,
):
    overall_rows = [
        fake_only_metric_row(
            frame_predictions,
            "Frame",
            decision_threshold,
            uncertainty_threshold,
            score_inverted,
        ),
        fake_only_metric_row(
            unit_predictions,
            "Unit",
            decision_threshold,
            uncertainty_threshold,
            score_inverted,
        ),
    ]

    method_rows = []

    for level, dataframe in [
        ("Frame", frame_predictions),
        ("Unit", unit_predictions),
    ]:
        if "method" not in dataframe.columns:
            continue

        for method, method_df in dataframe.groupby(
            "method",
            dropna=False,
        ):
            method_rows.append(
                fake_only_metric_row(
                    method_df,
                    level,
                    decision_threshold,
                    uncertainty_threshold,
                    score_inverted,
                    method=str(method),
                )
            )

    return (
        pd.DataFrame(overall_rows),
        pd.DataFrame(method_rows),
    )


def main():
    cli_args = parse_args()

    project_root = Path.cwd().resolve()

    checkpoint_path = resolve_path(
        cli_args.checkpoint,
        project_root,
    )

    model_script_path = resolve_path(
        cli_args.model_script,
        project_root,
    )

    ood_manifest_path = resolve_path(
        cli_args.ood_metadata_csv,
        project_root,
    )

    indomain_dir = (
        resolve_path(
            cli_args.indomain_dir,
            project_root,
        )
        if cli_args.indomain_dir
        else checkpoint_path.parent
    )

    if cli_args.out_dir:
        out_dir = resolve_path(
            cli_args.out_dir,
            project_root,
        )
    else:
        out_dir = (
            checkpoint_path.parent
            / "ood_fake_test"
        )

    out_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    for required_path in [
        checkpoint_path,
        model_script_path,
        ood_manifest_path,
    ]:
        if not required_path.exists():
            raise FileNotFoundError(
                required_path
            )

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    module = import_model_module(
        model_script_path
    )

    checkpoint = load_checkpoint(
        checkpoint_path,
        device,
    )

    model_args = checkpoint_arguments(
        checkpoint
    )

    stream_names = recover_stream_names(
        checkpoint,
        model_args,
    )

    model = module.build_model(
        model_args,
        stream_names,
    ).to(device)

    model.load_state_dict(
        checkpoint["model"],
        strict=True,
    )

    model.eval()

    if cli_args.decision_threshold is None:
        (
            decision_threshold,
            threshold_source,
        ) = recover_threshold(
            indomain_dir
        )
    else:
        decision_threshold = float(
            cli_args.decision_threshold
        )

        threshold_source = (
            "explicit_command_line"
        )

    (
        invert_scores,
        score_direction_source,
    ) = recover_score_inversion(
        indomain_dir,
        cli_args.score_direction,
    )

    raw_manifest = pd.read_csv(
        ood_manifest_path
    )

    ood_dataframe = prepare_ood_manifest(
        raw_manifest,
        project_root,
        model_args,
    )

    if (
        cli_args.frames_per_unit > 0
        or cli_args.limit_units > 0
    ):
        ood_dataframe = (
            module.sample_frames_per_unit(
                ood_dataframe,
                frames_per_unit=(
                    cli_args.frames_per_unit
                    if cli_args.frames_per_unit > 0
                    else None
                ),
                limit_units=(
                    cli_args.limit_units
                    if cli_args.limit_units > 0
                    else None
                ),
                seed=cli_args.seed,
                desc="Sample OOD Fake_Test",
                balanced_units=False,
            )
        )

    batch_size = (
        cli_args.batch_size
        if cli_args.batch_size is not None
        else int(model_args.batch_size)
    )

    num_workers = (
        cli_args.num_workers
        if cli_args.num_workers is not None
        else int(model_args.num_workers)
    )

    dataset = module.GRFNDataset(
        ood_dataframe,
        image_size=int(
            model_args.image_size
        ),
        train=False,
        train_aug=False,
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=(
            device.type == "cuda"
        ),
    )

    start_time = time.perf_counter()

    predictions = module.predict_loader(
        model,
        loader,
        model_args,
        device,
        split_name="ood_fake_test",
    )

    elapsed_seconds = (
        time.perf_counter()
        - start_time
    )

    if invert_scores:
        predictions = predictions.copy()

        predictions["fake_prob"] = (
            1.0
            - predictions["fake_prob"]
        )

    predictions = add_decisions(
        predictions,
        decision_threshold,
        cli_args.uncertainty_threshold,
    )

    units = aggregate_units(
        predictions
    )

    units = add_decisions(
        units,
        decision_threshold,
        cli_args.uncertainty_threshold,
    )

    (
        overall_metrics,
        per_method_metrics,
    ) = build_fake_only_metrics(
        predictions,
        units,
        decision_threshold,
        cli_args.uncertainty_threshold,
        invert_scores,
    )

    timing = pd.DataFrame(
        [
            {
                "Evaluation": (
                    "DF40 Fake_Test OOD"
                ),
                "Device": str(device),
                "Fusion Mode": (
                    model_args.fusion_mode
                ),
                "Image Size": int(
                    model_args.image_size
                ),
                "Batch Size": batch_size,
                "Frames Scored": len(
                    predictions
                ),
                "Units Scored": len(units),
                "Total Seconds": (
                    elapsed_seconds
                ),
                "Time Per Frame (ms)": (
                    elapsed_seconds
                    / max(
                        len(predictions),
                        1,
                    )
                    * 1000.0
                ),
                "FPS": (
                    len(predictions)
                    / max(
                        elapsed_seconds,
                        1e-8,
                    )
                ),
            }
        ]
    )

    run_configuration = {
        "checkpoint": str(
            checkpoint_path
        ),
        "model_script": str(
            model_script_path
        ),
        "ood_metadata_csv": str(
            ood_manifest_path
        ),
        "indomain_dir": str(
            indomain_dir
        ),
        "fusion_mode": str(
            model_args.fusion_mode
        ),
        "stream_names": stream_names,
        "decision_threshold": (
            decision_threshold
        ),
        "threshold_source": (
            threshold_source
        ),
        "score_inverted": (
            invert_scores
        ),
        "score_direction_source": (
            score_direction_source
        ),
        "uncertainty_threshold": (
            cli_args.uncertainty_threshold
        ),
        "frames_per_unit": (
            cli_args.frames_per_unit
        ),
        "limit_units": (
            cli_args.limit_units
        ),
        "seed": cli_args.seed,
        "device": str(device),
    }

    predictions.to_csv(
        out_dir
        / "ood_frame_predictions.csv",
        index=False,
    )

    units.to_csv(
        out_dir
        / "ood_video_predictions.csv",
        index=False,
    )

    overall_metrics.to_csv(
        out_dir
        / "ood_fake_only_metrics.csv",
        index=False,
    )

    per_method_metrics.to_csv(
        out_dir
        / "ood_per_method_metrics.csv",
        index=False,
    )

    timing.to_csv(
        out_dir
        / "ood_inference_time.csv",
        index=False,
    )

    ood_dataframe.to_csv(
        out_dir
        / "ood_sampled_manifest.csv",
        index=False,
    )

    with open(
        out_dir
        / "ood_run_configuration.json",
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            run_configuration,
            handle,
            indent=2,
        )

    diagnostic_paths = (
        module.save_diagnostic_csvs(
            predictions,
            out_dir,
        )
    )

    print("\n" + "=" * 80)
    print("A-GRFN FAKE-ONLY OOD EVALUATION COMPLETE")
    print("=" * 80)

    print("\nOutput directory:")
    print(out_dir)

    print("\nCheckpoint:")
    print(checkpoint_path)

    print("\nDecision threshold:")
    print(decision_threshold)

    print("\nThreshold source:")
    print(threshold_source)

    print("\nScore inverted:")
    print(invert_scores)

    print("\nFake-only metrics:")
    print(
        overall_metrics.to_string(
            index=False
        )
    )

    print("\nInference timing:")
    print(
        timing.to_string(
            index=False
        )
    )

    print("\nDiagnostic files:")

    for filename in sorted(
        diagnostic_paths
    ):
        print(" ", filename)

    print(
        "\nAUCROC, AUPRC, balanced accuracy, and EER "
        "are intentionally omitted because the OOD "
        "partition contains only fake samples."
    )


if __name__ == "__main__":
    main()
'''

evaluator_path.write_text(
    textwrap.dedent(
        evaluator_code
    ).lstrip(),
    encoding="utf-8",
)

# ============================================================
# Validate the generated script
# ============================================================

py_compile.compile(
    str(evaluator_path),
    doraise=True,
)

help_result = subprocess.run(
    [
        sys.executable,
        str(evaluator_path),
        "--help",
    ],
    cwd=str(project_root),
    capture_output=True,
    text=True,
)

if help_result.returncode != 0:
    print(help_result.stdout)
    print(help_result.stderr)

    raise RuntimeError(
        "The generated evaluator failed its --help check."
    )

required_arguments = [
    "--checkpoint",
    "--ood_metadata_csv",
    "--indomain_dir",
    "--out_dir",
    "--decision_threshold",
    "--score_direction",
    "--frames_per_unit",
    "--limit_units",
]

missing_arguments = [
    argument
    for argument in required_arguments
    if argument not in help_result.stdout
]

if missing_arguments:
    raise RuntimeError(
        "Generated evaluator is missing arguments: "
        + ", ".join(missing_arguments)
    )

print("\n" + "=" * 80)
print("REUSABLE OOD EVALUATOR SCRIPT READY")
print("=" * 80)

print("\nCreated:")
print(evaluator_path)

print("\nPASS: Python compilation succeeded.")
print("PASS: command-line help succeeded.")

print("\nExample full OOD command:")

print(
    f'{sys.executable} "{evaluator_path}" '
    '--checkpoint "/path/to/best_checkpoint.pt" '
    '--indomain_dir "/path/to/experiment_output" '
    f'--ood_metadata_csv "{project_root / "outputs_df40/df40_manifest_ood_fake_test.csv"}" '
    '--frames_per_unit 0 '
    '--limit_units 0'
)


REUSABLE OOD EVALUATOR SCRIPT READY

Created:
/home/jovyan/Deepfake_Low Resource_Low Resolution/evaluate_agrfn_ood.py

PASS: Python compilation succeeded.
PASS: command-line help succeeded.

Example full OOD command:
/opt/conda/bin/python "/home/jovyan/Deepfake_Low Resource_Low Resolution/evaluate_agrfn_ood.py" --checkpoint "/path/to/best_checkpoint.pt" --indomain_dir "/path/to/experiment_output" --ood_metadata_csv "/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40/df40_manifest_ood_fake_test.csv" --frames_per_unit 0 --limit_units 0


In [51]:
from pathlib import Path
from datetime import datetime
import shutil
import subprocess
import sys
import textwrap

import pandas as pd


# ============================================================
# Project paths
# ============================================================

project_root = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution"
)

training_script = (
    project_root
    / "agrfn_adaptive_routing_df40.py"
)

manifest_path = (
    project_root
    / "outputs_df40"
    / "df40_manifest_all_clean.csv"
)

plan_path = (
    project_root
    / "agrfn_ablation_plan.csv"
)

runner_path = (
    project_root
    / "run_agrfn_ablation_suite.py"
)

output_root = (
    project_root
    / "outputs_agrfn_ablation_suite"
)

if not training_script.exists():
    raise FileNotFoundError(training_script)

if not manifest_path.exists():
    raise FileNotFoundError(manifest_path)


# ============================================================
# Back up existing files
# ============================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

for path in [
    plan_path,
    runner_path,
]:
    if not path.exists():
        continue

    backup_path = path.with_name(
        f"{path.stem}_backup_{timestamp}{path.suffix}"
    )

    shutil.copy2(
        path,
        backup_path,
    )

    print("Backup created:")
    print(backup_path)


# ============================================================
# Define the paper ablation matrix
# ============================================================

experiments = [
    {
        "experiment_name": "E0_late_fusion",
        "fusion_mode": "late",
        "streams": "rgb,fingerprint,jpeg,freq",
        "use_prototypes": False,
        "use_uncertainty": False,
        "lambda_aux": 0.05,
        "lambda_route": 0.00,
        "lambda_cons": 0.02,
        "lambda_pred_cons": 0.02,
        "lambda_proto": 0.00,
        "lambda_unc": 0.00,
        "enabled": True,
        "description": (
            "Original GRFN-CSF late decision-aware fusion baseline."
        ),
    },
    {
        "experiment_name": "E1_average_fusion",
        "fusion_mode": "average",
        "streams": "rgb,fingerprint,jpeg,freq",
        "use_prototypes": False,
        "use_uncertainty": False,
        "lambda_aux": 0.05,
        "lambda_route": 0.00,
        "lambda_cons": 0.02,
        "lambda_pred_cons": 0.02,
        "lambda_proto": 0.00,
        "lambda_unc": 0.00,
        "enabled": True,
        "description": (
            "Non-adaptive multi-stream comparison baseline."
        ),
    },
    {
        "experiment_name": "E2_adaptive_scalar",
        "fusion_mode": "adaptive_scalar",
        "streams": "rgb,fingerprint,jpeg,freq",
        "use_prototypes": False,
        "use_uncertainty": False,
        "lambda_aux": 0.05,
        "lambda_route": 0.01,
        "lambda_cons": 0.02,
        "lambda_pred_cons": 0.02,
        "lambda_proto": 0.00,
        "lambda_unc": 0.00,
        "enabled": True,
        "description": (
            "Fingerprint-conditioned scalar evidence routing."
        ),
    },
    {
        "experiment_name": "E3_adaptive_film",
        "fusion_mode": "adaptive_film",
        "streams": "rgb,fingerprint,jpeg,freq",
        "use_prototypes": False,
        "use_uncertainty": False,
        "lambda_aux": 0.05,
        "lambda_route": 0.01,
        "lambda_cons": 0.02,
        "lambda_pred_cons": 0.02,
        "lambda_proto": 0.00,
        "lambda_unc": 0.00,
        "enabled": True,
        "description": (
            "Fingerprint-conditioned scalar routing and FiLM."
        ),
    },
    {
        "experiment_name": "E4_adaptive_transformer",
        "fusion_mode": "adaptive_transformer",
        "streams": "rgb,fingerprint,jpeg,freq",
        "use_prototypes": False,
        "use_uncertainty": False,
        "lambda_aux": 0.05,
        "lambda_route": 0.01,
        "lambda_cons": 0.02,
        "lambda_pred_cons": 0.02,
        "lambda_proto": 0.00,
        "lambda_unc": 0.00,
        "enabled": True,
        "description": (
            "Fingerprint and evidence token interaction through "
            "transformer routing."
        ),
    },
    {
        "experiment_name": "E5_transformer_prototypes",
        "fusion_mode": "adaptive_transformer",
        "streams": "rgb,fingerprint,jpeg,freq",
        "use_prototypes": True,
        "use_uncertainty": False,
        "lambda_aux": 0.05,
        "lambda_route": 0.01,
        "lambda_cons": 0.02,
        "lambda_pred_cons": 0.02,
        "lambda_proto": 0.10,
        "lambda_unc": 0.00,
        "enabled": True,
        "description": (
            "Transformer routing with real/fake prototype learning."
        ),
    },
    {
        "experiment_name": "E6_full_agrfn",
        "fusion_mode": "adaptive_transformer",
        "streams": "rgb,fingerprint,jpeg,freq",
        "use_prototypes": True,
        "use_uncertainty": True,
        "lambda_aux": 0.05,
        "lambda_route": 0.01,
        "lambda_cons": 0.02,
        "lambda_pred_cons": 0.02,
        "lambda_proto": 0.10,
        "lambda_unc": 0.10,
        "enabled": True,
        "description": (
            "Complete A-GRFN with transformer routing, FiLM, "
            "prototypes, and learned uncertainty."
        ),
    },
]

plan_df = pd.DataFrame(
    experiments
)

plan_df.to_csv(
    plan_path,
    index=False,
)

print("\nAblation plan created:")
print(plan_path)


# ============================================================
# Create the reusable suite runner
# ============================================================

runner_code = r'''
#!/usr/bin/env python3

import argparse
import shlex
import subprocess
import sys
from datetime import datetime
from pathlib import Path

import pandas as pd


def parse_args():
    parser = argparse.ArgumentParser(
        description=(
            "Run the standardized A-GRFN ablation suite."
        )
    )

    parser.add_argument(
        "--project_root",
        type=str,
        default=".",
    )

    parser.add_argument(
        "--train_script",
        type=str,
        default="agrfn_adaptive_routing_df40.py",
    )

    parser.add_argument(
        "--metadata_csv",
        type=str,
        default=(
            "outputs_df40/"
            "df40_manifest_all_clean.csv"
        ),
    )

    parser.add_argument(
        "--plan_csv",
        type=str,
        default="agrfn_ablation_plan.csv",
    )

    parser.add_argument(
        "--output_root",
        type=str,
        default="outputs_agrfn_ablation_suite",
    )

    parser.add_argument(
        "--experiments",
        type=str,
        default="all",
        help=(
            "Comma-separated experiment names or 'all'."
        ),
    )

    parser.add_argument(
        "--execute",
        action="store_true",
        help=(
            "Actually execute training. Without this flag, "
            "the runner performs a dry run."
        ),
    )

    parser.add_argument(
        "--skip_completed",
        action="store_true",
        help=(
            "Skip experiments that already contain "
            "test_metrics.csv and experiment_summary.csv."
        ),
    )

    parser.add_argument(
        "--resume_existing",
        action="store_true",
        help=(
            "Resume from latest_checkpoint.pt when present."
        ),
    )

    parser.add_argument(
        "--stop_on_error",
        action="store_true",
    )

    parser.add_argument(
        "--python_executable",
        type=str,
        default=sys.executable,
    )

    parser.add_argument(
        "training_arguments",
        nargs=argparse.REMAINDER,
        help=(
            "Additional arguments forwarded to the training "
            "script. Place them after '--'."
        ),
    )

    return parser.parse_args()


def resolve_path(
    value,
    project_root,
):
    path = Path(value).expanduser()

    if not path.is_absolute():
        path = project_root / path

    return path.resolve()


def as_bool(value):
    if isinstance(value, bool):
        return value

    return str(value).strip().lower() in {
        "1",
        "true",
        "yes",
        "y",
    }


def save_status(
    status_rows,
    output_root,
):
    output_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    status_path = (
        output_root
        / "ablation_run_status.csv"
    )

    pd.DataFrame(
        status_rows
    ).to_csv(
        status_path,
        index=False,
    )

    return status_path


def main():
    args = parse_args()

    project_root = Path(
        args.project_root
    ).expanduser().resolve()

    train_script = resolve_path(
        args.train_script,
        project_root,
    )

    metadata_csv = resolve_path(
        args.metadata_csv,
        project_root,
    )

    plan_csv = resolve_path(
        args.plan_csv,
        project_root,
    )

    output_root = resolve_path(
        args.output_root,
        project_root,
    )

    for required_path in [
        train_script,
        metadata_csv,
        plan_csv,
    ]:
        if not required_path.exists():
            raise FileNotFoundError(
                required_path
            )

    plan = pd.read_csv(
        plan_csv
    )

    required_columns = {
        "experiment_name",
        "fusion_mode",
        "streams",
        "use_prototypes",
        "use_uncertainty",
        "lambda_aux",
        "lambda_route",
        "lambda_cons",
        "lambda_pred_cons",
        "lambda_proto",
        "lambda_unc",
        "enabled",
    }

    missing_columns = (
        required_columns
        - set(plan.columns)
    )

    if missing_columns:
        raise RuntimeError(
            "Ablation plan is missing columns: "
            + ", ".join(
                sorted(missing_columns)
            )
        )

    plan = plan[
        plan["enabled"].map(as_bool)
    ].copy()

    if args.experiments.strip().lower() != "all":
        requested = {
            value.strip()
            for value in args.experiments.split(",")
            if value.strip()
        }

        available = set(
            plan["experiment_name"]
            .astype(str)
        )

        unknown = (
            requested - available
        )

        if unknown:
            raise ValueError(
                "Unknown experiments: "
                + ", ".join(sorted(unknown))
            )

        plan = plan[
            plan["experiment_name"]
            .astype(str)
            .isin(requested)
        ].copy()

    if len(plan) == 0:
        raise RuntimeError(
            "No enabled experiments were selected."
        )

    extra_arguments = list(
        args.training_arguments
    )

    if (
        extra_arguments
        and extra_arguments[0] == "--"
    ):
        extra_arguments = (
            extra_arguments[1:]
        )

    status_rows = []

    print("=" * 80)
    print("A-GRFN ABLATION SUITE")
    print("=" * 80)

    print("\nExecution mode:")
    print(
        "EXECUTE"
        if args.execute
        else "DRY RUN"
    )

    print("\nSelected experiments:")
    for name in plan["experiment_name"]:
        print(" ", name)

    for _, experiment in plan.iterrows():
        experiment_name = str(
            experiment["experiment_name"]
        )

        experiment_dir = (
            output_root
            / experiment_name
        )

        completed_files = [
            experiment_dir
            / "test_metrics.csv",

            experiment_dir
            / "experiment_summary.csv",
        ]

        if (
            args.skip_completed
            and all(
                path.exists()
                for path in completed_files
            )
        ):
            status_rows.append(
                {
                    "experiment_name": (
                        experiment_name
                    ),
                    "fusion_mode": (
                        experiment["fusion_mode"]
                    ),
                    "status": "skipped_completed",
                    "return_code": 0,
                    "output_dir": str(
                        experiment_dir
                    ),
                    "command": "",
                    "started_at": "",
                    "finished_at": "",
                }
            )

            save_status(
                status_rows,
                output_root,
            )

            print(
                f"\nSKIP: {experiment_name}"
            )

            continue

        command = [
            args.python_executable,
            str(train_script),

            "--metadata_csv",
            str(metadata_csv),

            "--out_dir",
            str(experiment_dir),

            "--dataset_name",
            "DF40",

            "--fusion_mode",
            str(
                experiment[
                    "fusion_mode"
                ]
            ),

            "--streams",
            str(
                experiment["streams"]
            ),

            "--lambda_aux",
            str(
                experiment[
                    "lambda_aux"
                ]
            ),

            "--lambda_route",
            str(
                experiment[
                    "lambda_route"
                ]
            ),

            "--lambda_cons",
            str(
                experiment[
                    "lambda_cons"
                ]
            ),

            "--lambda_pred_cons",
            str(
                experiment[
                    "lambda_pred_cons"
                ]
            ),

            "--lambda_proto",
            str(
                experiment[
                    "lambda_proto"
                ]
            ),

            "--lambda_unc",
            str(
                experiment[
                    "lambda_unc"
                ]
            ),
        ]

        if as_bool(
            experiment[
                "use_prototypes"
            ]
        ):
            command.append(
                "--use_prototypes"
            )

        if as_bool(
            experiment[
                "use_uncertainty"
            ]
        ):
            command.append(
                "--use_uncertainty"
            )

        latest_checkpoint = (
            experiment_dir
            / "latest_checkpoint.pt"
        )

        if (
            args.resume_existing
            and latest_checkpoint.exists()
        ):
            command.extend(
                [
                    "--resume",
                    str(
                        latest_checkpoint
                    ),
                ]
            )

        # Forwarded arguments are appended last so they can
        # override suite-level defaults when argparse permits it.
        command.extend(
            extra_arguments
        )

        printable_command = shlex.join(
            command
        )

        print("\n" + "-" * 80)
        print(experiment_name)
        print("-" * 80)
        print(printable_command)

        started_at = datetime.now().isoformat(
            timespec="seconds"
        )

        if not args.execute:
            return_code = 0
            status = "planned"

        else:
            experiment_dir.mkdir(
                parents=True,
                exist_ok=True,
            )

            result = subprocess.run(
                command,
                cwd=str(project_root),
            )

            return_code = int(
                result.returncode
            )

            status = (
                "completed"
                if return_code == 0
                else "failed"
            )

        finished_at = datetime.now().isoformat(
            timespec="seconds"
        )

        status_rows.append(
            {
                "experiment_name": (
                    experiment_name
                ),
                "fusion_mode": str(
                    experiment[
                        "fusion_mode"
                    ]
                ),
                "status": status,
                "return_code": (
                    return_code
                ),
                "output_dir": str(
                    experiment_dir
                ),
                "command": (
                    printable_command
                ),
                "started_at": (
                    started_at
                ),
                "finished_at": (
                    finished_at
                ),
            }
        )

        status_path = save_status(
            status_rows,
            output_root,
        )

        if (
            return_code != 0
            and args.stop_on_error
        ):
            raise RuntimeError(
                f"Experiment failed: "
                f"{experiment_name}"
            )

    print("\n" + "=" * 80)
    print("ABLATION SUITE PLAN COMPLETE")
    print("=" * 80)

    print("\nStatus file:")
    print(status_path)

    if not args.execute:
        print(
            "\nNo training was started. Add --execute "
            "when the final training configuration is ready."
        )


if __name__ == "__main__":
    main()
'''

runner_path.write_text(
    textwrap.dedent(
        runner_code
    ).lstrip(),
    encoding="utf-8",
)

print("\nAblation runner created:")
print(runner_path)


# ============================================================
# Compile the runner
# ============================================================

compile_result = subprocess.run(
    [
        sys.executable,
        "-m",
        "py_compile",
        str(runner_path),
    ],
    cwd=str(project_root),
    capture_output=True,
    text=True,
)

if compile_result.returncode != 0:
    print(compile_result.stdout)
    print(compile_result.stderr)

    raise RuntimeError(
        "Ablation runner compilation failed."
    )

print("\nPASS: runner compilation succeeded.")


# ============================================================
# Perform a safe dry run
# ============================================================

dry_run_command = [
    sys.executable,
    str(runner_path),

    "--project_root",
    str(project_root),

    "--train_script",
    str(training_script),

    "--metadata_csv",
    str(manifest_path),

    "--plan_csv",
    str(plan_path),

    "--output_root",
    str(output_root),

    "--experiments",
    "all",
]

dry_run = subprocess.run(
    dry_run_command,
    cwd=str(project_root),
    capture_output=True,
    text=True,
)

print("\n" + dry_run.stdout)

if dry_run.returncode != 0:
    print(dry_run.stderr)

    raise RuntimeError(
        "Ablation-suite dry run failed."
    )


# ============================================================
# Validate the generated status file
# ============================================================

status_path = (
    output_root
    / "ablation_run_status.csv"
)

if not status_path.exists():
    raise RuntimeError(
        "Dry run did not create ablation_run_status.csv."
    )

status_df = pd.read_csv(
    status_path
)

if len(status_df) != len(
    experiments
):
    raise RuntimeError(
        "Dry-run status row count does not match "
        "the ablation plan."
    )

if set(
    status_df["status"]
) != {"planned"}:
    raise RuntimeError(
        "Dry run unexpectedly executed or skipped experiments."
    )

expected_names = {
    experiment[
        "experiment_name"
    ]
    for experiment in experiments
}

observed_names = set(
    status_df[
        "experiment_name"
    ]
)

if observed_names != expected_names:
    raise RuntimeError(
        "Dry-run experiment names do not match the plan."
    )


# ============================================================
# Final report
# ============================================================

print("\n" + "=" * 80)
print("A-GRFN ABLATION SUITE READY")
print("=" * 80)

print("\nPlan:")
print(plan_path)

print("\nRunner:")
print(runner_path)

print("\nStatus:")
print(status_path)

print("\nExperiments:")
display(
    plan_df[
        [
            "experiment_name",
            "fusion_mode",
            "use_prototypes",
            "use_uncertainty",
            "description",
        ]
    ]
)

print(
    "\nPASS: all seven experiments were generated "
    "and validated in dry-run mode."
)

print(
    "PASS: no training jobs were started."
)


Ablation plan created:
/home/jovyan/Deepfake_Low Resource_Low Resolution/agrfn_ablation_plan.csv

Ablation runner created:
/home/jovyan/Deepfake_Low Resource_Low Resolution/run_agrfn_ablation_suite.py

PASS: runner compilation succeeded.

A-GRFN ABLATION SUITE

Execution mode:
DRY RUN

Selected experiments:
  E0_late_fusion
  E1_average_fusion
  E2_adaptive_scalar
  E3_adaptive_film
  E4_adaptive_transformer
  E5_transformer_prototypes
  E6_full_agrfn

--------------------------------------------------------------------------------
E0_late_fusion
--------------------------------------------------------------------------------
/opt/conda/bin/python '/home/jovyan/Deepfake_Low Resource_Low Resolution/agrfn_adaptive_routing_df40.py' --metadata_csv '/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40/df40_manifest_all_clean.csv' --out_dir '/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_ablation_suite/E0_late_fusion' --dataset_name DF40 --fusion_mode late --strea

,experiment_name,fusion_mode,use_prototypes,use_uncertainty,description
0,E0_late_fusion,late,False,False,Original GRFN-CSF late decision-aware fusion b...
1,E1_average_fusion,average,False,False,Non-adaptive multi-stream comparison baseline.
2,E2_adaptive_scalar,adaptive_scalar,False,False,Fingerprint-conditioned scalar evidence routing.
3,E3_adaptive_film,adaptive_film,False,False,Fingerprint-conditioned scalar routing and FiLM.
4,E4_adaptive_transformer,adaptive_transformer,False,False,Fingerprint and evidence token interaction thr...
5,E5_transformer_prototypes,adaptive_transformer,True,False,Transformer routing with real/fake prototype l...
6,E6_full_agrfn,adaptive_transformer,True,True,"Complete A-GRFN with transformer routing, FiLM..."



PASS: all seven experiments were generated and validated in dry-run mode.
PASS: no training jobs were started.


In [52]:
from pathlib import Path
from datetime import datetime
import subprocess
import sys

import numpy as np
import pandas as pd


# ============================================================
# Paths
# ============================================================

project_root = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution"
)

runner_path = (
    project_root
    / "run_agrfn_ablation_suite.py"
)

training_script = (
    project_root
    / "agrfn_adaptive_routing_df40.py"
)

plan_path = (
    project_root
    / "agrfn_ablation_plan.csv"
)

manifest_path = (
    project_root
    / "outputs_df40"
    / "df40_manifest_all_clean.csv"
)

required_paths = [
    runner_path,
    training_script,
    plan_path,
    manifest_path,
]

for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(path)


# ============================================================
# Create an isolated smoke-test output root
# ============================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

smoke_output_root = (
    project_root
    / f"outputs_agrfn_ablation_micro_smoke_{timestamp}"
)

print("=" * 80)
print("RUNNING SEVEN-EXPERIMENT MICRO ABLATION SUITE")
print("=" * 80)

print("\nSmoke output root:")
print(smoke_output_root)


# ============================================================
# Execute all seven tiny experiments
# ============================================================

command = [
    sys.executable,
    str(runner_path),

    "--project_root",
    str(project_root),

    "--train_script",
    str(training_script),

    "--metadata_csv",
    str(manifest_path),

    "--plan_csv",
    str(plan_path),

    "--output_root",
    str(smoke_output_root),

    "--experiments",
    "all",

    "--execute",
    "--stop_on_error",

    "--",

    "--epochs",
    "1",

    "--frames_per_video",
    "1",

    "--limit_train_units",
    "8",

    "--limit_val_units",
    "4",

    "--limit_test_units",
    "4",

    "--balanced_unit_limits",

    "--image_size",
    "64",

    "--batch_size",
    "2",

    "--num_workers",
    "0",

    "--feat_dim",
    "64",

    "--controller_dim",
    "64",

    "--router_heads",
    "4",

    "--router_layers",
    "1",

    "--router_dropout",
    "0.10",

    "--lr",
    "1e-4",

    "--balance_strategy",
    "auto",

    "--decision_threshold",
    "0.5",

    "--score_direction",
    "normal",

    "--seed",
    "42",
]

print("\nCommand:")
print(
    " ".join(
        f'"{value}"'
        if " " in value
        else value
        for value in command
    )
)

result = subprocess.run(
    command,
    cwd=str(project_root),
)

if result.returncode != 0:
    raise RuntimeError(
        "Micro ablation suite failed with return code "
        f"{result.returncode}."
    )


# ============================================================
# Validate suite status
# ============================================================

status_path = (
    smoke_output_root
    / "ablation_run_status.csv"
)

if not status_path.exists():
    raise RuntimeError(
        "The suite did not create ablation_run_status.csv."
    )

status = pd.read_csv(
    status_path
)

expected_experiments = [
    "E0_late_fusion",
    "E1_average_fusion",
    "E2_adaptive_scalar",
    "E3_adaptive_film",
    "E4_adaptive_transformer",
    "E5_transformer_prototypes",
    "E6_full_agrfn",
]

missing_status_rows = [
    experiment
    for experiment in expected_experiments
    if experiment
    not in set(
        status["experiment_name"]
        .astype(str)
    )
]

if missing_status_rows:
    raise RuntimeError(
        "Missing experiment status rows: "
        + ", ".join(missing_status_rows)
    )

failed_rows = status[
    status["status"] != "completed"
]

if len(failed_rows) > 0:
    display(failed_rows)

    raise RuntimeError(
        "One or more micro experiments did not complete."
    )

if not (
    status["return_code"]
    .fillna(-1)
    .astype(int)
    == 0
).all():
    raise RuntimeError(
        "One or more experiments returned a nonzero code."
    )


# ============================================================
# Validate common output files
# ============================================================

common_output_files = [
    "best_checkpoint.pt",
    "latest_checkpoint.pt",
    "training_history.csv",
    "test_frame_predictions.csv",
    "test_video_predictions.csv",
    "test_metrics.csv",
    "test_inference_time.csv",
    "experiment_summary.csv",
]

missing_outputs = []

for experiment_name in expected_experiments:
    experiment_dir = (
        smoke_output_root
        / experiment_name
    )

    for filename in common_output_files:
        output_path = (
            experiment_dir
            / filename
        )

        if not output_path.exists():
            missing_outputs.append(
                str(output_path)
            )

if missing_outputs:
    raise RuntimeError(
        "Missing micro-suite outputs:\n"
        + "\n".join(missing_outputs)
    )


# ============================================================
# Validate training histories
# ============================================================

history_rows = []

for experiment_name in expected_experiments:
    experiment_dir = (
        smoke_output_root
        / experiment_name
    )

    history_path = (
        experiment_dir
        / "training_history.csv"
    )

    history = pd.read_csv(
        history_path
    )

    if len(history) != 1:
        raise RuntimeError(
            f"{experiment_name} should contain exactly "
            "one smoke-test epoch."
        )

    numeric_columns = history.select_dtypes(
        include=[np.number]
    ).columns

    if not np.isfinite(
        history[
            numeric_columns
        ].to_numpy(dtype=float)
    ).all():
        raise RuntimeError(
            f"{experiment_name} training history "
            "contains NaN or infinity."
        )

    row = {
        "experiment_name": experiment_name,
        "train_total": float(
            history[
                "train_total"
            ].iloc[-1]
        ),
        "train_cls": float(
            history[
                "train_cls"
            ].iloc[-1]
        ),
        "train_proto": float(
            history[
                "train_proto"
            ].iloc[-1]
        )
        if "train_proto" in history.columns
        else 0.0,
        "train_unc": float(
            history[
                "train_unc"
            ].iloc[-1]
        )
        if "train_unc" in history.columns
        else 0.0,
    }

    history_rows.append(row)

history_summary = pd.DataFrame(
    history_rows
)


# ============================================================
# Validate prototype and uncertainty activation
# ============================================================

prototype_experiments = {
    "E5_transformer_prototypes",
    "E6_full_agrfn",
}

uncertainty_experiments = {
    "E6_full_agrfn",
}

for _, row in history_summary.iterrows():
    experiment_name = row[
        "experiment_name"
    ]

    prototype_loss = float(
        row["train_proto"]
    )

    uncertainty_loss = float(
        row["train_unc"]
    )

    if experiment_name in prototype_experiments:
        if prototype_loss <= 0.0:
            raise RuntimeError(
                f"{experiment_name} did not activate "
                "prototype learning."
            )
    elif abs(prototype_loss) > 1e-10:
        raise RuntimeError(
            f"{experiment_name} unexpectedly activated "
            "prototype learning."
        )

    if experiment_name in uncertainty_experiments:
        if uncertainty_loss <= 0.0:
            raise RuntimeError(
                f"{experiment_name} did not activate "
                "uncertainty learning."
            )
    elif abs(uncertainty_loss) > 1e-10:
        raise RuntimeError(
            f"{experiment_name} unexpectedly activated "
            "uncertainty learning."
        )


# ============================================================
# Validate specialized diagnostic files
# ============================================================

for experiment_name in prototype_experiments:
    prototype_path = (
        smoke_output_root
        / experiment_name
        / "prototype_distances.csv"
    )

    if not prototype_path.exists():
        raise RuntimeError(
            f"{experiment_name} did not save "
            "prototype_distances.csv."
        )

full_uncertainty_path = (
    smoke_output_root
    / "E6_full_agrfn"
    / "uncertainty_breakdown.csv"
)

if not full_uncertainty_path.exists():
    raise RuntimeError(
        "E6_full_agrfn did not save "
        "uncertainty_breakdown.csv."
    )


# ============================================================
# Final report
# ============================================================

print("\n" + "=" * 80)
print("A-GRFN MICRO ABLATION SUITE PASSED")
print("=" * 80)

print("\nSmoke output root:")
print(smoke_output_root)

print("\nExecution status:")
display(
    status[
        [
            "experiment_name",
            "fusion_mode",
            "status",
            "return_code",
        ]
    ]
)

print("\nTraining-loss activation:")
display(history_summary)

print(
    "\nPASS: all seven launcher commands completed "
    "real train, validation, checkpoint, and test cycles."
)

print(
    "PASS: prototypes activated only for E5 and E6."
)

print(
    "PASS: learned uncertainty activated only for E6."
)

print(
    "\nThese outputs are plumbing tests only and should "
    "not be used as paper results."
)

RUNNING SEVEN-EXPERIMENT MICRO ABLATION SUITE

Smoke output root:
/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_ablation_micro_smoke_20260730_222509

Command:
/opt/conda/bin/python "/home/jovyan/Deepfake_Low Resource_Low Resolution/run_agrfn_ablation_suite.py" --project_root "/home/jovyan/Deepfake_Low Resource_Low Resolution" --train_script "/home/jovyan/Deepfake_Low Resource_Low Resolution/agrfn_adaptive_routing_df40.py" --metadata_csv "/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40/df40_manifest_all_clean.csv" --plan_csv "/home/jovyan/Deepfake_Low Resource_Low Resolution/agrfn_ablation_plan.csv" --output_root "/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_ablation_micro_smoke_20260730_222509" --experiments all --execute --stop_on_error -- --epochs 1 --frames_per_video 1 --limit_train_units 8 --limit_val_units 4 --limit_test_units 4 --balanced_unit_limits --image_size 64 --batch_size 2 --num_workers 0 --feat_dim 64 --controller_dim 6

Sample test: 100%|██████████| 4/4 [00:00<00:00, 353.10unit/s]


Sample test: frames=4, units=4, real=2, fake=2

Balancing strategy
Train class counts [real, fake]: [4, 4]
Train imbalance ratio: 1.000
Use class-weighted loss: False
Use weighted sampler: False

Building dataloaders
Train batches: 4
Val batches: 2
Test batches: 2

Building model
Model mode: grfn
Loss mode: grfn
Active streams: rgb, fingerprint, jpeg, freq
Total params: 1,566,603
Trainable params: 1,566,603

Training


Predict val: 100%|██████████| 2/2 [00:00<00:00, 10.45it/s]


New best val frame AUCROC: 0.7500

Validation metrics:
Level  Decision Threshold  Accuracy  Balanced Accuracy    AUPRC  AUCROC  EER (Equal Error Rate)  Precision (0)  Recall (0)  F1 (0)  Precision (1)  Recall (1)   F1 (1)  Uncertain Rate (%)  Predicted Real Count  Predicted Fake Count
Frame                 0.5       0.5                0.5 0.833333    0.75                     0.5            0.0         0.0     0.0            0.5         1.0 0.666667               100.0                     0                     4
Video                 0.5       0.5                0.5 0.833333    0.75                     0.5            0.0         0.0     0.0            0.5         1.0 0.666667               100.0                     0                     4

Final test evaluation
Loading best checkpoint for final test: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_ablation_micro_smoke_20260730_222509/E0_late_fusion/best_checkpoint.pt


Predict test: 100%|██████████| 2/2 [00:00<00:00, 15.45it/s]


... (37 outputs left out)

,experiment_name,fusion_mode,status,return_code
0,E0_late_fusion,late,completed,0
1,E1_average_fusion,average,completed,0
2,E2_adaptive_scalar,adaptive_scalar,completed,0
3,E3_adaptive_film,adaptive_film,completed,0
4,E4_adaptive_transformer,adaptive_transformer,completed,0
5,E5_transformer_prototypes,adaptive_transformer,completed,0
6,E6_full_agrfn,adaptive_transformer,completed,0



Training-loss activation:


,experiment_name,train_total,train_cls,train_proto,train_unc
0,E0_late_fusion,0.742385,0.705832,0.000000,0.000000
1,E1_average_fusion,0.746010,0.709519,0.000000,0.000000
2,E2_adaptive_scalar,0.781239,0.743819,0.000000,0.000000
3,E3_adaptive_film,0.802322,0.765207,0.000000,0.000000
4,E4_adaptive_transformer,0.835702,0.798926,0.000000,0.000000
5,E5_transformer_prototypes,0.978172,0.758405,1.836885,0.000000
6,E6_full_agrfn,0.997938,0.719672,1.723973,0.693966



PASS: all seven launcher commands completed real train, validation, checkpoint, and test cycles.
PASS: prototypes activated only for E5 and E6.
PASS: learned uncertainty activated only for E6.

These outputs are plumbing tests only and should not be used as paper results.


In [53]:
from pathlib import Path
from datetime import datetime
import hashlib
import json

import numpy as np
import pandas as pd


# ============================================================
# Paths
# ============================================================

project_root = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution"
)

manifest_path = (
    project_root
    / "outputs_df40"
    / "df40_manifest_all_clean.csv"
)

audit_dir = (
    project_root
    / "outputs_agrfn_split_audit"
)

audit_dir.mkdir(
    parents=True,
    exist_ok=True,
)

if not manifest_path.exists():
    raise FileNotFoundError(manifest_path)


# ============================================================
# Load canonical in-domain manifest
# ============================================================

manifest = pd.read_csv(
    manifest_path
)

if len(manifest) == 0:
    raise RuntimeError(
        "The canonical manifest is empty."
    )

print("=" * 80)
print("AUDITING CANONICAL DF40 SPLIT")
print("=" * 80)

print("\nManifest:")
print(manifest_path)

print("\nRows:")
print(len(manifest))


# ============================================================
# Detect required columns
# ============================================================

split_candidates = [
    "split",
    "dataset_split",
    "subset",
    "partition",
]

split_column = next(
    (
        column
        for column in split_candidates
        if column in manifest.columns
    ),
    None,
)

if split_column is None:
    raise RuntimeError(
        "No split column was found. Expected one of:\n"
        + ", ".join(split_candidates)
        + "\n\nAvailable columns:\n"
        + ", ".join(manifest.columns)
    )

group_candidates = [
    "sample_unit_id",
    "group_id",
    "video_id",
    "clip_id",
]

group_column = next(
    (
        column
        for column in group_candidates
        if column in manifest.columns
    ),
    None,
)

if group_column is None:
    raise RuntimeError(
        "No unit/group column was found. Expected one of:\n"
        + ", ".join(group_candidates)
    )

path_candidates = [
    "image_path",
    "frame_path",
    "path",
    "filepath",
    "file_path",
]

path_column = next(
    (
        column
        for column in path_candidates
        if column in manifest.columns
    ),
    None,
)

if path_column is None:
    raise RuntimeError(
        "No image-path column was found."
    )

if "label" not in manifest.columns:
    raise RuntimeError(
        "The canonical manifest has no label column."
    )

print("\nDetected columns:")
print("Split:", split_column)
print("Group:", group_column)
print("Path:", path_column)
print("Label: label")


# ============================================================
# Ensure the manifest contains no OOD rows
# ============================================================

if "dataset_partition" in manifest.columns:
    partition_values = (
        manifest["dataset_partition"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    ood_mask = partition_values.str.contains(
        "ood",
        regex=False,
    )

    if ood_mask.any():
        examples = (
            manifest.loc[
                ood_mask,
                [
                    path_column,
                    "dataset_partition",
                ],
            ]
            .head(10)
        )

        display(examples)

        raise RuntimeError(
            "The canonical in-domain manifest contains "
            f"{int(ood_mask.sum())} OOD rows."
        )


# ============================================================
# Normalize labels and split names
# ============================================================

def normalize_label(value):
    normalized = str(value).strip().lower()

    real_values = {
        "0",
        "0.0",
        "real",
        "authentic",
        "genuine",
    }

    fake_values = {
        "1",
        "1.0",
        "fake",
        "manipulated",
        "deepfake",
    }

    if normalized in real_values:
        return 0

    if normalized in fake_values:
        return 1

    raise ValueError(
        f"Unsupported label value: {value!r}"
    )


def normalize_split(value):
    normalized = str(value).strip().lower()

    split_mapping = {
        "train": "train",
        "training": "train",

        "val": "val",
        "valid": "val",
        "validation": "val",
        "dev": "val",

        "test": "test",
        "testing": "test",
    }

    if normalized not in split_mapping:
        raise ValueError(
            f"Unsupported split value: {value!r}"
        )

    return split_mapping[normalized]


canonical = manifest.copy()

canonical["label"] = canonical[
    "label"
].map(normalize_label)

canonical["split"] = canonical[
    split_column
].map(normalize_split)

canonical["group_id_canonical"] = (
    canonical[group_column]
    .astype(str)
    .str.strip()
)

canonical["image_path_canonical"] = (
    canonical[path_column]
    .astype(str)
    .str.strip()
)

if canonical[
    "group_id_canonical"
].eq("").any():
    raise RuntimeError(
        "One or more rows have an empty group identifier."
    )

if canonical[
    "image_path_canonical"
].eq("").any():
    raise RuntimeError(
        "One or more rows have an empty image path."
    )


# ============================================================
# Required split and class checks
# ============================================================

required_splits = {
    "train",
    "val",
    "test",
}

observed_splits = set(
    canonical["split"].unique()
)

missing_splits = (
    required_splits
    - observed_splits
)

if missing_splits:
    raise RuntimeError(
        "Missing canonical splits: "
        + ", ".join(
            sorted(missing_splits)
        )
    )

for split_name in [
    "train",
    "val",
    "test",
]:
    split_labels = set(
        canonical.loc[
            canonical["split"] == split_name,
            "label",
        ].unique()
    )

    if split_labels != {0, 1}:
        raise RuntimeError(
            f"Split '{split_name}' does not contain "
            "both real and fake classes. "
            f"Observed labels: {sorted(split_labels)}"
        )


# ============================================================
# Leakage checks
# ============================================================

group_split_counts = (
    canonical.groupby(
        "group_id_canonical"
    )["split"]
    .nunique()
)

leaking_groups = (
    group_split_counts[
        group_split_counts > 1
    ]
)

if len(leaking_groups) > 0:
    leakage_examples = (
        canonical[
            canonical[
                "group_id_canonical"
            ].isin(
                leaking_groups.index
            )
        ][
            [
                "group_id_canonical",
                "split",
                "label",
                "image_path_canonical",
            ]
        ]
        .sort_values(
            [
                "group_id_canonical",
                "split",
            ]
        )
        .head(30)
    )

    display(leakage_examples)

    raise RuntimeError(
        f"Group leakage detected for "
        f"{len(leaking_groups)} groups."
    )

path_split_counts = (
    canonical.groupby(
        "image_path_canonical"
    )["split"]
    .nunique()
)

leaking_paths = (
    path_split_counts[
        path_split_counts > 1
    ]
)

if len(leaking_paths) > 0:
    raise RuntimeError(
        f"{len(leaking_paths)} image paths occur "
        "in multiple splits."
    )

group_label_counts = (
    canonical.groupby(
        "group_id_canonical"
    )["label"]
    .nunique()
)

mixed_label_groups = (
    group_label_counts[
        group_label_counts > 1
    ]
)

if len(mixed_label_groups) > 0:
    raise RuntimeError(
        f"{len(mixed_label_groups)} groups contain "
        "both real and fake labels."
    )


# ============================================================
# Duplicate checks
# ============================================================

exact_duplicate_rows = int(
    canonical.duplicated(
        subset=[
            "image_path_canonical",
            "split",
            "label",
        ]
    ).sum()
)

duplicate_path_rows = int(
    canonical.duplicated(
        subset=[
            "image_path_canonical",
        ]
    ).sum()
)

if duplicate_path_rows > 0:
    duplicate_examples = (
        canonical[
            canonical.duplicated(
                subset=[
                    "image_path_canonical",
                ],
                keep=False,
            )
        ][
            [
                "image_path_canonical",
                "split",
                "label",
                "group_id_canonical",
            ]
        ]
        .sort_values(
            "image_path_canonical"
        )
        .head(30)
    )

    display(duplicate_examples)

    raise RuntimeError(
        f"Duplicate image paths detected: "
        f"{duplicate_path_rows} duplicate rows."
    )


# ============================================================
# Split summaries
# ============================================================

frame_summary = (
    canonical.groupby(
        [
            "split",
            "label",
        ],
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size": "frames",
        }
    )
)

unit_summary = (
    canonical[
        [
            "split",
            "label",
            "group_id_canonical",
        ]
    ]
    .drop_duplicates()
    .groupby(
        [
            "split",
            "label",
        ],
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size": "units",
        }
    )
)

split_summary = frame_summary.merge(
    unit_summary,
    on=[
        "split",
        "label",
    ],
    how="outer",
)

split_summary["label_name"] = (
    split_summary["label"].map(
        {
            0: "real",
            1: "fake",
        }
    )
)

split_summary = split_summary[
    [
        "split",
        "label",
        "label_name",
        "frames",
        "units",
    ]
].sort_values(
    [
        "split",
        "label",
    ]
)


# ============================================================
# Overall unit ratios
# ============================================================

units_by_split = (
    canonical[
        [
            "split",
            "group_id_canonical",
        ]
    ]
    .drop_duplicates()
    .groupby(
        "split"
    )
    .size()
    .reindex(
        [
            "train",
            "val",
            "test",
        ]
    )
)

total_units = int(
    units_by_split.sum()
)

unit_ratios = (
    units_by_split
    / max(total_units, 1)
)

ratio_table = pd.DataFrame({
    "split": units_by_split.index,
    "units": units_by_split.values,
    "unit_ratio": unit_ratios.values,
    "target_ratio": [
        0.80,
        0.10,
        0.10,
    ],
})

ratio_table[
    "absolute_difference"
] = np.abs(
    ratio_table["unit_ratio"]
    - ratio_table["target_ratio"]
)

large_ratio_deviation = bool(
    (
        ratio_table[
            "absolute_difference"
        ] > 0.08
    ).any()
)


# ============================================================
# Optional method and domain summaries
# ============================================================

distribution_columns = [
    column
    for column in [
        "method",
        "domain",
        "top_level_folder",
    ]
    if column in canonical.columns
]

distribution_tables = {}

for column in distribution_columns:
    table = (
        canonical.groupby(
            [
                "split",
                "label",
                column,
            ],
            dropna=False,
            as_index=False,
        )
        .agg(
            frames=(
                "image_path_canonical",
                "size",
            ),
            units=(
                "group_id_canonical",
                "nunique",
            ),
        )
    )

    distribution_tables[column] = table


# ============================================================
# Freeze the canonical split manifest
# ============================================================

preferred_columns = [
    "split",
    "label",
    "label_name",
    group_column,
    path_column,
    "group_id",
    "sample_unit_id",
    "video_id",
    "method",
    "domain",
    "top_level_folder",
    "dataset_partition",
    "relative_path",
]

export_columns = []

for column in preferred_columns:
    if (
        column in canonical.columns
        and column not in export_columns
    ):
        export_columns.append(column)

canonical_export = canonical[
    export_columns
].copy()

canonical_export = canonical_export.sort_values(
    [
        "split",
        group_column,
        path_column,
    ],
    kind="stable",
).reset_index(
    drop=True
)

frozen_manifest_path = (
    audit_dir
    / "agrfn_canonical_split_manifest.csv"
)

canonical_export.to_csv(
    frozen_manifest_path,
    index=False,
)


# ============================================================
# Create stable SHA-256 split fingerprint
# ============================================================

hash_columns = [
    "split",
    "label",
    group_column,
    path_column,
]

hash_dataframe = canonical[
    hash_columns
].copy()

hash_dataframe = hash_dataframe.sort_values(
    hash_columns,
    kind="stable",
).reset_index(
    drop=True
)

hash_payload = hash_dataframe.to_csv(
    index=False,
    lineterminator="\n",
).encode("utf-8")

split_sha256 = hashlib.sha256(
    hash_payload
).hexdigest()

hash_path = (
    audit_dir
    / "agrfn_canonical_split_sha256.txt"
)

hash_path.write_text(
    split_sha256 + "\n",
    encoding="utf-8",
)


# ============================================================
# Save reports
# ============================================================

summary_path = (
    audit_dir
    / "agrfn_split_summary.csv"
)

ratio_path = (
    audit_dir
    / "agrfn_split_ratios.csv"
)

split_summary.to_csv(
    summary_path,
    index=False,
)

ratio_table.to_csv(
    ratio_path,
    index=False,
)

for column, table in distribution_tables.items():
    table.to_csv(
        audit_dir
        / f"agrfn_split_by_{column}.csv",
        index=False,
    )

audit_report = {
    "created_at": datetime.now().isoformat(
        timespec="seconds"
    ),
    "source_manifest": str(
        manifest_path
    ),
    "frozen_manifest": str(
        frozen_manifest_path
    ),
    "split_column": split_column,
    "group_column": group_column,
    "path_column": path_column,
    "total_frames": int(
        len(canonical)
    ),
    "total_units": total_units,
    "split_sha256": split_sha256,
    "observed_splits": sorted(
        observed_splits
    ),
    "group_leakage_count": int(
        len(leaking_groups)
    ),
    "path_leakage_count": int(
        len(leaking_paths)
    ),
    "mixed_label_group_count": int(
        len(mixed_label_groups)
    ),
    "duplicate_path_rows": (
        duplicate_path_rows
    ),
    "exact_duplicate_rows": (
        exact_duplicate_rows
    ),
    "large_ratio_deviation_warning": (
        large_ratio_deviation
    ),
}

audit_json_path = (
    audit_dir
    / "agrfn_split_audit.json"
)

audit_json_path.write_text(
    json.dumps(
        audit_report,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# Final report
# ============================================================

print("\n" + "=" * 80)
print("CANONICAL DF40 SPLIT AUDIT PASSED")
print("=" * 80)

print("\nFrozen split manifest:")
print(frozen_manifest_path)

print("\nSplit SHA-256:")
print(split_sha256)

print("\nFrame and unit counts:")
display(split_summary)

print("\nUnit split ratios:")
display(ratio_table)

print("\nLeakage checks:")
print("Group leakage:", len(leaking_groups))
print("Path leakage:", len(leaking_paths))
print("Mixed-label groups:", len(mixed_label_groups))
print("Duplicate paths:", duplicate_path_rows)

if large_ratio_deviation:
    print(
        "\nWARNING: at least one unit ratio differs from "
        "the 80/10/10 target by more than 0.08."
    )
else:
    print(
        "\nPASS: unit ratios are consistent with "
        "the intended 80/10/10 partition."
    )

print("\nSaved audit files:")

for path in sorted(
    audit_dir.iterdir()
):
    print(" ", path.name)

print(
    "\nPASS: the canonical split is leakage-free "
    "and ready for identical use across all ablations."
)

AUDITING CANONICAL DF40 SPLIT

Manifest:
/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40/df40_manifest_all_clean.csv

Rows:
292907

Detected columns:
Split: split
Group: group_id
Path: frame_path
Label: label

CANONICAL DF40 SPLIT AUDIT PASSED

Frozen split manifest:
/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_split_audit/agrfn_canonical_split_manifest.csv

Split SHA-256:
f5fae79deb56e831bf4e6758b30db83c066d3b4912b78121f80fe78258aac350

Frame and unit counts:


,split,label,label_name,frames,units
0,test,0,real,5999,189
1,test,1,fake,22717,792
2,train,0,real,48132,1512
3,train,1,fake,187508,6331
4,val,0,real,5997,189
5,val,1,fake,22554,791



Unit split ratios:


,split,units,unit_ratio,target_ratio,absolute_difference
0,train,7843,0.799980,0.8,0.000020
1,val,980,0.099959,0.1,0.000041
2,test,981,0.100061,0.1,0.000061



Leakage checks:
Group leakage: 0
Path leakage: 0
Mixed-label groups: 0
Duplicate paths: 0

PASS: unit ratios are consistent with the intended 80/10/10 partition.

Saved audit files:
  agrfn_canonical_split_manifest.csv
  agrfn_canonical_split_sha256.txt
  agrfn_split_audit.json
  agrfn_split_by_domain.csv
  agrfn_split_by_method.csv
  agrfn_split_by_top_level_folder.csv
  agrfn_split_ratios.csv
  agrfn_split_summary.csv

PASS: the canonical split is leakage-free and ready for identical use across all ablations.


In [54]:
from pathlib import Path
from datetime import datetime
from types import SimpleNamespace
import copy
import hashlib
import importlib.util
import json
import sys

import pandas as pd
import torch


# ============================================================
# Project files
# ============================================================

project_root = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution"
)

training_script = (
    project_root
    / "agrfn_adaptive_routing_df40.py"
)

ablation_plan_path = (
    project_root
    / "agrfn_ablation_plan.csv"
)

ablation_runner_path = (
    project_root
    / "run_agrfn_ablation_suite.py"
)

ood_evaluator_path = (
    project_root
    / "evaluate_agrfn_ood.py"
)

indomain_manifest_path = (
    project_root
    / "outputs_df40"
    / "df40_manifest_all_clean.csv"
)

ood_manifest_path = (
    project_root
    / "outputs_df40"
    / "df40_manifest_ood_fake_test.csv"
)

split_audit_dir = (
    project_root
    / "outputs_agrfn_split_audit"
)

frozen_split_path = (
    split_audit_dir
    / "agrfn_canonical_split_manifest.csv"
)

split_hash_path = (
    split_audit_dir
    / "agrfn_canonical_split_sha256.txt"
)

output_dir = (
    project_root
    / "outputs_agrfn_reproducibility"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

required_paths = [
    training_script,
    ablation_plan_path,
    ablation_runner_path,
    ood_evaluator_path,
    indomain_manifest_path,
    ood_manifest_path,
    frozen_split_path,
    split_hash_path,
]

for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(path)


# ============================================================
# SHA-256 helper
# ============================================================

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


# ============================================================
# Hash all important files
# ============================================================

tracked_files = {
    "training_script": training_script,
    "ablation_plan": ablation_plan_path,
    "ablation_runner": ablation_runner_path,
    "ood_evaluator": ood_evaluator_path,
    "indomain_manifest": indomain_manifest_path,
    "ood_manifest": ood_manifest_path,
    "frozen_split_manifest": frozen_split_path,
}

hash_rows = []

for role, path in tracked_files.items():
    hash_rows.append({
        "role": role,
        "filename": path.name,
        "absolute_path": str(path),
        "size_bytes": int(path.stat().st_size),
        "sha256": sha256_file(path),
    })

hash_table = pd.DataFrame(hash_rows)

hash_table_path = (
    output_dir
    / "agrfn_code_data_hashes.csv"
)

hash_table.to_csv(
    hash_table_path,
    index=False,
)


# ============================================================
# Verify the previously frozen split hash
# ============================================================

stored_split_hash = (
    split_hash_path
    .read_text(encoding="utf-8")
    .strip()
)

frozen_manifest = pd.read_csv(
    frozen_split_path
)

split_column = "split"

group_candidates = [
    "sample_unit_id",
    "group_id",
    "video_id",
    "clip_id",
]

path_candidates = [
    "image_path",
    "frame_path",
    "path",
    "filepath",
    "file_path",
]

group_column = next(
    (
        column
        for column in group_candidates
        if column in frozen_manifest.columns
    ),
    None,
)

path_column = next(
    (
        column
        for column in path_candidates
        if column in frozen_manifest.columns
    ),
    None,
)

if group_column is None:
    raise RuntimeError(
        "Could not identify the frozen split group column."
    )

if path_column is None:
    raise RuntimeError(
        "Could not identify the frozen split path column."
    )

hash_columns = [
    split_column,
    "label",
    group_column,
    path_column,
]

split_payload = (
    frozen_manifest[
        hash_columns
    ]
    .sort_values(
        hash_columns,
        kind="stable",
    )
    .reset_index(drop=True)
    .to_csv(
        index=False,
        lineterminator="\n",
    )
    .encode("utf-8")
)

recomputed_split_hash = hashlib.sha256(
    split_payload
).hexdigest()

if recomputed_split_hash != stored_split_hash:
    raise RuntimeError(
        "The frozen split SHA-256 no longer matches. "
        "The canonical split may have changed."
    )


# ============================================================
# Import the current training module
# ============================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

module_name = (
    "agrfn_reproducibility_"
    + timestamp
)

spec = importlib.util.spec_from_file_location(
    module_name,
    training_script,
)

module = importlib.util.module_from_spec(spec)

if spec is None or spec.loader is None:
    raise RuntimeError(
        "Could not import the A-GRFN training script."
    )

spec.loader.exec_module(module)


# ============================================================
# Recover current parser defaults
# ============================================================

original_argv = list(sys.argv)

try:
    sys.argv = [
        str(training_script)
    ]

    parser_defaults = module.parse_args()

finally:
    sys.argv = original_argv

default_values = vars(
    parser_defaults
).copy()

tracked_default_names = [
    "epochs",
    "batch_size",
    "image_size",
    "num_workers",
    "frames_per_video",
    "lr",
    "weight_decay",
    "feat_dim",
    "controller_dim",
    "routing_temperature",
    "film_scale",
    "router_heads",
    "router_layers",
    "router_dropout",
    "lambda_stream",
    "lambda_aux",
    "lambda_pred_cons",
    "lambda_feat_cons",
    "lambda_cons",
    "lambda_route",
    "lambda_proto",
    "lambda_unc",
    "lambda_supcon",
    "balance_strategy",
    "decision_threshold",
    "score_direction",
    "seed",
    "amp",
]

training_defaults = {
    name: default_values[name]
    for name in tracked_default_names
    if name in default_values
}


# ============================================================
# Load and validate the seven-experiment plan
# ============================================================

plan = pd.read_csv(
    ablation_plan_path
)

expected_experiments = [
    "E0_late_fusion",
    "E1_average_fusion",
    "E2_adaptive_scalar",
    "E3_adaptive_film",
    "E4_adaptive_transformer",
    "E5_transformer_prototypes",
    "E6_full_agrfn",
]

observed_experiments = (
    plan["experiment_name"]
    .astype(str)
    .tolist()
)

if observed_experiments != expected_experiments:
    raise RuntimeError(
        "The ablation plan experiment order or names changed."
    )


def as_bool(value):
    if isinstance(value, bool):
        return value

    return str(value).strip().lower() in {
        "1",
        "true",
        "yes",
        "y",
    }


# ============================================================
# Build each model and count parameters
# ============================================================

model_rows = []

for _, experiment in plan.iterrows():
    model_args = SimpleNamespace(
        **copy.deepcopy(default_values)
    )

    model_args.fusion_mode = str(
        experiment["fusion_mode"]
    )

    model_args.streams = str(
        experiment["streams"]
    )

    model_args.use_prototypes = as_bool(
        experiment["use_prototypes"]
    )

    model_args.use_uncertainty = as_bool(
        experiment["use_uncertainty"]
    )

    for name in [
        "lambda_aux",
        "lambda_route",
        "lambda_cons",
        "lambda_pred_cons",
        "lambda_proto",
        "lambda_unc",
    ]:
        setattr(
            model_args,
            name,
            float(experiment[name]),
        )

    stream_names = [
        value.strip()
        for value in model_args.streams.split(",")
        if value.strip()
    ]

    model = module.build_model(
        model_args,
        stream_names,
    )

    total_parameters = sum(
        parameter.numel()
        for parameter in model.parameters()
    )

    trainable_parameters = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )

    model_parameter_bytes = sum(
        parameter.numel()
        * parameter.element_size()
        for parameter in model.parameters()
    )

    model_buffer_bytes = sum(
        buffer.numel()
        * buffer.element_size()
        for buffer in model.buffers()
    )

    estimated_state_mb = (
        model_parameter_bytes
        + model_buffer_bytes
    ) / (1024 ** 2)

    model_rows.append({
        "experiment_name": str(
            experiment["experiment_name"]
        ),
        "fusion_mode": str(
            experiment["fusion_mode"]
        ),
        "streams": ",".join(stream_names),
        "use_prototypes": bool(
            model_args.use_prototypes
        ),
        "use_uncertainty": bool(
            model_args.use_uncertainty
        ),
        "total_parameters": int(
            total_parameters
        ),
        "trainable_parameters": int(
            trainable_parameters
        ),
        "estimated_model_state_mb": float(
            estimated_state_mb
        ),
    })

    del model

model_table = pd.DataFrame(
    model_rows
)

model_table_path = (
    output_dir
    / "agrfn_model_parameter_counts.csv"
)

model_table.to_csv(
    model_table_path,
    index=False,
)


# ============================================================
# Record split sizes
# ============================================================

split_counts = (
    frozen_manifest.groupby(
        [
            "split",
            "label",
        ],
        as_index=False,
    )
    .agg(
        frames=(
            path_column,
            "size",
        ),
        units=(
            group_column,
            "nunique",
        ),
    )
)

split_counts_path = (
    output_dir
    / "agrfn_frozen_split_counts.csv"
)

split_counts.to_csv(
    split_counts_path,
    index=False,
)


# ============================================================
# Hardware metadata
# ============================================================

hardware = {
    "torch_version": torch.__version__,
    "cuda_available": bool(
        torch.cuda.is_available()
    ),
    "cuda_version": torch.version.cuda,
    "cudnn_version": (
        torch.backends.cudnn.version()
        if torch.backends.cudnn.is_available()
        else None
    ),
}

if torch.cuda.is_available():
    hardware.update({
        "gpu_name": torch.cuda.get_device_name(0),
        "gpu_count": int(
            torch.cuda.device_count()
        ),
        "gpu_memory_gb": round(
            torch.cuda.get_device_properties(
                0
            ).total_memory
            / (1024 ** 3),
            3,
        ),
    })
else:
    hardware.update({
        "gpu_name": None,
        "gpu_count": 0,
        "gpu_memory_gb": 0.0,
    })


# ============================================================
# Save final reproducibility manifest
# ============================================================

reproducibility_manifest = {
    "created_at": datetime.now().isoformat(
        timespec="seconds"
    ),
    "project_root": str(project_root),
    "experiment_count": int(len(plan)),
    "experiment_names": expected_experiments,
    "canonical_split_sha256": stored_split_hash,
    "training_defaults": training_defaults,
    "hardware": hardware,
    "tracked_files": {
        row["role"]: {
            "path": row["absolute_path"],
            "size_bytes": row["size_bytes"],
            "sha256": row["sha256"],
        }
        for row in hash_rows
    },
}

manifest_output_path = (
    output_dir
    / "agrfn_reproducibility_manifest.json"
)

manifest_output_path.write_text(
    json.dumps(
        reproducibility_manifest,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)


# ============================================================
# Final validation
# ============================================================

required_outputs = [
    hash_table_path,
    model_table_path,
    split_counts_path,
    manifest_output_path,
]

for path in required_outputs:
    if not path.exists():
        raise RuntimeError(
            f"Missing reproducibility output: {path}"
        )

if len(model_table) != 7:
    raise RuntimeError(
        "Parameter table does not contain seven models."
    )

if (
    model_table[
        "trainable_parameters"
    ] <= 0
).any():
    raise RuntimeError(
        "At least one model has no trainable parameters."
    )


# ============================================================
# Report
# ============================================================

print("=" * 80)
print("A-GRFN REPRODUCIBILITY FREEZE PASSED")
print("=" * 80)

print("\nCanonical split SHA-256:")
print(stored_split_hash)

print("\nTracked file hashes:")
display(
    hash_table[
        [
            "role",
            "filename",
            "size_bytes",
            "sha256",
        ]
    ]
)

print("\nModel parameter counts:")
display(model_table)

print("\nFrozen split counts:")
display(split_counts)

print("\nTraining defaults:")
display(
    pd.DataFrame(
        [
            {
                "setting": name,
                "value": value,
            }
            for name, value
            in training_defaults.items()
        ]
    )
)

print("\nHardware:")
display(
    pd.DataFrame(
        [
            {
                "property": name,
                "value": value,
            }
            for name, value in hardware.items()
        ]
    )
)

print("\nSaved files:")

for path in required_outputs:
    print(" ", path)

print(
    "\nPASS: code, data, split, ablation plan, "
    "model sizes, defaults, and hardware were frozen."
)

print(
    "PASS: no training jobs were started."
)

A-GRFN REPRODUCIBILITY FREEZE PASSED

Canonical split SHA-256:
f5fae79deb56e831bf4e6758b30db83c066d3b4912b78121f80fe78258aac350

Tracked file hashes:


,role,filename,size_bytes,sha256
0,training_script,agrfn_adaptive_routing_df40.py,147107,07d58775ee332bbf86cdc9b66ac806de0c2c94d3d48374...
1,ablation_plan,agrfn_ablation_plan.csv,1323,91253c2ccd639ef9adef44d53d6a1705e406d23c5a4bcd...
2,ablation_runner,run_agrfn_ablation_suite.py,11589,50870d1658b6b9bfe8e7de4891796cc7ed1446a640b8b4...
3,ood_evaluator,evaluate_agrfn_ood.py,24399,7f2d25b4faa405894cac4d2f931eff5f117df577a5f312...
4,indomain_manifest,df40_manifest_all_clean.csv,159915209,a2ba1786b7361ae32d5047a26dccbeac4e4a58671dbe95...
5,ood_manifest,df40_manifest_ood_fake_test.csv,101697832,b6450959dd051236a53722c302648ab32ab6693a4e6898...
6,frozen_split_manifest,agrfn_canonical_split_manifest.csv,93181748,f0e10566e6e5c405682862d311d21f54cd72b22129cc17...



Model parameter counts:


,experiment_name,fusion_mode,streams,use_prototypes,use_uncertainty,total_parameters,trainable_parameters,estimated_model_state_mb
0,E0_late_fusion,late,"rgb,fingerprint,jpeg,freq",False,False,1662731,1662731,6.359112
1,E1_average_fusion,average,"rgb,fingerprint,jpeg,freq",False,False,1580872,1580872,6.046844
2,E2_adaptive_scalar,adaptive_scalar,"rgb,fingerprint,jpeg,freq",False,False,1681741,1681741,6.431629
3,E3_adaptive_film,adaptive_film,"rgb,fingerprint,jpeg,freq",False,False,1780813,1780813,6.809559
4,E4_adaptive_transformer,adaptive_transformer,"rgb,fingerprint,jpeg,freq",False,False,2044747,2044747,7.816387
5,E5_transformer_prototypes,adaptive_transformer,"rgb,fingerprint,jpeg,freq",True,False,2045003,2045003,7.817364
6,E6_full_agrfn,adaptive_transformer,"rgb,fingerprint,jpeg,freq",True,True,2062540,2062540,7.884262



Frozen split counts:


,split,label,frames,units
0,test,0,5999,189
1,test,1,22717,792
2,train,0,48132,1512
3,train,1,187508,6331
4,val,0,5997,189
5,val,1,22554,791



Training defaults:


,setting,value
0,epochs,15
1,batch_size,16
2,image_size,224
3,num_workers,0
4,frames_per_video,32
5,lr,0.0001
6,weight_decay,0.0001
7,feat_dim,128
8,controller_dim,128
9,routing_temperature,1.0



Hardware:


,property,value
0,torch_version,2.13.0+cu130
1,cuda_available,True
2,cuda_version,13.0
3,cudnn_version,92000
4,gpu_name,NVIDIA L40
5,gpu_count,1
6,gpu_memory_gb,44.394



Saved files:
  /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_reproducibility/agrfn_code_data_hashes.csv
  /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_reproducibility/agrfn_model_parameter_counts.csv
  /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_reproducibility/agrfn_frozen_split_counts.csv
  /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_reproducibility/agrfn_reproducibility_manifest.json

PASS: code, data, split, ablation plan, model sizes, defaults, and hardware were frozen.
PASS: no training jobs were started.


In [ ]:
from pathlib import Path
from collections import deque
from IPython.display import HTML, display

import codecs
import hashlib
import html
import json
import os
import re
import subprocess
import sys
import time

import numpy as np
import pandas as pd


# ============================================================
# Paths
# ============================================================

project_root = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution"
)

training_script = (
    project_root
    / "agrfn_adaptive_routing_df40.py"
)

runner_path = (
    project_root
    / "run_agrfn_ablation_suite.py"
)

plan_path = (
    project_root
    / "agrfn_ablation_plan.csv"
)

frozen_manifest_path = (
    project_root
    / "outputs_agrfn_split_audit"
    / "agrfn_canonical_split_manifest.csv"
)

reproducibility_path = (
    project_root
    / "outputs_agrfn_reproducibility"
    / "agrfn_reproducibility_manifest.json"
)

output_root = (
    project_root
    / "outputs_agrfn_ablation_suite"
)

experiment_name = "E0_late_fusion"

experiment_dir = (
    output_root
    / experiment_name
)

launcher_log_path = (
    output_root
    / "E0_late_fusion_launcher.log"
)

required_paths = [
    training_script,
    runner_path,
    plan_path,
    frozen_manifest_path,
    reproducibility_path,
]

for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(path)


# ============================================================
# Verify frozen training-script hash
# ============================================================

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):
    digest = hashlib.sha256()

    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


with open(
    reproducibility_path,
    "r",
    encoding="utf-8",
) as handle:
    reproducibility = json.load(handle)

expected_script_hash = (
    reproducibility[
        "tracked_files"
    ][
        "training_script"
    ][
        "sha256"
    ]
)

current_script_hash = sha256_file(
    training_script
)

if current_script_hash != expected_script_hash:
    raise RuntimeError(
        "The training script changed after the "
        "reproducibility freeze.\n"
        f"Frozen hash:  {expected_script_hash}\n"
        f"Current hash: {current_script_hash}"
    )


# ============================================================
# Initial report
# ============================================================

print("=" * 80)
print("STARTING FULL E0 LATE-FUSION BASELINE")
print("=" * 80)

print("\nTraining-script SHA-256 verified:")
print(current_script_hash)

print("\nFrozen manifest:")
print(frozen_manifest_path)

print("\nExperiment output:")
print(experiment_dir)

print("\nLauncher log:")
print(launcher_log_path)


# ============================================================
# Build E0 command
# ============================================================

command = [
    sys.executable,

    # Prevent Python print statements from being buffered
    # while their output is being captured through a pipe.
    "-u",

    str(runner_path),

    "--project_root",
    str(project_root),

    "--train_script",
    str(training_script),

    "--metadata_csv",
    str(frozen_manifest_path),

    "--plan_csv",
    str(plan_path),

    "--output_root",
    str(output_root),

    "--experiments",
    experiment_name,

    "--execute",
    "--resume_existing",
    "--skip_completed",
    "--stop_on_error",
]

print("\nCommand:")
print(
    " ".join(
        f'"{value}"'
        if " " in value
        else value
        for value in command
    )
)

print(
    "\nTraining will update one live progress area below. "
    "The complete output is preserved in the launcher log.\n"
)


# ============================================================
# Terminal-output processing helpers
# ============================================================

# Remove terminal color, cursor movement, and other ANSI codes.
ansi_pattern = re.compile(
    rb"\x1b(?:"
    rb"[@-Z\\-_]"
    rb"|"
    rb"\[[0-?]*[ -/]*[@-~]"
    rb")"
)

decoder = codecs.getincrementaldecoder(
    "utf-8"
)(
    errors="replace"
)

# Keep only the latest useful non-progress messages visible.
recent_messages = deque(
    maxlen=10
)

latest_progress = ""
line_buffer = ""

last_render_time = 0.0
render_interval_seconds = 0.20


def clean_terminal_text(value):
    """Remove terminal control codes from one output line."""

    encoded = value.encode(
        "utf-8",
        errors="replace",
    )

    encoded = ansi_pattern.sub(
        b"",
        encoded,
    )

    cleaned = encoded.decode(
        "utf-8",
        errors="replace",
    )

    cleaned = (
        cleaned
        .replace("\b", "")
        .replace("\x00", "")
        .strip()
    )

    return cleaned


def looks_like_progress_line(value):
    """Return True for tqdm-style progress output."""

    normalized = value.lower()

    progress_markers = [
        "%",
        "it/s",
        "s/it",
        "epoch",
        "train",
        "predict",
        "validation",
        "valid",
        "test",
        "batch",
    ]

    has_progress_marker = any(
        marker in normalized
        for marker in progress_markers
    )

    has_progress_shape = any(
        marker in value
        for marker in [
            "|",
            "█",
            "━",
            "▏",
            "▎",
            "▍",
            "▌",
            "▋",
            "▊",
            "▉",
        ]
    )

    has_counter_shape = (
        "/" in value
        and (
            "[" in value
            or "<" in value
        )
    )

    return (
        has_progress_marker
        and (
            has_progress_shape
            or has_counter_shape
        )
    )


# ============================================================
# Create the one-area notebook display
# ============================================================

live_display = display(
    HTML(
        "<pre style='"
        "white-space: pre-wrap;"
        "font-family: monospace;"
        "line-height: 1.35;"
        "padding: 10px;"
        "border: 1px solid #cccccc;"
        "border-radius: 6px;"
        "max-height: 420px;"
        "overflow-y: auto;"
        "'>"
        "Starting E0 late-fusion training..."
        "</pre>"
    ),
    display_id=True,
)


def render_live_output(
    final_status=None,
    force=False,
):
    """Refresh one notebook output area."""

    global last_render_time

    current_time = time.monotonic()

    if (
        not force
        and (
            current_time
            - last_render_time
        )
        < render_interval_seconds
    ):
        return

    last_render_time = current_time

    display_lines = [
        "E0 late-fusion baseline",
        "=" * 78,
    ]

    if recent_messages:
        display_lines.extend(
            list(recent_messages)
        )

    if latest_progress:
        display_lines.extend(
            [
                "",
                latest_progress,
            ]
        )

    if final_status:
        display_lines.extend(
            [
                "",
                "=" * 78,
                final_status,
            ]
        )

    rendered_text = "\n".join(
        display_lines
    )

    live_display.update(
        HTML(
            "<pre style='"
            "white-space: pre-wrap;"
            "font-family: monospace;"
            "line-height: 1.35;"
            "padding: 10px;"
            "border: 1px solid #cccccc;"
            "border-radius: 6px;"
            "max-height: 420px;"
            "overflow-y: auto;"
            "'>"
            + html.escape(rendered_text)
            + "</pre>"
        )
    )


def process_completed_line(
    raw_line,
    delimiter,
):
    """Handle one carriage-return or newline-delimited item."""

    global latest_progress

    cleaned_line = clean_terminal_text(
        raw_line
    )

    if not cleaned_line:
        return

    is_progress = (
        delimiter == "\r"
        or looks_like_progress_line(
            cleaned_line
        )
    )

    if is_progress:
        latest_progress = cleaned_line

    else:
        # Avoid showing the same message repeatedly.
        if (
            not recent_messages
            or recent_messages[-1]
            != cleaned_line
        ):
            recent_messages.append(
                cleaned_line
            )

    render_live_output()


# ============================================================
# Run E0 with one live progress display
# ============================================================

output_root.mkdir(
    parents=True,
    exist_ok=True,
)

process = subprocess.Popen(
    command,
    cwd=str(project_root),

    # Binary mode allows carriage returns from tqdm to be
    # handled separately from true newline characters.
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=0,
)

if process.stdout is None:
    raise RuntimeError(
        "Could not capture the training process output."
    )

try:
    with open(
        launcher_log_path,
        "wb",
    ) as log_handle:

        while True:
            chunk = os.read(
                process.stdout.fileno(),
                4096,
            )

            if not chunk:
                break

            # Preserve the full original process output.
            log_handle.write(chunk)
            log_handle.flush()

            decoded_chunk = decoder.decode(
                chunk
            )

            for character in decoded_chunk:
                if character == "\r":
                    process_completed_line(
                        line_buffer,
                        "\r",
                    )

                    line_buffer = ""

                elif character == "\n":
                    process_completed_line(
                        line_buffer,
                        "\n",
                    )

                    line_buffer = ""

                else:
                    line_buffer += character

        # Flush any remaining decoder bytes.
        decoder_tail = decoder.decode(
            b"",
            final=True,
        )

        line_buffer += decoder_tail

        if line_buffer.strip():
            process_completed_line(
                line_buffer,
                "\n",
            )

except KeyboardInterrupt:
    process.terminate()

    try:
        process.wait(
            timeout=15
        )

    except subprocess.TimeoutExpired:
        process.kill()
        process.wait()

    render_live_output(
        final_status=(
            "Training was interrupted by the user."
        ),
        force=True,
    )

    raise

return_code = process.wait()

if return_code == 0:
    final_process_status = (
        "Training process completed successfully."
    )

else:
    final_process_status = (
        "Training process failed with return code "
        f"{return_code}."
    )

render_live_output(
    final_status=final_process_status,
    force=True,
)

if return_code != 0:
    raise RuntimeError(
        "E0 late-fusion training failed with "
        f"return code {return_code}.\n"
        f"Launcher log: {launcher_log_path}"
    )


# ============================================================
# Validate suite status
# ============================================================

status_path = (
    output_root
    / "ablation_run_status.csv"
)

if not status_path.exists():
    raise RuntimeError(
        "ablation_run_status.csv was not created."
    )

status = pd.read_csv(
    status_path
)

experiment_status = status[
    status[
        "experiment_name"
    ].astype(str)
    == experiment_name
]

if len(experiment_status) != 1:
    raise RuntimeError(
        "Could not find exactly one E0 status row."
    )

observed_status = str(
    experiment_status[
        "status"
    ].iloc[0]
)

valid_statuses = {
    "completed",
    "skipped_completed",
}

if observed_status not in valid_statuses:
    display(
        experiment_status
    )

    raise RuntimeError(
        f"Unexpected E0 status: {observed_status}"
    )


# ============================================================
# Validate final experiment outputs
# ============================================================

required_outputs = [
    "best_checkpoint.pt",
    "latest_checkpoint.pt",
    "training_history.csv",
    "test_frame_predictions.csv",
    "test_video_predictions.csv",
    "test_metrics.csv",
    "test_inference_time.csv",
    "experiment_summary.csv",
]

missing_outputs = [
    filename
    for filename in required_outputs
    if not (
        experiment_dir
        / filename
    ).exists()
]

if missing_outputs:
    raise RuntimeError(
        "E0 is missing final outputs:\n"
        + "\n".join(
            missing_outputs
        )
    )


# ============================================================
# Load experiment outputs
# ============================================================

history = pd.read_csv(
    experiment_dir
    / "training_history.csv"
)

metrics = pd.read_csv(
    experiment_dir
    / "test_metrics.csv"
)

inference = pd.read_csv(
    experiment_dir
    / "test_inference_time.csv"
)

summary = pd.read_csv(
    experiment_dir
    / "experiment_summary.csv"
)


# ============================================================
# Numerical checks
# ============================================================

if len(history) == 0:
    raise RuntimeError(
        "E0 training history is empty."
    )

numeric_history = (
    history.select_dtypes(
        include=[np.number]
    )
)

if not np.isfinite(
    numeric_history.to_numpy(
        dtype=float
    )
).all():
    raise RuntimeError(
        "E0 training history contains NaN or infinity."
    )

if "train_proto" in history.columns:
    maximum_prototype_loss = float(
        history[
            "train_proto"
        ]
        .abs()
        .max()
    )

    if maximum_prototype_loss > 1e-10:
        raise RuntimeError(
            "The E0 baseline unexpectedly activated "
            "prototype learning."
        )

if "train_unc" in history.columns:
    maximum_uncertainty_loss = float(
        history[
            "train_unc"
        ]
        .abs()
        .max()
    )

    if maximum_uncertainty_loss > 1e-10:
        raise RuntimeError(
            "The E0 baseline unexpectedly activated "
            "learned uncertainty."
        )


# ============================================================
# Final report
# ============================================================

print("\n" + "=" * 80)
print("FULL E0 LATE-FUSION BASELINE PASSED")
print("=" * 80)

print("\nStatus:")
print(observed_status)

print("\nOutput directory:")
print(experiment_dir)

print("\nEpochs recorded:")
print(len(history))

print("\nFinal training-history row:")
display(
    history.tail(1)
)

print("\nTest metrics:")
display(metrics)

print("\nInference timing:")
display(inference)

print("\nExperiment summary:")
display(summary)

print("\nLauncher log:")
print(launcher_log_path)

print(
    "\nPASS: E0 used the frozen code and canonical split."
)

print(
    "PASS: the complete late-fusion baseline outputs "
    "are ready for the ablation table."
)

In [7]:
from pathlib import Path
from types import SimpleNamespace
from datetime import datetime
from torch.utils.data import DataLoader

import argparse
import importlib.util
import json
import time

import numpy as np
import pandas as pd
import torch


# ============================================================
# Configuration
# ============================================================

PROJECT_ROOT = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution"
)

EXPERIMENT_NAME = "E0_late_fusion"

# Keep both values at zero for the complete OOD evaluation.
LIMIT_UNITS = 0
FRAMES_PER_UNIT = 0

# None uses the values stored in the checkpoint.
BATCH_SIZE = None
NUM_WORKERS = None

UNCERTAINTY_THRESHOLD = 0.50


# ============================================================
# Paths
# ============================================================

training_script = (
    PROJECT_ROOT
    / "agrfn_adaptive_routing_df40.py"
)

experiment_dir = (
    PROJECT_ROOT
    / "outputs_agrfn_ablation_suite"
    / EXPERIMENT_NAME
)

checkpoint_path = (
    experiment_dir
    / "best_checkpoint.pt"
)

ood_manifest_path = (
    PROJECT_ROOT
    / "outputs_df40"
    / "df40_manifest_ood_fake_test.csv"
)

indomain_metrics_path = (
    experiment_dir
    / "test_metrics.csv"
)

indomain_predictions_path = (
    experiment_dir
    / "test_frame_predictions.csv"
)

experiment_summary_path = (
    experiment_dir
    / "experiment_summary.csv"
)

output_dir = (
    experiment_dir
    / "ood_fake_test"
)


# ============================================================
# Required-file checks
# ============================================================

for path in [
    training_script,
    checkpoint_path,
    ood_manifest_path,
]:
    if not path.exists():
        raise FileNotFoundError(path)

# Do not silently use 0.5 for the paper evaluation.
if not indomain_metrics_path.exists():
    raise RuntimeError(
        "The in-domain test evaluation is not complete because "
        "test_metrics.csv is missing.\n\n"
        "Wait for the current E0 process to finish before running "
        "the OOD evaluation.\n\n"
        f"Expected file:\n{indomain_metrics_path}"
    )

output_dir.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# Import the A-GRFN training module
# ============================================================

module_name = (
    "agrfn_ood_evaluation_"
    + datetime.now().strftime("%Y%m%d_%H%M%S")
)

spec = importlib.util.spec_from_file_location(
    module_name,
    training_script,
)

if spec is None or spec.loader is None:
    raise RuntimeError(
        "Could not import the A-GRFN training script."
    )

module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)


# ============================================================
# Recover the in-domain decision threshold
# ============================================================

def normalized_name(value):
    return "".join(
        character.lower()
        for character in str(value)
        if character.isalnum()
    )


def recover_threshold(csv_paths):
    for csv_path in csv_paths:
        if not csv_path.exists():
            continue

        table = pd.read_csv(csv_path)

        for column in table.columns:
            name = normalized_name(column)

            if (
                "threshold" in name
                and "uncertainty" not in name
            ):
                values = pd.to_numeric(
                    table[column],
                    errors="coerce",
                ).dropna()

                valid_values = values[
                    values.between(0.0, 1.0)
                ]

                if len(valid_values) > 0:
                    return (
                        float(valid_values.iloc[0]),
                        f"{csv_path.name}:{column}",
                    )

    raise RuntimeError(
        "Could not recover the validation-tuned decision "
        "threshold from the in-domain results."
    )


decision_threshold, threshold_source = recover_threshold(
    [
        indomain_metrics_path,
        experiment_summary_path,
    ]
)


# ============================================================
# Load checkpoint
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

try:
    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=False,
    )
except TypeError:
    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
    )

stored_args = checkpoint.get(
    "args",
    {},
)

if isinstance(
    stored_args,
    argparse.Namespace,
):
    stored_args = vars(stored_args)

if not isinstance(
    stored_args,
    dict,
):
    raise RuntimeError(
        "The checkpoint arguments are not a dictionary."
    )

args = SimpleNamespace(
    **stored_args
)


# ============================================================
# Safe defaults for older checkpoints
# ============================================================

default_args = {
    "fusion_mode": "late",
    "streams": "rgb,fingerprint,jpeg,freq",
    "feat_dim": 128,
    "controller_dim": 128,
    "routing_temperature": 1.0,
    "film_scale": 0.10,
    "router_heads": 4,
    "router_layers": 1,
    "router_dropout": 0.10,
    "stream_logit_residual_weight": 0.25,
    "use_prototypes": False,
    "prototype_temperature": 0.10,
    "use_uncertainty": False,
    "image_size": 224,
    "batch_size": 8,
    "num_workers": 0,
    "score_direction": "normal",
    "data_root": None,
}

for name, value in default_args.items():
    if not hasattr(args, name):
        setattr(
            args,
            name,
            value,
        )

stream_names = checkpoint.get(
    "stream_names"
)

if stream_names is None:
    stream_names = [
        value.strip()
        for value in str(
            args.streams
        ).split(",")
        if value.strip()
    ]


# ============================================================
# Recover score direction
# ============================================================

def recover_score_inversion():
    if indomain_predictions_path.exists():
        predictions = pd.read_csv(
            indomain_predictions_path
        )

        if "score_inverted" in predictions.columns:
            values = (
                predictions["score_inverted"]
                .astype(str)
                .str.strip()
                .str.lower()
                .isin(
                    [
                        "true",
                        "1",
                        "yes",
                    ]
                )
            )

            return (
                bool(values.mean() >= 0.50),
                "test_frame_predictions.csv:score_inverted",
            )

    if experiment_summary_path.exists():
        summary = pd.read_csv(
            experiment_summary_path
        )

        for column in summary.columns:
            column_name = normalized_name(
                column
            )

            if (
                "scoredirection" in column_name
                or "scoreinverted" in column_name
            ):
                value = str(
                    summary[column].iloc[0]
                ).strip().lower()

                if (
                    "invert" in value
                    or value in {"true", "1", "yes"}
                ):
                    return (
                        True,
                        f"experiment_summary.csv:{column}",
                    )

                if (
                    "normal" in value
                    or value in {"false", "0", "no"}
                ):
                    return (
                        False,
                        f"experiment_summary.csv:{column}",
                    )

    checkpoint_direction = str(
        getattr(
            args,
            "score_direction",
            "normal",
        )
    ).strip().lower()

    if checkpoint_direction == "inverted":
        return True, "checkpoint_args"

    if checkpoint_direction == "normal":
        return False, "checkpoint_args"

    raise RuntimeError(
        "The checkpoint used automatic score direction, but "
        "the final in-domain outputs do not record whether "
        "scores were inverted."
    )


invert_scores, score_direction_source = (
    recover_score_inversion()
)


# ============================================================
# Build and load the model
# ============================================================

model = module.build_model(
    args,
    stream_names,
).to(device)

state_dict = checkpoint.get(
    "model"
)

if state_dict is None:
    state_dict = checkpoint.get(
        "model_state_dict"
    )

if state_dict is None:
    raise RuntimeError(
        "The checkpoint contains no model state dictionary."
    )

model.load_state_dict(
    state_dict,
    strict=True,
)

model.eval()


# ============================================================
# Prepare the fake-only OOD manifest
# ============================================================

ood_df = pd.read_csv(
    ood_manifest_path
)

if len(ood_df) == 0:
    raise RuntimeError(
        "The OOD manifest is empty."
    )

path_candidates = [
    "image_path",
    "frame_path",
    "path",
    "filepath",
    "file_path",
]

path_column = next(
    (
        column
        for column in path_candidates
        if column in ood_df.columns
    ),
    None,
)

if path_column is None:
    raise RuntimeError(
        "Could not find an image-path column in "
        "the OOD manifest."
    )


def resolve_image_path(value):
    path = Path(str(value))

    if path.is_absolute():
        return str(path)

    data_root = getattr(
        args,
        "data_root",
        None,
    )

    if data_root:
        candidate = (
            Path(data_root)
            / path
        )

        if candidate.exists():
            return str(
                candidate.resolve()
            )

    return str(
        (
            PROJECT_ROOT
            / path
        ).resolve()
    )


ood_df = ood_df.copy()

ood_df["image_path"] = (
    ood_df[path_column]
    .map(resolve_image_path)
)

# Fake_Test is intentionally fake-only.
ood_df["label"] = 1
ood_df["label_name"] = "fake"
ood_df["split"] = "ood_fake_test"

group_candidates = [
    "sample_unit_id",
    "group_id",
    "video_id",
    "clip_id",
]

group_column = next(
    (
        column
        for column in group_candidates
        if column in ood_df.columns
    ),
    None,
)

if group_column is None:
    ood_df["group_id"] = (
        ood_df["image_path"]
        .astype(str)
    )
else:
    ood_df["group_id"] = (
        ood_df[group_column]
        .astype(str)
    )

if "sample_unit_id" not in ood_df.columns:
    ood_df["sample_unit_id"] = (
        ood_df["group_id"]
        .astype(str)
    )
else:
    ood_df["sample_unit_id"] = (
        ood_df["sample_unit_id"]
        .astype(str)
    )

if "method" not in ood_df.columns:
    ood_df["method"] = "unknown"

if "domain" not in ood_df.columns:
    ood_df["domain"] = "ood_fake_test"

if "dataset_partition" not in ood_df.columns:
    ood_df[
        "dataset_partition"
    ] = "ood_fake_test"


# ============================================================
# Optional smoke-test sampling
# ============================================================

if (
    LIMIT_UNITS > 0
    or FRAMES_PER_UNIT > 0
):
    ood_df = module.sample_frames_per_unit(
        ood_df,
        frames_per_unit=(
            FRAMES_PER_UNIT
            if FRAMES_PER_UNIT > 0
            else None
        ),
        limit_units=(
            LIMIT_UNITS
            if LIMIT_UNITS > 0
            else None
        ),
        seed=42,
        desc="Sample OOD Fake_Test",
        balanced_units=False,
    )


# ============================================================
# Validate image paths
# ============================================================

missing_mask = ~ood_df[
    "image_path"
].map(
    lambda value: Path(value).exists()
)

if missing_mask.any():
    missing_examples = (
        ood_df.loc[
            missing_mask,
            "image_path",
        ]
        .head(10)
        .tolist()
    )

    raise FileNotFoundError(
        f"{int(missing_mask.sum())} OOD images "
        "were not found.\n\nExamples:\n"
        + "\n".join(missing_examples)
    )


# ============================================================
# Create OOD loader
# ============================================================

batch_size = (
    BATCH_SIZE
    if BATCH_SIZE is not None
    else int(args.batch_size)
)

num_workers = (
    NUM_WORKERS
    if NUM_WORKERS is not None
    else int(args.num_workers)
)

ood_dataset = module.GRFNDataset(
    ood_df,
    image_size=int(
        args.image_size
    ),
    train=False,
    train_aug=False,
)

ood_loader = DataLoader(
    ood_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=(
        device.type == "cuda"
    ),
)


# ============================================================
# Run OOD inference
# ============================================================

print("=" * 80)
print("RUNNING FULL DF40 FAKE_TEST OOD EVALUATION")
print("=" * 80)

print("\nExperiment:")
print(EXPERIMENT_NAME)

print("\nCheckpoint:")
print(checkpoint_path)

print("\nOOD frames:")
print(len(ood_df))

print("\nOOD units:")
print(
    ood_df[
        "sample_unit_id"
    ].nunique()
)

print("\nDecision threshold:")
print(decision_threshold)

print("\nThreshold source:")
print(threshold_source)

print("\nScore inverted:")
print(invert_scores)

print("\nOutput directory:")
print(output_dir)

if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

predictions = module.predict_loader(
    model,
    ood_loader,
    args,
    device,
    split_name="ood_fake_test",
)

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed_seconds = (
    time.perf_counter()
    - start_time
)


# ============================================================
# Apply the in-domain score policy
# ============================================================

predictions = predictions.copy()

predictions["raw_fake_prob"] = (
    predictions["fake_prob"]
)

if invert_scores:
    predictions["fake_prob"] = (
        1.0
        - predictions["fake_prob"]
    )

predictions["predicted_label"] = (
    predictions["fake_prob"]
    >= decision_threshold
).astype(int)

predictions["detected_fake"] = (
    predictions["predicted_label"]
    == 1
).astype(int)

predictions["missed_fake"] = (
    predictions["predicted_label"]
    == 0
).astype(int)

predictions["is_uncertain"] = (
    predictions["uncertainty"]
    >= UNCERTAINTY_THRESHOLD
).astype(int)


# ============================================================
# Aggregate frame predictions by unit/video
# ============================================================

aggregation = {
    "label": "first",
    "fake_prob": "mean",
    "raw_fake_prob": "mean",
    "uncertainty": "mean",
    "method": "first",
    "domain": "first",
    "group_id": "first",
    "image_path": "count",
}

aggregation = {
    column: operation
    for column, operation in aggregation.items()
    if column in predictions.columns
}

unit_predictions = (
    predictions
    .groupby(
        "sample_unit_id",
        as_index=False,
    )
    .agg(aggregation)
)

if "image_path" in unit_predictions.columns:
    unit_predictions = unit_predictions.rename(
        columns={
            "image_path": "num_frames"
        }
    )

unit_predictions["predicted_label"] = (
    unit_predictions["fake_prob"]
    >= decision_threshold
).astype(int)

unit_predictions["detected_fake"] = (
    unit_predictions["predicted_label"]
    == 1
).astype(int)

unit_predictions["missed_fake"] = (
    unit_predictions["predicted_label"]
    == 0
).astype(int)

unit_predictions["is_uncertain"] = (
    unit_predictions["uncertainty"]
    >= UNCERTAINTY_THRESHOLD
).astype(int)


# ============================================================
# Fake-only OOD metrics
# ============================================================

def fake_only_metrics(
    dataframe,
    level,
    method="ALL",
):
    return {
        "Level": level,
        "Method": method,
        "Samples": int(len(dataframe)),
        "Decision Threshold": float(
            decision_threshold
        ),
        "Score Inverted": bool(
            invert_scores
        ),
        "Fake Detection Rate": float(
            dataframe[
                "detected_fake"
            ].mean()
        ),
        "Recall (1)": float(
            dataframe[
                "detected_fake"
            ].mean()
        ),
        "Missed Fake Rate": float(
            dataframe[
                "missed_fake"
            ].mean()
        ),
        "Mean Fake Probability": float(
            dataframe[
                "fake_prob"
            ].mean()
        ),
        "Median Fake Probability": float(
            dataframe[
                "fake_prob"
            ].median()
        ),
        "Mean Uncertainty": float(
            dataframe[
                "uncertainty"
            ].mean()
        ),
        "Uncertain Rate": float(
            dataframe[
                "is_uncertain"
            ].mean()
        ),
        "Uncertainty Threshold": float(
            UNCERTAINTY_THRESHOLD
        ),
    }


overall_metrics = pd.DataFrame(
    [
        fake_only_metrics(
            predictions,
            "Frame",
        ),
        fake_only_metrics(
            unit_predictions,
            "Unit",
        ),
    ]
)

per_method_rows = []

for level, dataframe in [
    ("Frame", predictions),
    ("Unit", unit_predictions),
]:
    if "method" not in dataframe.columns:
        continue

    for method, method_df in dataframe.groupby(
        "method",
        dropna=False,
    ):
        per_method_rows.append(
            fake_only_metrics(
                method_df,
                level,
                method=str(method),
            )
        )

per_method_metrics = pd.DataFrame(
    per_method_rows
)


# ============================================================
# Inference timing
# ============================================================

inference_timing = pd.DataFrame(
    [
        {
            "Experiment": EXPERIMENT_NAME,
            "Evaluation": "DF40 Fake_Test OOD",
            "Device": str(device),
            "Fusion Mode": str(
                args.fusion_mode
            ),
            "Image Size": int(
                args.image_size
            ),
            "Batch Size": int(
                batch_size
            ),
            "Frames Scored": int(
                len(predictions)
            ),
            "Units Scored": int(
                len(unit_predictions)
            ),
            "Total Seconds": float(
                elapsed_seconds
            ),
            "Time Per Frame (ms)": float(
                elapsed_seconds
                / max(
                    len(predictions),
                    1,
                )
                * 1000.0
            ),
            "FPS": float(
                len(predictions)
                / max(
                    elapsed_seconds,
                    1e-8,
                )
            ),
        }
    ]
)


# ============================================================
# Save outputs
# ============================================================

predictions.to_csv(
    output_dir
    / "ood_frame_predictions.csv",
    index=False,
)

unit_predictions.to_csv(
    output_dir
    / "ood_video_predictions.csv",
    index=False,
)

overall_metrics.to_csv(
    output_dir
    / "ood_fake_only_metrics.csv",
    index=False,
)

per_method_metrics.to_csv(
    output_dir
    / "ood_per_method_metrics.csv",
    index=False,
)

inference_timing.to_csv(
    output_dir
    / "ood_inference_time.csv",
    index=False,
)

ood_df.to_csv(
    output_dir
    / "ood_sampled_manifest.csv",
    index=False,
)

configuration = {
    "experiment_name": EXPERIMENT_NAME,
    "checkpoint": str(checkpoint_path),
    "model_script": str(training_script),
    "ood_metadata_csv": str(
        ood_manifest_path
    ),
    "indomain_dir": str(
        experiment_dir
    ),
    "fusion_mode": str(
        args.fusion_mode
    ),
    "stream_names": list(
        stream_names
    ),
    "decision_threshold": float(
        decision_threshold
    ),
    "threshold_source": (
        threshold_source
    ),
    "score_inverted": bool(
        invert_scores
    ),
    "score_direction_source": (
        score_direction_source
    ),
    "uncertainty_threshold": float(
        UNCERTAINTY_THRESHOLD
    ),
    "frames_per_unit": int(
        FRAMES_PER_UNIT
    ),
    "limit_units": int(
        LIMIT_UNITS
    ),
    "total_ood_frames": int(
        len(predictions)
    ),
    "total_ood_units": int(
        len(unit_predictions)
    ),
}

with open(
    output_dir
    / "ood_run_configuration.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        configuration,
        handle,
        indent=2,
    )

if hasattr(
    module,
    "save_diagnostic_csvs",
):
    module.save_diagnostic_csvs(
        predictions,
        output_dir,
    )


# ============================================================
# Final validation
# ============================================================

required_output_files = [
    "ood_frame_predictions.csv",
    "ood_video_predictions.csv",
    "ood_fake_only_metrics.csv",
    "ood_per_method_metrics.csv",
    "ood_inference_time.csv",
    "ood_sampled_manifest.csv",
    "ood_run_configuration.json",
]

missing_outputs = [
    filename
    for filename in required_output_files
    if not (
        output_dir
        / filename
    ).exists()
]

if missing_outputs:
    raise RuntimeError(
        "Missing OOD output files:\n"
        + "\n".join(
            missing_outputs
        )
    )

if not predictions[
    "fake_prob"
].between(
    0.0,
    1.0,
).all():
    raise RuntimeError(
        "OOD probabilities are outside [0, 1]."
    )

if not np.isfinite(
    predictions[
        [
            "fake_prob",
            "uncertainty",
        ]
    ].to_numpy(
        dtype=float
    )
).all():
    raise RuntimeError(
        "OOD predictions contain NaN or infinity."
    )


# ============================================================
# Final report
# ============================================================

print("\n" + "=" * 80)
print("DF40 FAKE_TEST OOD EVALUATION COMPLETE")
print("=" * 80)

print("\nOutput directory:")
print(output_dir)

print("\nOverall fake-only metrics:")
display(overall_metrics)

print("\nPer-method metrics:")
display(per_method_metrics)

print("\nInference timing:")
display(inference_timing)

print(
    "\nAUCROC, AUPRC, balanced accuracy, and EER "
    "are not calculated because Fake_Test contains "
    "only fake samples."
)

print(
    "\nPASS: full fake-only OOD evaluation completed "
    "using the in-domain decision policy."
)

RUNNING FULL DF40 FAKE_TEST OOD EVALUATION

Experiment:
E0_late_fusion

Checkpoint:
/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_ablation_suite/E0_late_fusion/best_checkpoint.pt

OOD frames:
175667

OOD units:
5513

Decision threshold:
0.6306639599999999

Threshold source:
test_metrics.csv:Decision Threshold

Score inverted:
False

Output directory:
/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_ablation_suite/E0_late_fusion/ood_fake_test


Predict ood_fake_test:   0%|          | 0/10980 [00:00<?, ?it/s]

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [2]:
!pip install numpy pandas scikit-learn openpyxl matplotlib torch

  Using cached numpy-2.4.6-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached pandas-3.0.5-cp311-cp311-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached scikit_learn-1.9.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached matplotlib-3.11.1-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (80 kB)
  Using cached torch-2.13.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (38 kB)
  Using cached scipy-1.17.1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached narwhals-2.23.0-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
  Using cached contourpy-1.3.3-cp311-cp311-manyli